# F-LMM Phase 1 — feasibility-first reproduction
Private, bounded, manifest-driven. Run All obeys integrity/resource gates; it does not run every benchmark. Original 20-image regression is immutable. No research improvement.

## 1. Configuration
120-minute walltime budget, 75-minute evaluation budget, 20-minute export reserve; disk floor 3 GiB. Larger evaluations disabled by default. Resume accepts only validated matching fingerprints.

In [ ]:
import base64,json,os,subprocess,sys,time
from pathlib import Path
phase=Path('/kaggle/working/phase1'); phase.mkdir(exist_ok=True)
payload={'bootstrap.py': 'IiIiTm90ZWJvb2sgY29udHJvbCBwbGFuZS4gUmVzdG9yZSBhIHZlcmlmaWVkIENQVSBjYWNoZSBvbmNlOyBib3VuZGVkIEdQVSBzdGFnZXMuIiIiCmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHNodXRpbAppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgc3lzCmltcG9ydCB0aW1lCmZyb20gY29tbW9uIGltcG9ydCBhdG9taWNfanNvbixyZWFkX2pzb24sc2hhMjU2CgpIRVJFPVBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQKT1VUPVBhdGgoJy9rYWdnbGUvd29ya2luZy9yZXN1bHRzJyk7IE9VVC5ta2RpcihleGlzdF9vaz1UcnVlKQoKZGVmIGluaXRpYWxpemUoKToKICAgIHN0YXJ0ZWQ9dGltZS50aW1lKCkKICAgIGF0b21pY19qc29uKE9VVC8nc2Vzc2lvbi5qc29uJyxkaWN0KHN0YXJ0ZWRfdW5peD1zdGFydGVkKSkKICAgIGlmIChIRVJFLydkb2NzJykuZXhpc3RzKCk6IHNodXRpbC5jb3B5dHJlZShIRVJFLydkb2NzJyxPVVQucGFyZW50Lydkb2NzJyxkaXJzX2V4aXN0X29rPVRydWUpCiAgICBjYWNoZXM9bGlzdChQYXRoKCcva2FnZ2xlL2lucHV0Jykucmdsb2IoJ2NhY2hlL2Vudmlyb25tZW50Lmpzb24nKSkKICAgIGlmIGxlbihjYWNoZXMpIT0xOiByYWlzZSBSdW50aW1lRXJyb3IoZidFeHBlY3RlZCBleGFjdGx5IG9uZSB2ZXJpZmllZCBlbnZpcm9ubWVudCBjYWNoZSwgZm91bmQge2NhY2hlc30nKQogICAgY2FjaGU9Y2FjaGVzWzBdLnBhcmVudAogICAgZW52aW5mbz1yZWFkX2pzb24oY2FjaGUvJ2Vudmlyb25tZW50Lmpzb24nKQogICAgaWYgc2hhMjU2KGNhY2hlLydlbnZpcm9ubWVudC50YXIuZ3onKSE9ZW52aW5mb1snc2hhMjU2J106IHJhaXNlIFZhbHVlRXJyb3IoJ0Vudmlyb25tZW50IGNhY2hlIGNoZWNrc3VtIG1pc21hdGNoJykKICAgIFBhdGgoJy9rYWdnbGUvdGVtcCcpLm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKQogICAgaWYgbm90IFBhdGgoJy9rYWdnbGUvdGVtcC9mbG1tLWVudi9iaW4vcHl0aG9uJykuZXhpc3RzKCk6CiAgICAgICAgc3VicHJvY2Vzcy5ydW4oWyd0YXInLCcteHpmJyxzdHIoY2FjaGUvJ2Vudmlyb25tZW50LnRhci5neicpLCctQycsJy9rYWdnbGUvdGVtcCddLGNoZWNrPVRydWUsdGltZW91dD05MDApCiAgICBvcy5lbnZpcm9uWydGTE1NX0NBQ0hFJ109c3RyKGNhY2hlKQogICAgb3MuZW52aXJvblsnUFlUSE9OUEFUSCddPXN0cihIRVJFKSsnOicrc3RyKGNhY2hlLydGLUxNTScpCiAgICBvcy5lbnZpcm9uWydDVURBX1ZJU0lCTEVfREVWSUNFUyddPScwJwogICAgb3MuZW52aXJvblsnTERfTElCUkFSWV9QQVRIJ109Jy9rYWdnbGUvdGVtcC9mbG1tLWVudi9saWIvcHl0aG9uMy4xMC9zaXRlLXBhY2thZ2VzL3RvcmNoL2xpYjonK29zLmVudmlyb24uZ2V0KCdMRF9MSUJSQVJZX1BBVEgnLCcnKQogICAgb3MuZW52aXJvblsnSEZfSFVCX09GRkxJTkUnXT0nMSc7IG9zLmVudmlyb25bJ1RSQU5TRk9STUVSU19PRkZMSU5FJ109JzEnOyBvcy5lbnZpcm9uWydIRl9IVUJfRElTQUJMRV9YRVQnXT0nMScKICAgICMgQW4gZXhwbGljaXQgY29tcGxldGVkIHByaW9yIGtlcm5lbCBvdXRwdXQgaXMgdGhlIGNyb3NzLXNlc3Npb24gY2hlY2twb2ludC4KICAgIGM9cmVhZF9qc29uKEhFUkUvJ2NvbmZpZy5qc29uJykKICAgIHByaW9yPVtdCiAgICBpZiBjLmdldCgnUkVTVU1FX0lOUFVUJyk6CiAgICAgICAgcHJpb3I9W3AgZm9yIHAgaW4gUGF0aCgnL2thZ2dsZS9pbnB1dCcpLnJnbG9iKCdyZXN1bHRzL3J1bl9jb250cmFjdC5qc29uJykgaWYgY1snUkVTVU1FX0lOUFVUJ10gaW4gcC5wYXJ0c10KICAgICAgICBpZiBsZW4ocHJpb3IpIT0xOiByYWlzZSBSdW50aW1lRXJyb3IoJ1JlcXVlc3RlZCByZXN1bWUgb3V0cHV0IG5vdCBmb3VuZCB1bmlxdWVseScpCiAgICBmb3IgcCBpbiBwcmlvcjoKICAgICAgICBzb3VyY2U9cC5wYXJlbnQKICAgICAgICBmb3IgaXRlbSBpbiBzb3VyY2UuaXRlcmRpcigpOgogICAgICAgICAgICBpZiBpdGVtLm5hbWUgaW4gWydzZXNzaW9uLmpzb24nLCdyZXNvdXJjZV9wcmVmbGlnaHQuanNvbicsJ2dvX25vX2dvLmpzb24nLCdydW5fc3VtbWFyeS5qc29uJywnZXZhbHVhdGlvbl9idWRnZXQuanNvbiddOiBjb250aW51ZQogICAgICAgICAgICBkZXN0PU9VVC9pdGVtLm5hbWUKICAgICAgICAgICAgaWYgaXRlbS5pc19kaXIoKTogc2h1dGlsLmNvcHl0cmVlKGl0ZW0sZGVzdCxkaXJzX2V4aXN0X29rPVRydWUpCiAgICAgICAgICAgIGVsaWYgaXRlbS5zdWZmaXggaW4gWycuanNvbicsJy5qc29ubCcsJy5jc3YnXTogc2h1dGlsLmNvcHkyKGl0ZW0sZGVzdCkKICAgICAgICBkb2NzPXNvdXJjZS5wYXJlbnQvJ2RvY3MnCiAgICAgICAgaWYgZG9jcy5pc19kaXIoKTogc2h1dGlsLmNvcHl0cmVlKGRvY3MsT1VULnBhcmVudC8nZG9jcycsZGlyc19leGlzdF9vaz1UcnVlKQogICAgc2Vzc2lvbj1kaWN0KHN0YXJ0ZWRfdW5peD1zdGFydGVkLHJlc3RvcmVfcz10aW1lLnRpbWUoKS1zdGFydGVkLGNhY2hlX3NvdXJjZT1zdHIoY2FjaGUpLHJlc3VtZV9zb3VyY2U9W3N0cihwLnBhcmVudCkgZm9yIHAgaW4gcHJpb3JdKQogICAgYXRvbWljX2pzb24oT1VULydzZXNzaW9uLmpzb24nLHNlc3Npb24pCiAgICBpbXBvcnQgcHN1dGlsCiAgICBwPXN1YnByb2Nlc3MucnVuKFsnbnZpZGlhLXNtaScsJy0tcXVlcnktZ3B1PW5hbWUsbWVtb3J5LnRvdGFsLG1lbW9yeS51c2VkJywnLS1mb3JtYXQ9Y3N2LG5vaGVhZGVyLG5vdW5pdHMnXSxjYXB0dXJlX291dHB1dD1UcnVlLHRleHQ9VHJ1ZSx0aW1lb3V0PTMwKQogICAgYXRvbWljX2pzb24oT1VULydyZXNvdXJjZV9wcmVmbGlnaHQuanNvbicsZGljdChzdGF0dXM9J1BBU1MnIGlmIHAucmV0dXJuY29kZT09MCBlbHNlICdVTlZFUklGSUVEJywKICAgICAgICAgICAgICAgIG9ic2VydmVkX2F0X3VuaXg9dGltZS50aW1lKCksZ3B1X3F1ZXJ5PXAuc3Rkb3V0LnN0cmlwKCksZ3B1X3F1ZXJ5X2Vycm9yPXAuc3RkZXJyLnN0cmlwKCksCiAgICAgICAgICAgICAgICByYW1fdG90YWxfYnl0ZXM9cHN1dGlsLnZpcnR1YWxfbWVtb3J5KCkudG90YWwscmFtX2F2YWlsYWJsZV9ieXRlcz1wc3V0aWwudmlydHVhbF9tZW1vcnkoKS5hdmFpbGFibGUsCiAgICAgICAgICAgICAgICB3b3JraW5nX2Rpc2s9ZGljdCh6aXAoWyd0b3RhbCcsJ3VzZWQnLCdmcmVlJ10sc2h1dGlsLmRpc2tfdXNhZ2UoJy9rYWdnbGUvd29ya2luZycpKSksCiAgICAgICAgICAgICAgICBzY3JhdGNoX2Rpc2s9ZGljdCh6aXAoWyd0b3RhbCcsJ3VzZWQnLCdmcmVlJ10sc2h1dGlsLmRpc2tfdXNhZ2UoJy9rYWdnbGUvdGVtcCcpKSksCiAgICAgICAgICAgICAgICBvdXRwdXRfc3RvcmFnZT0nd29ya2luZyBleHBvcnRlZCBvbmx5IHdoZW4gS2FnZ2xlIGNvbW1pdHMgY29tcGxldGVkIGpvYiBvdXRwdXQ7IHRlbXAgZXBoZW1lcmFsOyBpbnB1dCByZWFkLW9ubHkgcHJpb3IgY29tbWl0dGVkIG91dHB1dHMnLAogICAgICAgICAgICAgICAgZ3B1X3F1b3RhX3NuYXBzaG90PXJlYWRfanNvbihIRVJFLydxdW90YV9zbmFwc2hvdC5qc29uJyksb3BlcmF0aW9uYWxfYnVkZ2V0cz1jLAogICAgICAgICAgICAgICAgcmVxdWVzdGVkX2tlcm5lbF90aW1lb3V0X3M9NzIwMCxleHRlcm5hbF9wbGF0Zm9ybV9tYXhfc2Vzc2lvbl9saW1pdD0nVU5WRVJJRklFRCBmcm9tIGR5bmFtaWMgcHVibGljIGRvY3M7IG5vdCBhc3N1bWVkIGZvciBzY2hlZHVsaW5nJywKICAgICAgICAgICAgICAgIGNhY2hlX21hbmlmZXN0PXJlYWRfanNvbihjYWNoZS5wYXJlbnQvJ3Jlc3VsdHMvY2FjaGUtbWFuaWZlc3QuanNvbicpLG5ldHdvcmtfZHVyaW5nX2V2YWx1YXRpb249RmFsc2UsCiAgICAgICAgICAgICAgICByZXN0b3JlZF9jYWNoZV9zaGEyNTY9ZW52aW5mb1snc2hhMjU2J10sc2V0dXBfcmVzdG9yZWRfd2l0aG91dF9pbnN0YWxsPVRydWUpKQogICAgc2h1dGlsLmNvcHkyKGNhY2hlLnBhcmVudC8ncmVzdWx0cy9kYXRhc2V0X21hbmlmZXN0Lmpzb24nLE9VVC8nZGF0YXNldF9tYW5pZmVzdC5qc29uJykKICAgIHJldHVybiBjYWNoZQoKZGVmIHN0YWdlKG5hbWUpOgogICAgY2ZnPXJlYWRfanNvbihIRVJFLydjb25maWcuanNvbicpOyBzZXNzaW9uPXJlYWRfanNvbihPVVQvJ3Nlc3Npb24uanNvbicpCiAgICByZW1haW5pbmc9KGNmZ1snTUFYX1RPVEFMX1dBTExUSU1FX01JTiddLWNmZ1snU0hVVERPV05fUkVTRVJWRV9NSU4nXSkqNjAtKHRpbWUudGltZSgpLXNlc3Npb25bJ3N0YXJ0ZWRfdW5peCddKQogICAgaWYgcmVtYWluaW5nPDYwOiByYWlzZSBUaW1lb3V0RXJyb3IoJ0luc3VmZmljaWVudCB0aW1lIGJlZm9yZSBleHBvcnQgcmVzZXJ2ZScpCiAgICBwcmludCgnU1RBR0UnLG5hbWUsJ2J1ZGdldCByZW1haW5pbmcnLHJvdW5kKHJlbWFpbmluZyksJ3NlY29uZHMnLGZsdXNoPVRydWUpCiAgICB3aXRoIChPVVQvKG5hbWUrJy1zdGFnZS5sb2cnKSkub3BlbigndycpIGFzIGY6CiAgICAgICAgcmVzdWx0PXN1YnByb2Nlc3MucnVuKFsnL2thZ2dsZS90ZW1wL2ZsbW0tZW52L2Jpbi9weXRob24nLHN0cihIRVJFLydlbmdpbmUucHknKSxuYW1lXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc3Rkb3V0PWYsc3RkZXJyPXN1YnByb2Nlc3MuU1RET1VULGVudj1vcy5lbnZpcm9uLmNvcHkoKSx0aW1lb3V0PXJlbWFpbmluZykKICAgIHByaW50KChPVVQvKG5hbWUrJy1zdGFnZS5sb2cnKSkucmVhZF90ZXh0KGVycm9ycz0ncmVwbGFjZScpWy0yNTAwOl0sZmx1c2g9VHJ1ZSkKICAgIGlmIHJlc3VsdC5yZXR1cm5jb2RlOgogICAgICAgIGF0b21pY19qc29uKE9VVC8obmFtZSsnLWNvbnRyb2xsZXItZmFpbHVyZS5qc29uJyksZGljdChzdGF0dXM9J0ZBSUwnLHJldHVybmNvZGU9cmVzdWx0LnJldHVybmNvZGUpKQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmJ3tuYW1lfSBzdGFnZSBmYWlsZWQ7IGVhcmxpZXIgdmFsaWQgYXJ0aWZhY3RzIHByZXNlcnZlZCcpCgo=', 'common.py': 'IiIiUHVyZS1zdGFuZGFyZC1saWJyYXJ5IGludGVncml0eSwgYnVkZ2V0LCBhbmQgcmVzdW1hYmlsaXR5IHByaW1pdGl2ZXMuIiIiCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHRpbWUKCmRlZiBjYW5vbmljYWwodmFsdWUpOgogICAgcmV0dXJuIGpzb24uZHVtcHModmFsdWUsIHNvcnRfa2V5cz1UcnVlLCBzZXBhcmF0b3JzPSgnLCcsICc6JyksIGVuc3VyZV9hc2NpaT1GYWxzZSkKCmRlZiBkaWdlc3QodmFsdWUpOgogICAgcmV0dXJuIGhhc2hsaWIuc2hhMjU2KGNhbm9uaWNhbCh2YWx1ZSkuZW5jb2RlKCkpLmhleGRpZ2VzdCgpCgpkZWYgc2hhMjU2KHBhdGgpOgogICAgaD1oYXNobGliLnNoYTI1NigpCiAgICB3aXRoIG9wZW4ocGF0aCwncmInKSBhcyBmOgogICAgICAgIGZvciBiIGluIGl0ZXIobGFtYmRhOmYucmVhZCg4KjEwMjQqMTAyNCksYicnKTogaC51cGRhdGUoYikKICAgIHJldHVybiBoLmhleGRpZ2VzdCgpCgpkZWYgYXRvbWljX2pzb24ocGF0aCx2YWx1ZSk6CiAgICBwYXRoPVBhdGgocGF0aCk7IHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKQogICAgdGVtcD1wYXRoLndpdGhfbmFtZShwYXRoLm5hbWUrJy50bXAnKQogICAgd2l0aCB0ZW1wLm9wZW4oJ3cnLGVuY29kaW5nPSd1dGYtOCcpIGFzIGY6CiAgICAgICAgZi53cml0ZShqc29uLmR1bXBzKHZhbHVlLGluZGVudD0yLGVuc3VyZV9hc2NpaT1GYWxzZSkpOyBmLmZsdXNoKCk7IG9zLmZzeW5jKGYuZmlsZW5vKCkpCiAgICBvcy5yZXBsYWNlKHRlbXAscGF0aCkKCmRlZiByZWFkX2pzb24ocGF0aCk6CiAgICByZXR1cm4ganNvbi5sb2FkcyhQYXRoKHBhdGgpLnJlYWRfdGV4dChlbmNvZGluZz0ndXRmLTgnKSkKCmRlZiBldmVudChyb290LHNhbXBsZV9pZCxzdGF0dXMsKipleHRyYSk6CiAgICBhc3NlcnQgc3RhdHVzIGluIHsnUEVORElORycsJ1JVTk5JTkcnLCdET05FJywnRkFJTEVEJywnU0tJUFBFRCd9CiAgICBwYXRoPVBhdGgocm9vdCkvJ3J1bl9tYW5pZmVzdC5qc29ubCc7IHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKQogICAgd2l0aCBwYXRoLm9wZW4oJ2EnLGVuY29kaW5nPSd1dGYtOCcpIGFzIGY6CiAgICAgICAgZi53cml0ZShjYW5vbmljYWwoZGljdCh0aW1lX3VuaXg9dGltZS50aW1lKCksc2FtcGxlX2lkPXNhbXBsZV9pZCxzdGF0dXM9c3RhdHVzLCoqZXh0cmEpKSsnXG4nKQogICAgICAgIGYuZmx1c2goKTsgb3MuZnN5bmMoZi5maWxlbm8oKSkKCmRlZiB2YWxpZF9yZWNvcmQocGF0aCxmaW5nZXJwcmludCxzYW1wbGVfaGFzaCxjaGVja19tYXNrcz1UcnVlKToKICAgIHRyeToKICAgICAgICByPXJlYWRfanNvbihwYXRoKQogICAgICAgIGlmIHJbJ3N0YXR1cyddIT0nRE9ORScgb3IgclsnZmluZ2VycHJpbnQnXSE9ZmluZ2VycHJpbnQgb3Igclsnc2FtcGxlX2hhc2gnXSE9c2FtcGxlX2hhc2g6CiAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgaWYgci5nZXQoJ3JlY29yZF9oYXNoJykhPWRpZ2VzdCh7azp2IGZvciBrLHYgaW4gci5pdGVtcygpIGlmIGshPSdyZWNvcmRfaGFzaCd9KTogcmV0dXJuIE5vbmUKICAgICAgICBpZiBub3QgKGxlbihyWydpbnRlcnNlY3Rpb24nXSk9PWxlbihyWyd1bmlvbiddKT09bGVuKHJbJ2V4cHJlc3Npb25zJ10pKTogcmV0dXJuIE5vbmUKICAgICAgICBpZiBhbnkoaTwwIG9yIHU8MCBvciBpPnUgZm9yIGksdSBpbiB6aXAoclsnaW50ZXJzZWN0aW9uJ10sclsndW5pb24nXSkpOiByZXR1cm4gTm9uZQogICAgICAgIGlmIGNoZWNrX21hc2tzIGFuZCByLmdldCgnbWFza19maWxlJyk6CiAgICAgICAgICAgIG1hc2s9UGF0aChwYXRoKS5wYXJlbnQvclsnbWFza19maWxlJ10KICAgICAgICAgICAgaWYgbm90IG1hc2suaXNfZmlsZSgpIG9yIHNoYTI1NihtYXNrKSE9clsnbWFza19zaGEyNTYnXTogcmV0dXJuIE5vbmUKICAgICAgICByZXR1cm4gcgogICAgZXhjZXB0IChPU0Vycm9yLFZhbHVlRXJyb3IsS2V5RXJyb3IsVHlwZUVycm9yKTogcmV0dXJuIE5vbmUKCmRlZiBhZ2dyZWdhdGUocmVjb3Jkcyk6CiAgICBzZWVuPXNldCgpOyBpbnRlcnNlY3Rpb25zPVtdOyB1bmlvbnM9W10KICAgIGZvciByIGluIHJlY29yZHM6CiAgICAgICAgaWYgclsnc2FtcGxlX2lkJ10gaW4gc2VlbjogcmFpc2UgVmFsdWVFcnJvcignRHVwbGljYXRlIHNhbXBsZSBjb3VudGVkJykKICAgICAgICBzZWVuLmFkZChyWydzYW1wbGVfaWQnXSk7IGludGVyc2VjdGlvbnMuZXh0ZW5kKHJbJ2ludGVyc2VjdGlvbiddKTsgdW5pb25zLmV4dGVuZChyWyd1bmlvbiddKQogICAgdG90YWxfaT1zdW0oaW50ZXJzZWN0aW9ucyk7IHRvdGFsX3U9c3VtKHVuaW9ucykKICAgIGlmIG5vdCB1bmlvbnM6IHJldHVybiBkaWN0KGltYWdlX2NvdW50PTAsbWFza19jb3VudD0wLGNJb1U9Tm9uZSxtSW9VPU5vbmUpCiAgICAjIE1hdGNoIFJlZlNlZ01ldHJpYzogZW1wdHkgR1QvcHJlZCBoYXMgSW9VIDE7IGNJb1UgdXNlcyBnbG9iYWwgc3Vtcy4KICAgIHBlcj1baS91IGlmIHUgZWxzZSAxLjAgZm9yIGksdSBpbiB6aXAoaW50ZXJzZWN0aW9ucyx1bmlvbnMpXQogICAgcmV0dXJuIGRpY3QoaW1hZ2VfY291bnQ9bGVuKHNlZW4pLG1hc2tfY291bnQ9bGVuKHVuaW9ucyksaW50ZXJzZWN0aW9uPXRvdGFsX2ksdW5pb249dG90YWxfdSwKICAgICAgICAgICAgICAgIGNJb1U9MTAwKnRvdGFsX2kvdG90YWxfdSBpZiB0b3RhbF91IGVsc2UgMTAwLjAsbUlvVT0xMDAqc3VtKHBlcikvbGVuKHBlcikpCgpkZWYgYnVkZ2V0X2RlY2lzaW9uKGNvdW50LHNlY29uZHNfcGVyX2ltYWdlLHJlbWFpbmluZ193YWxsX3MsZXZhbF9yZW1haW5pbmdfcyxmcmVlX2J5dGVzLAogICAgICAgICAgICAgICAgICAgIG91dHB1dF9ieXRlc19wZXJfaW1hZ2UscXVvdGFfcmVtYWluaW5nX3MsY29uZmlnKToKICAgIG5lZWRlZD1jb3VudCpzZWNvbmRzX3Blcl9pbWFnZSpjb25maWdbJ1NBRkVUWV9GQUNUT1InXQogICAgcmVzZXJ2ZT1jb25maWdbJ1NIVVRET1dOX1JFU0VSVkVfTUlOJ10qNjAKICAgIGZsb29yPWNvbmZpZ1snTUlOX0ZSRUVfRElTS19HQiddKjEwMjQqKjMKICAgIGRpc2tfY2FwPW1heCgwLGludCgoZnJlZV9ieXRlcy1mbG9vcikvbWF4KDEsb3V0cHV0X2J5dGVzX3Blcl9pbWFnZSpjb25maWdbJ1NBRkVUWV9GQUNUT1InXSkpKQogICAgdGltZV9jYXA9bWF4KDAsaW50KG1pbihtYXgoMCxyZW1haW5pbmdfd2FsbF9zLXJlc2VydmUpLGV2YWxfcmVtYWluaW5nX3MpL21heCgwLjAwMSxzZWNvbmRzX3Blcl9pbWFnZSpjb25maWdbJ1NBRkVUWV9GQUNUT1InXSkpKQogICAgcXVvdGFfY2FwPXRpbWVfY2FwIGlmIHF1b3RhX3JlbWFpbmluZ19zIGlzIE5vbmUgZWxzZSBtYXgoMCxpbnQobWF4KDAscXVvdGFfcmVtYWluaW5nX3MtcmVzZXJ2ZSkvbWF4KC4wMDEsc2Vjb25kc19wZXJfaW1hZ2UqY29uZmlnWydTQUZFVFlfRkFDVE9SJ10pKSkKICAgIGNhcD1taW4oY291bnQsZGlza19jYXAsdGltZV9jYXAscXVvdGFfY2FwKQogICAgc3RhdHVzPSdHTycgaWYgY2FwPT1jb3VudCBlbHNlICgnUkVEVUNFX1NDT1BFJyBpZiBjYXA+MCBlbHNlICdCTE9DS0VEJykKICAgIGlmIHF1b3RhX3JlbWFpbmluZ19zIGlzIE5vbmUgYW5kIGNhcD4wOiBzdGF0dXM9J1VOS05PV04nOyBjYXA9bWluKGNhcCwxMDApCiAgICByZXR1cm4gZGljdChzdGF0dXM9c3RhdHVzLHNlbGVjdGVkX21heF9pbWFnZXM9Y2FwLHRhcmdldF9pbWFnZXM9Y291bnQsZXN0aW1hdGVkX2luZmVyZW5jZV93aXRoX21hcmdpbl9zPW5lZWRlZCwKICAgICAgICAgICAgICAgIHRpbWVfY2FwYWNpdHlfaW1hZ2VzPXRpbWVfY2FwLGRpc2tfY2FwYWNpdHlfaW1hZ2VzPWRpc2tfY2FwLHF1b3RhX2NhcGFjaXR5X2ltYWdlcz1xdW90YV9jYXAsCiAgICAgICAgICAgICAgICBzaHV0ZG93bl9yZXNlcnZlX3M9cmVzZXJ2ZSxyZWFzb249J01lYXN1cmVkIGxhdGVuY3kvb3V0cHV0IHNpemUgd2l0aCBzYWZldHkgZmFjdG9yOyBxdW90YSBVTktOT1dOIGNhcHMgc2NvcGUgYXQgMTAwJyBpZiBxdW90YV9yZW1haW5pbmdfcyBpcyBOb25lIGVsc2UgJ01lYXN1cmVkIGxhdGVuY3kvb3V0cHV0IHNpemUgYW5kIG9ic2VydmVkIHF1b3RhIHNuYXBzaG90JykK', 'engine.py': '...', 'prepare.py': 'IiIiQ1BVLW9ubHkgcHJlcGFyYXRpb246IHZlcmlmaWVkIGNhY2hlLCBvZmZpY2lhbCBsb2FkZXIgbWFuaWZlc3QsIHJldXNhYmxlIGVudmlyb25tZW50LiIiIgppbXBvcnQgY29uY3VycmVudC5mdXR1cmVzCmltcG9ydCBpbnNwZWN0CmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHJhbmRvbQppbXBvcnQgc2h1dGlsCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRhcmZpbGUKaW1wb3J0IHRpbWUKaW1wb3J0IHVybGxpYi5yZXF1ZXN0CmltcG9ydCB6aXBmaWxlCmZyb20gY29tbW9uIGltcG9ydCBhdG9taWNfanNvbixkaWdlc3QscmVhZF9qc29uLHNoYTI1NgoKSEVSRT1QYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50CkM9cmVhZF9qc29uKEhFUkUvJ2NvbmZpZy5qc29uJykKV09SSz1QYXRoKCcva2FnZ2xlL3dvcmtpbmcnKTsgQ0FDSEU9V09SSy8nY2FjaGUnOyBSRVNVTFRTPVdPUksvJ3Jlc3VsdHMnCgpkZWYgZG93bmxvYWQodXJsLHBhdGgsZXhwZWN0ZWQ9Tm9uZSk6CiAgICBwYXRoPVBhdGgocGF0aCk7IHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKQogICAgaWYgcGF0aC5pc19maWxlKCkgYW5kIChleHBlY3RlZCBpcyBOb25lIG9yIHNoYTI1NihwYXRoKT09ZXhwZWN0ZWQpOiByZXR1cm4KICAgIGZvciBhdHRlbXB0IGluIHJhbmdlKENbJ01BWF9SRVRSSUVTX1BFUl9ET1dOTE9BRCddKToKICAgICAgICB0ZW1wPXBhdGgud2l0aF9zdWZmaXgocGF0aC5zdWZmaXgrJy5wYXJ0JykKICAgICAgICB0cnk6CiAgICAgICAgICAgIHdpdGggdXJsbGliLnJlcXVlc3QudXJsb3Blbih1cmwsdGltZW91dD1DWydET1dOTE9BRF9USU1FT1VUX1MnXSkgYXMgcix0ZW1wLm9wZW4oJ3diJykgYXMgZjoKICAgICAgICAgICAgICAgIHN0YXJ0PXRpbWUubW9ub3RvbmljKCkKICAgICAgICAgICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgICAgICAgICAgYj1yLnJlYWQoMTAyNCoxMDI0KQogICAgICAgICAgICAgICAgICAgIGlmIG5vdCBiOiBicmVhawogICAgICAgICAgICAgICAgICAgIGYud3JpdGUoYikKICAgICAgICAgICAgICAgICAgICBpZiB0aW1lLm1vbm90b25pYygpLXN0YXJ0PjkwMDogcmFpc2UgVGltZW91dEVycm9yKCdEb3dubG9hZCB0b3RhbCBkZWFkbGluZScpCiAgICAgICAgICAgIGlmIGV4cGVjdGVkIGFuZCBzaGEyNTYodGVtcCkhPWV4cGVjdGVkOiByYWlzZSBWYWx1ZUVycm9yKCdEb3dubG9hZCBjaGVja3N1bSBtaXNtYXRjaCcpCiAgICAgICAgICAgIG9zLnJlcGxhY2UodGVtcCxwYXRoKTsgcmV0dXJuCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgaWYgYXR0ZW1wdD09Q1snTUFYX1JFVFJJRVNfUEVSX0RPV05MT0FEJ10tMTogcmFpc2UKICAgICAgICAgICAgdGltZS5zbGVlcCgyKiphdHRlbXB0KQoKZGVmIHdvcmtlcigpOgogICAgZnJvbSBodWdnaW5nZmFjZV9odWIgaW1wb3J0IHNuYXBzaG90X2Rvd25sb2FkLGhmX2h1Yl9kb3dubG9hZAogICAgZnJvbSBtbWRldC5kYXRhc2V0cyBpbXBvcnQgUmVmQ29jb0RhdGFzZXQKICAgIGZyb20gUElMIGltcG9ydCBJbWFnZQogICAgaW1wb3J0IG1tZW5naW5lCiAgICBSRVNVTFRTLm1rZGlyKGV4aXN0X29rPVRydWUpOyBDQUNIRS5ta2RpcihleGlzdF9vaz1UcnVlKQogICAgc3RhcnRlZD10aW1lLm1vbm90b25pYygpCiAgICBhbm5vdGF0aW9uPUNBQ0hFLydyZWZjb2NvLnppcCcKICAgIGRvd25sb2FkKCdodHRwczovL3dlYi5hcmNoaXZlLm9yZy93ZWIvMjAyMjA0MTMwMTE3MThpZF8vaHR0cHM6Ly9idmlzaW9ud2ViMS5jcy51bmMuZWR1L2xpY2hlbmcvcmVmZXJpdC9kYXRhL3JlZmNvY28uemlwJyxhbm5vdGF0aW9uLENbJ0FOTk9UQVRJT05fU0hBMjU2J10pCiAgICBkYXRhPUNBQ0hFLydkYXRhL2NvY28nCiAgICB3aXRoIHppcGZpbGUuWmlwRmlsZShhbm5vdGF0aW9uKSBhcyB6OgogICAgICAgIGZvciBuYW1lIGluIFsncmVmY29jby9pbnN0YW5jZXMuanNvbicsJ3JlZmNvY28vcmVmcyh1bmMpLnAnXToKICAgICAgICAgICAgdGFyZ2V0PWRhdGEvbmFtZTsgdGFyZ2V0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSk7IHRhcmdldC53cml0ZV9ieXRlcyh6LnJlYWQobmFtZSkpCiAgICBkYXRhc2V0PVJlZkNvY29EYXRhc2V0KGRhdGFfcm9vdD1zdHIoZGF0YSksYW5uX2ZpbGU9J3JlZmNvY28vaW5zdGFuY2VzLmpzb24nLHNwbGl0X2ZpbGU9J3JlZmNvY28vcmVmcyh1bmMpLnAnLAogICAgICAgICAgICAgICAgICAgICAgICAgIGRhdGFfcHJlZml4PWRpY3QoaW1nX3BhdGg9J3RyYWluMjAxNC8nKSxzcGxpdD0ndmFsJyx0ZXh0X21vZGU9J3NlbGVjdF9maXJzdCcscGlwZWxpbmU9W10pCiAgICByZWZzPW1tZW5naW5lLmxvYWQoZGF0YS8ncmVmY29jby9yZWZzKHVuYykucCcsZmlsZV9mb3JtYXQ9J3BrbCcpCiAgICBpbnN0YW5jZXM9cmVhZF9qc29uKGRhdGEvJ3JlZmNvY28vaW5zdGFuY2VzLmpzb24nKQogICAgYW5uX2J5X2lkPXthWydpZCddOmEgZm9yIGEgaW4gaW5zdGFuY2VzWydhbm5vdGF0aW9ucyddfQogICAgc2VsZWN0ZWQ9e30KICAgIGZvciByZWYgaW4gcmVmczoKICAgICAgICBpZiByZWZbJ3NwbGl0J109PSd2YWwnOiBzZWxlY3RlZFtyZWZbJ2Fubl9pZCddXT1yZWYKICAgIGdyb3VwZWQ9e30KICAgIGZvciByZWYgaW4gc2VsZWN0ZWQudmFsdWVzKCk6IGdyb3VwZWQuc2V0ZGVmYXVsdChyZWZbJ2ltYWdlX2lkJ10sW10pLmFwcGVuZChyZWYpCiAgICByZWNvcmRzPVtdCiAgICBmb3IgaWR4IGluIHJhbmdlKGxlbihkYXRhc2V0KSk6CiAgICAgICAgaW5mbz1kYXRhc2V0LmdldF9kYXRhX2luZm8oaWR4KTsgZXhwcmVzc2lvbnM9W10KICAgICAgICBtYXRjaGluZz1ncm91cGVkW2luZm9bJ2ltZ19pZCddXQogICAgICAgIGlmIGxlbihtYXRjaGluZykhPWxlbihpbmZvWyd0ZXh0J10pOiByYWlzZSBWYWx1ZUVycm9yKCdSZWZlcmVuY2UgbWFwcGluZyBjb3VudCBtaXNtYXRjaCcpCiAgICAgICAgZm9yIG1hc2tfaWQsKHJlZix0ZXh0LGluc3QpIGluIGVudW1lcmF0ZSh6aXAobWF0Y2hpbmcsaW5mb1sndGV4dCddLGluZm9bJ2luc3RhbmNlcyddKSk6CiAgICAgICAgICAgIHNlbnRlbmNlPXJlZlsnc2VudGVuY2VzJ11bMF0KICAgICAgICAgICAgaWYgc2VudGVuY2VbJ3JhdyddLmxvd2VyKCkhPXRleHQgb3IgZGlnZXN0KGluc3RbJ21hc2snXSkhPWRpZ2VzdChhbm5fYnlfaWRbcmVmWydhbm5faWQnXV1bJ3NlZ21lbnRhdGlvbiddKToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ1JlZmVyZW5jZSBtYXBwaW5nIGRpZmZlcnMgZnJvbSBvZmZpY2lhbCBsb2FkZXInKQogICAgICAgICAgICBleHByZXNzaW9ucy5hcHBlbmQoZGljdChpbWFnZV9pZD1pbmZvWydpbWdfaWQnXSxkYXRhc2V0X2luZGV4PWlkeCxyZWZfaWQ9cmVmLmdldCgncmVmX2lkJyksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHNlbnRfaWQ9c2VudGVuY2UuZ2V0KCdzZW50X2lkJyksZXhwcmVzc2lvbj10ZXh0LG1hc2tfaWQ9bWFza19pZCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgYW5ub3RhdGlvbl9pZD1yZWZbJ2Fubl9pZCddLHByZWRpY3Rpb25faWQ9ZiJyZWZjb2NvLXZhbC17aW5mb1snaW1nX2lkJ119OnttYXNrX2lkfSIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGd0X3NlZ21lbnRhdGlvbl9zaGEyNTY9ZGlnZXN0KGluc3RbJ21hc2snXSksY2F0ZWdvcnlfaWQ9cmVmLmdldCgnY2F0ZWdvcnlfaWQnKSkpCiAgICAgICAgcmVjb3JkPWRpY3Qoc2FtcGxlX2lkPWYicmVmY29jby12YWwte2luZm9bJ2ltZ19pZCddfSIsaW1hZ2VfaWQ9aW5mb1snaW1nX2lkJ10sZGF0YXNldF9pbmRleD1pZHgsCiAgICAgICAgICAgICAgICAgICAgZmlsZV9uYW1lPVBhdGgoaW5mb1snaW1nX3BhdGgnXSkubmFtZSxleHByZXNzaW9ucz1leHByZXNzaW9ucywKICAgICAgICAgICAgICAgICAgICBndF9pbnN0YW5jZXNfc2hhMjU2PWRpZ2VzdChpbmZvWydpbnN0YW5jZXMnXSkpCiAgICAgICAgcmVjb3JkWydzYW1wbGVfaGFzaCddPWRpZ2VzdChyZWNvcmQpOyByZWNvcmRzLmFwcGVuZChyZWNvcmQpCiAgICByZWdyZXNzaW9uX2lkcz1beFsnaW1nX2lkJ10gZm9yIHggaW4gcmVhZF9qc29uKEhFUkUvJ3JlZ3Jlc3Npb24tbWFuaWZlc3QuanNvbicpXQogICAgYnlfaW1hZ2U9e3hbJ2ltYWdlX2lkJ106eCBmb3IgeCBpbiByZWNvcmRzfQogICAgaWYgW3hbJ2ltYWdlX2lkJ10gZm9yIHggaW4gcmVjb3Jkc1s6MjBdXSE9cmVncmVzc2lvbl9pZHM6IHJhaXNlIFZhbHVlRXJyb3IoJ1JlZ3Jlc3Npb24gbG9hZGVyIG9yZGVyIGNoYW5nZWQnKQogICAgcmVncmVzc2lvbj1bYnlfaW1hZ2VbaV1bJ3NhbXBsZV9pZCddIGZvciBpIGluIHJlZ3Jlc3Npb25faWRzXQogICAgcmVtYWluaW5nPXNvcnRlZChbciBmb3IgciBpbiByZWNvcmRzIGlmIHJbJ3NhbXBsZV9pZCddIG5vdCBpbiByZWdyZXNzaW9uXSxrZXk9bGFtYmRhIHI6clsnaW1hZ2VfaWQnXSkKICAgIHJhbmRvbS5SYW5kb20oQ1snU0VFRCddKS5zaHVmZmxlKHJlbWFpbmluZykKICAgIGRldj1yZW1haW5pbmdbOkNbJ0RFVkVMT1BNRU5UX0lNQUdFUyddXQogICAgaG9sZG91dD1yZW1haW5pbmdbQ1snREVWRUxPUE1FTlRfSU1BR0VTJ106Q1snREVWRUxPUE1FTlRfSU1BR0VTJ10rQ1snSE9MRE9VVF9JTUFHRVMnXV0KICAgIHNldHM9ZGljdChyZWdyZXNzaW9uPXJlZ3Jlc3Npb24scHJvYmU9W3JbJ3NhbXBsZV9pZCddIGZvciByIGluIGRldls6Q1snUFJPQkVfSU1BR0VTJ11dXSwKICAgICAgICAgICAgICBzdGFiaWxpdHk9W3JbJ3NhbXBsZV9pZCddIGZvciByIGluIGRldls6Q1snU1RBQklMSVRZX0lNQUdFUyddXV0sCiAgICAgICAgICAgICAgZGV2ZWxvcG1lbnQ9W3JbJ3NhbXBsZV9pZCddIGZvciByIGluIGRldl0saG9sZG91dD1bclsnc2FtcGxlX2lkJ10gZm9yIHIgaW4gaG9sZG91dF0sCiAgICAgICAgICAgICAgZnVsbF92YWxpZGF0aW9uPVtyWydzYW1wbGVfaWQnXSBmb3IgciBpbiByZWNvcmRzXSkKICAgIGFzc2VydCBub3Qgc2V0KHNldHNbJ2RldmVsb3BtZW50J10pJnNldChzZXRzWydob2xkb3V0J10pCiAgICBoaXN0PXt9CiAgICBmb3IgbmFtZSxpZHMgaW4gc2V0cy5pdGVtcygpOgogICAgICAgIGNob3Nlbj1bciBmb3IgciBpbiByZWNvcmRzIGlmIHJbJ3NhbXBsZV9pZCddIGluIHNldChpZHMpXQogICAgICAgIGhpc3RbbmFtZV09ZGljdChpbWFnZXM9bGVuKGNob3Nlbiksc2VsZWN0ZWRfbWFza3M9c3VtKGxlbihyWydleHByZXNzaW9ucyddKSBmb3IgciBpbiBjaG9zZW4pLAogICAgICAgICAgICAgICAgICAgICAgIG9uZV90YXJnZXQ9c3VtKGxlbihyWydleHByZXNzaW9ucyddKT09MSBmb3IgciBpbiBjaG9zZW4pLAogICAgICAgICAgICAgICAgICAgICAgIHRocmVlX29yX21vcmVfdGFyZ2V0cz1zdW0obGVuKHJbJ2V4cHJlc3Npb25zJ10pPj0zIGZvciByIGluIGNob3NlbikpCiAgICBtYW5pZmVzdD1kaWN0KHN0YXR1cz0nUEFTUycsc2VlZD1DWydTRUVEJ10sc3BsaXQ9J1JlZkNPQ08gVU5DIHZhbCcsdGV4dF9tb2RlPSdzZWxlY3RfZmlyc3QnLGNvbmNhdD1UcnVlLAogICAgICAgICAgICAgICAgICBzZWxlY3Rpb25fcnVsZT0nU2VlZGVkIHNpbXBsZSByYW5kb20gc2h1ZmZsZSBvZiBzb3J0ZWQgaW1hZ2UgSURzLCBleGNsdWRpbmcgMjAgZGVidWcgaW1hZ2VzOyBwcm9iZSBuZXN0ZWQgaW4gc3RhYmlsaXR5IG5lc3RlZCBpbiBkZXZlbG9wbWVudDsgaG9sZG91dCBkaXNqb2ludCcsCiAgICAgICAgICAgICAgICAgIHVuaXQ9J29uZSBpbWFnZSBpbmZlcmVuY2U7IG9uZSBzZWxlY3RlZCBmaXJzdCBzZW50ZW5jZSBwZXIgcmV0YWluZWQgYW5ub3RhdGlvbi9yZWZlcmVuY2UnLAogICAgICAgICAgICAgICAgICBjb3VudHM9ZGljdChpbWFnZXM9bGVuKHJlY29yZHMpLHJhd192YWxfcmVmZXJlbmNlcz1zdW0oclsnc3BsaXQnXT09J3ZhbCcgZm9yIHIgaW4gcmVmcyksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJhd192YWxfc2VudGVuY2VzPXN1bShsZW4oclsnc2VudGVuY2VzJ10pIGZvciByIGluIHJlZnMgaWYgclsnc3BsaXQnXT09J3ZhbCcpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICByZXRhaW5lZF9yZWZlcmVuY2VzPWxlbihzZWxlY3RlZCksc2VsZWN0ZWRfdGFyZ2V0X21hc2tzPXN1bShsZW4oclsnZXhwcmVzc2lvbnMnXSkgZm9yIHIgaW4gcmVjb3JkcykpLAogICAgICAgICAgICAgICAgICBhbm5vdGF0aW9uX3ppcF9zaGEyNTY9c2hhMjU2KGFubm90YXRpb24pLGluc3RhbmNlc19zaGEyNTY9c2hhMjU2KGRhdGEvJ3JlZmNvY28vaW5zdGFuY2VzLmpzb24nKSwKICAgICAgICAgICAgICAgICAgcmVmc19zaGEyNTY9c2hhMjU2KGRhdGEvJ3JlZmNvY28vcmVmcyh1bmMpLnAnKSxzYW1wbGVzPXJlY29yZHMsc3Vic2V0cz1zZXRzLGRpc3RyaWJ1dGlvbj1oaXN0KQogICAgbWFuaWZlc3RbJ21hbmlmZXN0X2hhc2gnXT1kaWdlc3QobWFuaWZlc3QpCiAgICBhdG9taWNfanNvbihSRVNVTFRTLydkYXRhc2V0X21hbmlmZXN0Lmpzb24nLG1hbmlmZXN0KQogICAgKFJFU1VMVFMvJ3JlZmNvY28tbG9hZGVyLnB5Jykud3JpdGVfdGV4dChpbnNwZWN0LmdldHNvdXJjZShSZWZDb2NvRGF0YXNldCkpCiAgICAoUkVTVUxUUy8nbG9hZGVyLW1vZHVsZS5weScpLndyaXRlX3RleHQoUGF0aChpbnNwZWN0LmdldGZpbGUoUmVmQ29jb0RhdGFzZXQpKS5yZWFkX3RleHQoKSkKICAgIHJlcXVpcmVkPXNldChzZXRzWydyZWdyZXNzaW9uJ10rc2V0c1snZGV2ZWxvcG1lbnQnXStzZXRzWydob2xkb3V0J10pCiAgICBpZiBDWydSVU5fRlVMTF9WQUxJREFUSU9OJ106IHJlcXVpcmVkPXNldChzZXRzWydmdWxsX3ZhbGlkYXRpb24nXSkKICAgIGRlZiBpbWFnZV90YXNrKHJlY29yZCk6CiAgICAgICAgdGFyZ2V0PWRhdGEvJ3RyYWluMjAxNCcvcmVjb3JkWydmaWxlX25hbWUnXQogICAgICAgIGRvd25sb2FkKCdodHRwczovL3MzLmFtYXpvbmF3cy5jb20vaW1hZ2VzLmNvY29kYXRhc2V0Lm9yZy90cmFpbjIwMTQvJytyZWNvcmRbJ2ZpbGVfbmFtZSddLHRhcmdldCkKICAgICAgICB3aXRoIEltYWdlLm9wZW4odGFyZ2V0KSBhcyBpbTogaW0udmVyaWZ5KCkKICAgICAgICB3aXRoIEltYWdlLm9wZW4odGFyZ2V0KSBhcyBpbTogc2hhcGU9bGlzdChpbS5zaXplKQogICAgICAgIHJldHVybiBkaWN0KHNhbXBsZV9pZD1yZWNvcmRbJ3NhbXBsZV9pZCddLGZpbGVfbmFtZT1yZWNvcmRbJ2ZpbGVfbmFtZSddLHNoYTI1Nj1zaGEyNTYodGFyZ2V0KSxieXRlcz10YXJnZXQuc3RhdCgpLnN0X3NpemUsc2l6ZT1zaGFwZSkKICAgIHdpdGggY29uY3VycmVudC5mdXR1cmVzLlRocmVhZFBvb2xFeGVjdXRvcihtYXhfd29ya2Vycz02KSBhcyBwb29sOgogICAgICAgIGltYWdlcz1saXN0KHBvb2wubWFwKGltYWdlX3Rhc2ssW3IgZm9yIHIgaW4gcmVjb3JkcyBpZiByWydzYW1wbGVfaWQnXSBpbiByZXF1aXJlZF0pKQogICAgYXRvbWljX2pzb24oUkVTVUxUUy8naW1hZ2UtY2FjaGUuanNvbicsZGljdChzdGF0dXM9J1BBU1MnLGltYWdlcz1pbWFnZXMsdG90YWxfYnl0ZXM9c3VtKHhbJ2J5dGVzJ10gZm9yIHggaW4gaW1hZ2VzKSkpCiAgICBtb2RlbF9kaXI9Q0FDSEUvJ21vZGVscyc7IG1vZGVsX2Rpci5ta2RpcihleGlzdF9vaz1UcnVlKQogICAgb3MuZW52aXJvblsnSEZfSFVCX0VUQUdfVElNRU9VVCddPSc0NSc7IG9zLmVudmlyb25bJ0hGX0hVQl9ET1dOTE9BRF9USU1FT1VUJ109JzQ1JwogICAgc25hcHNob3Q9c25hcHNob3RfZG93bmxvYWQoJ2RlZXBzZWVrLWFpL2RlZXBzZWVrLXZsLTEuM2ItY2hhdCcscmV2aXNpb249Q1snQkFDS0JPTkVfUkVWSVNJT04nXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGxvY2FsX2Rpcj1zdHIobW9kZWxfZGlyLydiYWNrYm9uZScpLGFsbG93X3BhdHRlcm5zPVsnKi5qc29uJywnKi5zYWZldGVuc29ycyddLG1heF93b3JrZXJzPTMpCiAgICBiYWNrYm9uZT1QYXRoKHNuYXBzaG90KS8nbW9kZWwuc2FmZXRlbnNvcnMnCiAgICBpZiBzaGEyNTYoYmFja2JvbmUpIT1DWydCQUNLQk9ORV9TSEEyNTYnXTogcmFpc2UgVmFsdWVFcnJvcignQmFja2JvbmUgTEZTIGhhc2ggbWlzbWF0Y2gnKQogICAgZmxtbT1oZl9odWJfZG93bmxvYWQoJ3d1c2l6ZS9mbG1tJywnZnJvemVuX2RlZXBzZWVrX3ZsXzFfM2JfY2hhdF91bmV0X3NhbV9sX3JlZmNvY29fcG5nLnB0aCcscmV2aXNpb249Q1snRkxNTV9SRVZJU0lPTiddLGxvY2FsX2Rpcj1zdHIobW9kZWxfZGlyLydmbG1tJykpCiAgICBpZiBzaGEyNTYoZmxtbSkhPUNbJ0ZMTU1fU0hBMjU2J106IHJhaXNlIFZhbHVlRXJyb3IoJ0YtTE1NIGhhc2ggbWlzbWF0Y2gnKQogICAgZG93bmxvYWQoJ2h0dHBzOi8vZGwuZmJhaXB1YmxpY2ZpbGVzLmNvbS9zZWdtZW50X2FueXRoaW5nL3NhbV92aXRfbF8wYjMxOTUucHRoJyxtb2RlbF9kaXIvJ3NhbV92aXRfbF8wYjMxOTUucHRoJyxDWydTQU1fU0hBMjU2J10pCiAgICBhdG9taWNfanNvbihSRVNVTFRTLydjYWNoZS1tYW5pZmVzdC5qc29uJyxkaWN0KHN0YXR1cz0nUEFTUycsbW9kZWxzPVtkaWN0KHBhdGg9c3RyKHAucmVsYXRpdmVfdG8oQ0FDSEUpKSxzaGEyNTY9c2hhMjU2KHApLGJ5dGVzPXAuc3RhdCgpLnN0X3NpemUpIGZvciBwIGluIFtiYWNrYm9uZSxQYXRoKGZsbW0pLG1vZGVsX2Rpci8nc2FtX3ZpdF9sXzBiMzE5NS5wdGgnXV0sZGF0YV9wcmVwYXJhdGlvbl9zZWNvbmRzPXRpbWUubW9ub3RvbmljKCktc3RhcnRlZCkpCgpkZWYgbWFpbigpOgogICAgUkVTVUxUUy5ta2RpcihleGlzdF9vaz1UcnVlKTsgQ0FDSEUubWtkaXIoZXhpc3Rfb2s9VHJ1ZSkKICAgIGVudj1vcy5lbnZpcm9uLmNvcHkoKTsgZW52LnVwZGF0ZShVVl9DQUNIRV9ESVI9Jy9rYWdnbGUvdGVtcC91di1jYWNoZScsVVZfUFlUSE9OX0lOU1RBTExfRElSPScva2FnZ2xlL3RlbXAvcHl0aG9uJyxQSVBfTk9fQ0FDSEVfRElSPScxJyxEU19CVUlMRF9PUFM9JzAnLEhGX0hVQl9ESVNBQkxFX1hFVD0nMScsSEZfSE9NRT0nL2thZ2dsZS90ZW1wL2hmJykKICAgIHN0ZXBzPVtdOyBzdGFydD10aW1lLm1vbm90b25pYygpCiAgICBkZWYgcnVuKG5hbWUsY21kLHRpbWVvdXQ9OTAwKToKICAgICAgICBwcmludCgnU1RBUlQnLG5hbWUsZmx1c2g9VHJ1ZSk7IHQ9dGltZS5tb25vdG9uaWMoKQogICAgICAgIHdpdGggKFJFU1VMVFMvKG5hbWUrJy5sb2cnKSkub3BlbigndycpIGFzIGY6CiAgICAgICAgICAgIHA9c3VicHJvY2Vzcy5ydW4oY21kLGVudj1lbnYsc3Rkb3V0PWYsc3RkZXJyPXN1YnByb2Nlc3MuU1RET1VULHRpbWVvdXQ9dGltZW91dCkKICAgICAgICBzdGVwcy5hcHBlbmQoZGljdChzdGFnZT1uYW1lLHN0YXR1cz0nUEFTUycgaWYgcC5yZXR1cm5jb2RlPT0wIGVsc2UgJ0ZBSUwnLHNlY29uZHM9dGltZS5tb25vdG9uaWMoKS10KSkKICAgICAgICBhdG9taWNfanNvbihSRVNVTFRTLydwcmVwYXJlLXN0YWdlcy5qc29uJyxzdGVwcykKICAgICAgICBwcmludCgoUkVTVUxUUy8obmFtZSsnLmxvZycpKS5yZWFkX3RleHQoZXJyb3JzPSdyZXBsYWNlJylbLTE1MDA6XSxmbHVzaD1UcnVlKQogICAgICAgIGlmIHAucmV0dXJuY29kZTogcmFpc2UgUnVudGltZUVycm9yKGYne25hbWV9OiBleGl0IHtwLnJldHVybmNvZGV9JykKICAgIHJ1bigndXYnLFtzeXMuZXhlY3V0YWJsZSwnLW0nLCdwaXAnLCdpbnN0YWxsJywndXYnXSkKICAgIHV2PVtzeXMuZXhlY3V0YWJsZSwnLW0nLCd1diddOyBweT0nL2thZ2dsZS90ZW1wL2ZsbW0tZW52L2Jpbi9weXRob24nCiAgICBpZiBub3QgUGF0aChweSkuZXhpc3RzKCk6IHJ1bignZW52aXJvbm1lbnQnLHV2K1sndmVudicsJy0tcHl0aG9uJywnMy4xMC4yMicsJy9rYWdnbGUvdGVtcC9mbG1tLWVudiddKQogICAgIyBSZXVzZSB0aGUgc3VjY2Vzc2Z1bCBmcmVlemUgYXMgY29uc3RyYWludHMsIGluY2x1ZGluZyBhbGwgdHJhbnNpdGl2ZSB2ZXJzaW9ucy4KICAgIHBpcD11ditbJ3BpcCcsJ2luc3RhbGwnLCctLXB5dGhvbicscHksJy1jJyxzdHIoSEVSRS8nY29uc3RyYWludHMudHh0JyldCiAgICAjIFB5VG9yY2gncyB3aGVlbCBpbmRleCBkb2VzIG5vdCBtaXJyb3IgZXZlcnkgdHJhbnNpdGl2ZSBQeVBJIHZlcnNpb24gaW4gdGhlCiAgICAjIHN1Y2Nlc3NmdWwgZnJlZXplLiBJbnN0YWxsIGl0cyB3aGVlbHMgZmlyc3QsIHRoZW4gY29uc3RyYWluIFB5UEkgZGVwZW5kZW5jaWVzLgogICAgcnVuKCd0b3JjaCcsdXYrWydwaXAnLCdpbnN0YWxsJywnLS1weXRob24nLHB5LCd0b3JjaD09Mi4xLjIrY3UxMTgnLCd0b3JjaHZpc2lvbj09MC4xNi4yK2N1MTE4JywnLS1pbmRleC11cmwnLCdodHRwczovL2Rvd25sb2FkLnB5dG9yY2gub3JnL3dobC9jdTExOCddKQogICAgcnVuKCdkZXBlbmRlbmNpZXMnLHBpcCtbJ251bXB5PT0xLjI2LjQnLCd0cmFuc2Zvcm1lcnM9PTQuMzkuMScsJ2FjY2VsZXJhdGU9PTAuMjguMCcsJ21tZW5naW5lPT0wLjEwLjUnLCdtbWRldD09My4zLjAnLCdtbXNlZ21lbnRhdGlvbj09MS4yLjInLCd4dHVuZXI9PTAuMS4yMycsJ3BlZnQ9PTAuMTAuMCcsJ2RhdGFzZXRzPT0yLjE5LjInLCdiaXRzYW5kYnl0ZXM9PTAuNDMuMScsJ3RpbW09PTAuOS4xNicsJ2F0dHJkaWN0JywnZWlub3BzJywnc2VudGVuY2VwaWVjZScsJ2Z0ZnknLCdvcGVuY3YtcHl0aG9uLWhlYWRsZXNzPT00LjEwLjAuODQnLCdzZXR1cHRvb2xzPDgxJywncGlwJywnd2hlZWwnLCdwYWNrYWdpbmcnLCduaW5qYScsJ3BzdXRpbCcsJ3B5LWNwdWluZm8nLCdoanNvbicsJ3B5ZGFudGljPDMnXSkKICAgIHJ1bignbW1jdicscGlwK1snbW1jdj09Mi4xLjAnLCctLW9ubHktYmluYXJ5JywnbW1jdicsJy0tZmluZC1saW5rcycsJ2h0dHBzOi8vZG93bmxvYWQub3Blbm1tbGFiLmNvbS9tbWN2L2Rpc3QvY3UxMTgvdG9yY2gyLjEvaW5kZXguaHRtbCddKQogICAgcnVuKCdkZWVwc3BlZWQnLHBpcCtbJy0tbm8tYnVpbGQtaXNvbGF0aW9uJywnZGVlcHNwZWVkPT0wLjE0LjInXSkKICAgIHJlcG89Q0FDSEUvJ0YtTE1NJwogICAgaWYgbm90IHJlcG8uZXhpc3RzKCk6IHJ1bignc291cmNlJyxbJ2dpdCcsJ2Nsb25lJywnaHR0cHM6Ly9naXRodWIuY29tL3d1c2l6ZS9GLUxNTS5naXQnLHN0cihyZXBvKV0pCiAgICBydW4oJ3JldmlzaW9uJyxbJ2dpdCcsJy1DJyxzdHIocmVwbyksJ2NoZWNrb3V0JyxDWydTT1VSQ0VfU0hBJ11dKQogICAgZW52WydQWVRIT05QQVRIJ109c3RyKHJlcG8pKyc6JytzdHIoSEVSRSkKICAgIGVudlsnTERfTElCUkFSWV9QQVRIJ109Jy9rYWdnbGUvdGVtcC9mbG1tLWVudi9saWIvcHl0aG9uMy4xMC9zaXRlLXBhY2thZ2VzL3RvcmNoL2xpYjonK2Vudi5nZXQoJ0xEX0xJQlJBUllfUEFUSCcsJycpCiAgICBydW4oJ2RhdGEtbW9kZWwtY2FjaGUnLFtweSxzdHIoSEVSRS8ncHJlcGFyZS5weScpLCd3b3JrZXInXSx0aW1lb3V0PTI0MDApCiAgICBydW4oJ2ZyZWV6ZScsdXYrWydwaXAnLCdmcmVlemUnLCctLXB5dGhvbicscHldKQogICAgZm9yIHAgaW4gW1BhdGgoJy9rYWdnbGUvdGVtcC9mbG1tLWVudicpLFBhdGgoJy9rYWdnbGUvdGVtcC9weXRob24nKV06CiAgICAgICAgcHJpbnQoJ0NBQ0hFX0VOVl9CWVRFUycscCxzdW0oeC5zdGF0KCkuc3Rfc2l6ZSBmb3IgeCBpbiBwLnJnbG9iKCcqJykgaWYgeC5pc19maWxlKCkgYW5kIG5vdCB4LmlzX3N5bWxpbmsoKSksZmx1c2g9VHJ1ZSkKICAgIHJ1bigncGFja2FnZS1lbnZpcm9ubWVudCcsWyd0YXInLCctSScsJ2d6aXAgLTEnLCctY2YnLHN0cihDQUNIRS8nZW52aXJvbm1lbnQudGFyLmd6JyksJy1DJywnL2thZ2dsZS90ZW1wJywnZmxtbS1lbnYnLCdweXRob24nXSx0aW1lb3V0PTEyMDApCiAgICBhdG9taWNfanNvbihDQUNIRS8nZW52aXJvbm1lbnQuanNvbicsZGljdChzdGF0dXM9J1BBU1MnLHNoYTI1Nj1zaGEyNTYoQ0FDSEUvJ2Vudmlyb25tZW50LnRhci5neicpLGJ5dGVzPShDQUNIRS8nZW52aXJvbm1lbnQudGFyLmd6Jykuc3RhdCgpLnN0X3NpemUscHl0aG9uPSczLjEwLjIyJyxyZXN0b3JlX3RvPScva2FnZ2xlL3RlbXAnLGZyZWV6ZV9zaGEyNTY9c2hhMjU2KFJFU1VMVFMvJ2ZyZWV6ZS5sb2cnKSkpCiAgICBhdG9taWNfanNvbihSRVNVTFRTLydwcmVwYXJhdGlvbl9zdW1tYXJ5Lmpzb24nLGRpY3Qoc3RhdHVzPSdQQVNTJyxzZWNvbmRzPXRpbWUubW9ub3RvbmljKCktc3RhcnQsY2FjaGVfYnl0ZXM9c3VtKHAuc3RhdCgpLnN0X3NpemUgZm9yIHAgaW4gQ0FDSEUucmdsb2IoJyonKSBpZiBwLmlzX2ZpbGUoKSBhbmQgbm90IHAuaXNfc3ltbGluaygpKSxzdGVwcz1zdGVwcykpCiAgICBwcmludCgnUFJFUEFSQVRJT05fUEFTUycsZmx1c2g9VHJ1ZSkKCmlmIF9fbmFtZV9fPT0nX19tYWluX18nOgogICAgaWYgbGVuKHN5cy5hcmd2KT4xIGFuZCBzeXMuYXJndlsxXT09J3dvcmtlcic6IHdvcmtlcigpCiAgICBlbHNlOgogICAgICAgIHRyeTogbWFpbigpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgICAgICAgICBhdG9taWNfanNvbihSRVNVTFRTLydwcmVwYXJhdGlvbl9zdW1tYXJ5Lmpzb24nLGRpY3Qoc3RhdHVzPSdGQUlMJyxlcnJvcj1yZXByKGUpKSkKICAgICAgICAgICAgcmFpc2UK', 'config.json': 'ewogICJTT1VSQ0VfU0hBIjogIjQ3ZDNjMmE2ZjYyOGRkNDhiYTc2YTFjZmRmMjliNmIyYWE4ZjllYjMiLAogICJCQUNLQk9ORV9SRVZJU0lPTiI6ICI4ZjEzYThlMDBkYmRjMzgxZDYxNGE5ZDI5ZDYxYjA3ZThmZTkxYjNmIiwKICAiRkxNTV9SRVZJU0lPTiI6ICJkNzBlNWIzMjg5Y2E0MjU1Y2I2YTAxN2E3OTI4NzBhYWQyNmMwMWNlIiwKICAiRkxNTV9TSEEyNTYiOiAiMDJiZGY5ZTg3ODg0N2M2OGFjYWEyMDdhOTE2MjkxY2IxNTY4YWIxMTljYmM2NThhNGRmYTk0ZjM5OWRmZWQwZSIsCiAgIlNBTV9TSEEyNTYiOiAiM2FkY2M0MzE1YjY0MmE0ZDIxMDExMjhmNjExNjg0ZTg3MzRjNDEyMzJhMTdjNjQ4ZWQxNjkzNzAyYTQ5YTYyMiIsCiAgIkFOTk9UQVRJT05fU0hBMjU2IjogIjdmOTI0YmI3ZWQ4ZGM0NTY4MDU4ZTRmZjYyNjI4MTkxOGQ1NmU1MjA2ZjRjODY4YzVhODBmMDg4ZjM4YzhiZjAiLAogICJCQUNLQk9ORV9TSEEyNTYiOiAiYjFjMzgyYzZkY2VjYmMzZTE3YzE2NDgyNTNlNTA4NjUyMzYyNjcxOWUwNWU1MmU5MDVmYmZkMTEzNjk0ZTY1ZSIsCiAgIlNFRUQiOiAyMDI2MTEwNiwKICAiU1RPUF9BRlRFUl9HQVRFIjogMCwKICAiTUFYX1RPVEFMX1dBTExUSU1FX01JTiI6IDEyMCwKICAiTUFYX0VWQUxVQVRJT05fTUlOIjogNzUsCiAgIlNIVVRET1dOX1JFU0VSVkVfTUlOIjogMjAsCiAgIk1JTl9GUkVFX0RJU0tfR0IiOiAzLAogICJNQVhfUkVUUklFU19QRVJfRE9XTkxPQUQiOiAzLAogICJET1dOTE9BRF9USU1FT1VUX1MiOiA0NSwKICAiU0FGRVRZX0ZBQ1RPUiI6IDEuNSwKICAiUFJPQkVfSU1BR0VTIjogNTAsCiAgIlNUQUJJTElUWV9JTUFHRVMiOiAxMDAsCiAgIkRFVkVMT1BNRU5UX0lNQUdFUyI6IDI1MCwKICAiSE9MRE9VVF9JTUFHRVMiOiA0MDAsCiAgIlJVTl9ERVZFTE9QTUVOVCI6IGZhbHNlLAogICJSVU5fSE9MRE9VVCI6IGZhbHNlLAogICJSVU5fRlVMTF9WQUxJREFUSU9OIjogZmFsc2UsCiAgIk1BWF9JTUFHRVNfVEhJU19KT0IiOiAxMDAsCiAgIkVYUE9SVF9FVkVSWV9JTUFHRVMiOiAyNSwKICAiU1RPUF9BRlRFUl9JTUFHRVMiOiBudWxsLAogICJSRVNVTUVfSU5QVVQiOiBudWxsCn0K', 'docs/checkpoint_integrity.md': 'IyBDaGVja3BvaW50IGludGVncml0eSDigJQgVU5WRVJJRklFRCBwZW5kaW5nIEdQVSBHMAoKU3RhdGljIGV2aWRlbmNlIGNvbmZpcm1zIHRoZSBvbGQgbG9nIGhhcyA1MjkgbWlzc2luZyBgZGVlcHNlZWtfdmwuKmAga2V5cywgMzQ1IG1pc3NpbmcgYHNhbS5tb2RlbC5pbWFnZV9lbmNvZGVyLipgIGtleXMgYW5kIG5vIHVuZXhwZWN0ZWQga2V5cy4gUHJlZml4IGFjY291bnRpbmcgYWxvbmUgaXMgaW5zdWZmaWNpZW50IHByb29mIG9mIGluaXRpYWxpemVkIHRlbnNvciB2YWx1ZXMuCgpOZXcgYXVkaXQgdmVyaWZpZXMgZnVsbCBjaGVja3BvaW50IFNIQTI1NiwgYmFja2JvbmUgcHJldHJhaW5lZCBsb2FkaW5nIGluZm8sIGFsbCBwcmV0cmFpbmVkIGJhY2tib25lL1NBTSBlbmNvZGVyIHRlbnNvcnMgYWdhaW5zdCBkaXNrLCB1bmNoYW5nZWQgbWlzc2luZyB0ZW5zb3JzIGFmdGVyIHRyYWluZWQgY2hlY2twb2ludCBsb2FkaW5nLCBhbmQgZXhhY3QgbG9hZGVkIHZhbHVlcyBmb3IgZXZlcnkgdHJhaW5lZCB0ZW5zb3IuIEl0IGFsc28gcmVjb3JkcyBtb2R1bGUgZHR5cGVzIGFuZCBhdHRlbnRpb24gaW1wbGVtZW50YXRpb24uIFJ1bnRpbWUgcGVyLWtleSBwcm9vZiBpcyB3cml0dGVuIHRvIGByZXN1bHRzL2NoZWNrcG9pbnRfaW50ZWdyaXR5Lmpzb25gLgoKVGhlIGhpc3RvcmljYWwgc21va2UgYmFja2JvbmUgcmV2aXNpb24gd2FzIG5vdCByZWNvcmRlZCBhbmQgcmVtYWlucyBVTlZFUklGSUVELiBBIG5ldyBwaW5uZWQgYmFja2JvbmUgbWF5IGJlIGVxdWl2YWxlbnQgaWYgaXRzIDIwIHNhdmVkIHJlZ3Jlc3Npb24gbWFza3MgbWF0Y2ggZXhhY3RseSwgYnV0IHRoaXMgZG9lcyBub3QgcmV0cm9hY3RpdmVseSBlc3RhYmxpc2ggdGhlIG9sZCByZXZpc2lvbi4K', 'docs/evaluation_protocol.md': 'IyBGLUxNTSBQaGFzZSAxIGV2YWx1YXRpb24gcHJvdG9jb2wg4oCUIGZpeGVkIGJlZm9yZSBuZXcgcmVzdWx0cwoKRGF0ZTogMjAyNi0xMS0wNiAoSUNUKS4gTm8gbWV0aG9kIGltcHJvdmVtZW50cywgdHJhaW5pbmcsIHRocmVzaG9sZCB0dW5pbmcsIHByZWNpc2lvbiBjaGFuZ2VzIG9yIHJlc2l6aW5nIGV4cGVyaW1lbnRzIGluIFBoYXNlIDEuCgojIyBHYXRlcyBhbmQgaW50ZXJwcmV0YXRpb24KCjEuICoqRzA6KiogcHJvdmUgY2hlY2twb2ludCB0ZW5zb3IgaW5pdGlhbGl6YXRpb24vbG9hZGluZyBhbmQgY29uZGl0aW9uYWwgZXZhbHVhdG9yIGVxdWl2YWxlbmNlIG9uIHRoZSBvcmlnaW5hbCAyMCBkZWJ1ZyBpbWFnZXMuIEdhdGUgaXMgRkFJTC9CTE9DS0VEIHVudGlsIG1lYXN1cmVkLCBuZXZlciBpbmZlcnJlZCBmcm9tIGltcG9ydCBzdWNjZXNzIG9yIGtleSBwcmVmaXhlcyBhbG9uZS4KMi4gKipQcm9iZToqKiA1MCBzZWVkZWQgaW1hZ2VzLCBvdXRzaWRlIHRoZSBvcmlnaW5hbCAyMDsgQ1VEQS1zeW5jaHJvbml6ZWQgdGltaW5ncyBhZnRlciBzZXBhcmF0ZSB3YXJtLXVwLCBwZWFrIFZSQU0vUkFNLCBvdXRwdXQgYnl0ZXMsIGNvbGQgbG9hZCB0aW1lIGFuZCBkYXRhIHByZXBhcmF0aW9uIGNvc3QuCjMuICoqRzE6KiogbWVhc3VyZWQgcDk1IHBsdXMgcHJlcHJvY2Vzc2luZy9wb3N0cHJvY2Vzc2luZywgMS41eCBzYWZldHkgZmFjdG9yLCAyMC1taW51dGUgc2h1dGRvd24gcmVzZXJ2ZSwgZGlzayBmbG9vciBhbmQgb2JzZXJ2ZWQgcXVvdGEuIFVua25vd24gY3JpdGljYWwgcmVzb3VyY2VzIGNhcCBzY29wZSBjb25zZXJ2YXRpdmVseS4KNC4gKipTdGFiaWxpdHkvcmVzdW1lOioqIG5lc3RlZCAxMDAtaW1hZ2Ugc2V0LCBmaXJzdCBqb2IgY29tbWl0cyBhdCA3NSwgbmV4dCBzZXNzaW9uIG1vdW50cyBpdHMgY29tcGxldGVkIG91dHB1dCBhbmQgZmluaXNoZXMgdGhlIHJlbWFpbmluZyAyNS4gTWF0Y2ggZnVsbCBmaW5nZXJwcmludCBhbmQgZmlsZSBjaGVja3N1bXMgYmVmb3JlIHNraXBwaW5nLgo1LiAqKkxhcmdlciBiYXNlbGluZToqKiBvbmx5IGFmdGVyIEcwL0cxL0cyIGFuZCBhIGZyZXNoIGJ1ZGdldCBnYXRlLiBGdWxsIHZhbGlkYXRpb24gYWRkaXRpb25hbGx5IHJlcXVpcmVzIGFsbCBpbWFnZXMgY2FjaGVkLiBEZXZlbG9wbWVudC9ob2xkb3V0L2Z1bGwgZGlzYWJsZWQgaW4gdGhlIGluaXRpYWwgbm90ZWJvb2suIERpc2FibGVkIHN0YWdlcyBhcmUgVU5WRVJJRklFRCwgbm90IFBBU1MuCgojIyBGaXhlZCBjb21wdXRhdGlvbgoKLSBPZmZpY2lhbCBzb3VyY2UgY29tbWl0OiBgNDdkM2MyYTZmNjI4ZGQ0OGJhNzZhMWNmZGYyOWI2YjJhYThmOWViM2AuCi0gT3JpZ2luYWwgRGVlcFNlZWtWTC0xLjNCIEYtTE1NIGNvbmZpZywgU0FNIFZpVC1MLCBvcmlnaW5hbCByZXNpemUvcG9zdHByb2Nlc3MgYW5kIHRocmVzaG9sZCAwLjUsIGJhdGNoIG9uZSBpbWFnZS4KLSBCYWNrYm9uZSBwaW5uZWQgdG8gYDhmMTNhOGUwMGRiZGMzODFkNjE0YTlkMjlkNjFiMDdlOGZlOTFiM2ZgIGFuZCB2ZXJpZmllZCBhZ2FpbnN0IGl0cyBzYWZldGVuc29ycyBMRlMgU0hBMjU2LgotIEYtTE1NIHRyYWluZWQgY2hlY2twb2ludCByZXZpc2lvbiBgZDcwZTViMzI4OWNhNDI1NWNiNmEwMTdhNzkyODcwYWFkMjZjMDFjZWAgYW5kIG9yaWdpbmFsIHN1Y2Nlc3NmdWwgU0hBMjU2OyBTQU0gaGFzaCB1bmNoYW5nZWQuCi0gTG9jYWwgY2FjaGUgcGF0aHMgcmVwbGFjZSBVUkxzOyBpbmZlcmVuY2UgcnVucyB3aXRoIEludGVybmV0IGRpc2FibGVkIGFuZCBIdWdnaW5nIEZhY2Ugb2ZmbGluZS4KLSBUZXNsYSBUNCBtYXRoIFNEUEEgYmFja2VuZDsgQkYxNiBiYWNrYm9uZSBhbmQgb3JpZ2luYWwgRlAzMiBtb2R1bGVzIHJldGFpbmVkLiBObyBjbGFpbSBvZiBuYXRpdmUgVDQgQkYxNiB0ZW5zb3ItY29yZSBvcHRpbWl6YXRpb24uIE1vZHVsZSBkdHlwZXMgYW5kIGFjdHVhbCBsYW5ndWFnZSBhdHRlbnRpb24gY2xhc3MgYXJlIHJlY29yZGVkLgotIFJhbmRvbSBzZWVkIGAyMDI2MTEwNmA7IG5vIG1vZGVsIG9yIHRocmVzaG9sZCBzZWxlY3Rpb24gb24gcGlsb3QvaG9sZG91dCByZXN1bHRzLgoKIyMgRXZhbHVhdGlvbiB1bml0IGFuZCBzYW1wbGUgc2VsZWN0aW9uCgpUaGUgaW5zdGFsbGVkIGBtbWRldD09My4zLjBgIGxvYWRlciBhbmQgYW5ub3RhdGlvbiBmaWxlcyBhcmUgYXV0aG9yaXRhdGl2ZS4gVGhlIG1hbmlmZXN0IHJlY29yZHMgYWN0dWFsIGltYWdlL3JlZmVyZW5jZS9zZW50ZW5jZS9hbm5vdGF0aW9uIGNvdW50czsgaXQgZG9lcyBub3QgYXNzdW1lIHRoZXkgYXJlIGVxdWFsLiBgc2VsZWN0X2ZpcnN0YCB1c2VzIHRoZSBmaXJzdCBzZW50ZW5jZSBvZiBlYWNoIHJldGFpbmVkIHJlZmVyZW5jZSwgYW5kIGAtLWNvbmNhdGAgY29tYmluZXMgdGhlbSBpbiBvbmUgaW1hZ2UgaW5mZXJlbmNlLiBNdWx0aXBsZSByZWZlcmVuY2VzIHBvaW50aW5nIHRvIHRoZSBzYW1lIGFubm90YXRpb24gbWF5IGJlIGNvbGxhcHNlZCBieSB0aGUgbG9hZGVyLiBBbGwgZXhjbHVkZWQgc2VudGVuY2VzIHJlbWFpbiBvdXRzaWRlIHRoaXMgcHJvdG9jb2wncyBjb3ZlcmFnZS4KCk1hcCBtYXNrIHBvc2l0aW9uIGJhY2sgdG8gYGltYWdlX2lkYCwgYGRhdGFzZXRfaW5kZXhgLCBgcmVmX2lkYCwgYHNlbnRfaWRgLCBsb3dlcmNhc2VkIGV4cHJlc3Npb24sIGFubm90YXRpb24gSUQgYW5kIHByZWRpY3Rpb24gSUQgYnkgcmVwcm9kdWNpbmcgdGhlIGxvYWRlcidzIG1hcHBpbmcgYW5kIHZlcmlmeWluZyBib3RoIHRleHQgYW5kIHBvbHlnb24vUkxFIGlkZW50aXR5LiBNaXNzaW5nIHNvdXJjZSBmaWVsZHMgYXJlIG51bGwsIG5ldmVyIGludmVudGVkLgoKUHJlc2VydmUgb3JpZ2luYWwgMjAgaW1hZ2VzIGFzIHJlZ3Jlc3Npb24gb25seS4gRXhjbHVkZSB0aGVtIGZyb20gYSBzZWVkZWQgc2ltcGxlIHJhbmRvbSBzaHVmZmxlIG9mIHNvcnRlZCByZW1haW5pbmcgaW1hZ2UgSURzLiBGaXJzdCAyNTAgYmVjb21lIGRldmVsb3BtZW50OyBuZXh0IDQwMCBiZWNvbWUgaG9sZG91dC4gVGhlIGZpcnN0IDUwLzEwMCBkZXZlbG9wbWVudCBJRHMgZm9ybSBuZXN0ZWQgZmVhc2liaWxpdHkvc3RhYmlsaXR5IHNldHMuIERlc2NyaWJlIG9iamVjdC1jb3VudCBkaXN0cmlidXRpb25zOyByYW5kb20gc2VsZWN0aW9uIGlzIG5vdCBhIGNsYWltIG9mIGJhbGFuY2VkIHNhbXBsaW5nLiBIb2xkb3V0IGlzIG5vdCB1c2VkIGZvciB0dW5pbmcgb3IgZXhwbG9yYXRvcnkgZXhhbXBsZSBzZWxlY3Rpb24uIFRlc3RBL3Rlc3RCIHJlbWFpbiB1bnVzZWQuCgojIyBSZXN1bWUgYW5kIGFnZ3JlZ2F0aW9uCgpTdGFibGUgc2FtcGxlIElEOiBgcmVmY29jby12YWwte2ltYWdlX2lkfWAuIEVhY2ggb3V0cHV0IHJlY29yZHMgc291cmNlL2NvbmZpZy9lbmdpbmUvY2hlY2twb2ludC9hbm5vdGF0aW9uL21hbmlmZXN0IGZpbmdlcnByaW50LCBzYW1wbGUgaGFzaCwgbWFzayBmaWxlIGhhc2gsIGludGVyc2VjdGlvbi91bmlvbiBhcnJheXMgYW5kIG1ldGFkYXRhLiBBdG9taWMgTlBaIGZvbGxvd2VkIGJ5IGF0b21pYyBKU09OOyB2YWxpZGF0ZSBiZWZvcmUgRE9ORS4gSW52YWxpZC9pbmNvbXBsZXRlL21pc21hdGNoZWQgcmVzdWx0cyByZXJ1biBvciBibG9jayBpbnRlZ3JpdHkgdmVyaWZpY2F0aW9uLiBFdmVudCB0cmFpbCBpcyBhcHBlbmQtb25seS4gRHVwbGljYXRlIHNhbXBsZSBJRHMgY2Fubm90IGJlIGFnZ3JlZ2F0ZWQuCgpjSW9VID0gMTAwICogc3VtKGludGVyc2VjdGlvbikgLyBzdW0odW5pb24pLiBtSW9VIGF2ZXJhZ2VzIHBlci1zZWxlY3RlZC1tYXNrIElvVSwgY2hlY2tlZCBhZ2FpbnN0IFJlZlNlZ01ldHJpYy4gRG8gbm90IGF2ZXJhZ2UgY2h1bmsgY0lvVXMuIFBlci1leHByZXNzaW9uIGxhdGVuY3kgaXMgZXhwbGljaXRseSBhbW9ydGl6ZWQgaW1hZ2UgbGF0ZW5jeSBkaXZpZGVkIGJ5IHNlbGVjdGVkIG1hc2tzLCBub3QgaW5kZXBlbmRlbnQgc2luZ2xlLWV4cHJlc3Npb24gZXhlY3V0aW9uLgoKRXZlcnkgMjUgbmV3IGltYWdlcyBjcmVhdGVzIGFuIGF0b21pYyBleHBvcnQtcmVhZHkgYXJjaGl2ZS4gYC9rYWdnbGUvd29ya2luZ2AgYWxvbmUgaXMgbm90IGEgY3Jvc3Mtc2Vzc2lvbiBkdXJhYmlsaXR5IGd1YXJhbnRlZS4gQ29tcGxldGVkIEthZ2dsZSBOb3RlYm9vayBPdXRwdXQgaXMgdGhlIGR1cmFibGUgYm91bmRhcnk7IGJvdW5kZWQgc2VwYXJhdGUgam9icyBhbmQgdGhlIHJlc3VtZSB0ZXN0IGVzdGFibGlzaCB0aGF0IGJvdW5kYXJ5LiBgL2thZ2dsZS90ZW1wYCBpcyBzY3JhdGNoIG9ubHkuCgojIyBGYWlsdXJlIHBvbGljeQoKTmV0d29yayBwcmVwYXJhdGlvbjogdGhyZWUgYXR0ZW1wdHMsIGJvdW5kZWQgcmVhZCB0aW1lb3V0L2JhY2tvZmYsIG92ZXJhbGwgd29ya2VyIGRlYWRsaW5lLCBpbWFnZSBkZWNvZGUgY2hlY2tzLCBjaGVja3BvaW50IGhhc2hlcy4gTm8gbmV0d29yayBpbiBHUFUgZXZhbHVhdGlvbi4KClN0b3Agb24gZmlyc3QgaW5mZXJlbmNlIGVycm9yL09PTTsgcHJlc2VydmUgbG9ncyBhbmQgY29tcGxldGVkIHNhbXBsZXMuIE5ldmVyIHNpbGVudGx5IHJlZHVjZSByZXNvbHV0aW9uLCBkdHlwZSwgYmF0Y2ggb3IgdGhyZXNob2xkLiBTdG9wIGJlZm9yZSB0aW1lIHJlc2VydmUgb3IgZGlzayBmbG9vci4gVW51c2VkIHN0YWdlcyByZXRhaW4gVU5WRVJJRklFRCBzdGF0dXMuIEEgZmFpbGVkIHN0YWdlIG11c3Qgc3VyZmFjZSBmYWlsdXJlIGluIG5vdGVib29rIGV4ZWN1dGlvbjsgZWFybGllciB2YWxpZCBhcnRpZmFjdHMgcmVtYWluIHVzYWJsZS4KCiMjIEJ1ZGdldHMgYW5kIHNvdXJjZSBjYXZlYXRzCgpPcGVyYXRpb25hbCBkZWZhdWx0czogd2FsbHRpbWUgMTIwIG1pbiwgZXZhbHVhdGlvbiA3NSBtaW4sIHJlc2VydmUgMjAgbWluLCBkaXNrIGZsb29yIDMgR2lCLiBUaGVzZSBhcmUgdXNlci1zZWxlY3RlZCBvcGVyYXRpbmcgbGltaXRzLCBub3QgbW9kZWwgbWluaW1hIG9yIGd1YXJhbnRlZWQgS2FnZ2xlIHF1b3Rhcy4KClF1b3RhIHdhcyBtZWFzdXJlZCB0aHJvdWdoIG9mZmljaWFsIEthZ2dsZSBBUEkgb24gMjAyNi0xMS0wNjogMjguNDkvMzAgR1BVIGhvdXJzIHJlbWFpbmluZyBhdCBzbmFwc2hvdC4gUmVjaGVjayBhdCBzdWJtaXNzaW9uIGFuZCB0cmVhdCBzdGFsZSBzbmFwc2hvdHMgYXMgVU5LTk9XTi4gS2FnZ2xlJ3MgZHluYW1pYyBkb2NzIGRpZCBub3QgZXhwb3NlIGEgdmVyaWZpYWJsZSBjdXJyZW50IHBsYXRmb3JtIG1heGltdW0gdGhyb3VnaCB0aGUgYXZhaWxhYmxlIHJlYWRlcjsgc2NoZWR1bGluZyByZWxpZXMgb24gdGhlIGV4cGxpY2l0bHkgcmVxdWVzdGVkIDcsMjAwLXNlY29uZCBBUEkgdGltZW91dCBhbmQgYSBzdHJpY3RlciAxMDAtbWludXRlIHByb2Nlc3NpbmcgY3V0b2ZmLCBub3QgYW4gYXNzdW1lZCAxMi1ob3VyIGxpbWl0LgoKU291cmNlczogW0YtTE1NXShodHRwczovL2dpdGh1Yi5jb20vd3VzaXplL0YtTE1NKSwgW0thZ2dsZSBub3RlYm9vayBkb2NzXShodHRwczovL3d3dy5rYWdnbGUuY29tL2RvY3Mvbm90ZWJvb2tzKSwgW0thZ2dsZSBDTEldKGh0dHBzOi8vZ2l0aHViLmNvbS9LYWdnbGUva2FnZ2xlLWNsaSksIFthdXRob3IgY2hlY2twb2ludCByZXBvc2l0b3J5XShodHRwczovL2h1Z2dpbmdmYWNlLmNvL3d1c2l6ZS9mbG1tKSwgW2JhY2tib25lXShodHRwczovL2h1Z2dpbmdmYWNlLmNvL2RlZXBzZWVrLWFpL2RlZXBzZWVrLXZsLTEuM2ItY2hhdCkuCg==', 'docs/evaluator_equivalence.md': 'IyBFdmFsdWF0b3IgZXF1aXZhbGVuY2Ug4oCUIFVOVkVSSUZJRUQgcGVuZGluZyBHUFUgRzAKClN0YXRpYyByZXZpZXc6IHRoZSBzdWJzZXQgc2NyaXB0IHJlc3RyaWN0cyBzcGxpdHMvc2FtcGxlIElEcywgbG9ncyBjaGVja3BvaW50IGtleXMgYW5kIHdyaXRlcyBjb21wcmVzc2VkIG1hc2sgYXJ0aWZhY3RzLiBQcmVkaWN0aW9uLCBpbnRlcnBvbGF0aW9uLCBzaWdtb2lkIHRocmVzaG9sZCBhbmQgUmVmU2VnTWV0cmljIG1hdGNoIHVwc3RyZWFtLiBTdGF0aWMgc2ltaWxhcml0eSBkb2VzIG5vdCBlc3RhYmxpc2ggbWVhc3VyZWQgZXF1aXZhbGVuY2UuCgpUaGUgY29udHJvbGxlZCBvZmZpY2lhbCBldmFsdWF0b3IgYW5kIHByZXNlcnZlZCBzdWJzZXQgZXZhbHVhdG9yIHdpbGwgZXhlY3V0ZSBvbiB0aGUgc2FtZSAyMCBJRHMsIHBpbm5lZCB3ZWlnaHRzIGFuZCBtYXRoIGF0dGVudGlvbiBiYWNrZW5kLiBPbmx5IHNlbGVjdGlvbi9sb2NhbC1wYXRoL21vZGVsLXNoYXJpbmcgaGFybmVzcyBjaGFuZ2VzIGFyZSBwZXJtaXR0ZWQsIGFuZCBleGFjdCBkaWZmcyBhcmUgc2F2ZWQuIE1hc2tzIGFuZCBHVCBtdXN0IG1hdGNoIGJpdHdpc2UsIG1ldHJpY3Mgd2l0aGluIDAuMDAxIHBlcmNlbnRhZ2UgcG9pbnQuIFRoZSByZXN1bWFibGUgZW5naW5lIG11c3QgYWxzbyBtYXRjaCBhbGwgMjAgcmVmZXJlbmNlIG91dHB1dHMuCgpUaGlzIGlzIGEgY29uZGl0aW9uYWwgc2FtZS1zZWxlY3Rpb24gZXF1aXZhbGVuY2UgdGVzdCwgbm90IGEgY2xhaW0gdGhhdCBhbiB1bm1vZGlmaWVkIGVpZ2h0LXNwbGl0IGV2YWx1YXRvciB3YXMgcnVuIG9yIHRoYXQgZXZlcnkgcmVmZXJyaW5nIHNlbnRlbmNlIGluIFJlZkNPQ08gd2FzIGNvdmVyZWQuCg==', 'docs/feasibility_report.md': 'IyBGZWFzaWJpbGl0eSDigJQgVU5WRVJJRklFRCBwZW5kaW5nIDUwLWltYWdlIHByb2JlCgpFeGlzdGluZyBzbW9rZTogMjAgaW1hZ2VzIC8gNDcgbWFza3MsIGFib3V0IDM5IHNlY29uZHMgaW4gdGhlIGxvYWRlci9pbmZlcmVuY2Uvd3JpdGUgbG9vcC4gVGhpcyBpcyBub3QgaXNvbGF0ZWQgd2FybWVkLXVwIGxhdGVuY3kuIExpbmVhciBleHRyYXBvbGF0aW9uIGlzIHBsYW5uaW5nIGluZm9ybWF0aW9uIG9ubHkuCgpUaGUgbmV3IENQVSBqb2IgcHJlcGFyZXMgcmV1c2FibGUgZW52aXJvbm1lbnQsIGV4YWN0IG1vZGVsIHdlaWdodHMsIGFubm90YXRpb25zIGFuZCBzZWxlY3RlZCBpbWFnZXMgYmVmb3JlIEdQVSBzY2hlZHVsaW5nLiBUaGUgR1BVIG5vdGVib29rIHJlc3RvcmVzIHRoZXNlIGlucHV0cywgYXVkaXRzIEcwLCB0aGVuIG1lYXN1cmVzIGEgc2VlZGVkIDUwLWltYWdlIHByb2JlLiBJdCByZWNvcmRzIGNvbGQgbG9hZGluZywgd2FybS11cCwgc3luY2hyb25pemVkIGluZmVyZW5jZSBtZWRpYW4vcDk1L21heCwgcHJlcHJvY2Vzc2luZy9wb3N0cHJvY2Vzc2luZywgcGVhayBhbGxvY2F0ZWQvcmVzZXJ2ZWQgVlJBTSwgUkFNIGFuZCBwZXItc2FtcGxlIG91dHB1dCBzaXplLiBBIGZyZXNoIGdvL25vLWdvIHVzZXMgbWVhc3VyZWQgdmFsdWVzLCBxdW90YSwgZnJlZSBkaXNrLCBzYWZldHkgZmFjdG9yIGFuZCBzaHV0ZG93biByZXNlcnZlLgoKTm8gbGFyZ2UgYmVuY2htYXJrIHNob3VsZCBzdGFydCB3aGlsZSBHMCBvciB0aGUgcHJvYmUgaXMgdW5yZXNvbHZlZC4gQ3VycmVudCBjb2RlIGRlZmF1bHRzIHRvIDEwMCBzdGFiaWxpdHkgaW1hZ2VzIGF0IG1vc3Q7IGRldmVsb3BtZW50IGFuZCBob2xkb3V0IGFyZSBkZWZpbmVkIGJ1dCBub3QgZXZhbHVhdGVkLgo=', 'docs/observation_report.md': 'IyBPYnNlcnZhdGlvbnMg4oCUIG5vIG1ldGhvZCBzZWxlY3Rpb24gaW4gUGhhc2UgMQoKVGhlIGV4aXN0aW5nIHNtb2tlIGNvbnRhaW5zIHN1Y2Nlc3NmdWwgbWFza3MgYW5kIHdyb25nLXJlZ2lvbiBwcmVkaWN0aW9ucy4gVGhlc2UgYXJlIGV4cGxvcmF0b3J5IGV4YW1wbGVzIGZyb20gZGVidWcgZGF0YTsgdGhleSBkbyBub3QgZXN0YWJsaXNoIGEgZmFpbHVyZSByYXRlLCBjYWxpYnJhdGVkIGNvbmZpZGVuY2UsIG9yIGEgc2VsZWN0aXZlLVNBTSBpbXByb3ZlbWVudCBoeXBvdGhlc2lzLgoKTmV3IHByb2JlL3N0YWJpbGl0eSBkYXRhIGFyZSBmb3IgZmVhc2liaWxpdHkgYW5kIGJhc2VsaW5lIGludGVncml0eS4gSG9sZG91dCByZW1haW5zIHVudG91Y2hlZC4gUmVjb3JkIG9ic2VydmVkIGZhaWx1cmVzIGFuZCByZXNvdXJjZSBjb3N0cyBiZWZvcmUgZHJhZnRpbmcgYW4gaW1wcm92ZW1lbnQgcHJvdG9jb2wuIE5vIHJlc2VhcmNoIGltcHJvdmVtZW50IGlzIGltcGxlbWVudGVkIG9yIHNlbGVjdGVkIGJ5IHRoaXMgbm90ZWJvb2suCg==', 'quota_snapshot.json': 'ew0KICAib2JzZXJ2ZWRfYXRfdXRjIjogIjIwMjYtMTEtMDZUMDI6MDQ6MjEuMzk1MDUzKzAwOjAwIiwNCiAgInJlZnJlc2hfYXQiOiAiMjAyNi0xMS0wNyAwMDowMDowMCIsDQogICJncHUiOiB7DQogICAgInVzZWRfc2Vjb25kcyI6IDU0MzYuMTYzLA0KICAgICJ0b3RhbF9zZWNvbmRzIjogMTA4MDAwLjAsDQogICAgInJlbWFpbmluZ19zZWNvbmRzIjogMTAyNTYzLjgzNw0KICB9LA0KICAidHB1Ijogew0KICAgICJ1c2VkX3NlY29uZHMiOiAwLjAsDQogICAgInRvdGFsX3NlY29uZHMiOiA3MjAwMC4wLA0KICAgICJyZW1haW5pbmdfc2Vjb25kcyI6IDcyMDAwLjANCiAgfQ0KfQ==', 'constraints.txt': 'YWNjZWxlcmF0ZT09MC4yOC4wCmFkZGljdD09Mi40LjAKYWlvaGFwcHlleWViYWxscz09Mi43LjEKYWlvaHR0cD09My4xNC41CmFpb3NpZ25hbD09MS40LjAKYWx0YWlyPT02LjIuMgphbm5vdGF0ZWQtdHlwZXM9PTAuOC4wCmFueWlvPT00LjE2LjAKYXJnb24yLWNmZmk9PTI1LjEuMAphcmdvbjItY2ZmaS1iaW5kaW5ncz09MjYuMS4wCmFycm93PT0xLjQuMAphcnhpdj09NC4wLjIKYXN0dG9rZW5zPT0zLjAuMgphc3luYy1scnU9PTIuNC4wCmFzeW5jLXRpbWVvdXQ9PTUuMC4xCmF0dHJkaWN0PT0yLjAuMQphdHRycz09MjYuMS4wCmJhYmVsPT0yLjE4LjAKYmVhdXRpZnVsc291cDQ9PTQuMTUuMApiaXRzYW5kYnl0ZXM9PTAuNDMuMQpibGVhY2g9PTYuNC4wCmJyb3RsaT09MS4yLjEKY2VydGlmaT09MjAyNi4xMC4yMwpjZmZpPT0yLjEuMQpjaGFyc2V0LW5vcm1hbGl6ZXI9PTIuMS4xCmNsaWNrPT04LjUuMApjb2xvcmFtYT09MC40LjYKY29tbT09MC4yLjMKY29udG91cnB5PT0xLjMuMgpjeWNsZXI9PTAuMTIuMQpkYXRhc2V0cz09Mi4xOS4yCmRlYnVncHk9PTEuOC4yMgpkZWNvcmF0b3I9PTUuMy4xCmRlZXBzcGVlZD09MC4xNC4yCmRlZnVzZWR4bWw9PTAuNy4xCmRpbGw9PTAuMy44CmRpc3Rybz09MS45LjAKZHVja2R1Y2tnby1zZWFyY2g9PTUuMy4xYjEKZWlub3BzPT0wLjguMgpldC14bWxmaWxlPT0yLjAuMApleGNlcHRpb25ncm91cD09MS4zLjEKZXhlY3V0aW5nPT0yLjMuMApmYXN0anNvbnNjaGVtYT09Mi4yMi4zCmZpbGVsb2NrPT00LjEuMApmb250dG9vbHM9PTQuNjUuMApmcWRuPT0xLjYuMApmcm96ZW5saXN0PT0xLjguMApmc3NwZWM9PTIwMjQuMy4xCmZ0Znk9PTYuMy4xCmZ1bmMtdGltZW91dD09NC4zLjUKZ2Rvd249PTYuNS4xCmdyaWZmZT09MC40OS4wCmgxMT09MC4xNi4wCmgyPT00LjQuMQpoZi14ZXQ9PTEuOC4wCmhqc29uPT0zLjEuMApocGFjaz09NC4yLjAKaHR0cGNvcmU9PTEuMC45Cmh0dHB0b29scz09MC45LjAKaHR0cHg9PTAuMjguMQpodWdnaW5nZmFjZS1odWI9PTAuMzYuMgpoeXBlcmZyYW1lPT02LjEuMAppZG5hPT0zLjQKaW1hZ2Vpbz09Mi4zOC4wCmlweWtlcm5lbD09Ny4zLjAKaXB5dGhvbj09OC4zOS4wCmlweXdpZGdldHM9PTguMS45Cmlzb2R1cmF0aW9uPT0yMC4xMS4wCml0c2Rhbmdlcm91cz09Mi4yLjAKamVkaT09MC4yMC4xCmppbmphMj09My4xLjYKanNvbjU9PTAuMTcuMwpqc29ucG9pbnRlcj09My44Cmpzb25zY2hlbWE9PTQuMjYuMApqc29uc2NoZW1hLXNwZWNpZmljYXRpb25zPT0yMDI1LjkuMQpqdXB5dGVyPT0xLjEuMQpqdXB5dGVyLWJ1aWxkZXI9PTEuMi40Cmp1cHl0ZXItY2xpZW50PT04LjEwLjAKanVweXRlci1jb25zb2xlPT02LjYuMwpqdXB5dGVyLWNvcmU9PTUuOS4xCmp1cHl0ZXItZXZlbnRzPT0wLjEyLjEKanVweXRlci1sc3A9PTIuMy4xCmp1cHl0ZXItc2VydmVyPT0yLjIxLjEKanVweXRlci1zZXJ2ZXItdGVybWluYWxzPT0wLjUuNApqdXB5dGVybGFiPT00LjYuNQpqdXB5dGVybGFiLXB5Z21lbnRzPT0wLjMuMApqdXB5dGVybGFiLXNlcnZlcj09Mi4yOC4xCmp1cHl0ZXJsYWItd2lkZ2V0cz09My4wLjE3Cmtpd2lzb2x2ZXI9PTEuNS4xCmxhZ2VudD09MC4yLjQKbGFyaz09MS4zLjEKbGF6eS1sb2FkZXI9PTAuNgpseG1sPT02LjEuMwptYXJrZG93bi1pdC1weT09NC4yLjAKbWFya3Vwc2FmZT09My4wLjMKbWF0cGxvdGxpYj09My4xMC45Cm1hdHBsb3RsaWItaW5saW5lPT0wLjIuMgptZHVybD09MC4xLjIKbWlzdHVuZT09My40LjAKbW1jdj09Mi4xLjAKbW1kZXQ9PTMuMy4wCm1tZW5naW5lPT0wLjEwLjUKbW1zZWdtZW50YXRpb249PTEuMi4yCm1wbWF0aD09MS4zLjAKbXVsdGlkaWN0PT03LjEuMAptdWx0aXByb2Nlc3M9PTAuNzAuMTYKbmFyd2hhbHM9PTIuMjguMApuYmNsaWVudD09MC4xMS4wCm5iY29udmVydD09Ny4xNy4yCm5iZm9ybWF0PT01LjExLjEKbmVzdC1hc3luY2lvMj09MS43LjQKbmV0d29ya3g9PTMuNC4yCm5pbmphPT0xLjEzLjIKbm90ZWJvb2s9PTcuNi40Cm5vdGVib29rLXNoaW09PTAuMi40Cm51bXB5PT0xLjI2LjQKbnZpZGlhLW1sLXB5PT0xMy42MTUuNzEKb3BlbmN2LXB5dGhvbj09NC4xMS4wLjg2Cm9wZW5jdi1weXRob24taGVhZGxlc3M9PTQuMTAuMC44NApvcGVucHl4bD09My4xLjUKb3ZlcnJpZGVzPT03LjcuMApwYWNrYWdpbmc9PTI2LjMKcGFuZGFzPT0yLjMuMwpwYW5kb2NmaWx0ZXJzPT0xLjUuMQpwYXJzbz09MC44LjcKcGVmdD09MC4xMC4wCnBleHBlY3Q9PTQuOS4wCnBoeC1jbGFzcy1yZWdpc3RyeT09NC4xLjAKcGlsbG93PT0xMi4zLjAKcGlwPT0yNi4zCnBsYXRmb3JtZGlycz09NC4xMi40CnByZXR0eXRhYmxlPT0zLjE4LjAKcHJvbWV0aGV1cy1jbGllbnQ9PTAuMjYuMApwcm9tcHQtdG9vbGtpdD09My4wLjUzCnByb3BjYWNoZT09MC41LjQKcHJvdG9idWY9PTcuMzYuMgpwc3V0aWw9PTcuMi4yCnB0eXByb2Nlc3M9PTAuNy4wCnB1cmUtZXZhbD09MC4yLjQKcHktY3B1aW5mbz09OS4wLjAKcHlhcnJvdz09MjUuMC4xCnB5YXJyb3ctaG90Zml4PT0wLjcKcHljb2NvdG9vbHM9PTIuMC4xMQpweWNwYXJzZXI9PTMuMTEKcHlkYW50aWM9PTIuMTQuMwpweWRhbnRpYy1jb3JlPT0yLjUwLjQKcHlkZWNrPT0wLjkuMwpweWdtZW50cz09Mi4yMS4wCnB5bnZtbD09MTMuMC4xCnB5cGFyc2luZz09My4zLjMKcHlzb2Nrcz09MS43LjEKcHl0aG9uLWRhdGV1dGlsPT0zLjAuMApweXRob24tanNvbi1sb2dnZXI9PTQuMi4wCnB5dGhvbi1tdWx0aXBhcnQ9PTAuMC4zMgpweXR6PT0yMDI2LjYKcHl5YW1sPT02LjAuMwpweXptcT09MjcuMy4wCnJlZmVyZW5jaW5nPT0wLjM3LjAKcmVnZXg9PTIwMjYuOS4yOQpyZXF1ZXN0cz09Mi4zNC4yCnJmYzMzMzktdmFsaWRhdG9yPT0wLjEuNApyZmMzOTg2LXZhbGlkYXRvcj09MC4xLjEKcmZjMzk4Ny1zeW50YXg9PTEuMS4wCnJpY2g9PTE1LjAuMApycGRzLXB5PT0wLjMwLjAKc2FmZXRlbnNvcnM9PTAuOC4wCnNjaWtpdC1pbWFnZT09MC4yNS4yCnNjaXB5PT0xLjE1LjMKc2VuZDJ0cmFzaD09Mi4xLjAKc2VudGVuY2VwaWVjZT09MC4yLjMKc2V0dXB0b29scz09ODAuMTAuMgpzaGFwZWx5PT0yLjEuMgpzaXg9PTEuMTcuMApzb2Nrc2lvPT0xLjAuMApzb3Vwc2lldmU9PTIuMTAKc3RhY2stZGF0YT09MC42LjMKc3RhcmxldHRlPT0xLjcuMApzdHJlYW1saXQ9PTEuNjUuMApzeW1weT09MS4xNC4wCnRlcm1jb2xvcj09My4zLjAKdGVybWluYWRvPT0wLjE4LjEKdGVybWluYWx0YWJsZXM9PTMuMS4xMAp0aWZmZmlsZT09MjAyNS41LjEwCnRpa3Rva2VuPT0wLjE0LjAKdGltZW91dC1kZWNvcmF0b3I9PTAuNS4wCnRpbW09PTAuOS4xNgp0aW55Y3NzMj09MS41LjEKdG9rZW5pemVycz09MC4xNS4yCnRvbWw9PTAuMTAuMgp0b21saT09Mi41LjAKdG9yY2g9PTIuMS4yK2N1MTE4CnRvcmNodmlzaW9uPT0wLjE2LjIrY3UxMTgKdG9ybmFkbz09Ni41LjEwCnRxZG09PTQuNzAuMQp0cmFpdGxldHM9PTUuMTYuMQp0cmFuc2Zvcm1lcnM9PTQuMzkuMQp0cmFuc2Zvcm1lcnMtc3RyZWFtLWdlbmVyYXRvcj09MC4wLjUKdHJpdG9uPT0yLjEuMAp0eXBpbmctZXh0ZW5zaW9ucz09NC4xNi4wCnR5cGluZy1pbnNwZWN0aW9uPT0wLjQuNQp0emRhdGE9PTIwMjYuNgp1cmktdGVtcGxhdGU9PTEuMy4wCnVybGxpYjM9PTEuMjYuMTMKdXZpY29ybj09MC41NC4wCndhdGNoZG9nPT02LjAuMAp3Y3dpZHRoPT0wLjkuNAp3ZWJjb2xvcnM9PTI1LjEwLjAKd2ViZW5jb2RpbmdzPT0wLjYuMQp3ZWJzb2NrZXQtY2xpZW50PT0xLjkuMgp3ZWJzb2NrZXRzPT0xNi4xLjEKd2hlZWw9PTAuNDguMAp3aWRnZXRzbmJleHRlbnNpb249PTQuMC4xNgp4dHVuZXI9PTAuMS4yMwp4eGhhc2g9PTQuMC4yCnlhcGY9PTAuNDMuMAp5YXJsPT0xLjI1LjEK', 'regression-manifest.json': 'WwogIHsKICAgICJkYXRhc2V0X2luZGV4IjogMCwKICAgICJpbWdfaWQiOiAzNzY4NDgsCiAgICAiaW1nX3BhdGgiOiAiZGF0YS9jb2NvL3RyYWluMjAxNC9DT0NPX3RyYWluMjAxNF8wMDAwMDAzNzY4NDguanBnIgogIH0sCiAgewogICAgImRhdGFzZXRfaW5kZXgiOiAxLAogICAgImltZ19pZCI6IDI3ODU0OSwKICAgICJpbWdfcGF0aCI6ICJkYXRhL2NvY28vdHJhaW4yMDE0L0NPQ09fdHJhaW4yMDE0XzAwMDAwMDI3ODU0OS5qcGciCiAgfSwKICB7CiAgICAiZGF0YXNldF9pbmRleCI6IDIsCiAgICAiaW1nX2lkIjogNTMyNTA1LAogICAgImltZ19wYXRoIjogImRhdGEvY29jby90cmFpbjIwMTQvQ09DT190cmFpbjIwMTRfMDAwMDAwNTMyNTA1LmpwZyIKICB9LAogIHsKICAgICJkYXRhc2V0X2luZGV4IjogMywKICAgICJpbWdfaWQiOiA0MzQyMDEsCiAgICAiaW1nX3BhdGgiOiAiZGF0YS9jb2NvL3RyYWluMjAxNC9DT0NPX3RyYWluMjAxNF8wMDAwMDA0MzQyMDEuanBnIgogIH0sCiAgewogICAgImRhdGFzZXRfaW5kZXgiOiA0LAogICAgImltZ19pZCI6IDQwMTQzOSwKICAgICJpbWdfcGF0aCI6ICJkYXRhL2NvY28vdHJhaW4yMDE0L0NPQ09fdHJhaW4yMDE0XzAwMDAwMDQwMTQzOS5qcGciCiAgfSwKICB7CiAgICAiZGF0YXNldF9pbmRleCI6IDUsCiAgICAiaW1nX2lkIjogMjcwMzY5LAogICAgImltZ19wYXRoIjogImRhdGEvY29jby90cmFpbjIwMTQvQ09DT190cmFpbjIwMTRfMDAwMDAwMjcwMzY5LmpwZyIKICB9LAogIHsKICAgICJkYXRhc2V0X2luZGV4IjogNiwKICAgICJpbWdfaWQiOiAxMjI5MTYsCiAgICAiaW1nX3BhdGgiOiAiZGF0YS9jb2NvL3RyYWluMjAxNC9DT0NPX3RyYWluMjAxNF8wMDAwMDAxMjI5MTYuanBnIgogIH0sCiAgewogICAgImRhdGFzZXRfaW5kZXgiOiA3LAogICAgImltZ19pZCI6IDQxMDA1LAogICAgImltZ19wYXRoIjogImRhdGEvY29jby90cmFpbjIwMTQvQ09DT190cmFpbjIwMTRfMDAwMDAwMDQxMDA1LmpwZyIKICB9LAogIHsKICAgICJkYXRhc2V0X2luZGV4IjogOCwKICAgICJpbWdfaWQiOiA1MDc5NTIsCiAgICAiaW1nX3BhdGgiOiAiZGF0YS9jb2NvL3RyYWluMjAxNC9DT0NPX3RyYWluMjAxNF8wMDAwMDA1MDc5NTIuanBnIgogIH0sCiAgewogICAgImRhdGFzZXRfaW5kZXgiOiA5LAogICAgImltZ19pZCI6IDQwOTY1MywKICAgICJpbWdfcGF0aCI6ICJkYXRhL2NvY28vdHJhaW4yMDE0L0NPQ09fdHJhaW4yMDE0XzAwMDAwMDQwOTY1My5qcGciCiAgfSwKICB7CiAgICAiZGF0YXNldF9pbmRleCI6IDEwLAogICAgImltZ19pZCI6IDMxOTU0MywKICAgICJpbWdfcGF0aCI6ICJkYXRhL2NvY28vdHJhaW4yMDE0L0NPQ09fdHJhaW4yMDE0XzAwMDAwMDMxOTU0My5qcGciCiAgfSwKICB7CiAgICAiZGF0YXNldF9pbmRleCI6IDExLAogICAgImltZ19pZCI6IDM1MjMxMiwKICAgICJpbWdfcGF0aCI6ICJkYXRhL2NvY28vdHJhaW4yMDE0L0NPQ09fdHJhaW4yMDE0XzAwMDAwMDM1MjMxMi5qcGciCiAgfSwKICB7CiAgICAiZGF0YXNldF9pbmRleCI6IDEyLAogICAgImltZ19pZCI6IDI3MDM5MSwKICAgICJpbWdfcGF0aCI6ICJkYXRhL2NvY28vdHJhaW4yMDE0L0NPQ09fdHJhaW4yMDE0XzAwMDAwMDI3MDM5MS5qcGciCiAgfSwKICB7CiAgICAiZGF0YXNldF9pbmRleCI6IDEzLAogICAgImltZ19pZCI6IDEzOTMyNCwKICAgICJpbWdfcGF0aCI6ICJkYXRhL2NvY28vdHJhaW4yMDE0L0NPQ09fdHJhaW4yMDE0XzAwMDAwMDEzOTMyNC5qcGciCiAgfSwKICB7CiAgICAiZGF0YXNldF9pbmRleCI6IDE0LAogICAgImltZ19pZCI6IDU0ODkzNSwKICAgICJpbWdfcGF0aCI6ICJkYXRhL2NvY28vdHJhaW4yMDE0L0NPQ09fdHJhaW4yMDE0XzAwMDAwMDU0ODkzNS5qcGciCiAgfSwKICB7CiAgICAiZGF0YXNldF9pbmRleCI6IDE1LAogICAgImltZ19pZCI6IDQ1ODgyNywKICAgICJpbWdfcGF0aCI6ICJkYXRhL2NvY28vdHJhaW4yMDE0L0NPQ09fdHJhaW4yMDE0XzAwMDAwMDQ1ODgyNy5qcGciCiAgfSwKICB7CiAgICAiZGF0YXNldF9pbmRleCI6IDE2LAogICAgImltZ19pZCI6IDE2NDY1LAogICAgImltZ19wYXRoIjogImRhdGEvY29jby90cmFpbjIwMTQvQ09DT190cmFpbjIwMTRfMDAwMDAwMDE2NDY1LmpwZyIKICB9LAogIHsKICAgICJkYXRhc2V0X2luZGV4IjogMTcsCiAgICAiaW1nX2lkIjogNDQyNDYxLAogICAgImltZ19wYXRoIjogImRhdGEvY29jby90cmFpbjIwMTQvQ09DT190cmFpbjIwMTRfMDAwMDAwNDQyNDYxLmpwZyIKICB9LAogIHsKICAgICJkYXRhc2V0X2luZGV4IjogMTgsCiAgICAiaW1nX2lkIjogMTM5MzU5LAogICAgImltZ19wYXRoIjogImRhdGEvY29jby90cmFpbjIwMTQvQ09DT190cmFpbjIwMTRfMDAwMDAwMTM5MzU5LmpwZyIKICB9LAogIHsKICAgICJkYXRhc2V0X2luZGV4IjogMTksCiAgICAiaW1nX2lkIjogMTE0Nzg2LAogICAgImltZ19wYXRoIjogImRhdGEvY29jby90cmFpbjIwMTQvQ09DT190cmFpbjIwMTRfMDAwMDAwMTE0Nzg2LmpwZyIKICB9Cl0=', 'reference/metrics.json': 'ewogICJzcGxpdCI6ICJyZWZjb2NvX3ZhbCIsCiAgInNhbXBsZV9jb3VudCI6IDIwLAogICJtZXRyaWNzIjogewogICAgImNJb1UiOiA3OS4xODY5NDMwNTQxOTkyMiwKICAgICJtSW9VIjogODEuNjgyNDcyMjI5MDAzOQogIH0KfQ==', 'reference/subset-evaluator.py': 'ZnJvbSBtbWRldC5kYXRhc2V0cyBpbXBvcnQgUmVmQ29jb0RhdGFzZXQNCmZyb20gZmxtbS5kYXRhc2V0cy50cmFuc2Zvcm1zIGltcG9ydCBQSUxMb2FkSW1hZ2VGcm9tRmlsZSwgUmVmQ09DTzJQTkcNCmZyb20gbW1kZXQuZGF0YXNldHMudHJhbnNmb3JtcyBpbXBvcnQgTG9hZEFubm90YXRpb25zDQpmcm9tIG1tZGV0LmV2YWx1YXRpb24gaW1wb3J0IFJlZlNlZ01ldHJpYw0KaW1wb3J0IGFyZ3BhcnNlDQppbXBvcnQganNvbg0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQpmcm9tIG1tZW5naW5lLmNvbmZpZyBpbXBvcnQgQ29uZmlnDQpmcm9tIHh0dW5lci5tb2RlbC51dGlscyBpbXBvcnQgZ3Vlc3NfbG9hZF9jaGVja3BvaW50DQpmcm9tIHh0dW5lci5yZWdpc3RyeSBpbXBvcnQgQlVJTERFUg0KZnJvbSB4dHVuZXIudXRpbHMuY29uc3RhbnRzIGltcG9ydCBERUZBVUxUX0lNQUdFX1RPS0VODQpmcm9tIGFjY2VsZXJhdGUgaW1wb3J0IEFjY2VsZXJhdG9yDQpmcm9tIGFjY2VsZXJhdGUudXRpbHMgaW1wb3J0IGdhdGhlcl9vYmplY3QNCmZyb20gbW1kZXQuc3RydWN0dXJlcy5tYXNrIGltcG9ydCBCaXRtYXBNYXNrcw0KDQpmcm9tIHRxZG0gaW1wb3J0IHRxZG0NCmltcG9ydCB0b3JjaA0KaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRg0KZnJvbSB0aW1lIGltcG9ydCB0aW1lDQoNCg0KaWYgX19uYW1lX18gPT0gJ19fbWFpbl9fJzoNCiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihmb3JtYXR0ZXJfY2xhc3M9YXJncGFyc2UuUmF3VGV4dEhlbHBGb3JtYXR0ZXIpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgnY29uZmlnJywgaGVscD0nY29uZmlnIGZpbGUgcGF0aC4nKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoJy0tY2hlY2twb2ludCcsIGRlZmF1bHQ9Tm9uZSwgdHlwZT1zdHIpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgnLS1kZWJ1ZycsIGFjdGlvbj0nc3RvcmVfdHJ1ZScpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgnLS1jZXBoJywgYWN0aW9uPSdzdG9yZV90cnVlJykNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCctLWNvbmNhdCcsIGFjdGlvbj0nc3RvcmVfdHJ1ZScpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgnLS1tYXgtc2FtcGxlcycsIHR5cGU9aW50LCBkZWZhdWx0PTEpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgnLS1vdXRwdXQtZGlyJywgZGVmYXVsdD0nL2thZ2dsZS93b3JraW5nL3Ntb2tlLXJlc3VsdHMnKQ0KICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpDQogICAgaWYgYXJncy5tYXhfc2FtcGxlcyA8IDE6DQogICAgICAgIHBhcnNlci5lcnJvcignLS1tYXgtc2FtcGxlcyBtdXN0IGJlIHBvc2l0aXZlJykNCiAgICBvdXRwdXRfZGlyID0gUGF0aChhcmdzLm91dHB1dF9kaXIpDQogICAgb3V0cHV0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpDQoNCiAgICAjIyMgSW5pdGlhbGl6ZSBhY2NlbGVyYXRvcg0KICAgIGFjY2VsZXJhdG9yID0gQWNjZWxlcmF0b3IoKQ0KICAgICMgZWFjaCBHUFUgY3JlYXRlcyBhIHN0cmluZw0KICAgIG1lc3NhZ2UgPSBbZiJIZWxsbyB0aGlzIGlzIEdQVSB7YWNjZWxlcmF0b3IucHJvY2Vzc19pbmRleH0iXQ0KICAgICMgY29sbGVjdCB0aGUgbWVzc2FnZXMgZnJvbSBhbGwgR1BVcw0KICAgIG1lc3NhZ2VzID0gZ2F0aGVyX29iamVjdChtZXNzYWdlKQ0KICAgICMgb3V0cHV0IHRoZSBtZXNzYWdlcyBvbmx5IG9uIHRoZSBtYWluIHByb2Nlc3Mgd2l0aCBhY2NlbGVyYXRvci5wcmludCgpDQogICAgYWNjZWxlcmF0b3IucHJpbnQobWVzc2FnZXMpDQoNCiAgICBjZmcgPSBDb25maWcuZnJvbWZpbGUoYXJncy5jb25maWcpDQogICAgcHJvbXB0X3RlbXBsYXRlID0gY2ZnLnByb21wdF90ZW1wbGF0ZQ0KICAgIHRva2VuaXplciA9IGNmZy50b2tlbml6ZXINCiAgICBpbWFnZV9wcm9jZXNzb3IgPSBjZmcuaW1hZ2VfcHJvY2Vzc29yDQogICAgcHJvbXB0ID0gY2ZnLmdldCgncHJvbXB0JywgTm9uZSkNCiAgICB0b3JjaC5iYWNrZW5kcy5jdWRhLmVuYWJsZV9mbGFzaF9zZHAoRmFsc2UpDQogICAgdG9yY2guYmFja2VuZHMuY3VkYS5lbmFibGVfbWVtX2VmZmljaWVudF9zZHAoRmFsc2UpDQogICAgdG9yY2guYmFja2VuZHMuY3VkYS5lbmFibGVfbWF0aF9zZHAoVHJ1ZSkNCiAgICBtb2RlbCA9IEJVSUxERVIuYnVpbGQoY2ZnLm1vZGVsKQ0KDQogICAgaWYgYXJncy5jaGVja3BvaW50IGlzIG5vdCBOb25lOg0KICAgICAgICBzdGF0ZV9kaWN0ID0gZ3Vlc3NfbG9hZF9jaGVja3BvaW50KGFyZ3MuY2hlY2twb2ludCkNCiAgICAgICAgbWlzc2luZywgdW5leHBlY3RlZCA9IG1vZGVsLmxvYWRfc3RhdGVfZGljdChzdGF0ZV9kaWN0LCBzdHJpY3Q9RmFsc2UpDQogICAgICAgIGFjY2VsZXJhdG9yLnByaW50KGYiVW5leHBlY3RlZCBwYXJhbWV0ZXJzOiB7dW5leHBlY3RlZH0iKQ0KICAgICAgICBhY2NlbGVyYXRvci5wcmludChmIk1pc3NpbmcgcGFyYW1ldGVyczoge21pc3Npbmd9IikNCiAgICAgICAgKG91dHB1dF9kaXIgLyAiY2hlY2twb2ludC1rZXlzLmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoZGljdChtaXNzaW5nPW1pc3NpbmcsIHVuZXhwZWN0ZWQ9dW5leHBlY3RlZCksIGluZGVudD0yKSkNCg0KICAgIHByaW50KGYiU3RhcnQgbW92aW5nIG1vZGVsIHRvIGRldmljZToge2FjY2VsZXJhdG9yLmRldmljZX0iLCBmbHVzaD1UcnVlKQ0KICAgIHRpayA9IHRpbWUoKQ0KICAgIG1vZGVsID0gbW9kZWwudG8oZGV2aWNlPWFjY2VsZXJhdG9yLmRldmljZSkNCiAgICBwcmludChmIkZpbmlzaGVkIG1vdmluZyBtb2RlbCB0byBkZXZpY2U6IHthY2NlbGVyYXRvci5kZXZpY2V9LCB0aW1lIHVzZWQ6IHt0aW1lKCkgLSB0aWt9IiwgZmx1c2g9VHJ1ZSkNCiAgICBtb2RlbC5ldmFsKCkNCg0KICAgIGlmIGFyZ3MuY2VwaDoNCiAgICAgICAgYmFja2VuZF9hcmdzID0gZGljdCgNCiAgICAgICAgICAgIGJhY2tlbmQ9J3BldHJlbCcsDQogICAgICAgICAgICBwYXRoX21hcHBpbmc9ZGljdCh7DQogICAgICAgICAgICAgICAgJ2RhdGEvY29jby90cmFpbjIwMTQvJzogJ29wZW5tbWxhYjpzMzovL29wZW5tbWxhYi9kYXRhc2V0cy9kZXRlY3Rpb24vY29jby90cmFpbjIwMTQvJw0KICAgICAgICAgICAgfSkpDQogICAgZWxzZToNCiAgICAgICAgYmFja2VuZF9hcmdzID0gTm9uZQ0KDQogICAgcmVmY29jbzJwbmdfcGFyYW1zID0gZGljdCgNCiAgICAgICAgdHlwZT1SZWZDT0NPMlBORywNCiAgICAgICAgaW1hZ2VfcHJvY2Vzc29yPWltYWdlX3Byb2Nlc3NvciwNCiAgICAgICAgdG9rZW5pemVyPXRva2VuaXplciwNCiAgICAgICAgcHJvbXB0X3RlbXBsYXRlPXByb21wdF90ZW1wbGF0ZSwNCiAgICAgICAgY29uY2F0PWFyZ3MuY29uY2F0LA0KICAgICAgICBpbWFnZTJ0ZW5zb3I9Y2ZnLmdldCgnaW1hZ2UydGVuc29yJywgVHJ1ZSksDQogICAgICAgIGFkZF9pbWFnZV90b2tlbj1jZmcuZ2V0KCdhZGRfaW1hZ2VfdG9rZW4nLCBGYWxzZSksDQogICAgICAgIGltYWdlX3Rva2VuPWNmZy5nZXQoJ2ltYWdlX3Rva2VuJywgREVGQVVMVF9JTUFHRV9UT0tFTikNCiAgICApDQogICAgYWNjZWxlcmF0b3IucHJpbnQoZiJEbyBjb25jYXRlbmF0aW9uPyB7YXJncy5jb25jYXR9IikNCiAgICBpZiBwcm9tcHQgaXMgbm90IE5vbmU6DQogICAgICAgIHJlZmNvY28ycG5nX3BhcmFtcy51cGRhdGUocHJvbXB0PXByb21wdCkNCg0KICAgICMgcmVmX2NvY28gZGF0YSBwaXBlbGluZQ0KICAgIHRlc3RfcGlwZWxpbmUgPSBbDQogICAgICAgIGRpY3QodHlwZT1QSUxMb2FkSW1hZ2VGcm9tRmlsZSwgYmFja2VuZF9hcmdzPWJhY2tlbmRfYXJncyksDQogICAgICAgIGRpY3QoDQogICAgICAgICAgICB0eXBlPUxvYWRBbm5vdGF0aW9ucywNCiAgICAgICAgICAgIHdpdGhfbWFzaz1UcnVlLA0KICAgICAgICAgICAgd2l0aF9iYm94PUZhbHNlLA0KICAgICAgICAgICAgd2l0aF9zZWc9RmFsc2UsDQogICAgICAgICAgICB3aXRoX2xhYmVsPUZhbHNlKSwNCiAgICAgICAgcmVmY29jbzJwbmdfcGFyYW1zDQogICAgXQ0KDQogICAgcmVmY29jb19zdWJzZXRzID0gZGljdCgpDQogICAgZm9yIHNwbGl0IGluIFsndmFsJywgJ3Rlc3RBJywgJ3Rlc3RCJ106DQogICAgICAgIHJlZmNvY29fc3Vic2V0c1tmJ3JlZmNvY29fe3NwbGl0fSddID0gZGljdCgNCiAgICAgICAgICAgIGFubl9maWxlPSdyZWZjb2NvL2luc3RhbmNlcy5qc29uJywNCiAgICAgICAgICAgIHNwbGl0X2ZpbGU9J3JlZmNvY28vcmVmcyh1bmMpLnAnLA0KICAgICAgICAgICAgc3BsaXQ9c3BsaXQpDQoNCiAgICBmb3Igc3BsaXQgaW4gWyd2YWwnLCAndGVzdEEnLCAndGVzdEInXToNCiAgICAgICAgcmVmY29jb19zdWJzZXRzW2YncmVmY29jbytfe3NwbGl0fSddID0gZGljdCgNCiAgICAgICAgICAgIGFubl9maWxlPSdyZWZjb2NvKy9pbnN0YW5jZXMuanNvbicsDQogICAgICAgICAgICBzcGxpdF9maWxlPSdyZWZjb2NvKy9yZWZzKHVuYykucCcsDQogICAgICAgICAgICBzcGxpdD1zcGxpdCkNCg0KICAgIGZvciBzcGxpdCBpbiBbJ3ZhbCcsICd0ZXN0J106DQogICAgICAgIHJlZmNvY29fc3Vic2V0c1tmJ3JlZmNvY29nX3tzcGxpdH0nXSA9IGRpY3QoDQogICAgICAgICAgICBhbm5fZmlsZT0ncmVmY29jb2cvaW5zdGFuY2VzLmpzb24nLA0KICAgICAgICAgICAgc3BsaXRfZmlsZT0ncmVmY29jb2cvcmVmcyh1bWQpLnAnLA0KICAgICAgICAgICAgc3BsaXQ9c3BsaXQpDQoNCiAgICByZWZjb2NvX3N1YnNldHMgPSB7InJlZmNvY29fdmFsIjogcmVmY29jb19zdWJzZXRzWyJyZWZjb2NvX3ZhbCJdfQ0KICAgIGZvciBuYW1lLCBzdWJzZXQgaW4gcmVmY29jb19zdWJzZXRzLml0ZW1zKCk6DQogICAgICAgIGFjY2VsZXJhdG9yLnByaW50KGYiU3RhcnQgZXZhbHVhdGluZyB7bmFtZX0iKQ0KICAgICAgICBkYXRhc2V0ID0gUmVmQ29jb0RhdGFzZXQoDQogICAgICAgICAgICBkYXRhX3Jvb3Q9J2RhdGEvY29jby8nLA0KICAgICAgICAgICAgZGF0YV9wcmVmaXg9ZGljdChpbWdfcGF0aD0ndHJhaW4yMDE0LycpLA0KICAgICAgICAgICAgdGV4dF9tb2RlPSdzZWxlY3RfZmlyc3QnLA0KICAgICAgICAgICAgcGlwZWxpbmU9dGVzdF9waXBlbGluZSwNCiAgICAgICAgICAgICoqc3Vic2V0DQogICAgICAgICkNCiAgICAgICAgIyBzeW5jIEdQVXMgYW5kIHN0YXJ0IHRoZSB0aW1lcg0KICAgICAgICBhY2NlbGVyYXRvci53YWl0X2Zvcl9ldmVyeW9uZSgpDQoNCiAgICAgICAgZGF0YV9pZHMgPSBsaXN0KHJhbmdlKGxlbihkYXRhc2V0KSkpDQogICAgICAgIGlmIGFyZ3MuZGVidWc6DQogICAgICAgICAgICBkYXRhX2lkcyA9IGRhdGFfaWRzWzoxMDBdDQoNCiAgICAgICAgZGF0YV9pZHMgPSBkYXRhX2lkc1s6YXJncy5tYXhfc2FtcGxlc10NCiAgICAgICAgbWFuaWZlc3QgPSBbZGljdChkYXRhc2V0X2luZGV4PWksIGltZ19pZD1kYXRhc2V0LmdldF9kYXRhX2luZm8oaSkuZ2V0KCJpbWdfaWQiKSwgaW1nX3BhdGg9ZGF0YXNldC5nZXRfZGF0YV9pbmZvKGkpLmdldCgiaW1nX3BhdGgiKSkgZm9yIGkgaW4gZGF0YV9pZHNdDQogICAgICAgIChvdXRwdXRfZGlyIC8gInNhbXBsZS1tYW5pZmVzdC5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKG1hbmlmZXN0LCBpbmRlbnQ9MikpDQogICAgICAgIHJlc3VsdHMgPSBbXQ0KICAgICAgICAjIGRpdmlkZSB0aGUgcHJvbXB0IGxpc3Qgb250byB0aGUgYXZhaWxhYmxlIEdQVXMNCiAgICAgICAgd2l0aCBhY2NlbGVyYXRvci5zcGxpdF9iZXR3ZWVuX3Byb2Nlc3NlcyhkYXRhX2lkcykgYXMgc3ViX2lkczoNCiAgICAgICAgICAgIGZvciBpZHggaW4gdHFkbShzdWJfaWRzLCBkaXNhYmxlPW5vdCBhY2NlbGVyYXRvci5pc19tYWluX3Byb2Nlc3MpOg0KICAgICAgICAgICAgICAgIGRhdGFfc2FtcGxlID0gZGF0YXNldFtpZHhdDQogICAgICAgICAgICAgICAgaWYgYXJncy5jb25jYXQ6DQogICAgICAgICAgICAgICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOg0KICAgICAgICAgICAgICAgICAgICAgICAgcHJlZF9tYXNrX2xvZ2l0cyA9IG1vZGVsLnByZWRpY3QoZGF0YV9zYW1wbGUpDQoNCiAgICAgICAgICAgICAgICAgICAgZ3RfbWFza3MgPSBkYXRhX3NhbXBsZVsnZ3RfbWFza3MnXS5udW1weSgpID4gMA0KICAgICAgICAgICAgICAgICAgICBwcmVkX21hc2tzID0gRi5pbnRlcnBvbGF0ZShwcmVkX21hc2tfbG9naXRzW05vbmVdLmZsb2F0KCkuc2lnbW9pZCgpLA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzaXplPWd0X21hc2tzLnNoYXBlWy0yOl0sIG1vZGU9J2JpbGluZWFyJylbMF0uY3B1KCkNCiAgICAgICAgICAgICAgICAgICAgcHJlZF9tYXNrcyA9IHByZWRfbWFza3MgPiAwLjUNCiAgICAgICAgICAgICAgICAgICAgaW1wb3J0IG51bXB5IGFzIG5wDQogICAgICAgICAgICAgICAgICAgIG5wLnNhdmV6X2NvbXByZXNzZWQob3V0cHV0X2RpciAvIGYic2FtcGxlLXtpZHh9Lm5weiIsIHByZWRpY3Rpb249cHJlZF9tYXNrcy5udW1weSgpLCBncm91bmRfdHJ1dGg9Z3RfbWFza3MpDQoNCiAgICAgICAgICAgICAgICAgICAgYXNzZXJ0IGxlbihwcmVkX21hc2tzKSA9PSBsZW4oZ3RfbWFza3MpDQogICAgICAgICAgICAgICAgICAgIG1hc2tfY250ID0gcHJlZF9tYXNrcy5zaGFwZVswXQ0KDQogICAgICAgICAgICAgICAgICAgICMgRm9ybXVsYXRlIHRoZSBvdXRwdXQgaW50byB0aGUgZm9ybWF0IHRoYXQgdGhlIGV2YWx1YXRvciBhY2NlcHRzDQogICAgICAgICAgICAgICAgICAgIHJlc3VsdHMuYXBwZW5kKGRpY3QocHJlZF9pbnN0YW5jZXM9ZGljdChtYXNrcz1wcmVkX21hc2tzKSwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBndF9tYXNrcz1CaXRtYXBNYXNrcyhtYXNrcz1ndF9tYXNrcywNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBoZWlnaHQ9Z3RfbWFza3Muc2hhcGVbMV0sDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgd2lkdGg9Z3RfbWFza3Muc2hhcGVbMl0pKQ0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICApDQogICAgICAgICAgICAgICAgZWxzZToNCiAgICAgICAgICAgICAgICAgICAgZm9yIHN1Yl9kYXRhX3NhbXBsZSBpbiBkYXRhX3NhbXBsZToNCiAgICAgICAgICAgICAgICAgICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOg0KICAgICAgICAgICAgICAgICAgICAgICAgICAgIHByZWRfbWFza19sb2dpdHMgPSBtb2RlbC5wcmVkaWN0KHN1Yl9kYXRhX3NhbXBsZSkNCg0KICAgICAgICAgICAgICAgICAgICAgICAgZ3RfbWFza3MgPSBzdWJfZGF0YV9zYW1wbGVbJ2d0X21hc2tzJ10ubnVtcHkoKSA+IDANCiAgICAgICAgICAgICAgICAgICAgICAgIHByZWRfbWFza3MgPSBGLmludGVycG9sYXRlKHByZWRfbWFza19sb2dpdHNbTm9uZV0uZmxvYXQoKS5zaWdtb2lkKCksDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzaXplPWd0X21hc2tzLnNoYXBlWy0yOl0sIG1vZGU9J2JpbGluZWFyJylbMF0uY3B1KCkNCiAgICAgICAgICAgICAgICAgICAgICAgIHByZWRfbWFza3MgPSBwcmVkX21hc2tzID4gMC41DQoNCiAgICAgICAgICAgICAgICAgICAgICAgIGFzc2VydCBsZW4ocHJlZF9tYXNrcykgPT0gbGVuKGd0X21hc2tzKQ0KICAgICAgICAgICAgICAgICAgICAgICAgbWFza19jbnQgPSBwcmVkX21hc2tzLnNoYXBlWzBdDQogICAgICAgICAgICAgICAgICAgICAgICBhc3NlcnQgbWFza19jbnQgPT0gMQ0KDQogICAgICAgICAgICAgICAgICAgICAgICAjIEZvcm11bGF0ZSB0aGUgb3V0cHV0IGludG8gdGhlIGZvcm1hdCB0aGF0IHRoZSBldmFsdWF0b3IgYWNjZXB0cw0KICAgICAgICAgICAgICAgICAgICAgICAgcmVzdWx0cy5hcHBlbmQoZGljdChwcmVkX2luc3RhbmNlcz1kaWN0KG1hc2tzPXByZWRfbWFza3MpLA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBndF9tYXNrcz1CaXRtYXBNYXNrcyhtYXNrcz1ndF9tYXNrcywNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgaGVpZ2h0PWd0X21hc2tzLnNoYXBlWzFdLA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB3aWR0aD1ndF9tYXNrcy5zaGFwZVsyXSkpDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICApDQogICAgICAgICAgICByZXN1bHRzID0gZ2F0aGVyX29iamVjdChyZXN1bHRzKQ0KICAgICAgICBpZiBhY2NlbGVyYXRvci5pc19tYWluX3Byb2Nlc3M6DQogICAgICAgICAgICBhY2NlbGVyYXRvci5wcmludChmIkNvbGxlY3RlZCB7bGVuKHJlc3VsdHMpfSByZXN1bHQgc2FtcGxlcyBmcm9tIGFsbCBncHVzIikNCiAgICAgICAgICAgIGV2YWx1YXRvciA9IFJlZlNlZ01ldHJpYyhtZXRyaWM9WydjSW9VJywgJ21Jb1UnXSkNCiAgICAgICAgICAgIGV2YWx1YXRvci5wcm9jZXNzKGRhdGFfYmF0Y2g9ZGljdCgpLCBkYXRhX3NhbXBsZXM9cmVzdWx0cykNCiAgICAgICAgICAgIG1ldHJpY3MgPSBldmFsdWF0b3IuY29tcHV0ZV9tZXRyaWNzKGV2YWx1YXRvci5yZXN1bHRzKQ0KICAgICAgICAgICAgYWNjZWxlcmF0b3IucHJpbnQoZiJFdmFsdWF0aW9uIHJlc3VsdHMgb24ge25hbWV9OiB7bWV0cmljc30iKQ0KICAgICAgICAgICAgKG91dHB1dF9kaXIgLyAibWV0cmljcy5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKGRpY3Qoc3BsaXQ9bmFtZSwgc2FtcGxlX2NvdW50PWxlbihkYXRhX2lkcyksIG1ldHJpY3M9e2s6IGZsb2F0KHYpIGZvciBrLCB2IGluIG1ldHJpY3MuaXRlbXMoKX0pLCBpbmRlbnQ9MikpDQogICAgICAgIGFjY2VsZXJhdG9yLnByaW50KGYiRmluaXNoZWQgZXZhbHVhdGluZyB7bmFtZX0iKQ0K', 'reference/sample-0.npz': 'UEsDBBQAAAAIAAAAIQDxW9NzhhQAAICXCQAOABQAcHJlZGljdGlvbi5ucHkBABAAgJcJAAAAAACGFAAAAAAAAO3dy45dSVbHYeeUp/CsQKoBjVpI8ADMQEwYMEINXYgBoltViAnwFLwwdfHdec7eEbFW/Pep832j8iV3xF7rp5SdtlX/93f/8Ld//48vb/7rzX9/8/vvfviX77/567ff/M8//+abb99+869/+P4/v//df/zTH77//Xc/ff/f/O7ff/jux+//4d9+98fvfvz2n/7Ft29/+1e/+fbtX/72z//s27f/+3bcn7wBAAAAAAAAAAAAAAAAAAAAAACey8vLS/oKPK2XF/2R8fJe+iI8nZdPpS/Dk3l50R8xL/oj5sv69MdGX+enP7Z5JT/9sctr+emPTV7NT39s8Xp9+mOLW/npj34369Mf/e7kpz+63ctPf3TTH0F389MfzfRH0P389Ecv/ZFzUJ/+aHRYn/7ocyI//dFGfwSdyU9/NDmVn/7ocS4//dHiZH76o8PZ/PRHB/0Rczo+/VFvJD/9UU1/BA3lpz+K6Y+gsfz0R6nB/PRHKf0RNJqf/ig0nJ/+qDOen/6oMlGf/qgylZ/+qDGXn/6ooT+CJvPTHwVm69MfBebz0x/LFvLTH6tW8tMfa5bq0x9LFuvTHyuW89Mf09br0x/TKvLTH3NK6tMfE4ra0x/jCuPTH4Nq69MfY/RHTnV9+uO8+vr0x2kd+emPs/RHkv4IaslPf5ykP4J68tMf5+iPJP0R1JSf/jhFfwR15ac/TmjLT3+coD+C+vLTH4ca89MfRzrz0x8HWvPTHwf0R1BvfvrjPv0R1Jyf/rhLfyTpjyT9kaQ/kvRHkv5I0h9J+iNJf0TpjyT9EdScn/64T38k6Y8k/ZGkP5L0R5L+SNIfSfojSX8k6Y8k/ZGkP5L0R5L+SNIfSfojSX8k6Y8k/ZGkP5L0R5L+SNIfUfojSX8k6Y8o/RGlP5L0R5L+iNIfUfojSX8k6Y8k/RGlP5L0R5L+SNIfSfojSn9E6Y8o/ZGkP6L0R5b+yNIfUfojSX8k6Y8k/ZGkP5L0R5T+SNIfSfojSn8k6Y8k/ZGkP5L0R5T+iNIfSfojSX8k6Y8o/RGlP5L0R5T+SNIfUfojSn8k6Y8k/RGlP5L0R5T+SNIfUfojSn9E6Y8k/ZGkP5L0R5T+SNIfUfojSn8k6Y8o/ZGkP6L0R5L+SNIfUfojSX9E6Y8k/ZGkP6L0R5L+SNIfUfojSX8k6Y8o/ZGkP5L0R5L+iNIfSfojSX9E6Y8k/RGlP6L0R5L+iNIfSfojSn9k6Y8s/RGlP9L0R5r+yNMfUfojSn+E6Y8k/ZGkP6L0R5L+iNIfUfojyOc/kvRHkv5I0h9Bt/LTHzvoj5yb9emPdnfq0x/N7tanP/ocpKc/upxJT38UO52d/ig1Xp7+WPZzQbPt6Y95K9Xpj0k12emPGcX16Y+73vVRnp3+ntOZ3ffFpr8ntjWr09JTYYd0ZbelJ8MG6cjuSI+GDdKR3ZEeDRukI7sjPRr6pRu7Jz0b2qUTuys9HLqlC7svPR26pQu7Lz0dmqUDO5AeD83SgR1Ij4de6b6OpOdDr3RfR9LzoVe6ryPp+dAr3deR9Hzole7rSHo+tErndSg9IFql8zqUHhCt0nkdSg+IVum8DqUHRKt0XofSA6JVOq9D6QHRu4R0X0caX50LSPd1JD0feqX7OpKeD83SgR1Ij+fpbdlCOLI7ut+c+z5fxfP9TqTxhTlh3yZO1dCV2e0Tyfls4f2HnaihK7PbJ5KzdR/b0zql7305tHUxI0WsdzVyGhl7NzQSxHROw6rejglbNzQSxHxPo6rejglbVzQUxHRPo4pejhk7VzRWxGRN42pejik7djRXxPBHzaoYI5M2LGmyiPEPm1QzSKZsWNJkEeMfNqlmkEzZsKXJIsY/bFLRJJmwY02TRQzfZPycxRdj2ZY9zRUxeY3hw/SXs2dRc0UsX+HzD6t/LZZNZNF+ysud/kbP/eSfVtW/Fssmsug/pqy/Xx5zeInp12LVTBc7Drvxs3tOX3srFgz2t7ys8fPKMml7MPN2V/CS+yqJ/q7oSh20Hqu/S3qQ/lrftODhzNndQiK8w4OrzmDITHurO9tzyhdHHh1bcQqjptNb2Fv38385ZPTIyXNYMpXc4t66nz93ZMk5DJpKbnFv3c9//cyjY2vOYcx0dvOL63369KFFBzFirriF3TU+evHcooMYMJHa4vL6nrx6bM1BjBgPbXV9Xc9dP7bmIEaMZ7a6wJ6nVhxbcxADxhurWGD9E0sOLDqM8+Y6W95g9fNq3m/5NEYNF1a0w11FdNydOuOBVS1xTxEtV6fMTGFFW9xSRMvNKTNVWNUW+4PoujlFJhsrW2NrD50Xp8R0ZGV77Kuh995UWMmsaJFtNfRemwpLmVVtsieG7ltTYLGzylVWNrDt0qypK+5Ku3zEOz+n0uYus8wHvPKTKk3uMut8uAs/rdLgLrPPR7vv8yrt7SILfbDrPrfC2q6x0ce67dMrS+0aO32ku/Jmw//c4IHeZutV+UVNZZdY6gNdlS98HH9Bc5mtPs5NuasgusRWH+em3FdQ3f69Pso9ObS+ysBiH+SaHFtfZWCzj3FLTlhf5f7VPsQlOaNglbt3+wh35KSSZW5d7gNckdNqtrlxu9e/IQOK1rlvvZe/ICOK1rlpvZe/IKPqVtq/36vfj3GFO+3e79Xvx4TKpXbv9+r3Y1zhTrsXfPHrMaNyqc0Lvvj1mFG51NYFX/x6zKlda9+Gr307ZhXvtW3D174ds4r32rbiS1+OacV7bVryha/GkvLNdmz5ujdjUf1q6/d81XtRoGG51Yu+6LWo0LDc4kVf9FqU6Nhu6aIvei2KtOy3cNEXvRZFWvZbt+iLXosqPQsuW/RFr0WVngWXbfqat6JOz4aLNn3NW1Gsac3Lq77kpWjRteuFbV/vRvS61rZbbrN0I/a4xLZLL1FyIwJi614/uPhC5Ozfd1FrZfchbPO+63KruQ9pexdel1vJdbiAjSuvq63gMlzHnp2X9rZ4Fy6mf+nFwa1chQvqXntxcPMX4ZK6114c3PxFuKTmtRf3Nn8RLqp378W9Td+Dq+pdfG1u09fgwho3X5zb5C24tMbNF+c2eQuurW/z1b3N3YKLa1t+bW1TV+AR9Gy/uLaJG/AgWrZfndv4DXgUDdsvz234BjyMhvVX1zZ8AR5Hw/qraxu+AA+kPIDi1kaP57GUF1Cb2ujpPJjyAmpTGz2dB1OeQGlpo4fzaKojKMxs9GgeUXEGdZUNHsxjKu6gLLLBc3lQxR2URTZ4Lo+qtoO6ysbO5WFVhlAX2dCxPLDKEsoaGzqVh1aXQlliI4fy4OpSKEts5FAeXFkKZYWNHMqjq4qhKK+RI/k1qOmhpK2B8/i1KAmioqyB4/gVKQmipK3zx/HrUVJERVrnT+PXpKCIkrROn8avy7561qVnRYN0VAPSo6JBOqoB6VHRIB3VgPSoaJCOakB6VDRIRzUgPSo6pKs6Lz0pOqSrOi89KTqkqzovPSk6pKs6Lz0pOqSrOi89KVqkszotPShapLM6LT0oeqS7Ois9J5qkwzopPSaapMM6KT0muqTLOic9JbqkyzonPSW6pMs6Jz0luqTLOic9Jdqk0zolPSTapNM6JT0k+qTbOiM9I/qk2zojPSP6pNs6Iz0jKn2+1WRXZ2XnRakvtxrs6qzovGiWrutYekK0Sud1KD0gWqXzOpQeEK3SeR1KD4he6b6OpOdDr3RfR9LzoVe6ryPp+dAr3deR9Hzole7rSHo+NEsHdiA9HpqlAzuQHg/N0oEdSI+HZunADqTHQ7N0YAfS46FZOrAD6fHQL93YPenZ0C/d2D3p2dAv3dg96dnQL93YPenZsEE6sjvSo2GHdGW3pSfDDunKbktPhk3Sod2QHgu7vLL4V/5rV3DSezafJrbymM+fN9beJyfL78m89llo5imvPG3805/+nk9Ffneeef/zXu2xAAAA614+/QrhJ1+QfPOm5zdQMPe1RKgxUZ8CKaM/gibz0x8l9EeS/ojSH0n6I0l/BE3mpz9K6I8k/ZEkPoJ88iNJf0RpjyT5EaU/gnz+I0l/JOmPJPmRpD+CBvPTH6X0R5L+SNIfQfIjSX8EjeanPyrpjyT9kaQ/kuRHkv5I0h9J8iNLfyTpj6Ch/PRHLfmRpD+S9EeS/kjSH0nyI0l/ZMmPJP2RpD+S5EeS/IiSH0n6I0l/JOmPJPmRJD+S9EeS/kg6058A6aI/kk71J0Ca6I8k/ZEkP5LkR5L+SNIfSfojSX0kqY8k/ZGkP5L0R5L8SJIfSfIjSX8k6Y8k+ZGkP5L0R5L+SNIfQfIjSX/k3KtPfnTTH0n6I0t+RH2Vm/bYyKc7kuRHkv5I0h9Z6gMAeC5+9UeO3/2S5MsvJPn6H0n+/JcoAZLkL/yR5G+ckuSvPJOkP5L8ow+S/LsjgvzDN5K+7k+A7KM/kl7pT4HsI0CSXu1PgGyiP5Je70+AbHEjPwGyxc3+JEi/e/kpkG4H/QmQXvojySdAko76EyCdDvsTII2O+xMgbU7kJ0DanOpPgTQ52Z8GaTHQnwApN9KfBKk22J8AqTUaoAKpNNyfBCk0058CqTLXnwCpMdmfACkx258AqTDdnwQpsNKfAFm11J8AWbTWnwBZs9ifBFmy3p8EmVfSnwaZVNafAplQ2J8EGVbbnwwZ09KfFjmpvT8Bckd/fyLktk39aZBXbexPgHxlZ38q5Ev7+5MgH0X6EyDvZPqTIL+I9SdA3iT7kyDh/gT49PRHUrY/AT67RHHy471AfvCe/Eja3Z8A+ZT+SNIfSdv7EyCf0h9J2/sTIJ/Y358A+SjQnwD5SIAk6Y+kRH8C5D39kRTpT4C8M5SMAimW6k+AvBkpavCnC5ATgv0JkGh/MmTs139tv13OzoCc0USaApTgkxruQ4AUGs+jLUAFPiH9kTQRhwApM9PGjR/WH8P0R9JUGzd+VH+MKulv5nn6Y/jP3179qOlH6u/ZzbZxUI3+OEV/JE23cT8a/XGK/oiabuN+MwLklOX+Vh+rv6emP5JW+1t+rPye2nwbd2PRH6c0taE/TrlUf03vyHXpj6RL9SfAp9PTxmR++ns6+iNJfyTpj6SWNmbz09/TaWhjuj79PZ/6Nhby09/TKW9jJT/9PR39kaQ/korbWKpPf8+nto3F/PT3dGrj0B9j9EdSaRyr+env6eiPpMI8luvT3/Mp66OgPv09n6o+SvLT39PRH0k1gdTUp7/nU1FIVX36ez76I6kgkbr89Pd09EfSYiI/flt+zFuKpDA99T2n2oRWpCdBQrq6j9KTICFd3UfpSZCQru6j9CRISFf3QXoQRKSz+yA9CCLS2X2QHgQR6ew+SA+CiHR2H6QHQUQ6u/fScyAj3d176TmQke7uvfQcyEh39056DISkw/tZegjEpNP7SXoG5KTbe5HfU0vH96K/p5aO70V/T016JAmPJPWRpD6SxEeS9khSHknSI0l9JKmPJPWRpD2S9EeS/EjSH0nyI0l9JOmPJPmRpD+S5EeQ/EiSH0nyI0l/JOmPpOL+BMgQ/ZGkP5L0R1J1fwJkhP5IKu9PgAzQH0n6I0l/JNX3J0DOa+hPgJymP5I6+hMgpwmQpJb+BMhJ+iOppz8Bco7+SGrqT4Cc0tWfAjmjrz8Bckx/JOmPpMb+BMihzv4UyJHe/hTIffojSX8kdfcnQO7RH0n6I6m9PwFyR39/AuS2Df0JkJt29CdAbtnSnwC5QX8k6Y+kPf0JkNdt6k+BvGpbfwLkFfojSX8k7etPgHxNfyTpj6SN/QmQr+iPpJ39CZAvbe1PgXxBfyRt7k+AfEZ/JO3uT4B8ant/CuQTgf4EyHuJ/PTHe/ojSoAk6Y+kSH8C5B39kaQ/kjL9KZBfpPoTID+J9SdA3uiPLP2RlOtPgCTzEyDZ/gT49LL9CfDZ6Y8o/ZGU7U+Az05/JOmPpHB/Anxy+iMp3Z8Cn1u6vhcBPrV0fC/6e2rp+F7099TS8f0kPQNy0u39LD0EYtLpvZMeAyHp8N5Jj4GQdHjvpedARrq799JzICPd3QfpQRCRzu6D9CCISGf3QXoQRKSz+yg9CRLS1X2UngQJ6eo+lZ4F+6Wb+0x6GGyXTu5z6WmwW7q4z6WnwW7p4j6Xnga7pYv7QnocbJYO7ivpgbBVOrevpSfCTunavpaeCDula/taeiLslK7tFemRsFE6tlekR8JG6dhekR4JG6Vje016JuyTbu016ZmwT7q1G9JjYZN0aDekx8Im6dBuSI+FTdKh3ZKeC3ukO7slPRf2SHd2U3owbJHO7I70aNggHdkd6dGwQTqye9KzoV+6sXvSs6FfurG70sOhXTqxu9LDoV06sfvS06FburD70tOhW7qwA+nx0Cwd2KH0gGiVzutYekJ0Std1LD0hOqXrOpaeEJ3SdZ2QHhGN0nGdkZ4RfdJtnZOeEl3SZZ2TnhJd0mWdk54SXdJlnZOeEl3SZZ2UHhNN0mGdlB4TTdJhnZWeEz3SXZ2VnhM90l2dlh4ULdJZDUiPigbpqAakR0WDdFQD0qOiQTqqEelZUS/d1Ij0rKiXbmpIeliUSyc1KD0uiqWDGpQeF8XSQY1Kz4ta6Z5GpedFrXRPw9IDo1Q6p2HpgVEqndOw9MAolc5pXHpiVErXNCE9MgqlY5qSHhpl0ilNSQ+NMumU5qSnRpV0SXPSU6NKuqQ56alRJV3SpPTYKJIOaVp6cJRIZzQtPThKpDOal54cFdIVzUtPjgrpihakR0eBdEQL0qOjQDqiBenRUSAd0Yr07FiXbmhFenasSze0KD0+FqUDWpQeH4vSAS1Kj49F6YAWpcfHonRAq9LzY026n3XpCbIiXU+B9AhZkI6nQnqGzEu3UyE9Qxak4ymQHiEL0vFUSM+Qeel2KqRnyLx0OxXSM2Reup0S6SEyr6OG+mceHsmjWt380PpLanv9HjyurSsvLm/tMlzCzlXrjxu2bFh/JLX0J0BO0h9JPf0JkJP0R5L+CGn805H0q3FxzX8wl349Lk5/pMmPNPlxPV92dCI51QEAAABX9/9QSwMEFAAAAAgAAAAhANCSJVp4FAAAgJcJABAAFABncm91bmRfdHJ1dGgubnB5AQAQAICXCQAAAAAAeBQAAAAAAADt272OtNl5ntHp1Ecx2dgAA1EgDNgH4MyCEgWOBNqi4cAwBVJQIukodMLm/HdXd1W9P3vvq6p7rYwE1O/u+7mG0lDf/Pvf/N1//9v/8fLNP3/zL9/9wx/+/L/+9N1//fa7f/2fv/3uN99+97//+Kd/+tPv/9/f//FP//CH7//9//b7//vnP/zl3//z//n9P/7hL//6P/71b7793X/57W++/c+/+6v/9Jtv/+3b/f7DNwAAAAAAAAAAAAAAAAAAAAAAFF7qB/CVveiPzMuL/si86I/Oi/7IvLzoj8yL/ui86I/My4v+yLzoj86L/si8vOiPzGV++mOdd/Xpj3U+yE9/rPJRfvpjjQ/r0x9rXMlPf6xwLT/9Md/V+vTHfDfy0x+T3apPf0x2Oz/9MdOd+vTHTHfz0x/z3M9Pf8yyoT79Mcum/PTHFNvq0x9TbM1Pf4y3uT79Md6O/PTHaHvy0x9j7apPf4y1Mz/9MdDe+vTHQPvz0x+jHKhPf4xyKD/9Mcax/PTHCAfr0x8jHM5Pf5x2vD79cdqZ/PTHOafqkx/nnMtPf5xxsj79ccbp/PTHcefz0x9HDahPfxw1JD/9cciY+vTHIaPy0x/7DatPf+w3MD/9sdPI+vTHTmPz0x97DK5Pf+wxPD/9sdn4+vTHZjPy0x/bTKlPf2ykP0KT8tMfm+iP0Kz89McW+iM0LT/9sYH+CM3LT3/cpz9CE/PTH3fpj5L+CM3MT3/coz9CU/PTH3foj9Dc/PTHbfojNDk//XGT/ijpj9Ds/PTHLfojND0//XGD/ijpj5L+KOmPkv4o6Y+S/kjpj5L+KOmPkv4o6Y+S/ijpj5L+KOmPkv5I6Y+S/ijpj5L+KOmPkv4o6Y+U/ijpj5T+KOmPlP4o6Y+S/kjpj5L+SOmPkv4o6Y+U/ijpj5T+KOmPkv5I6Y+S/kjpj5L+SOmPkvwo6Y+U/ijpj5T+KOmPkv5I6Y+S/kjpj5L+KOmPlP4o6Y+U/ijpj5L+SOmPkv5I6Y+S/ijpj5T+KOmPlP4o6Y+S/kjpj5L+SOmPkv4o6Y+U/ijpj5T+KOmPkv5I6Y+S/kjpj5L+KOmPlP4o6Y+U/ijpj5L+SOmPkv5I6Y+S/ijpj5T+KOmPlP4o6Y+S/kjpj5L+SOmPkv5I6Y+S/ijpj5T+KOmPkv5I6Y+S/ijpj5T+KOmPkv5I6Y+S/ijpj5T+KOmPkv5I6Y+S/ijpj5T+KOmPlP5I6Y+U/kjpj5T+aMmPlP5I6Y+U/mjpj5b+aOmPlv5I6Y+W/mjpj5b+aOmPlv5o6Y+W/oj4v/9I3Ph7X/0xz53w9McU27rTH4PtC09/DHIoPP1x1pnw9MdRA7rTH/sNDE9/bDc+PP2xwbTw9Mcts8PTHx9Z1J3+eGtxePrje012+vvC4uTeqLdgjbqza+pdWKPu7Jp6F9aoO7um3oUl6syuqodhiTqzq+phWKLO7Kp6GFaoK7uuXoYV6squq5dhhbqy6+plWKCO7IZ6GhaoI7uhnoYF6shuqKdhvrqxW+ptmK9u7JZ6G3408xB1Y7dM/LXZbuYl6sRumvZbs8PUU8SF3Tbrl2aPqbdoA7tj0u/MHlOPUdZ135RfmX2mXmNbBPMKu/tpYnOvsfX8k0O7/wAas8+x+e5LgtvwDpaafY8dF1/X3cTfl13mX2TPD16cn/5y80+y7yfq70t5vKPo7wt5yLPo78t40Lvo72t42Mvo70t43NPo7wt45OMsyU9/pcc+z/z44l/wy3vw+0yPL/79vrqHv9Dc9Orf7st7ghNNba/+5b66hz/SxO7qX409532Y9+19+N3/yeY345udV3yE9x14/r0fqb/Orv7WX2rL93e+9QF+K36xs7+1p9rx7e0vrH8pXtvb38pjTfls/hcVr+zPb921pnwz/kuKt470t+he6/LTX+ZYfwsONul7+nssB/uberKJH8v+cuJDh/ObdLTJn1r5q7DBmf4GX23Bhxb9Jmx2qr/5XejvkzvX35C7rfnK7S+N+gI7nezv9OFWfGPDt0Z9gn3O5nfycgs+se1roz7BPuf7O3G72T9/x+dGfYN9BuR39HiTf/y+7w36BjudTe/o9ab+8AMfHPMRdjoT3YnzTfzRB7845ivsdLy4E/eb95OPf3HEV9jtWGxnDzjr5x7/3oivsN/R2s7dcMoPPfW1AZ/hgIOpnT3ihB957lPnv8MBB0M7f8XhPzD/EAccDe38GddUMeHhDHS0s/N3XBLFhHcz0tHMBhxyfhRTns1ABxsbc8rJTcx5NCMdLGzQLacmMenNjHS0sEHHnFfEtCcz0tHAxl1zThAzX8w4JxIbeM7RNUx/MGOcrWzgPcelsOa9DDCktHEXHdLBstdy2sDaHuWkT/XYr25kbY9x02d665c3MLUHuekzvZVPF+ATPZUfDWvtAa76PC/llVGx5Wd9modyYVRu6Vmf5qF8ZFBx2V2f5Z1cMSa46rBP8kyuGpJbddnneCW3jKgtuuxzvJKbBsQWnfYpHsk958/Y3PYZ3sh9A+5YHPcJnsgmIy65/LhP8ES2GXLKxdd9/Bey2Zhjrjzvwz+QHQZdc919H/197DPqnqvu++jvY59R91x04Ad/HrsNu+iKCz/26zhg3Ennn/ihH8cxA486+cQP/TgOGnnVqTd+5Ldx3NC7zjvyAz+NM8YedtaVH/dlnDT4tFOu/Lgv46zRt51w5od9GAMMv+7wOz/quxhm+IkH3vlR38Vwj3jo0W8a9CzmeahLj3vMyFexwCOcesAbxj+KpcpTD4tu5KMoFLceXN6QN9FaeWz9cWnhtafkp78npz9Ky849pT79Pb1F954S34n38CAWHXxGeyeew8NYcvAp7R1/Dg9kxcWntHf4NTySBSefUd7hx/Bg5t98RnlH38KjmX7zGeEdfQuPZ/LRZ3R38Ck8oslXn5DdwZfwmKaefUJ1xx7Cw5p59hnZHXoID2vi3SdEd+gdPLJ5h58Q3ZFn8NCmXX58ckdewaObdfkJzR14BQ9vzuknJLf/ETyBObcfX9z+N/AUphx/fHG7n8BzmHH88cHtfgLPYsL1xwe39wU8jQnnH97b3gfwRMbff3hvO7/PMxl+/+G57fw+z2V0AMNz2/d5nszoAkbXtu/rPJ2xBQyvbdfXeT5jExgd266P84yGNjA6tj3f5ikNjWBwa3s+zZMaGMHo1nZ8mqc1roLRrW3/Ms9rXAaDU9v+YZ7ZqA4Gl7b5uzy3UR0MLm3zd3lyg0IYXNrWz/LsxpQwtrONH+UzGNLC2NC2fZNPYUgLY0Pb9k0+hxExjA1t0yf5JAbUMDSzTV/k8zifw9DMtnyQT+R8DkMz2/JBPpOzPQytbMP3+FzOBjEysg2f47M5WcTIyO5/jU/nZBEjI7v/NT6fU0mMbOzux/iMTjUxMLG73+JzOhPFwMTufYpP6kQUAwu79yk+reNVDCzszpf4xJZldFi9EDPVdd1XL8RUdV531QMxVZ3XXfVAzFX3dU+9D3PVfd1T78NkdWB31PMwWR3YHfU8zFYXdlu9DrPVhd1Wr8N0dWI31eMwX93YLfU2zFc3dku9DQvUkd1QT8MCdWQ31NOwQl3ZdfUyLFFndlU9DEvUmV1VD8MadWfX1LuwRt3ZNfUuLFKHdkU9C6vUpX2sXoVV6tI+Vq/CMnVqH6pHYZk6tQ/Vo7BO3dpH6k1YqI7tA/UkLFTH9oF6Elaqa3uvXoSV6treqxdhqTq3d+pBWKvu7VK9B2vVvV2q92CxOrgL9RwsVgd3oZ6D1eri3qrXYLk6uTfqMViuTu6NegzWq5t7rd6CRJ3dL+oheBz64+Hoj4ejPx6O/ng4+uPh6A8AAKgc+m9u/O0Lg+iP0vH+BMh5+qN0oj8BctqZ/gTIWfqjdKo/AXKS/iid60+AnHOyPwFyiv4one1PgJyhP0qn+xMgJ5zvT4Acpz9KiqOkP0r6o6Q/SvKjpD9K+qOkP0r6o6Q/SvqjpD9K+qMkP0r6o6Q/SvqjJD9K+qMkP0r6oyQ/SvqjJD9K+qMkP0r6oyQ/SvqjJD9K+qMkP0r6oyQ/SvqjJD9K+qMkP0r6oyQ/SvqjJD9K/vOP0sb+BMgU+qO0tT8BMsPm/gTIBPqjpD9K6qMkP0ryoyQ/SuqjJD9K+qMkP0r6oyQ/SvqjJD9K+qMkP0r6oyQ/SvqjJD9K+qMkP0r6oyQ/SvqjJD9K+qMkP0ryo6Q9StqjpD1K6qMkP0ryoyQ/SvKjJD9K8qMkP0ryo+R//1Ly9x+U/Pd/lPz/Pyj58weU/PErSv7wKSV/+pmSP39PyT98ROmyPwWy0vv+BMhCAqT0QX8KZJkP+xMgi3zcnwJZ41p/AmSFq/0pkAVu9CdAprvVnwKZ7XZ/AmSuO/0pkKnu9idAJrrfnwKZZ0t/AmSWTf0JkEm29adA5tjanwCZYXN/CmSCHf0JkOH29KdARtvXnwAZa2d/CmSo3f0JkIH296dAxjnSnwAZ5VB/CmSQg/0JkCGO9qdARjjenwI570x/AuSsU/1JkJPO9idBzhjQnwQ5bEx/EuSYYf1JkANG9qdB9hrdnwTZY0J/EmSzOf1JkG2m9adBNpjanwS5Y3Z/EuSWBf1pkKsW9SdBPrSuPw3y3tr+JMhby/vTIK8k/WmQn2T9aZBv4v4k+OXV/Qnwa6vzE+DXtro1AfLa2v5ePvxqOgCpxf1pjTf0R2l1fwLkNf1RWt6fAHlFf5T0R0l/lPRHaX1/AuRX+qOkP0r6o6Q/SkF/AuQXRX8C5GdJfwLkJ01/AuRHUX8K5AdZfwLkm/Sfv6x/dR5A2J8Cif/58/q3p9b2J8CvLu5PgV9cnd+LAr+0Or4f1COQqdP7ST0DkTq8n9U70Ki7+0U9BIk6u1/VS1Coq3ut3oL16ubeqMdguTq5t+o1WK0u7kI9B4vVwV2o52CxOrgL9RwsVgd3oZ6DxergLtRzsFgd3IV6Dharg7tQz8FidXCX6j1Yq+7tUr0Ha9W9Xar3YK26t0v1HqxV93ap3oO16t4u1XuwVt3bO/UgLFXn9k49CEvVub1TD8JSdW7v1IOwVJ3bO/UgLFXn9k49CEvVuV2q92CturdL9R6sVfd2qd6DtereLtV7sFbd26V6D9aqe7tQz8FidXAX6jlYrA7uQj0Hi9XBXajnYLE6uLfqNVitLu6teg1Wq4t7q16D1eri3qrXYLW6uLfqNVitLu6NegyWq5N7ox6D5erk3qjHYLk6uTfqMViuTu61egvWq5t7rd6C9ermXqu3YL26udfqLVivbu61egvWq5t7pZ6CQB3dK/UUBOroXqmnIFBH96t6CQp1dT+rd6BRd/eTegYidXg/qlegUpf3g3oEMnV6L+r70ur45Pe11fXJ72uTHyX1UZIdJc1REh0l+VHSHyX5UZIfJflR0h8l+VGSHyX5UdIfJflRkh8l/VGSHyX5UZIfJf1Rkh8l+VGSHyX9UdIfpcH9CZBd9EdpdH8CZI/h/QmQHfRHaXx/AmS7Cf0JkM30R2lGfwJkqyn9CZCN9EdpTn8CZJtJ/QmQTfRHaVZ/AmSLaf0JkA30R2lefwLkvon9CZC79EdpZn8C5J6p/QmQO/RHaW5/AuS2yf0JkJv0R2l2fwLklun9CZAb9Edpfn8C5LoF/QmQq/RHaUV/AuSaJf0JkCv0R2lNfwLkY4v6EyAf0h+lVf0JkI8s60+AfEB/lNb1J0DeW9ifAHlHf5RW9idALi3tT4Bc0B+ltf0JkLcW9ydA3tAfpdX9CZDXlvcnQF7RH6X1/QmQXwX9CZBf6I9S0Z8A+VnSnwD5if4oNf0JkB9F/QmQH+iPUtWfAPle1p8A+UZ/tLr+BEjanwDRHy0BUir7EyD6o5T2J8Avr+1PgF+d/ijF/Qnwi6v7E+DXVuenv6+tzk+AX1td34sAv7Q6vhf9fWl1fN+rN6BTt/eDegQydXo/qEcgU6f3o3oFKnV5P6lnIFKH95N6BiJ1eD+rd6BRd/eLeggSdXa/qIcgUWf3q3oJCnV1r9RTEKije6WegkAd3Wv1FqxXN/dGPQbL1cm9UY/BcnVyb9VrsFpd3IV6Dharg7tQz8FidXCX6j1Yq+7tnXoQlqpze6cehKXq3N6rF2GlurYP1JOwUB3bB+pJWKiO7SP1JqxTt/ahehSWqVP7UD0Ky9SpfaxehVXq0q6oZ2GROrQr6llYpA7tmnoX1qg7u6oehiXqzK6qh2GJOrPr6mVYoa7shnoaFqgju6GehgXqyG6pt2G+urGb6nGYrk7spnocpqsTu61eh9nqwu6o52GyOrA76nmYrA7snnof5qr7uqseiKnqvO6qB2KqOq/76oWYqa5rg3oiJqrj2qCeiInquLaoN2Keuq1N6pGYpk5rk3okpqnT2qZeiVnqsjaqZ2KSOqyN6pmYpA5rq3on5qi72qweiinqrDarh2KKOqvt6qWYoa5qh3oqJqij2qGeignqqPaot2K8uqld6rEYrk5ql3oshquT2qdei9Hqonaq52KwOqid6rkYrA5qr3ovxqp72q0ejKHqnHarB2OoOqf96sUYqa7pgHoyBqpjOqCejIHqmI6oN2OcuqVD6tEYpk7pkHo0hqlTOqZejVHqkg6qZ2OQOqSD6tkYpA7pqHo3xqg7OqwejiHqjA6rh2OIOqPj6uUYoa7ohHo6BqgjOqGejgHqiM6ot+O8uqFT6vE4rU7olHo8TqsTOqdej7Pqgk6q5+OkOqCT6vk4qQ7orHo/zqn7Oa0ekFPqfE6rB+SUOp/z6gU5o65ngHpCTqjjGaCekBPqeK7b/rx4Qk6YHNER+1+XrcdZU0va6fDr1s/GIFOD2ur06xYOxlAzszrdzbAfxKOa19aAWsb/RB7MvMgGVDLxR/MYprU2Io4V3yA1K7kRTaz8Fo0HzO7g20Z9lYe28OBR9vC9nfnpj6H29idARtIfod356Y+B9EdJfpT0R2h3fvpjIP1Rkh8l/VHSHyX5UdIfJf1Rkh8l/VHSHyX9UZLfF/D/AVBLAQIUAxQAAAAIAAAAIQDxW9NzhhQAAICXCQAOAAAAAAAAAAAAAACAAQAAAABwcmVkaWN0aW9uLm5weVBLAQIUAxQAAAAIAAAAIQDQkiVaeBQAAICXCQAQAAAAAAAAAAAAAACAAcYUAABncm91bmRfdHJ1dGgubnB5UEsFBgAAAAACAAIAegAAAIApAAAAAA==', 'reference/sample-1.npz': 'UEsDBBQAAAAIAAAAIQDioxAe6hIAAIBSCAAOABQAcHJlZGljdGlvbi5ucHkBABAAgFIIAAAAAADqEgAAAAAAAO3dsa5k2VWH8aqUp+isQZoALMsBD0AGIiEgQoM9FgHC1gwiwX4KXhj3dNe9NXWrztnn7LXWt88+3y+wrzvo3vVfn26PW+Px//3Tv/zjP//r9fI/l//9/Lsffvrtj5///tPnP/37333+7tPn3//hx//+8fv/+rc//Pi7H778+D98/58//fCXH//pP77/4w9/+c9//avvPv36V7/57tNvfv23f/Pdpz9/2u6vLpIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIO7Hq90k/QaV1v6IfolK4GKJT9CeV3QJGu/h4s0NUABbI/ga5XAxTn+oh+kM7kQ372pzof8zNA1bE/sexPKAMUyd+BRfIvAUV61p8BqsrT/gxQNZ7nZ3+q8aI/A1QJ+xPpVX8GqAov+7NAFbA/kRb6M0Clsz+R7E8k+xNpqT8DVDr7E2jx+58BKpn9ibTcnwEqmQGKZH8i2Z9AK/nZn3IZoEj2J5L9ibTWnwEqk/2JZH8CreZngUrU0p8BKov9idTUnwEqif0J1RagBSpHY38GqBSt/RmgEjTnZ3/KYIAitfdngIpnfyJt6M8AFW5LfwaoaJv6M0AF29afASqW/YlkfyLZn0j2J9LG/gxQoexPJPsTaWt/BqhIm/uzQAXa0Z8BKoz9iWR/ItmfSHv6M0BFsT+RdvVngApifwLty88AFcQARdrbnwEqgv2JZH8i2Z9I9ifQ7vwMUAE6+jNAdbM/kXr6M0D1sj+hDFAk+xOpqz8DVCf7E6gvP/tTH/sTyf5Esj+R7E8k+xPK/kTyG6BI9ieS/YlkfyLZn0j2J1BnfvanLvYnlP2JZH8i2Z9I9ieS/YlkfyJ19meA6tLbnwGqQ3d+BqgOAf0ZoHaL6M8AtVtIgDaonaL6M0DtEdafBWqHwP5sUJsF92eE2iSjPyNUq7T+LFANEvszQK3JzM/+tCo1wC8JWqEWJPf3LULpuYr+bFCvVPVngHqmrD8D1BN1/RmgPrI/kQr7M0B9UNnfz38c+BaiRaq6vw/ojy+Y/YkE92eAJ0f3Z4DnRud3tcBTo+O7oXcQg+7uht5BDLq7G3oHMejubugdxKC7e0MPIQSd3Rt6CCHo7N7RS4hAV/eOXkIEurp79BaqRzf3S/QaqkYX90v0GqpGF/eAnkPF6OAe0HOoGB3cA3oOFaODe0DPoWJ0cI/oPVSL7u0jehFVomt7gp5EhejYnqAnUSE6tmfoTVSHbu0ZehPVoVt7ht5EdejWnqJHURk6tRfoWVSEDu0lehiVoDNbQE+jAnRkC+hpVICObAm9jfLRjS2ht1E+urFl9DrKRhe2ht5Huei+1tD7KBfd1wp6HiWjA1tHL6RMdF0N6ImUiI6rBb2R8tBtNaFHUho6rUb0TEpCh9WK3kk56K7a0UspA13VFvRWikc3tQE9lRLQUbWid1IOuqtW9E7KQXfVjl5KGeiqtqC3Ujy6qQ3oqZSAjmoDeioloKPagJ5KCeiotqC3Ujy6qS3orRSPbmoLeivFo5tqRy+lDHRVzeihlILOqhk9lFLQWTWjh1IKOqtm9FBKQWfVjB5KKeismtFDKQWdVTN6KKWgs2pGD6UUdFat6J2Ugs6qFb2TctBdtaJ3Ug66q1b0TspBd9WInklJ6LAa0TMpCR1WG3olZaHLakKPpDR0Wi3ojZSHbqsFvZHy0G21oDdSHrqtBvRESkTH1YCeSInouBrQEykRHVcDeiIlouNqQE+kRHRcDeiJlIiOqwE9kRLRcTWgJ1IiOq4G9ETKRNe1jl5Imei61tELKRNd1zp6IWWi61pHL6RMdF3r6IWUia5rHb2QMtF1NaAnUiI6rgb0REpEx9WAnkiJ6Lga0BMpER1XA3oiJaLjakKPpDR0Wm3olZSFLqsJPZKy0GW1oVdSGjqtNvRKykKX1YieSUnosBrRMykJHVYjeiYlocNqRM+kJHRYreidlIPuqhk9lFLQWbWjl1IGuqp29FLKQFfVjB5KKeismtFDKQWdVSt6J+Wgu2pF76QcdFet6J2Ug+6qFb2TctBdNaJnUhI6rEb0TEpCh9WGXklJ6LAa0TMpCR1WI3omJaHDakOvpCx0WU3okZSGTqsFvZHy0G21oDdSHrqtFvRGykO31YCeSInouNbRCykTXdcqeiClovNaQ++jXHRfa+h9lIvuawU9j7LRhS2j11E2urBF9DhKRye2iB5H6ejEFtHjKB2d2CJ6HKWjE1tCb6N8dGOv0cuoAl3Za/QyqkBX9hI9jErQmb1ED6MSdGav0LuoCB3aC/QsqkF39gq9i2rQnb1Az6IidGgv0LOoCB3aU/QoKkOn9gy9ierQrT1Db6JCdGxP0JOoEB3bE/QkqkTX9gE9iGrRvT2i91A9url79BZi0N3d0DsIZn4agP2JZn9imZ9o5idaTmvmpy2C83v8ScFPpqOIzu/9J2U+j47mGhHh489pftokNL+L+WmfiPqkbuYnlv2JZn6i2Z9Y5qchWJ9w5iec9UmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJB3X9Qv6ETqh6yP6QTqPD/EZocq8rO8WoR0qx1p6fjdUlm3tGaFC7czPABVgd302qF6d8VmgdotozwS1U2B+FqjNYvuzQG0SnZ/9qV14fQaoZhn1GaDaJNVngmqQWZ/9aUVufgaop7Kzsz89VRaeDeoDoj770zdMfgaor+xPJPsTiMrPAPWF/YnE9WeCgvszwLNj87O/s4P7s8CTo/O7WuCZ0e39jB5BFLq8r+gVRKHL+4aeQRA6vG/oGcSgu7tDTyEAHd09egvVo5u7R2+henRz9+gtVI5O7pfoNVSNLu4RvYcq0bU9QU+iOnRrz9CbqA7d2lP0KCpDp/YUPYrK0Kk9RY+iMnRqT9GjqApd2gv0LCpCh/YKvYtq0J29Ri+jCnRlC+hpVICObAE9jfLRjS2ht1E+urFF9DjKRhe2jF5HyejAVtEDKRNd1zp6ISWi42pBb6Q8dFst6I2Uh26rBb2R8tBttaA3Uh66rRb0RspDt9WEHklp6LSa0CMpC11WG3olZaHLakOvpCx0WW3olZSFLqsNvZJy0F21ondSDrqrVvROSkFn1Y5eShnoqtrRSykBHdUG9FRKQEe1AT2V4tFNbUKPpXB0UpvQYykaXdRG9FwKRge1Fb2XQtE5bUYPpkh0TdvRiykSXdN29GKKRNe0Hb2YAtEx7UFvpjh0S3vQmykO3dIu9GgKQ6e0D72agtAh7UXvphh0R7vRwykAHVEHejr1oxvqQW+nfnRDPejt1I9uqAs9nrrRCXWhx1MvuqA+9HrqRifUhR5PveiCOtHzqRMdUCd6PnWiA+pF76c+dD+d6PnUiQ6oD72eetEF9aHXUy+6oE70fOpD99OL3k996H560fupEx1QJ3o+daID6kTPpz50P93oAdWFzqcbPaC60Pl0owdUFzqfXvR+6kP304veTx3oeALQE6oDHU8AekJ1oOMJQE+oDnQ8AegJ1YGOJwA9ofaj24lAb6j96HYi0BtqNzqdEPSI2osuJwQ9onaiwwlCz6id6HCC0DNqJzqcIPSM2ofuJgq9o/ahu4lC76h96G6C0DNqH7qbMPSQ2oXOJgw9pPagqwlDD6k96GrC0ENqD7qaMPSQ2oXOJg69pHagowlET6kd6Gji0EtqO7qZSPSW2oxOJhK9pTajkwlFj6mt6GJi0WtqK7qYWPSa2ogOJha9pjaigwlGz6lN6FzC0YNqC7qWePSi2oKuJR69qDagY4lHL6pmdCop6FHVjE4lBT2qWtGl5KBXVTM6lRT0qNqH7iYKvaN2osOJQa+ovehyYtArarNvN6PLCcJuqb3obqLQO2oPupo49JLagY4mEr2lNqOTCUWPqe3oZiLRW2oXOpsw9JDah+4mDr2k9qCriXQxwsOhm4lG76lt6F7C0YNqEzqXBPSkake3kooeV2voQgrQE2sBHUcRemY9R3dRhh5aT9FZlKGH1jN0FZXorfUBnUQ9enHdo2sA0JPrDh0Dgd5c7+gWEPToekOnwKBX1w1dAoYe/txu89MVcND5T+92AboCEH2CE6NPPwT6CCdFnx31/unpM5wWeH3U5dv/MOS9PBsE0Bmgvn7+byVe/SZYjg5gHF+noO9xLvDJh2J/5eCLj+XbGvRNzoM993DsrxR77IHRhzkH+sojo28zPfrAQ/L33yrsncfyi//Wa3wV4IsP5eKfOpejbz4U/9SvGnzwQdFXOQn6zOOiL3MK9JGHRh9nfvSFR0ffZ3L0ecdHX2hq9HGPgL7RxOjTHgJ9pHnRlx3d5fa34ysDfd7RfR0JPtLE6PsO7+eR6CtNi77uEfj7bxb6skdB32lS9FkPgz7UnOirHgd9qTnRVz0O+lJToo96IPSpJkSf9EjoW82Hvuj4/Pvv09CnHdf7OBf7y0Oe+Cje/8Gvikbf9kDoU82Ivulx0JeaE33VEX0d5mEc9kzTom48svtxLvc/ZoThyDsP6+dhPn77u/274jAHPjD6YHOhrzmyh3Fuv/1aYBzmsEN72+Ty4s+f/RYYhrnw0C4P1V3flnofjTnWhLgzj+vy+NVtqbvRsINNBjnw0bxNdXn8Sp3o047o5fe/691m4M0mUn/c8V3W/vrv+v6D6gOdeGhvs9wGepvqfjTmXrOhbjyyy+P3v6e//xpgCOLAg7s8fv977+/uK/sLQRx4aLdZFkeruMxJ0PceDH2O06EPPhj6HKdDH3ww9DnOh774WOhrnA998aHQxzgh+uRDoY9xRvTNR0Lf4ozomw+FPsYJ0ScfC32N86EvPhb6Gh+N+ao49MXHQt/i7YulB+LvjFR22kM4zinQl0ZKuuQxHecS6EsjJV3yoA5zCfKhsZIueUyHugP42FDxZzyuo12AeW+suOsd36bdRrhA2BtAYdebwNH2J94bLep2Mzjc/sSDgwWdbgqbZhth/Ig30CION4vVre5GG2L7iEfA+q82j/Wt3kcbYvqARwygf4dZrA11t9gQuwc8YgD9O8yifbJhdo94CCxkhymsDvW+2Ciz97+DFbPCJNanevyKn73vHbi4ISbQMNXbV6Os3vWOAQROcXwNS92+GmXznneMIHKLw2tYarTNI94DipxiAstbjbh4yJsowVtM4HCb5z84S/AQczjc6ukPThO9xByOtnr2e1NFjzGDg02e/Nxc0WNM4ViT5742U/QS8zjS6plvzRW9xEQONHreU9OFbzGP42ye9tJ84VvM4zibp700W/gSUznM5jkPTXY1vxXxkx/loRXu/89G9Uz84sd4Zw2//60JX/wg76yStccsjrJ39DtrZK0xj6MsHv3OElljzOQgi8c+s0jaGjOJ+qds5C4e9sJKaWtMZ/jFox5YKm2N6Qy/eNQDS6WtMZ3BF496Xq2kMSY0+OJRz6uUNMWcxh496nWlcqaY1dCbxzyuWtIYcxp68pjH1UqaYlojTx7wtnJpY0xq5LVjiiiVO8iERp47JolSuYNMaOS5Y5IolTvIjAaeO6iJMrlrzGngxWOiqJM6xqwGXjwoizK5a0xq3MWDqqiTusasxl08qIo6qWvMatjFg6IolLnGvIYdPKyLKrlzzGrYrWPjKFCwyYTGHTo8kGRFs0xm2KHD+8hWM8tshh06vI9kNatMZ9ih4wvJ8eWhRZNMqXnnUR+GKV9kTqOOnRtPhPpNZjTq1rntBKmfZTqDDp3bTZTyWeYz6s7J5cQAdpnO8r7YxFUJdaHGmcqg+5ZF1IHcZxqD7lsWUQdyn2mMuW9ZQ3vcHgjuM5OlmUd70wDIWaY3yrp0ZAvoaVSAjmwBPY0K0JEtobdRPrqxJfQ2ykc3toTeRvnoxpbQ2ygf3dgSehvloxtbQm+jfHRjq+iBlIiOqxE9k3LQXbWid1IOuqtm9FBKQWfVjl5KGeiqNqMHUyg6pz3ozRSHbqkHvZ360Q11ocdTNzqhTvR86kQHFIYeUnvQ1YSjB9UWdC0Z6E3Vjm4lCT2rGtGhZKF3VRu6k2T0vFpBB1KJ3lof0U3UoxfXPboGBr26bugSQPT0ogsYAH2Ck6LPPiD6JOdBX3pk9G1mR9/3COgbTYw+7YFcDTEefdQjom82FfqYh0UfbhL0GY+Nvt7x0Rc8PPqAB0efbw70FY+LvtxE6FPi9i0BXmxqSUceTMAIJdc4rewAOFE75OyuO3VRFImcIH5uPVEXR7K+j/9xh9iZ1aS0mFA9n/jVJ4/bVdvUttOh/5MufeTun127VZe0WcHnDf4ltEV1TxtVfNzoX0MbVUe17pJVxdNfTEOpjg0KwfIO6uvhGmL6+i/waxsc4ImSJEmSzuX/AVBLAwQUAAAACAAAACEAbbIfZ8gRAACAUggAEAAUAGdyb3VuZF90cnV0aC5ucHkBABAAgFIIAAAAAADIEQAAAAAAAO3Ssa5kSVZG4UqXpyivQGoDRqMxeAA8EA4GFmqYRhiIGXUjHOApeGGmpzWlqrqZJyJO7n+vjHPX593j3J1/rP/7u3/427//x9uH//rw359++8NP//Ljp7/++Ol//vmvPn338dO//u7H//zx+//4p9/9+Nsffv7+N9//+08//OH7T//2/e9/+MPff/6r7z7++le/+e7jb379l3/x3cf//bjuzz5IkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkrZwu9EX6D273SxQnJsBCnS7WaA4t5sFinO7WaA4t5sFinMzQIHsT6SbAQp0M0CBvu3PANXpTX8GqEZv+zNA9bE/ke70Z4Bqc68/C1SX+/0ZoHo86M8A1eJRfwaoDg/7M0A1sD+R7E+kx/0ZoPLsTyT7E+mgPwNUnP2JZH8iHfVngEqzP5HsT6TD/gxQYfYnkv2JdNyfASrL/kSyP5EG/Rmgokb9GaCShv0ZoILG/RmgcuxPpIn+DFAxM/0ZoFKm+jNAhcz1Z4DKmOzPABUx258BKsH+RJruzwAVMN+fAaqe/Ym00J8Bqpz9ibTSnwGqmv2JtNSfAaqY/YlkfyKt9WeAqmV/Ii32Z4AqZX8i2Z9Iq/0ZoCrZn0jL/RmgCq33Z4CqY38inejPAFXmTH8GqCqn+jNAFbE/kc71Z4CqcbI/A1QJ+xPpbH8GqAr2J9Lp/gxQBc73Z4B6nv2J9ER/Bqin2Z9I9ifSM/0ZoJ5lfyLZn0j2J9JT/RmgnmR/ItmfSPYn0nP9GaCeY38i2Z9I9ieS/YlkfyLZn0j2J5L9iWR/ItmfSPYnkv2JZH8i2Z9I9ieS/YlkfyLZn0j2J5L9ifRkf2aop9T0Z4M6p64/C9S6yv5MUKuK+7NALSnvzwS1INGfBWpWpj8z1JxofyaoAfsTKd2fAepIvD8D1IF8fwaoxxr6M0A9ZH8idfRngHqkpT8D1AM9/Rmg7mvqzwB1V1d/Bqh72vqzQN3R2J8B6o3O/gxQ32rtzwD1jd7+DFBfa+7vdv8/c79frO7+Prd2/6vemf7+/tja/a96d9rje4yeQgA6ui/RW6gf3dxX6DHUjk7ua/Qa6kYX9w16DjWjg/sGPYea0cF9i95Dveje3qAHUSs6tzfoQdSKzu0tehF1omt7i15Eneja7qAnUSM6tjvoSdSIju0eehP1oVu7h95EfejW7qJHURs6tbvoUdSGTu0+ehV1oUu7j15FXejSHqBnURM6tAfoWdSEDu0Rehf1oDt7iB5GLejMHqKHUQs6s8foZdSBruwxehl1oCs7QE+jBnRkB+hp1ICO7Ai9jfLoxo7Q2yiPbuwQPY7i6MQO0eMojk7sGL2O0ujCjtHrKI0ubICeR2F0YAP0PAqjAxuh91EW3dcIvY+y6L6G6IEURec1RA+kKDqvMXohJdF1jdELKYmua4xeSEl0XRPoiRRExzWBnkhBdFwT6IkURMc1g95IOXRbM+iNlEO3NYPeSDl0WzPojZRDtzWFHkkxdFpT6JEUQ6c1hR5JMXRac+iVlEKXNYdeSSl0WXPolZRClzWJnkkhdFiT6JkUQoc1iZ5JIXRYk+iZFEKHNYmeSSF0WHPolZRClzWHXkkpdFlT6JEUQ6c1hR5JMXRaM+iNlEO3NYPeSDl0WzPojZRDtzWBnkhBdFwT6IkURMc1gZ5IQXRcY/RCSqLrGqMXUhJd1xi9kJLouobogRRF5zVGL6Qkuq4xeiEl0XWN0Qspia5rAj2Rgui4JtATKYiOawI9kYLouGbQGymHbmsGvZFy6LZm0Bsph25rCj2SYui0ptAjKYZOawo9kmLotObQKymFLmsOvZJS6LLm0CsphS5rEj2TQuiwZtE7KYPuaha9kzLorqbRQymCzmoaPZQi6Kzm0Uspga5qAT2VAuioVtBbqR7d1BJ6LJWjk1pCj6VydFJL6LFUjk5qDb2WqtFFraHXUjW6qDX0WqpGF7WGXkvV6KKW0GOpHJ3UEnoslaOTWkFvpXp0UyvorVSPbmoBPZUC6KgW0FMpgI5qHr2UEuiq5tFLKYGuaho9lCLorKbRQymCzmoWvZMy6K5m0Tspg+5qEj2TQuiwJtEzKYQOaw69klLosubQKymFLmsKPZJi6LSm0CMphk5rCj2SYui0ZtAbKYduawI9kYLouMbohZRE1zVED6QoOq8Reh9l0X0N0PMojA7sGL2O0ujCDtHjKI5O7BA9juLoxI7Q2yiPbuwIvY3y6MaO0Nsoj27sAD2NGtCRHaCnUQM6sgP0NGpAR3aAnkYN6MgO0NOoAR3ZY/Qy6kBX9hi9jDrQlT1GL6MOdGWP0cuoC13aXfQoakXn9gY9iPrRzX2J3kIQOrxf0CsIRednfyIjpH+6XoX9CWd/wpmfcOYnnP0JZ37C2Z9w1iec9QlnfcJZn3DmJ5r5CWd+wtmfcOYnnPkJZ3/CmZ9w5idJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJkiRJ0oZuX6Gv0btyO0Rfp0s7js8CFTQRnwkqYzo+C1S1tfgsUIVOxGeBKnK2PgvU056JzwL1nKfrs0CdVlKfBeqMsvgsUMtq67NALanPzwA1K1GfBWpOqj4L1FiyPgvUQDo/A9Rj+fosUI/01GeBuqevPgPUG635GaC+0dyfBepL7fkZoL4A9GeB+gzpzwD1CyY/C9QvsP4MUB/I/ixQaH4GKLY/C3zv6P4M8F2j6/sZvYE4dHt/RI8gCl3en9A7iEF39yf0DmLQ3X1GDyEEnd1n9BAi0NV9gZ5CADq6L9BTCEBH9yV6C7Wjk/sKPYba0cl9jV5D3ejivkavoWZ0cN+i91Avurdv0XuoF93bG/Qg6kTX9ha9iDrRtb1FL6JOdG130JOoD93aPfQm6kO3dhc9itrQqd1Fj6IudGn30auoC13affQq6kKX9gA9i3rQnT1C76IedGeP0LuoB93ZQ/Qw6kBX9hi9jDrQlT1GL6MGdGQH6GnUgI7sCL2N4ujEDtHjKI5O7BA9juLoxI7R6yiMDmyAnkdhdGAD9DzKovsaofdRFt3XCL2Poui8huiBFEXnNUQPpCg6ryF6ICXRdY3RCymJrmuMXkhBdFwT6IkURMc1gZ5IOXRbM+iNlEO3NYPeSDl0WzPojRRDpzWFHkkxdFpT6JEUQ6c1hR5JKXRZc+iVlEKXNYUeSTF0WlPokZRClzWHXkkpdFlz6JWUQpc1h15JIXRYk+iZFEKHNYdeSSl0WXPolRRChzWJnkkhdFiT6JmUQXc1i95JGXRXk+iZFEKHNYmeSRl0V7PonZRBdzWL3kkRdFbT6KEUQWc1i95JEXRW0+ihFEFnNY0eShF0VtPooZRAVzWPXkoJdFXT6KGUQFc1j15KCXRV8+ilFEBHtYCeSgF0VAvoqRRARzWPXkoBdFQL6KkUQEe1gJ5K9eimVtBbqR7d1Ap6K5Wjk1pCj6VydFIr6K1Uj25qBb2V6tFNraC3Ujk6qSX0WCpHJ7WEHkvl6KRW0FupHt3UCnorlaOTWkKPpXJ0UkvosVSOTmoJPZaq0UWtoddSNbqoJfRYKkcntYQeS+XopJbQY6kaXdQaei1Vo4taQ6+lanRRa+i1VIwOahE9l4rRQS2i51ItuqdF9FwqRge1iJ5LteieVtF7qRbd0yp6L5Wic1pGD6ZSdE7L6MFUia5pHb2YKtE1raMXUyW6pmX0YKpE17SOXkyV6JrW0YupEB3TCfRkKkTHdAI9merQLZ1Bb6Y6dEtn0JupDJ3SKfRoKkOndAa9merQLZ1Bb6YydEqn0KOpDJ3SKfRoqkKXdA69mqrQJZ1Dr6YidEgn0bOpCB3SSfRsKkKHdA69morQIZ1Ez6YidEgn0bOpBt3RWfRuqkF3dBa9m0rQGZ1F76YadEdn0bupBJ3RafRwKkFndBo9nErQGZ1F76YSdEan0cOpBJ3RafRwqkBXdB69nCrQFZ1GD6cKdEXn0cupAl3RefRyqkBXdBo9nArQET2Bnk5PoxN6Cj2enkQH9CR6Pj2FzudZ9H56Bl3P8+gFdR7dTgV6Q51Fl1OCHlEn0eEUoWfUKXQ2ZeghdQZdTR16Sa2jmylET6lldDKl6DG1iA6mGD2nltC5VKP31Aq6lnr0oppHt5JAb6pZdCkZ9KqaQ3cSQs+qOXQnKfSumkFXEkMPqxl0JTH0sJpBV5JDL6sxupEgelqN0Y0E0dNqiE4kid5WY3QjSfS2GqITSaK31RCdSBK9rYboRJLobTVEJxJFj6shOpEkelsN0Ykk0dtqiE4kid5WQ3QiUfS4GqELiaLH1RCdSBK9rcboRpLobTVGNxJFj6shOpEoelwN0YlE0eNqiE4kih5XQ3QiUfS4GqITiaLH1RCdSBQ9rsboRqLocTVEJxJFj6shOpEoelwN0YlE0eNqiE4kih5XQ3QiUfS4GqITiaLH1RCdSBa9rkboQrLodTVCF5JFr6shOpEoelwN0YlE0eNqiE4ki15XQ3QiYfS8GqADSaP31QAdSBq9r47RfcTRA+sQnUcevbAO0Xnk0QvrCF1HA3riZ23/Aw7RdXSgN37G9j9gAE6jDz30GXtfP4crgkCv/ci9w/a4/FlMByh68jfuXLXD2TV63/510Lt/9vailz+5Ustjvyp6/A/f7P/mw+vdWy76wK+OHn95ffreeoln3QY9/on16Yur1T/qTnYcH765WvGL7mXL7eGji9U+6G623J49ulrpe+5mz+XRq6sVvuaG9lyevLpc3WPuaNPdubPLFT3kpnbdnbu7Ws07bmvP2amrA2qecVt7zk5dHVDzjPvacnbo6ISaV9zXlqszR0eUPOLOdlwduTmj5A13tuHoxMkpJW+4tf1GBy6OKXnCvW23efvBSRUvuLftJm8/OKjiAXe32+bd9yZVvN/uNpu8+dyoiufb3mabN5+bVPF6F7DX5q3XRhU83SXsNXrrtUkFL3cRW43eeGzU80tcxk6rN94aVfBs17HR7H2nRhU82oVsNHvbqVHP73ApG+3edmrU8ztcyz6r91waVjHEpeyzesulYSVDXMsuqzfcGVcyxMXssnr+zriaIa5mk9XTZ+YVDXE1m6wePjOvaoir2WT17JkNypa4mj1GT17ZoW6Jq9lj89yVLSqnuJotNk8d2aN0iqvZYvPQkU1qt7iaHSaP3NileIur2WHyxI1dqre4nA0mrz+xTfkWl7PB5OUn9qkf42o2WLz6xD6BMS7n9QcvvbBTZI2ref3BKy9slZnjal5+77oDe4XmuJqX37vswF6pOS7n1fcuuq9ZbI7LefW9a+7rltvjal597pL7ugX3uJwXn7vgvHbJPS7nxdcuyKFbdpDLefG1S5JolV7kYl587JomGsUXuZrX3rqsiyYNk1zNS29dmEaHjkku55Wnrs0jrmeUq3nhocsLiepa5Wped+hAJDltq1zOq+6c6SSlb5fLec2ZU6Fk9O1yQS+4cjCVhL5hLunlVo7GEtC3zDW92MbpXMo1bnNNr7RwQy/VGte5qNcZuCWYYo3zXNWrbNtYTZ3mjS7pNdbkGnoGuZjq0B2dRe+mEnRGp9HDqQJd0Xn0cipAR/Qcej09h+6nAD2hzqPbqUGvqJPocOrQS2od3Uwxek7No1sJoWfVATqOHvTKuo/uog+9tN6im2hGz62v0T0A6Mn1GZ0Chd5dP6MrQNHjv3t0AC+BfoR3i37410K/xntDv/fLoh/mfaBfeQP0E10Y/bR7oV/rauj33Bb9cNdAv+L+6BfcG/16F0I/5Z7oV7sg+km3Qj/WldFvuwP6jd4F+pFfGf027wn91q+Gfo93in52Fr2+PrzLBOnJ9RU6h1b02LqHrqIHvbIeo9tIo/fVEJ1IDD2sJtGhBNCTag3dSyV6S51CZ1OCHlFPofN5Br2dKtE1LaCnUhJd12P0MmpF5/YZPYRgdiVJkiRJkqSX8P9QSwECFAMUAAAACAAAACEA4qMQHuoSAACAUggADgAAAAAAAAAAAAAAgAEAAAAAcHJlZGljdGlvbi5ucHlQSwECFAMUAAAACAAAACEAbbIfZ8gRAACAUggAEAAAAAAAAAAAAAAAgAEqEwAAZ3JvdW5kX3RydXRoLm5weVBLBQYAAAAAAgACAHoAAAA0JQAAAAA=', 'reference/sample-10.npz': 'UEsDBBQAAAAIAAAAIQBh4ob17ggAAIBgCQAOABQAcHJlZGljdGlvbi5ucHkBABAAgGAJAAAAAADuCAAAAAAAAO3dwY5cRxmG4ektVzG7AcmLgCKEuAB2RGxYsEKGGLFABNkoG+AquGFMPJ6ZjPt0n6rzV33dp59nNbHsU6e/euXYiuX895vf//Z3fzjcfX/3r4dv33348/uHX98//PtPP394c//wl+/e//P927//8bv33777/7f/5u3fPrz7+O0f/vr2H+8+/vNPf/Hm/utfffXm/pdff/WzN/f/uW/3kzsAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAIBdOTxLvwo35fCl9CtxE46Upz9GWkxOf4zwMaZ1zQmQKh3FCZCNtlenP7pUpqc/GpSnJ0HOG5ad/m7X86U/fTW+MwFyFw3thPQqTJHO7LT0OoyWLmyF9ESMk25rtfRQDJHOqlF6Lmqle+qSHo0y6ZS6pYejRDqjfunlqJCuaJv0emyVLqhAekI2SMdTIj0iXdLZlEkPSat0MbXSa9IiXcsY6VVZKR3KOOllb8TGndOVDFSzLydt3TkdyUBFC3OS/pYUDcwpW3dORzJQ0cKcsHnpbCJDlY3Mku1TJwMZrG5mlmyeOlrIWHUrs2jr1NlEhqobmUVbh05HMlDVxJzk978LivZlpHQkA6WnZYV0JAOlp2WFdCQDpadlhXQkA6WnZYV0JAOlp70Rfv+7oGhfTtPfgqJ9Ocmff1lUtDADpRsZKD0tK6QjGSg97dUpX+3w+f85dO7QXapc8ibUj6Y/Vhsw2uH8U5OFjFU75Q2oH+1w/qnZRkaqnXLvHlMpfuhBf6yy4qeqjocezkX9dFXBUAYpXfI2VK92/nlPV/V0aaFa6pUueSsG/Av4zHf4fFNPXyWbqVS75I0YsNrzVTzdyun7SUZTqH7J3RsxWvslzcljuPopd09/heqn3L0Rm7Vf0pw8hhuwJe3aL2lOHsPNHprjmi9pUh+jzd6ZEz7/SYQ1dzQ7lEEmL8xZK+9kQhnDjzjo72rNiGOC9Iz0SXdTJb0jnWZ0MfiMp3O4Ro/XN6WJ+kOWTuLKzKmi/pSFg7gyc7KoP0V+127Un8Q6ftacc7g2c7qoP0V/+zCnizmncH3mlFHxzHNncI22lnG8leJTvnjIzIUYaWMZi7HUnXLsIdPmYbCNsZ3qpeaYIw8avwqzVFW31Ev/MXN3IONS+5u7AjH6I0l/JM3pr/WYqRMQNKm/xoMmDkDURfY38fOTpT+SpvXXctK8j0/YtPzWHzbng3MRZua36rApn5qLMa++VYeN/8BcFP2RpD+SLqq/8R+XCzMzvxOHDf+cXKZsf8M/Hhcu2N/wz8blS/U3/INxFab2Bz8mP5LkR5L8SNIfSeIjSW0kyY8k/ZGkP6LkR5L4SFIfYfojSn8kqY8g9ZEkPqL0R5L8SBIfScIjSX4E+emPJPkRpT6S1EeS/EjSH0n6I0p/ZMmPKP2RpD+i9EeS/kiSH1HyI0l9JMkPAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAOAmHQ7pN+A2HI6ldjj6rVDueGmHT6a/DTdmobLDs+mvxC15HdjhqMy7cXOO5ydApljKT39MoT+S9EeS/gha/PWfAJlAfyTpjyT9kbTcnwCp8CKmI2Wd6E+AbPeipS/LOlWfANnqxzEdCUuAjHMmrIP+GGRFWGe+S1eFr36ghm/QqqRWx1cnvQszBMJaLb0No6ULOyM9D2Ol+1olPRKjpMtaL70U1dJFNUrPRbF0UI3Sc1Er3VOr9F7USvfUKr0XpdI5NUsPRqV0Te3Si1GpJomSp6w9jB2pSaLkKWsPY0dqkih5ytrD2JGZ5dRIL0aldE3t0otRKV1Tu/RiVErX1C69GJXSNbVLL0aldE3t0otRKV1Tu/RiVErX1C69GJXSNbVLL0aldE3t0otRKV1Tu/RiVErX1CE9GYXK22j7Czr6zmA3xrRR/NSjZ7AHg9oofuzRM9iB8jZ+6KP4qa/PYDeGljJGejIqpWtql16MQumYOqQno1A6pg7pySiUjqlDejIKpWPqlJ6NIumQOqVno0g6pE7p2SiSDqlTejaKpEPqlJ6NKumS+qRXo0rv/df21H4+u9B9/bU9dbwAe9B9/bU9dbwAe5DNqFt6NoqkQ+p1p8FdSHe0QXo6CqQj2iA9HQXSEW2Qno4C6Yg2SE9HgXREG6Sno0K6ok5+/7sP6Y566W8f0h31evyLPrhy6Y56+flvH9Id9dLfTqRDOu3U+6WXo8C8lMqlp6NAOqIN0tNRIB1Rr/Ru1Eh31Ck9G0XSIXVKz0aRdEh90qtRJV1Sn/RqVEmX1CE9GYXSMXVIT0ahdEwd0pNRKB1Th/RkFErH1CE9GYXSMXVIT0ahdEzt0otRKV1Tu/RiVKqLoSSutiO5dvUdlHXWeC7XaGwDpd01ns01mHD58mPRjJv//MP1x2tzbv7Tj9Yfr029ef3x2tRb1x+vzb5v/fHS9LvWHy8k7lp/PApdtf74JHjV+mOpgst6m/x7Mc7FXfPjfyy5e/E6l/FiDHFp/T15fpsLezFKXWp/P7zZ0xeX9F6UutT8uA3iI0p+JOmPKPkBAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAALzwP1BLAwQUAAAACAAAACEAR5DjdKIIAACAYAkAEAAUAGdyb3VuZF90cnV0aC5ucHkBABAAgGAJAAAAAACiCAAAAAAAAO3dvY4kVx2HYXfKVWy2IG1gkIUQF0AGIiEgQgYvIkAYrREJcBXcMLaQPzC7PV3n/Kvf+XiebCaoOvXrV93R9Pz7V7/55a9/e/no7x/94/Vnb7/4w7vXP3/1+p+///HrN69e//Hzd3979+lffvf5u8/efvX7X3z65y/efvn7L/706V/ffvnzD3/y5tUnP/v4zauffvLxj968+ter437wEQAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAADwYlzqA/ACXb5VH4Xn7PKw+og8TzekJ0BOcXN7AmTWsfT0x5SF8gTItvXuBMiGgfAEyHGD4QmQm53QnQJ5yKnZKZAPuFd3CuQ77p+dAvlKWt7X6hGI1OF9ox6C+6ub+1/1GtxVndt71JNwL3VpH1Lvwh3UkV1Vj8O56r4eVi/Eaeq0blTPxBnqqo6p12JUndOKejOG1CGtq5djX93Qpno+dtT1TKg3ZFEdzpR6RxbU0Qyqp+SouphZ9ZocU/cyrh6Uw+pkRtVjsqKuZlA9JYvqcIbUM7KhjmdAPSF76n521fuxr25oSz0eI+qMltXDMaUuaU29GpPqmo6rF2NaXdQx9VqMq5M6pl6LaXVRx9RrMa5O6pB6LMbVSR1Sj8W4OqlD6rGYVzd1SD0W4+qkDqnHYlyd1CH1WMyrmzqi3op5dVNH1Fsxr27qkHosxtVJHVKPxby6qSPqrZhXN3VEvRXz6qaOqLfiBHVUB9RTcYI6qgPqqTjBQBQTbd14K56dkSZmArvtXjwrs0kMpXbbzXgG5pMYyu22m/HUnVPEWHU33Y2n68Qg5tq76XY8RSf3oD+u0x+lc3PQH9edXIP+uO7cGOTHdfqjdHIM+uO6c1vQH9edm4L+uO7kFPTHdeeWoD+uOzcE9XHduSVojwc83v7mn5XH53H2N/+cPE6PsL/5h+TxemT9zT8gj9qZbWiPhzyW/k54NJ6C8xLRHg877T1Ke9wg7O+cB+JpOSlA7XGTc/rTHre5b3+nPQZP1RkBio9bndCf+LjZXfo78wF42sYDFB8HTPcnPo4Yf/+DIwRISX+UDvQnQOaJj5L6KImPlPooiY+S+kiJj5L6KImPlPooiY+S+kipj5L4KKkPAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAADgxbtcLvUReKEu/1Ufg5fo8q36KLwwl++pz8ML8v349MfdvCc+AXIXH2hPgJzvWnz640wPtCdAznNLfALkDLe2pz/GHYlPgIw6Gp/+mLLQngCZsRqfANm1057+2LIbnwBZNdGeDFkyHJ8Oud158cmQ6+7Rng55r3vHp0O+Frb3HfUKJOrs/k89CPdU1/Zh9TLcQ13Zg+qBOFWd183qoThFndVh9WCMqnNaVM/GkDqkVfVuzKg7WlXvxoy6o2X1cIyoM1pWD8eIOqNl9XCMqDNaVy/HhLqidfVyTKgr2lBPx4A6og31dAyoI9pRb8e+uqEd9XbsqxvaUW/HvrqhLfV4bKsT2lKPx7Y6oT31euyqC9pTr8euuqA99XrsqgvaVM/HpjqgTfV8bKoD2lXvx566n131fuyp+9lV78eeup9t9YBsqfPZVg/IljqfffWC7Kjr2VcvyI66nn31guyo6xlQT8iGOp4B9YRsqOOZUG/IurqdCfWGrKvbmVBvyLq6nRH1iCyr0xlRj8iyOp0Z9YqsqsuZUa/IqrqcGfWKLKvTmVGvyKq6nBn1iqyqyxlSz8iiOpwh9YwsqsMZUs/IojqcKfWOrKm7mVLvyJq6mzH1kCypsxlTD8mSOpsx9ZAsqbOZUy/JirqaOfWSrKirGVRPyYI6mkH1lCyooxlUT8mCOppJ9ZYcVzczqd6S4+pmRtVjclidzKh6TA6rk5lUb8lxdTOT6i05rm5mUD0lC+po5tRLsqKuZk69JCvqasbUQ7KkzmZKvSNr6m6m1Duypu5mSD0ji+pwZtQrsqou53bXjlqvyKoDr//QdVZducldluIMB179uYutuHKDM/fhXEde/dErHnTt6ifMwp0cevGHL3vElSsPrsG9HXrtT7j20fzec9WBFaicmN+t1z98BPk9H+fWd+tNDp5Afs/GPfJ78E47F5s7IIF71ffe2w1caOxkJO6b3zf3nLmK/J68IL8pT+CIPOTJ1veVp3BGrnrK+fH0yY+S+ijJj5L6KMmPkvwoqY+S/Cipj5L8KKmPkvwoyY+S+ijJj5L6KMkPAAAAAAAAAAAAAAAA+AB/fkRJf5T0R8mfn5PSHyX9UfIBTEl/lHz9GiX9UdIfJd9/Skl/lPRHyfffk9IfJW+AlPRHSX+U/P83SvqjpD9K/gErJf1R8g/QKemPlP4oeQOkpD9K+qN0ESAh/VHSH6WLAAnpj9JFgJT0R0l/lHwAU9IfJf1RugiQkP4oXQRISH+U9EfpIkBK+qOkP0o+gCnpj9JFgHQu+iOkP0rf70+A3JP+KOmPkPwoqY+S+ijJj5L6CMmPkvooyY+S+ijJj5L6CKkPAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAcv8BUEsBAhQDFAAAAAgAAAAhAGHihvXuCAAAgGAJAA4AAAAAAAAAAAAAAIABAAAAAHByZWRpY3Rpb24ubnB5UEsBAhQDFAAAAAgAAAAhAEeQ43SiCAAAgGAJABAAAAAAAAAAAAAAAIABLgkAAGdyb3VuZF90cnV0aC5ucHlQSwUGAAAAAAIAAgB6AAAAEhIAAAAA', 'reference/sample-11.npz': 'UEsDBBQAAAAIAAAAIQDSTDdRNwoAAIBlCQAOABQAcHJlZGljdGlvbi5ucHkBABAAgGUJAAAAAAA3CgAAAAAAAO3dvY4cWQGGYU/KVTgzSA5YtEKICyADkRAQoQWMCBCLvIgEuApumB3b659x90xVnarzbm89TzY9XTV1vnrz+d9vfvfr3/7+7tm/nv37xZ9fffOn1y9++fzFf/74xYuXz1/85evX/3z91d//8PXrP7+6//xXX/3tm1fffv7NX7/6x6tvf/7xz14+//IXX7x8/vMvf/qTl8//+3y9Hz0DAAAAAAAAAAAAAAAAAAAAAAAAAAAAuF13n6sfifO4kJ8EmeVKfgpkhuv5CZCjPVafBjnY0/lpkMMszE+AHGJxfwJkf8vzkyB7W1efANnV6vwkyI429adA9rExP/2xi639CZBxm+uTIMPG6hMgY4b7EyDbjeenQLbbpT8Bssk+9QmQLXarT4BssGd/AmSlXfNTICvpj87e9QmQ5Q6oT4Asdkx/EmSRw/LTH087Lj8B8pQj69Mfjzu2PgnyuAn9CZBrZuQnQK7RH5k58QmQi+blJ0A+N7M/BfLA3PwEyCdm56c/3psenwD5IMlPgLwR1adAnqX1CZA2P/2dW1yfAM+tju+NegQqdXnv1DMQqcP7Tr0DiTq79+ohmK9u7hP1GMxWF/dAPQeT1cF9ph6EierYLqgnYZ66tUvqTZilLu2KehYOVef1tHohDlTHtUA9Ecep21qkHonD1GktVM/EQeqwFqpn4iB1WEvVO3GMuqvF6qE4RJ3VYvVQHKLOarF6KI5Rd7VYPRTHqLtarl6KI9RVrVBPxQHqqFaop+IAdVQr1FNxgDqqNeqt2F/d1Cr1WOyuTmqdei32Vhe1Uj0X+6p7Wqvei33VPa1V78Wu6pzWqxdjR3VMG9STsaM6pg3qydhRHdMW9Wbsp25pi3oz9lO3tE29GnupS9qono2d1CFtVM/GTuqQtqp3Yx91R1vVu7GPuqPN6uHYR93RZvVw7KLOaLN6OHZRZ7RdvRx7qCsaUE/HDuqIBtTTMa5uaEg9HsPqhIbU4zGsTmhIPR7D6oSG1OMxrE5oSD0ew+qEhtTjMaxOaEg9HqPqgsbU6zGqLmhQPR+D6oAG1fMxqA5oUD0fg+qABtXzMagOaFA9H4PqgAbV8zGoDmhQPR9j6n5G1fsxpu5nVL0fY+p+RtX7MabuZ1A9H4PqgAbV8zGoDmhQPR+D6oAG1fMxqA5oUD0fg+qABtXzMagOaFA9H6PqgsbU6zGqLmhMvR6j6oLG1OsxrE5oSD0eO6gjGlBPxx7qirarl2MXdUab1cOxizqjzerh2Mfy131oTqvVu7GXxS/78KbWaDdjR0tf9JywlunWYm+LX/O0up7WLMUhlr7liX09JdmJYyx9x1MLe1ywEkdZ+nKL0K6YOQ9HW/hqo9YumbcNMyx6rWlxn5qzCt8rdXQfqacgUEf3kXoKCnV1H9RLUKir+6BegkJd3Qf1EhTq6t6rh6BRd/edegcadXfv1DMQqcN7p56BSB3eO/UMVOry3qpXoFKX91a9Apk6vXv1BnTq9u7VG9Cp27tXb0Cnbu9evQGdur179QaE6vju9HdqdXx3+ju1Oj75nVtdn/7Ora5Pf+dW16e/c6vr09+51fXp79zq+vR3cnV++ju3Oj/9nVudn/7Orc5Pf+dW56e/k9MfJf2R0h8l/VHSHyX9kdIfJf1R0h+ltj8Bnp3+SMUBKvDk9Eep7k+A51bnp79zq/PT37nV+env3Or8BHhydX76O7c6P/2dWl2f/s5tUSH64yALC9Efh9AfJf0RWlqI/tjdmkIEyM5W9aE/dvY96S9egYr+qKwO5Kj69HdCGwI5pr33t+dMNgRyTHvvb8+ZbAjkmPbe354T2RLI+kvkx0Ur87tbf9mKPzn36PTW5ne38tLuZNyC7fktubo5E7djrL+nri9OxC0Z7e/Rm0w+C7dnl/4u3HDCo3P7ds8P1pAfKflRkx/fD/IDAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAD4Ibv7SP0snM3dA/XzcCIP45Mgk1xOT4HM8Gh9GuRQC+rTH0dZlJ8AOcbC/ATIERbnJ0H2tio+AbKv1fnpjx2t70+A7GdDfwpkL5vyEyB72BifANnFQH8CZNRIfvpj1FB/CmTQYH8KZMhwfwJku/H89Md2O/SnQDbbpT8BstE+/UmRbXbvT4gsd1x+AuRp+qN0ZH8K5CnH9idBHnd4fwLkEcf3J0GumpKfALliUn8i5KKZ/amQh+b392mJcjy3sr936gkI1fHdqzegU7d3r96ATt3eG/UIZOr03qhHIFOn91a9ApW6vLfqFajU5b1XD0Gizu6DegkKdXUf1EtQqKv7SD0FgTq6j9VbMF/d3KfqNZitLu6Beg4mq4N7qN6DuereHqr3YLI6uM/UgzBVndvn6kWYqa7tgnoSJqpju6CehInq2C6oJ2GiOrYL6kmYqa7tgnoSJqpju6CehInq2C6pN2GeurWL6lGYpk7tsnoVZqlLu6xehVnq0q6oZ2GSOrQr6lmYpS7tsnoVZqlLu6KehUnq0K6oZ2GSOrRr6l2Yo+7sqnoYpqgzu65ehhnqyq6rl2GGurJH1NMwQ13ZdfUyzFBXdl29DDPUlV1XL8MMdWXX1cswRZ3ZVfUwzFF3dk29C3PUnV1Rz8IsdWmX1JswT93aBfUkTFTH9pl6EKaqc9PbuSW1iY535oYHn5IfJflR0h8p/VGSHyX9UdIfJf2R0h8l/VHSHyn9kdIfKf1R0h8l/VGSHyn9UdIfJflReiQe+XE4/VF6rB75cbRH85EfRxvp79Ev1AfjJuiP0lB/j32jPhg34anGtvZXn4vbMJTf9cvrY3EjDumvPhQ3Q3+UxvK7fHl8JG7IWH6Xr08PxE0Zq+/i9eFpuDmD/X1+fXcUbtHO/XUH4TaN9ffs6i9gkaH8nl39BSwz1N+za5/DQjv1Fz09t+9SYgv7e3b5U1juYmHr+guemh+MS3Wt6m/y8/LDtyy/t98rno8ftqX9+b+WHGFxf3AA/VHSHyX9UdIfJf1R0h+la/0JkBn0R0l/lPRHSX+U9EdJf5T0R0l/lPRHSX6U5EdKfZT0R0p+tORHTHUAAAAAAAAAAAAAAACwo/8DUEsDBBQAAAAIAAAAIQBLf/ATjQoAAIBlCQAQABQAZ3JvdW5kX3RydXRoLm5weQEAEACAZQkAAAAAAI0KAAAAAAAA7dfNbh1JAYbheMtVZBeQsmDQCCEugB2IDQtWaIAgFohBGcQGuApumMmPE8c+x+4+1V3vHPfz7GyprKqvXynK/37zu1//9vc3L/714t+v/vzmuz+9ffXLl6/+88evXr1++eov377959tv/v6Hb9/++c273//qm7999+b733/312/+8eb7n3/8s9cvv/7FV69f/vzrn/7k9cv/vlzvRy8AAAAAAAAAAAAAAAAAAAAAAAAA4Jm5+V59Bw7r5rP6KhzPzT31fTiU+/lJkIlO5adAJjmTnwKZ4Xx+EmR3j+enQHb1ZH4SZD+L8lMg+1ianwLZwYr8JMjWVuYnQLa0Oj8Fsp1L8hMgG7ksPwGyiUvzUyAbGMhPgIwayk+AjBnMT4GMGM9PgFxsi/wEyIW2yU+BXGSz/ATIBTbsT4CstWV+CmSljfMTIGtsnp8AWWyH+hTIUjvlJ0CW2C0/BfK0PfMTIE/YNz8B8qi981Mgj5iQnwA5Y0p9AuS0WfkpkBMm5idA7puanwD50uT8FMhd8/PTH7eC+gTIrSY/AfJelZ/+eBHmJ0DK+gRIm58AD67OT3+HVtd3I8Ajq9t7rx6BSB3eR/UMJOrsPquXYLo6uS/VazBX3dsD9SDMU7d2Sr0Jk9ShnVPvwgR1ZI+pt2FndWBPqOdhV3VeT6sXYkd1XEvUG7Gfuq1F6pHYT93WEvVG7Kdua5F6JPZTt7VIPRK7qdNapB6J3dRpLVOvxG7qtJapV2IvdVkL1TOxkzqsheqZ2Etd1kL1TOykDmupeid2UWe1WD0Uu6izWq5eij3UVa1QT8X26qZWqcdic3VSq9RjsbW6qJXqudhYHdRa9V5sq+5prXovNlXntF69GFuqa7pAPRnbqVu6SD0am6lTukg9GpupU7pMvRobqUO6VL0b26g7ulS9G9uoO7pYPRybqDO6XL0cm6gzuly9HFuoK7pcvRxbqCsaUE/HBuqIRtTbMa5uaES9HePqhobU4zGsTmhIPR7D6oSG1OMxrE5oSD0ew+qExtTrMaouaEy9HqPqgsbU6zGsTmhIPR7D6oTG1Osxqi5oTL0eo+qCxtTrMaouaEy9HqPqggbV8zGm7mdUvR9j6n5G1fsxpu5nVL0fY+p+RtX7MabuZ1S9H2PqfkbV+zGm7mdUvR9j6n5G1fsxpM5nWD0gQ+p8htUDMqTOZ1g9IGPqfgbV8zGoDmhQPR+j6oLG1Osxqi5oTL0ew+qEhtTjMaxOaEg9HsPqhIbU4zGubmhEvR3j6oYG1NOxhRXfe7+ULlIvxxZWfOxda1qvXo5NLP/Suxe1TroaW1n+mWdEtUI3GVta/o1nlbVMsxZbW/59Z9b1tPlLsYflH3dyYE+YPBN7Wfxlg8geMXMidrT8szahnTZrHXa3+KN2tT00ZRl+UOrm7qq3YL66ubvqLZivbu6uegsCdXR31FMQqKO7o56CQB3dHfUUBOroPquXIFFn90k9BIk6u0/qIUjU2X1SD0Gizu6TeggSdXaf1EPQqLu7Ve9Ao+7uVr0Djbq7W/UONOruPqpnIFKH91E9A5W6vA/qFajU5X1Qr0ClLu+DegUqdXkf1CuQqdN7rx6BTJ3ee/UIZOr03qtHIFOn9069AZ26vXfqDQjV8d3o79Dq+G70d2h1fDf6O7Q6vhv9HVtdn/6Ora5Pf8dW16e/Y6vr09+x1fXp7+Dq/PR3bHV+Ajy2uj79HVtdn/4Ors5Pf8dW56e/Y6vz09+x1fnp79jq/PR3cPqjVPcnwGOr89PfsS2uRH/sQH+k9EdpaSR79SfAY9MfJf2RWpiI/tiF/ijV/Qnw0JYWoj92sDgQ/bG95YHoj82tCER/bG1NIPv1J8CDWtWH/tjWujz0x6ZW5qE/trQ6j8vTkh/3rW1o5ZmzVemPF+tSWn9q1SV2eR8/aJflt/Bg9Caux8X9PX00eQ/XZaC/xw8Hb+H6jOT3yPHZz+BKjeV35g9MfQFXbDi/E39j3u25dhvkd/tXJt2YZ2gsP9iO/PjBkB8AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAw7Oau+jIcw81j6svx3D2anwDZ1xP5KZA9PZ2fANnNkvwEyE6W5SdAdrE0PwWygxX5CZCtrcpPgGxrZX4KZEvr8xMgm7kkPwGykcvyEyCbuDQ/BbKBgfwEyLCh/gTIoLH+FMiQ0fwEyIjx/gTIxTbIT4FcbJv+BMhlNupPi1xkh/6EyGJ79idAnrJrfwrkCTv3J0AetXd/AuQxu/cnQB6xf38K5KwZ+QmQc+b0J0BOm9SfBDlpXn8PGxQmc/u77e3+zxxV0N8D9QZ06vbeqTcgVMd3o79Dq+N7p96ATt3eO/UGdOr23qtHIFOn90G9ApW6vI/qGYjU4d2qd6BRd/dJPQSJOrvP6iUo1NXdUU9BoI7urnoL5qub+0I9BtPVyX2pXoPZ6uLuqedgsjq4++o9mKvu7YF6EKaqc3uoXoSZ6tpOqCdhojq2E+pJmKiO7YR6Emaqa3uoXoSZ6toeqhdhprq2h+pFmKmu7YR6EiaqYzul3oR56tZOqkdhnrq1k+pRmKZO7aR6FKapUzutXoVZ6tJOq1dhlrq0M+pZmKQO7Zx6F+aoOzun3oVJ6tDOqXdhjrqzc+pdmKPu7Kx6GKaoMzurHoYp6szOqodhijqzs+phmKPu7Jx6F+aoOzun3oU56s7OqXdhjrqzc+pdmKPu7LM7NyoHYaq6uvfqEehoj5L0KEmPkvYoSY+U9ijpj5T+KOmPkvxI6Y+S/kjJj5L+KOmPlPwo6Y+U/ijpj5L8SOmP0nhe+mPAnv3FT+MK6I+S/kjt11/8MK7CaH/n/0D7Lq6E/ihd3t8T/wDH7+JK7NRf/CquxT79xY/iauzSX/wmrsgO/cUv4ppc3N/NudP1i7gqQ/2dOB0/hyszlN/D0+1juD6Xx3ficPgOrtPl8T08XL2B6zVQ3/3DzQO4bpv111yfK7dVf83tuXoD/b0493tY6nRm6/pLLs6zcDqyVf0Ft+bZOFnYmv5mX5hn5kRfK/qbe1eeoYdxLe9v4jV5ts7/1/aJ/mZdkGNZ2F9yN56/Rf3BTvRHSX+U9EdJf4RO5ydA5tAfJf1ROtefAJlBf5T0R+lsfwJkAv1ROt+fAJlAfpTkR0t+tNRHS3605EdLfrTUR0t+AAAAAAAAAAAAAAAAAFfu/1BLAQIUAxQAAAAIAAAAIQDSTDdRNwoAAIBlCQAOAAAAAAAAAAAAAACAAQAAAABwcmVkaWN0aW9uLm5weVBLAQIUAxQAAAAIAAAAIQBLf/ATjQoAAIBlCQAQAAAAAAAAAAAAAACAAXcKAABncm91bmRfdHJ1dGgubnB5UEsFBgAAAAACAAIAegAAAEYVAAAAAA==', 'reference/sample-12.npz': 'UEsDBBQAAAAIAAAAIQBX2482AgkAAFi5BQAOABQAcHJlZGljdGlvbi5ucHkBABAAWLkFAAAAAAACCQAAAAAAAO3av44cWQHFYXfKU0w2IDlYQCskHoAMREJAhAxrRIDYlY1IgKfghbGnPT2z8r+u6u576tb5vqBHXdKoSud3K+v//e4Pv/39Hw8v/vXi3/ffvX77lzf3v767/8+ff37/8u7+r9+/+eebV//40/dvvnv9/vpvXv397et319/+7dUPr999/+kvXt798lffvrz79ptvfvby7r93y/3kBQAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA13U4pJ+AgQ6flH4qbujTyVXfr88Wl32vvpxc9705o7fsO3N+c9F3YkFyzXdhSXHRd2FhctV3YHlz0WeneZ0VyTWf25rkos9N8zrrkos+s7XNRZ/X6uaiT0vzPpr3Wd9c9Vld0lz0OWneR/M+mvfRvI/mfTTvo3mfi5qLPiXN+2heSPNCmvfRvI/mhTTvo3kfzfto3kfzPpr30byQ5n0076N5H80Lad5H8z6aF9K8kOaFNO+jeR/NC2leSPQ+mvdZ1Vz0ua2LrvrMVjZXfWKa91ndXPV5id7nguayT+qy5qrP6NLmsk/oCtF1n811oss+k2s1V30iove5XnPdZ3Hd5rLPQPNCojdSvc/Vm4u+faL3uX5z0bfuBs1V37ibNBd90zQvJHqf2zRXfcs073Or5qJv1s2Si75ZN2yu+kZp3uemzUXfotsm13yDbpxc9Q3SvM7tk4u+MSOSi74pY5JrviWDmou+HaOSa74Zw5KLvhkDm4u+EZrXGZlc823QvM/Q5qJvguZ9NO8ztrnoWzC4ueh5o5Nrnqd5H83rDE8uepzmfTSvE0iueZjmfTSvk0guepbmfTTvo3mdTHLNk0LNRQ9KNRc9R/M6seSap+SSa56ieR/N+wSbi56RTC56huZ9NO+jeR/N+2jeR/M+2eaqJ2jeR/M+mvdJNxd9vHRyzcdLJ9d8vHRyzcdLJ9d8vHRyzcdLJ9d8vHRyzcdLJxd9vHRxzcdLFz+IPlw6+EHz4dLBH6RHKJPOfZReoUu69oP0CGXSuY/SK1RJx36U3qFJuvVJeogi6dQn6SGapFs/Su/QJN36UXqHJunWj9I7NEm3fpTeoUm69Ul6iCLp1CfpIYqkU5+khyiSTn2SHqJIOvVJeogi6dRP0kv0SJd+kl6iR7r0k/QSPdKln6SXKJJOfZIeokg69Ul6iCLp1E/SS/RIl36SXqJHuvST9BI90qWfSU9RIx36mfQUNdKhn0lPUSMd+kl6iR7p0s+kp6iRDv1Meooa6dBP0ksUSac+SQ9RJJ36JD1EkXTqk/QQRdKpT9JDFEmnPkkPUSSd+iQ9RJF06pP0EE3SrT9Iz9AlXfsovUKXdO2j9Apd0rWP0itUScc+Sq/QJV37KL1Cl3Tto/QKXdK1j9IrlEnnfpAeoUw694P0CGXSuR+kR+iSrv0gPUKXdO2j9Apd0rWP0it0Sdc+Sq/QJV37vfQGbdK9D5IPlw5+0Hy4dPCD5sOlg0s+XDr4QfPh0sEPmg+XDn7QfLh0cMXHk7xOOrnm46WTaz5eOrnmw6WLaz5eurjm46WLaz5eurjmw6WDHzQfLh38oPlw6eAHzUdL9z5IPlq693vpDcqkcz9Ij1AmnftBeoQu6dpH6RW6pGsfpVfokq59lF6hSjr2B+kZqqRjf5CeoUm69aP0Dk3SrR+ld2iSbv0ovUOTdOtH6R2KpFOfpIcokk59kh6iR7r0k/QSLdKdn0tvUSKd+UfSY5RIZ/6R9BgVvlJgUOpnd+TmvhLg3ceo3I+35MbO2H9U7uf35IbOmX9YcMlHOGv+YcU1H+C89YcV13yA89YfVlzz2ztz/VHBJR/g/PlF34kF42u+E5r3WTK+5LuwaP0xzUW/rWXbD2ou+k0tm35Qcs1vatn0g5JrflPLhtd8DxburvkOLN1d8+kt3l3z2S3fXfPZaV5m3e6az+zzgz/8unnhf2m+fWv3Vnxa66fWfFofpl2+suSNrhBV8hl9lOlwuv7879dCat1H7EKSAwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAHzt8JP1E3NbHxUXfuU8m13zHPl1c9R37fHLV90rzOl9MrvkefTm56jv01eSi747mfc5oLvq+nJNc8105K7nou6J5H83rnJlc9B3RvI/mdc5OrvluaN5H8j6a99G8j+R9NO+jeR/NC4neR/M+mveRvI/mfTTvo3mdM5NrviOa99G8j+Z9JO+jeR/N+0hex2veR/I+mvfRvM15xSXfE83rSN5H8z5nNhd9N84trvl+aF7n/OSa78WC5qLvw5Lkou+D5nWWJdd8BxYmF30HNO+jeZ3FyTWf3vLmos9O8z6a99G8zorkmk9O8z6a99G8z5rmos9tVXPRp6Z5H837aN5nXXPRZ6Z5H837rGwu+sQ076N5H837aN5H8z6a91nbXPV5rW8u+qw076N5nwuaiz4pzfto3kfzPpc0PyO6g7FBFzX/fNDFh4NxNC90k+Znnw0SLmr+ccyFh4OE6zY//3CQo3mfqzVfdjgIuqz5irNBnOZ9NO+jeR/J+2jeR/I+kvdRvI/ifTTvo3kfyfto3kfzPpL3uU1hubfsJsnZNM37SN5H8T6a95G8j+J1FO8jeR/J+2jeR/M+mvfRvM/FzUWfjuZ9NO+jeZ/Lm6s+m2s0F30umhfSvM81mos+F837aN5H80KiFxK9j+aFNG8keiHNC2neR/NCmhe6KLjmU7qsefrpWeWS5kxK9D6SF5K8j+R9NC+keSHNC3nT+6xrLvrMNC+keR/NC4neR/NCmhfSvI/mfTTvo3mfNc1Fn5zofSTvo3mhNdFVn5vmhTTvo3khzft40fto3kfzPmuaqz43zQtp3kfyPt7zPpr30byP5n0k7yN5H837aN5nRXPRJ6d5H837aN5H8j6S99G8j+Z9ljcXfXaa95G8j+Z9NC+keR/J+2jeR/M+mvfRvI/mfTTvI3kfzfto3kfyQpr30byP5n00LyR5Jc05kRyqeM/hC/4PUEsDBBQAAAAIAAAAIQC4jKlNWAkAAFi5BQAQABQAZ3JvdW5kX3RydXRoLm5weQEAEABYuQUAAAAAAFgJAAAAAAAA7dS/jiNpGcXhdcpVdDYgTbCAVkhcABmIhIAIDewgAsSiGUQCXAU3zPxpemyPy+1y23WqvvM8mR19Or/X/u9vfvfr3/5+980/v/nXq+/fvv/Tu1e/fHj17z/+9NXrh1d//uHdP969+dsffnj3/duP3//qzV/fv/3w/fu/vPn72w+ff/yz1w8//8V3rx+++/bbn7x++M/DfD/6BgAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACA29l9kH4DC9odSL+G+9udlH4Vd3M6uOzDOhdc9SE9m1z14VzSXPWxXNZc9YFcmlz0cVzeXPVRzGmu+hjmNRd9ADOTqz6A+c1F37ormqu+cVc1F33Lrkuu+ZZd2Vz0DdO8z7XNRd8uzftc3Vz0zdK8j+Z9rm8u+lZp3kfzPi9oLvpGad5H8z6a93lJc9G3SfM+mvfRvM+Lmou+SZr30byP5oU076N5H8kLad5H8z6aF9K8j+R9NO+jeSHN+2jeR/JCmvfRvI/mhUTvc11z0bdM80JXRld9w65trvqGXR9d9a16QXPVN+pFzXXfphdHl31zbtBc9625UXTZN+RmzWXfjFs2F30jbhpd9U24bXPRt+DGzVXfAtEL3Ty66usnep/bNxd99e4QXfW1E72Q6IVE73OP5qKv212ai75q92ku+prdqbnoK3av5qKv192ai75a92su+lrdsbnoK3XP5qqv032bi75Gd24u+vrcO7no63P/5qKvzALJRV+XRZJrvirLNBd9RRZKrvmKaF5nqeSar8ZiyUVfDc3rLJhc83VYMrno66B5nWWTa74CCycXfQU0r7N4cs3Tlk+ueVqguehZieSaR0WSax6VaS56UCi55jmp5JrnxJqLnpJLrnlIMLnmIZr3STYXPUPzPpr3iTYXPULzPpr3yTYXPUHzPpr30bxOOLnoAenkmi8vnVzz5aWTi768dHHNl5curvny0sV3oi8uHXyn+eLSwXeaLy4dfKf54tLBP0pv0Cbd+6P0Bm3SvT9Kb9Am3fuT9Ahl0rk/SY9QJp37k/QIZdK5P0mP0CVd+7P0Cl3StT9Lr9AlXfuz9Apd0rUfpWeoko79KD1DlXTsR+kZqqRjP0rPUCUd+1F6hirp2I/SM1RJx36UnqFJuvX/pXdokm79JD1EkXTqJ+khiqRTP0kPUSSd+kl6iCLp1E/SQxRJp36SHqJIOvWT9BBF0qm/SC/RI136i/QSPdKlv0gvUSMdek96ihrp0HvSU9RIh96TnqJGOvSe9BQ10qH3pbdoke68L71Fi3TnfektWqQ770tv0SLdeV96ixbpzvvSW7RId96X3qJFuvO+9BYt0p0PpMcokc58ID1Gh3TlQ+k1OqQrH0qv0SFd+VB6jQ7pyofSa3RIVz6UXqNDuvKR9BwV0pGPpOeokI58JD1HhXTkI+k5KqQjH0nPUSEd+Uh6jgrpyEfSc1RIRz6W3qNCOvKR9BwV0pGPpOeokI58JD1HhXTkI+k5OqQrH0qv0SFd+VB6jQ7pyofSa3RIVz6SnqNCOvKR9Bwd0pUPpdfokK58KL1Gh3TlQ+k1OqQrH0qvUSKd+UB6jBLpzIfSa3RIVz6UXqNCOvKR9BwV0pGPpOeokI58JD1Hg3TjY+k9GqQbH0vvUSCd+CvpQQqkE38lPcj40oW/ll5kfOnCX0svMr504a+lFxleOvAJ6UmGlw58QnqS0aX7npLeZHTpvqekNxlduu8p6U0Gl857UnqUwaXznpQeZWzpuqelVxlbuu5p6VXGlq57WnqVoaXjTkjPMrR03AnpWUaWbjslvcvI0m2npHcZWDrtpPQwA0unnZQeZlzpstPSy4wrXXZaeplxpctOSy8zrHTYM9LTDCsddlp6mWGlw56RnmZY6bBnpKcZVbrrOeltRpXuekZ6mlGlu56T3mZU6a7npLcZVDrrOeltRpXuek56m0Gls56VHmdQ6axnpccZVDrrOeltRpXuek56m1Glu56T3mZU6a7npLcZVbrrGelphpUOe0Z6mmGlw56RnmZY6bDT0suMK112WnqZcaXLTksvM6502WnpZcaVLjspPczA0mknpYcZWDrtpPQwA0unnZLeZWjpuBPSswwtHXdCepaxpeuell5lcOm8p6Q3GV267ynpTYaXDnxCepLhpQOfkJ5kfNpWUgIAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAIBV2H2QfgML2j1Kv4Nl7PalH8P97Y6lH8RdfdVb9LGdDi76uKaLiz6os8lFH9IzzUUf0HPNRR/Ps81FH83zyUUfzSXNRR/LRc1FH8plzUUfiOR9NO+jeZ0Lk2s+EM37SN5H8zr+2vto3kfzPpL30byQ5n0076N5H837aN5H8z6a99G8j+Z9NO8jeR0/8z6a95G8jp95H8nr+Jn30byO5H00ryN5H83rXJZc85Fc2Fz0cVyaXPRxaF7n8uSij0LzPprXmZNc8zFo3kfzOrOSaz4EzevMS675CGY2F30AmteZm1zz7dO8j+Z1ZifXfPM0rzM/ueZbp3kfzfto3kfzPpr30byP5n0076N5H837aN5H8z6a99G8j+Z9NO+jeR/N+2je54rmom+c5n0076N5H837aN5H8z6a97lPc8exZndo7l9h7W7cfPaFsLxbNp97IGTcqvnc+yDnJs1n3gdZL28+7zxYgZc1n3UdrMQLms+6DtbjyuazjoN1uab5jNtgheY3v/g0WCvN+0heR/I+kvfRvI/kfSTvo3kfyftI3kfyPpr3kbyP5H0k76N5Hcn7SN5H8z6S95G8j+R9NO8jeR/J+2jeR/I+kveRvM5Nkmu+KZL3kbyP5H007yN5H8n7aN5I8kKSF9K8kOaNNC8keSHNC2neR/NCmvfRvJDmhTTvo3khyftoXkjzPpoX0ryP5n00L6R5H837SF5I8z6aF9K8j+Z9NC+keR/NC2neR/I+mhfSvI/mfTQvpHkfzQtp3kfzPpL30byP5n0076N5H837aN5H8z6a99G8j+R9NO+jeR/N+2jeR/M+mvfRvI/mfSTvo3kfzftoXue55JqPR/M+mvfRvI/mfSTvo3kfzfto3kfzPnr3kbuP3H307iN3H7kBAAAAgO36H1BLAQIUAxQAAAAIAAAAIQBX2482AgkAAFi5BQAOAAAAAAAAAAAAAACAAQAAAABwcmVkaWN0aW9uLm5weVBLAQIUAxQAAAAIAAAAIQC4jKlNWAkAAFi5BQAQAAAAAAAAAAAAAACAAUIJAABncm91bmRfdHJ1dGgubnB5UEsFBgAAAAACAAIAegAAANwSAAAAAA==', 'reference/sample-13.npz': 'UEsDBBQAAAAIAAAAIQD7uw7I1BEAAIBRCQAOABQAcHJlZGljdGlvbi5ucHkBABAAgFEJAAAAAADUEQAAAAAAAO3cu44kCVqG4R6XqxhvQBoD0IqVuAA8EA4GFlrYQRiIXc0iHOAquGG2D1mVh6iqPET8b+7G8xit6s5UdOb3v2qz/+/v/uFv//4fv/v0X5/++4df//S7f/n5h7/+/of/+ee/+OHH73/419/8/J8//+o//uk3P//6p89//je/+vff/fT7P//dv/3qtz/9/vd/+pc/fv+LX/7yx+//6hd//mc/fv+/39/uTz4BAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA/GH67kL9idiHy/JkyIwP2pMgW7ouPwGyiWvzEyBru749CbKqm9MTISt5oD0B8rBH+xMgD3g4PwFyvxXykyD3WSk+BXKPNfNTILfSH6WV+xMgt1g7P/1xi9X7UyDX2iA+BXI1/VHaqj8BcoXN8tMfV9iuPwXyoS3zUyAf0R+hjfPTH+/Yuj4B8o6B/PTHWybyEyBv0R8l/RGayU+ALJrKT4EsGMxPf1yY7E+AnNMfodH89MeZ2f4EyCn9ERrOT3+cmO5PgBzTH6Xx/hTIqyA//XFQ5Kc/DvRHKMlPgHyjP0JRfvrjC/1R0h+lqj8B8pn+KOmPkv4o6Y9S1p8AKfPTH2l/AkR/pPRHqexPgOiPlP4IlfnpD/1R0h+ltD8B7p7+KOmPkv4o6Y+S/ijpj5L+KOmPkv4o6Y+S/ijpj5L+KOmPkv4oyY+S/gil+elv9/RHSX+U9EdJf5T0R0l/lPRHSX6U9EdJf5TkR0p+lPRHSX+U9EdKf5T0R0p+lPRHSn+k9EdKf6TkR0l/lPRHSn/E5EdLf5TkR0p/lORHSn6k5EdJf5T0R0l9pNRHSn2U5EdJf6TkR0p/pORHSX+U9EdKf5TkR0t+pPRHSX+k5EdJf5T0R0l/lPRHad3+BMiNBEhp5f4EyE30R0l/lNbuT4DcYvX+FMj1NshPgFxNf5Q26U+AXEl/dLapT4FcZbv8BMiHtsxPf3xEf5Q27U+AvG/b/ATIO7aOT3+8YyA/AfIm/ZEZiU+BLJqrT4FcmM1PgByZjk9/HNMfnaA+AfJVE58A+ayrT4C7l8anv72r8xPgntXtfVGPQKPu7kg9BfPq5k7UYzCsDu5MPQez6t4u1IMwqI5tSb0JI+rM3lFPw8bqwD5S78Om6rw+Vi/Ehuq4rlFvxDbqrq5XL8Xa6qJuVe/FquqcblYPxqrqnG5XL8aK6pjuUW/GSuqQ7lYPxxrqih5QT8fD6oQeVe/HQ+p8HlYPyEPqfB5XL8gD6njWUG/I/ep2VlLPyH3qblZTD8ld6mzWUy/JPepqVlRPye3qZtZVr8lt6l5WVw/KTepcNlBPyvXqVjZRj8rV6lS2Ua/KtepSNlLPypXqUDZTD8tV6ky2Uy/LNepKNlRPy8fqRjZVj8uH6kS2Va/LR+pCtlbvy/vqPrZXL8w76jgG1BPzjjqOAfXEvKOOY0Q9Mm+q05hRr8wb6jCm1DuzrO5iSr0zy+ouptQ7s6zuYkq9M8vqLqbUO7Os7mJMPTSL6izG1EOzqM5iTD00i+osxtRDs6jOYkw9NIvqLMbUQ7OozmJMPTSL6izG1EOzqM5iTD00S+oq5tRLs6SuYk69NEvqKubUS7OkrmJOvTRL6irm1EuzoI5iUD01C+ooBtVTs6COYlA9NQvqKAbVU7OgjmJQPTWX6iYm1VtzqW5iUr01l+omJtVbc6luYlK9NZfqJibVW3OhTmJUPTYX6iRG1WNzoU5iVD025+oiZtVrc64uYla9NmfqIIbVc3OmDmJYPTdn6iCG1XNzpg5iWD03Z+oghtVzc6YOYlg9N6fqHqbVe3Oq7mFavTen6h6m1Xtzqu5hWr03p+oeptV7c6ruYVq9N6fqHqbVe3Oq7mFavTen6h6m1Xtzos5hXD04J+ocxtWDc6LOYVw9OCfqHMbVg3OizmFcPTgn6hzG1YNzos5hXD04J+ocxtWDc6yuYV69OMfqGubVi3OsrmFevTjH6hrm1YtzrK5hXr04R+oYCvXmvKpbCNSTc6SOIVBPzqu6hUK9Oa/qFgr15ryqWyjUm/OqbqFQb86ruoVCvTmv6hYK9ea8qlso1Jvzqm4hUY/OQV1Coh6dF3UKiXp0XtQpJOrReVGnkKhH50WdQqIenRd1Co16dQ7qEhr16hzUJTTq1TmoS2jUq3NQl9CoV+egLqFRr85BXUKjXp2DuoREPTov6hQa9ep8U4cQqWfnoC6hUa/OQV1Co16dg7qERr06B3UJjXp1DuoSGvXqHNQlNOrVOahLaNSrc1CX0KhX56AuoVGvzkFdQqNenYO6hEa9Oi/qFBL16LyoU0jUo/OiTiFRj86LOoVEPTov6hQS9egc1CU06tU5qEto1KtzUJfQqFfnoC6hUa/OQV1Co16db+oQIvXsfFOHEKln55s6hEg9O9/UIUTq2fmmDiFSz85XdQeVene+qDPI1MPzRZ1Bph6ez+oKOvXyfFZX0KmX57O6gk69PJ/2nJ/+nkEdQaienk/6o1VHEKqn55P+SNUNlOrt0R+tuoFSvT36o1U3UKq3R3+06gZK9fbsOj/99eoEUvX46I9UnUCrXp+6gFa9/u7VAcTq+XevDiBWz7939f1z9QF2rj5/rj7AztXnz9UH2Ln6/L36AvtWXz9XH2Df6uv36gvsW339Xn2Bfauv36svsG/19Xv1Bfatvn6vvsC+1dd/AvUJdq0+/hOoT7Br9fGfQH2CXauP/wTqE+xaffwnUJ9gz+rbP4P6BntW3/4J1CfYtfr4T6A+wa7Vx38G9Q32rL79M6hvsGP16Z9CfYQdq0//FOoj7Fh9+qdQH2HH6tM/hfoIO1af/hnUN9ix+vRPoT7CjtWnfwr1EXasPv1TqI+wY/Xpn0J9hB2rT/8U6iPsWH36p1AfYc/q2z+D+gZ7Vt/+GdQ32LP69s+gvsGe1bd/BvUN9qy+/TOob7Bn9e2fQX2DPatv/wzqG+xZffsnUJ9g3+rr5+oD7Fx9/lq9/+7VAbTq9bkpwKV3vz7l6NVPh18+v3T0N138tPyR3v5gx8+74bu8vnj6tntHY0UfdHX59su3/YH5w/3kAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAADApO9O1J+G3fjuTfUn44/e2/FJkG191J4G2dC1+QmQ9V1fnwBZ2031KZBV3VyfAFnJPe0pkHXcX58AedhD+QmQxzyYnwB5xMP5CZB7rRCfArnXWvkJkDusl5/+uN2K/QmQW62Zn/64zar1KZDbrJ6fALneBvkJkKvpj9Am+QmQ62yUn/64xlb5CZBr6I/Qdvnpj49t2J8A+ciW+SmQj+iP0sb9CZD3bJ2f/niP/iht3p8AeYf+CG2fnwB5m/4ITeQnQN4y058AWTSUnwBZpD9K+qM01p8AWaA/SnP9CZBL+qOkP0qD/QmQC5P9CZBz+qOkP0Kj+emPcwKkpD9K+qOkP0r6ozTbnwA5pT9Kw/0JkBP6o6Q/SvojJUBK+qOkP0r6I6U/SvqjpD9S+qMkP0r6I6U/UvqjJD9K+iOlP1L6o6U/SvqjpD9S+iOlP1r6o6Q/SvojpT9K+qOkP0r6o6Q/SvojpT9K+iOlP0r6I6U/WvojpT9K+qOkP1IbBqhAriFAUvqjpD9K+iOyWXoC5NzGsemPd8zXpz++KeLTH99E+emPT119AiStT4C0+Qlw7+r+BLhvdX7627W6Pv3tWd3eF/UIVOryvqpXoFKX91W9ApE6vG/qGYjU4R3UO5Cos3tRD0Gizu5VvQSFurpX9RIU6upe1UsQqKM7Vm/BuDq5E/UYTKuLO1WvwbS6uDP1HAyrgztTz8Gsurdz9R7Mqns7V+/BqDq3S/UiTKpru1QvwqS6tkv1IgyqY1tQT8KYOrVF9ShMqUtbVq/ClLq0ZfUqTKlLe0M9C0Pq0N5Qz8KMurO31cswoa7sHfU0bK9u7D31Nmyvbuw99TZsrk7sffU6bKwO7AP1PGyr7utD9UBsqs7rQ/VAbKrO62P1Qmyojusq9Uhspk7rOvVKbKQO61r1Tmyj7upa9U5sos7qavVQbKGu6nr1Umygjuom9VisrS7qRvVcrKwO6kb1XKysDupG9Vysq+7pZvVgrKmu6Q71ZKynbuke9Wasp27pLvVorKZO6S71aKylLulO9WyspA7pbvVwrKGu6H71cqyhruh+9XKsoa7ofvVyrKCO6BH1djysTuhB9Xw8qA7oUfV+PKTO52H1gDykzudh9YA8oq7ncfWCPKKuZwX1hNyvbmcN9Ybcr25nFfWI3K1OZxX1iNyrLmct9Y7cpc5mPfWS3KGOZk31lnxxyzXSXta28a5c5aZ7dLFsYPNpucItB8lS2cbAunzg+os0jWxramXecu1Bkjy2N7g0S649SBHHgMmpWXDdQYo0RsyOzYWrDlKUMWN4bS5ccZEijCnje3Pq44sEVcwJFufYxyeZj2JQMTlHPj7JfBSDisk59sFN5pOYFa3Oi/duMt/DtG53vnn7KPM5zCuX57O3jjKeQiIen8XO9lLfdwLs1QW06vXZd4D1+OiPVt1Aqd4e/RGrIwjV0/NZXUGmHp6v6g4q9e58VXdQqXfnq7qDSr07n9UVZOrh+ayuoFMvz57r098TqBMo1dtTF9Cq19+j+ubPpL7F3tT3fjb1PfalvvbTqQ+yM/W5n059kH2pr/186ovsS33tp1MfZGfqcz+b+h57U9/7ydTn2J364E+mPsfu1Ad/MvU59qe++FOpj7FP9dWfR32Jvarv/izqO+xYffpnUN9g1+rjP4H6BPtWXz9XH4C6gFa9PrsOsN6eXfdXT88n/dGqIwjV0/NJf7TqCDr18nxRZ5Cph+eLOoNMPTxf1R1U6t35pg4hUs/OQV1Co16dg7qERD06L+oUEvXovKhTSNSj86JOIVGPzos6hUQ9Ogd1CY16dQ7qEhL16LyoU0jUo3NQl1CoN+dFnUKiHp0XdQqJenRe1CkU6s15VbdQqDfnVd1Cod6cF3UKhXpzXtUtFOrNeVW3EKgn50gdw7x6cY7VNYyrB+dEncO0em9O1T1Mq/fmVN3DtHpvTtU9DKvn5kwdxLB6bs7UQYyqx+ZCncSkemsu1U1MqrfmUt3EoHpqFtRRDKqnZkEdxZx6aRbUUcypl2ZJXcWcemmW1FVMqXdmSV3FmHpoltRVjKmHZlGdxZh6aBbVWUypd2ZRncWUemcW1VkMqWfmDXUYQ+qZeUMdxox6Zd5QhzGknplFdRZT6p1ZVGcxpJ6ZN9RhDKlnZlndxZR6Z5bVXQypZ2ZRncWYemgW1FEMqqdmQR3FnHppFtRRDKqnZkEdxaB6ahbUUcypl+ZS3cSgemou1U0MqqfmUt3EpHprLtVNDKqn5lLdxKR6ay7VTQyqp+ZCncSoemwu1ElMqrfmQp3EpHprLtRJTKq35kKdxKR6ay7USYyqx+ZCncSoemzO1UXMqtfmTB3EsHpuTtU9DKvn5lTdw7R6b07VPUyr9+ZU3cO0em9O1DlMq/fmWF3DuHpwjtU1jKsH50Sdw7h6cI7VNYyrB+dEncO4enCO1TWMqwfnWF3DuHpwjtU1jKsH50Sdw7R6b07VPQyr5+ZU3cO0em9O1T0Mq+fmTB3ErHptztVFjKrH5lxdxKh6bC7USQyqp2ZBHcWcemmW1FVs4+VrvXzBemeWRYFs6vIbNtvysbqVDVx8xWJXrlPHsoF6Um5Qx7K+elFuUdeyvnpRblHXsrZ6T25T97Kyek5uVAezsnpOblQHs656TW5VF7OqekxuViezqnpMblYns6Z6S25XN7OeeknuUVezmnpI7lJns5p6SO5SZ7OaekjuUmezmnpI7lJns5Z6R+5Td7OWekfuU3ezlnpH7lN3s5Z6R+5Td7OSekbuVIezknpG7lSHs5J6Ru5Uh7OSekbuVIezknpG7lWXs4Z6Qx5Qx/O4ekEedNfRvx7+u8ee8jD/vcYfg3fu+3ro6x7z+ryjB58+ZemnT0shHb94+PE8P/4YvPmvyxrPXuUp7IpoAHjL/wNQSwMEFAAAAAgAAAAhAOXUoXKiEQAAgFEJABAAFABncm91bmRfdHJ1dGgubnB5AQAQAIBRCQAAAAAAohEAAAAAAADt0r2OLOtZhuG1Uo5iZxskB4AsLHEAZCASAiJk8EYECKNtRAIcBSeMZ83Mmu7p6pn+qXrvxnVdWVX06Xnv//2bv/vrv/37r1/+88t//fibn373Tz//+Jc//Pjf//hnP/7ihx//+bc//8fPv/63f/jtz7/56en/X/36X3/30+///+5ffv3vP/3++4///Bc//PJXv/rFD3/xyz/9k1/88D8/XO+PvgAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAH8QvtYPYK++vqofws58PVG/iJ04TU+BTDhbngTZ3uf5SZCtXFafBNnEFflJkLVdmZ8EWdEN9SmQtdyYnwRZwR31KZB73ZmfALnH3fkpkJutUZ8AudFK+QmQW6yWnwC52or1CZBrrZufALnK2vkpkGts0J8AudgW/QmQC22SnwC50Eb9CZBLbJWfALnEdv0JkE9tmJ/++NSW/QmQz2zanwD52Lb5CZCP6Y/Q1vkJkI9s358AOW+gPwFyzkR++uOckf4EyLKZ/ATIMv0RmspPfywZ60+ALNAfobn8BMgp/VHSH6XJ/gTIO6P56Y93ZvsTIMf0R0l/hIbzEyBH9EdJf5TG+xMgB/RHaD4/AfJGf5T0R6noT4C80h+hJD/98UJ/lPRHqelPgDzTHyX9UdIfJf1R0h+lqD8B8o3+KOmPkv4o6Y+U/ijpj1LUnwD5Rn+U9EdJf4Sq/PTHE/0RyvITIF/0R0t/hML89If+SOmPUJmf/kj7E+Du6Y9Qm5/+9k5/lPRHKM5PfzsnP0JxfvrbOflRkh+hNj/97Z78KOmPlPxIyY+U/kjJj5T+KMmPlPxI6Y+U/Ejpj5T8iMmPlv5oyY+S/ijpj5T8KOmPlP4o6Y+S/Ejpj5L+KOmPlPwo6Y+U/ijJj5L+SOmPkvhIqY+S+kipj5T6SOmP0nr9CZAbCJCUACmt2J8AuZ4AKa3ZnwK51rr9CZDrrNyfALnK2v0JkGus3p8AucL6/QmQy23QnwC52Bb9CZBLbdKfALnQNv0JkItslJ8A+dxm8emPT21ZnwD52Mb16Y8PbF6fADlvIj/9sWykPv2xaKg+AbJgrj79cWIyPwFybLY+/XFouj4BciDIT3+8KOoTIM+i+vTHl7A+/ZHmJ8DdS+vT387F9Qlw1+r2ntQbkKnTe1HPQKLO7k29BPPq5o7UYzCsDu5EPQiT6toW1JMwpk5tWb0KM+rOzquXYXt1Yx+qx2FjdWCfqfdhS3VdF6gnYjt1W5epV2IbdVcXq4diA3VU16i3Ym11Udeq92JNdU03qCdjNXVKt6lXYx11R7erl2MFdUT3qLfjXnVBd6rn4z51P3erB+QudT4rqCfkDnU8a6g35GZ1OuuoV+RWdTlrqXfkNnU3q6mH5BZ1NSuqp+QGdTRrqrfkenUz66rX5Ep1MGur9+Q6dS+rqwflGnUtW6g35XJ1K5uoR+VidSrbqFflQnUom6mH5SJ1Jtupl+USdSUbqqflc3Ujm6rH5VN1Ihur5+UTdSBbq/flY3Ufm6sH5kN1HgPqiflAHceEemPOq9sYUY/MWXUaM+qVOaMOY0g9M2fUYQypZ+aMOowh9cycUYcxpJ6ZM+owhtQzc0YdxpB6Zs6owxhSz8wZdRhD6plZVncxpd6ZZXUXU+qdWVZ3MaXemWV1F1PqnVlUZzGmHppFdRZj6qFZVGcxph6aRXUWY+qhWVJXMademiV1FXPqpVlSVzGnXpoldRVz6qVZUlcxp16aBXUUg+qpWVBHMaiemgV1FIPqqVlQRzGonppTdROT6q05VTcxqd6aU3UTk+qtOVU3ManemlN1E5PqrTlVNzGp3poTdRKj6rE5UScxqh6bE3USo+qxOVEnMaoemxN1EqPqsXmvLmJWvTbv1UXMqtfmvbqIWfXavFcXMayem3fqIIbVc3Os7mFavTfH6h6m1XtzrO5hWr03x+oeptV7c6zuYVq9N0fqHMbVg3OkzmFcPThH6hzG1YNzpM5hXD04R+ocxtWDc6iuYV69OIfqGubVi3OormFevTiH6hrm1YtzqK5hXr04B+oYAvXkHKhjCNSTc6COIVBPzoE6hkA9OQfqGAL15LypWyjUm/OmbqFQb86buoVCvTlv6hYK9ea8qVso1JvzXZ1Coh6d7+oUEvXofFenkKhH57s6hUQ9Ot/VKSTq0XlVl9CoV+dVXUKjXp1XdQmNenVe1SU06tV5VZfQqFfnRR1CpJ6dF3UIkXp2XtQhROrZeVGHEKln50UdQqSenWd1B5V6d57VHVTq3XlWd1Cpd+dZ3UGl3p0XdQiRenZe1CFE6tl5VZfQqFfnVV1Co16dV3UJjXp1vqtTSNSj812dQqIenTd1C4V6c97ULRTqzTlQxxCoJ+dAHUOgnpwDdQyBenLe1C0U6s15U7dQqDfnuzqFRD0639UpJOrReVWX0KhX51VdQqNenRd1CJF6dl7UIUTq2XlWd1Cpd+dZ3UGl3p1v6gwy9fB8U2eQqYfnSV1Bp16eJ3UFnXp5ntQVdOrleVJX0KmX50ldQadeni97zk9/j6COIFRPzxf90aojCNXT82XX/QnwAdQNlOrt0R+tuoFSvT27zk9/vTqBVD0++iNVJ5Cqx0d/pOoEUvX46I9UnUCqHp+6gFa9PnUBrXp96gJa9frUBbTq9XevDiBWz797dQCxev7dqwOI1fPvXX3/Wr3/3tX3r9X77119/1q9/87V58/VB9i5+vy5+gA7V58/Vx9g5+rz5+oD7Ft9/V59gX2rr9+rL7Bv9fV79QV2rT7+A6hPsGv18R9AfYJdq4//AOoT7Fl9+0dQ32DP6ts/gvoGe1bf/hHUN9iz+vaPoL7BjtWnfwj1EXasPv1DqI+wY/XpH0J9hP2qL/8Y6ivsV335x1BfYb/qyz+G+gq7VR/+QdRn2K368A+iPsNu1Yd/EPUZ9qq++6Oo77BX9d0fRX2Hvarv/ijqO+xVffdHUd9ht+rDP4j6DLtVH/5B1GfYr/ryj6G+wo7Vp38I9RF2rD79Q6iPsGf17R9BfYM9q2/fqFfnuzqFVdQjcru6nUX1KIzREylBUdITAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAD79vXr1/oJ7NfXr98C/Lqgfhp/8JayEyFDPs9Pg2zlwvoUyAauqE+ArO26/ATImq6tT4Cs6Ib8BMhKbqpPf6zixvoEyBpuz0+A3Oue+gTIne7MT4Dc4e76BMjN1qhPf9xonfwEyC3Wqk+A3GDF/ATIlVatT4BcZe36BMgVNshPf1xsi/4EyIU2yU+AXEh/lDbqT4BcYqv89MclNutPgHxuu/z0x+c27E+AfEp/hLbMT4B8Rn+Ets1PgHxMf4S2zk+AfER/lLbvT4Ccpz9KA/0JkLP0R2miPwFyjv4ojfQnQM7QH6WZ/gTIMv1RGupPgCzSH6Wp/gTIEv1RGutPgCzQH6W5/gTIKf1R0h+lwf4EyAn9UZrsT4C8pz9SAqSkP0r6I6U/SvqjpD9S8qOkP0r6o6Y/SvojpT9S+qOkP1L6IyU/SvojpT9K+iOlP1L6o6Q/SvojpT9K+qMkP0r6o6Q/UvqjpD9K+qOkP0r6I6U/UvojJT9S+iOlP1r6I6U/UvojpT9S+iMlQFr6IyVAUvpjE2uGpT+uFOQnQF4l+emPF01/AuSbKD/98aTKT4A80R+hLj8Boj9SZX4C3L02P/3tXdyfAPetzk9/u1bXp799q+v7KsA9q9t7Um9Ap27vm3oEKnV5z+oViNThvap3oFF396regUSd3Zt6CQJ1dAfqKQjU0R2qt2BcndyRegym1cW9U8/BsDq4d+o5mFX39l69B7Pq3k7UgzCpru1UvQiD6tiW1Jswp25tSb0JY+rUltWrMKQO7Yx6FobUoZ1T78KIOrOz6mEYUWd2Xr0MA+rIPlBPw4A6so/U27C5OrEP1eOwuTqxj9XrsLE6sM/U+7Ctuq/P1PuwrbqvT9UDsaW6rgvUE7GhOq4L1BOxnbqti9QjsZk6rYvUI7GVuqwL1TOxkTqsS9U7sYk6q4vVQ7GJOqvL1UuxgTqqa9Rbsb66qWvUW7G6Oqnr1GuxtrqoK9Vzsa66p2vVe7Guuqer1YOxqjqnq9WDsaa6phvUk7GiOqZb1Juxmjqlm9SjsZo6pdvUq7GSOqRb1buxjrqjW9W7sYo6o9vVy7GGuqI71NNxv7qhe9Tbcb+6obvU43GvuqA71fNxpzqgO9Xzcac6oHvV+3GXOp+71QNylzqf+9ULcoc6njXUG3K7up011Btyq7qcldQz8uaac9TdrGbDPbnOFfeIo1nRtpNyuSvu0Sazro1X5VKX3yPtZXWbD8slLr9H2MoWBrblcxffI0xlGxPr8plL79F1spWZffnYhffoMtnO0MJ85LJ7ZI1samxkzrroHlUgGxucmTMuuEeVx/ZGl2bRp/eo4pgwvDWnPrtHVMaM8bU58fE9qjKGzM/Nex/do+piTDE4x87fo6piTDM4x84dJMtiTDQ4R86cJGpiUDc5hxav0iQxKh2dNwtnaYqYFa/Od+8Pk+QwrR6dN3ULhXpz3tQtBOrJOVTXMK9enAN1DPPqxTlU1zCvXpwDdQyBenIO1DHMqxfnQB1DoJ6cA3UM8+rFOVDHEKgn50AdQ6CenO/qFAr15ryqS2jUq/Os7qBS786X/cb3VX+R+uyPor7DHtU3fyT1LfaovvkjqW+xR/XNH0l9i32qr/446kvsVH32h1EfYrfqwz+G+go7Vp/+IdRHYN8d1uPzpK6gUy/Pk7qCTr08T+oKOvXyPKkryNTD802dQaYenm/qDDL18HxTZ5Cph+ebOoNMPTzf1Blk6uH5ps6gUu/Os7qDSr07z+oOKvXuPKs7qNS786zuoFLvzrO6g0q9O8/qDiL17LyoQ4jUs/OiDiFSz86LOoRIPTsv6hAi9ey8qEOI1LPzog6hUa/Oq7qERr06r+oSGvXqvKpLaNSr86ouoVGvzqu6hEa9Oq/qEhL16HxXp1CoN+e7OoVCvTlv6hYC9eQcqGOYVy/OobqGcfXgHKlzmFbvzbG6h2H13LxTBzGrXpv36iJG1WNzok5iUr01p+omJtVbc6puYlA9NQvqKObUS7OgjmJOvTRL6irG1EOzpK5iTD00i+osptQ7s6jOYkq9M8vqLobUM7Os7mJIPTNn1GHMqFfmjDqMGfXKnFOXMaIemXPqMmbUK3NOXcaIemTOqcsYUY/MOXUZI+qROacuY0Q9MufUZYyoR+acuowR9cicU5cxoh6Zs+o0RtQjc05dxox6Zc6ow5hRr8wZdRhD6plZVncxpJ6ZZXUXU+qdWVRnMaXemUV1FmPqoVlUZzGmHpoldRVj6qFZUlcxp16aBXUUc+qlWVBHMaiemlN1E4PqqTlVNzGp3poTdRKj6rE5UScxqd6aE3USo+qxea8uYlQ9Nu/VRcyq1+adOohh9dwcq3sYVs/NsbqHafXeHKlzmFbvzbG6h2n13hypc5hW782ROodx9eAcqmsYVw/OobqGcfXgHKprmFcvzoE6hnn14hyoY5hXL86huoZ59eIcqGOYVy/OgTqGefXiHKprmFcvzqG6hnH14Bypc5hW782xuodp9d4cq3sYVs/Ne3URo+qxOVEnMaoemxN1EpPqrTlVNzGp3ppTdROD6qlZUEcxp16aJXUVM+qVOacuYzv1slyirmRt9Z5cp+5lZfWcXKkOZmX1nFypDmZd9ZpcrU5mVfWYXK1OZk31llyvbmZF9ZTcoI5mRfWU3KCOZj31ktyirmY99ZLcoq5mNfWQ3KTOZi31jtym7mYt9Y7cpu5mJfWM3KgOZyX1jNyoDmcd9YrcrE5nDfWG3K5uZw31htyubmcF9YTcoY5nBfWE3KGO5371gtyjrud+9YLco67nfvWC3KOuZwX1hKyr7ula9V48EP3x/5X+AAAA4NH8H1BLAQIUAxQAAAAIAAAAIQD7uw7I1BEAAIBRCQAOAAAAAAAAAAAAAACAAQAAAABwcmVkaWN0aW9uLm5weVBLAQIUAxQAAAAIAAAAIQDl1KFyohEAAIBRCQAQAAAAAAAAAAAAAACAARQSAABncm91bmRfdHJ1dGgubnB5UEsFBgAAAAACAAIAegAAAPgjAAAAAA==', 'reference/sample-14.npz': 'UEsDBBQAAAAIAAAAIQDa5ONo6hQAAIBSCAAOABQAcHJlZGljdGlvbi5ucHkBABAAgFIIAAAAAADqFAAAAAAAAO3dy5JkR1KH8TpbnqJ3AjMtYGxsFjwAO7DZsGCFCUYYC4wZkzA2A0/BC6NWV2Xl5VzixAn3v7vH91upW50nI8I/nezKuuj//uEf//73/7S8/ffbn7/7w48//+tP3/3tl+/+51/+5rvvv3z3b3/86b9++uE///mPP/3hx6+//3c//MfPP/7y+z//+w9/+vGXX//lb77/8tvf/O77L7/77V//1fdf/vfLeX/xBgAAAAAAAAAAAAAAAAAAAAAAAAAAAIy0qBcAAAAAAAAAHFt4HwtCy0KB0FkIEELkByFuf5BZ3qnXgTktBAihhQChszxRrwdTec6P/uDoJT8ChJ+V/OgPbugPUvQHLfqDFP1BiPsflOgPSt79fX0GKscH1/641+KZYxMvrccMMOaqqlLe/2IGGHJRZUlfgEOOOuSiyvK8J70+V8RZR1xTZStVLCZflr/2RAETDLeg6ja6GJ2J6cXHCbWYGbTlZ9tfoJlHWssUGvu7PJcc/UVayyRc6jt4nutXHyTUYubQ11/HmBIEGG09M+jqr2dI4fsLt6Ap9OTX09+p6ytEW88k6O+baOuZBv39Ktp65rEWw+OvB/R3MnB/0dYzj9XW3n+5MYrzQzroTz/wYMuZydq574/i/KDoD1u273+bo6A/DHPUxm6E7V8yEjvAWKuZzXL3VVd7jSwvDzs3sbgBhlrMpI4L3O4ve4CR1jKrc/E9Peb6cwzcyXmBljKtjvxG9iedepyVzKsjv8/HDXmSEbvoE2QZc+tN49TM9itXzZ/4AnjpoHEoo/tTVEB+cv09nBtcyADpT+5SDmdG1xSgdwr0J3Y1BoMCqWEGT5/z7f3Z5L/+qSFvxBDhPF6iu/vNcxWcreVEgf0Jbl2HwqN47m/j/td4AzwxzDP99RaydaUxV8cAHyM46K/9UmefeXyAm5u4emFYeIzu43d6xnR+oJcTPHOB5otC4H4gvWM6P9ELsVzPjv4ieRxMT359E+0o50p6DxuivzDuB7M5r4aLdD7xmXSG5Ud/YTyUE7m/EU6vEWZWRnNhZD3zNY6tbyPwMXpsXUO2au3CPuBi+OA6p2wTW98W4Gf48HonbVNc1xbgZ/zwOidtlFzPDuBn/PR6R20UXccO4Gj89PonbZce8UUVaoDkN51QI6S/2QQbIf1NJt4MyW8mBlMcMG/am4XBKIfMnPzmYDFL+kMro1lenzv9zSLuLIlvCkvzFxVrFkd/xW1/1WmEcdJfdbH7u1KgaME4JXp/3QXK1otz8gyU/irKN9Fcq8W+hDeVREvFkXz50V8l+fKL/fESzslXX8ua1StEq2ztfZVsudiTL7+DNasXh1MS3v/orxD6g1JDfwEne1vS8vIlPOql4ZQT/cUebYY14sWZ/mIPN8ES8aJOf90/gAZChfr7KsESca9Yf0imuTzyg4H7/wMXd77ROLcj7f1xlhiv4Tswv32fSPibYPDlYdN7ZqlfhKMuL+q6Irm1lTfAmMu7nduy8k+k+SF/f0GXd/iKcu1gPx8ccfPNPg+A/sZaOcGRJ1unv7t/zNhf1NW9Vrd9whcuH3T3HTLnF3B5Z/q7+P8uM1i9wM7hxN1i3PWd6+/2b89culB8bzv9qRe2J+76zvb3cNx7e1qdTtRT2PQ6OPob6+Pw7o6xPcC3jaReK3bc0UArq6e/sZ6re1u7J26f+OrRr/zRuAew52EDT7+Tpb2vYq9xub//tb7+LrdUjy64LB+fIM3mzEmEFn2NzQd95ugfok5oxBnEEH6NJv09XNdpIyONOgS98Es06i/ZmJ7Qnxub/pJN6Vmp/GKv0aY//v4XQaI1thx1V39O2xir+UDUC931fv4JFtlQ3d2ffWu5r2fOr8i7L+EX+O7ghJ+qe2py96o+6x+sOb7oGwy/wA9n+2uaQJbNv+oOMNiG4/8Xsu/uKy+7Hp1275ux3W1p47/HQH8nPLg/lPfxaVPxMjqsd/W6kYZbolCs1fhL21/Ta+3OHxQseU2ktShk3ftWfvQHF+1NRe4v0lpwSntSge+AcVaCkz47Ouppo78Ac4+yDpz31F/Ln6Q/jHEmos385HOPsg6cNaY/8eCDLAPnnUmI/jDYuYSiBhhjFTjvZEH0h6GG9ScdfYhFoMPpgCIWGGAJ6NQxO/rDMMv5927D3QD1K0CnpfHbC54fRH8YoauezRugaP7q50e/ztmFCpD+EusdXpx7oDp/XNE9vO1XYecM6C+z/uFFCZD+ErsyvBj96W68uO7S8ELcAekvs2vD2wnQqwXZfRcDLHc/6Kvv4eIAVXddDHGxP32A9Jfa5eHtBujQA/3lNmB4ygAFd1yMNWR4qgBfno4GsxmTiiZAyT0XIw2a22Z/pjnQX3qj5iYJ8PW5yC+ZgZm4B8i9r4CBo3MO0PdeCxsjR7cVoEkXrrdaWBk7OscA6a+EwaPzC3DnWegwDYNA5P1xH8zDKBCHALfutDa1w4bZyDQ3QPpLxu3+NPz6Ub4BAFf4vT4Of4JI3wSKTrYjM05iIz76y8P6luHfn8UTwYr11OgPexym5vP6Tn8puUzt6QnGPRn9ZaeY2bjnI7/sJPkNDvDtPsRRV4aPGv3xM/Czyt3f2+1a9JcT/UHKfWR2/b3xpX84NPjvf58XpT20oD8IDSxleX75BY7QH5QG97eMvigqG/o2ye1avPeCNgb98bkPtDO4/5Efmg1N5f1i9Idmw/vj5RdnmH3936jLAi0W+oPSrTr6gwL9Qejx/Rf6Qxujn3Y55qrAkbt3/egP3t5joz9IPH7YQX9wdWuN/iCw0Z94VZgF/UGJ/qD0GRv9wR/3Pwgta/3xow/g5Ha3494HhZX7n3hFmMpncPQHf/QHJV5/ofTSn3g9mAwvv1CiP+i8vv8iXhBmsvL5D/GKMJNbcXz4CwXuf1C4ZfbYH//Dabi474/vOofMck+9GMxhebj90R98LRsBKteEedAflOgPSg+p0R+cbfSnWxDmQn+Igv4gRXtQoj8o0R+U6A9K5Achbn8Q4q0XKH378VfqVWBW3PugRH9QIj8IcfuDEB/6Qon+oER/0OGtZ0jwXUcQ4rvOoUV/UCI/SNEflGgPSvQHJfqDEv1Bifzg7K44PvaFt+W1P+l6MJmH4ugPzh6Coz84oz8ILbz+QuihuK//RH9w9NjfG/3B0/LSH9/6CydPX2zF3/3g6uHLTXnvGc6WV+olYSL0ByX6gxL9QYn+oPRLbvQHkW9vNtMfFN5joz8oLOv9qZeFSXzkRn8QWOgPQp+53cdHf3Byu9tx74O3++LoD74eiiM/OKM/KD0WR3/w9fi1zvQHX7evOnj/FfHB08rrr3hFmMfzZzvoD56e8qM/eHq+/dEfPH3L7fn9P/WqMIeP7pYn6nVhDh+10R8UHj/qJT/4en3Xmf7gZ+WzHvQHN7zrAiX6g9JndfQHhbsfNU5/cMbrL3Q+3+/jfT/447MekKI/KD29/qqXg8lw+4MS/UHpoT36gzfufVAiPyjRH4R4+YUS/UGK/uDvMzf6g7/l/otOyQ/ObskRHwRun/GgPwh8VEd/UKA/aPFlL9AiPyg8fOhLf3BGf1DivWdIPd3+6A/OaA9K5Act+oMG/5M3KN3/uDX6gzf6Qwj0ByX6gxT9AZgQr70Q4r0/KJEfpOgPIh/fdqleB+ZEf1CiP6jRH5ToDyoLb/9Bhy+9hxL9Qepbe/QHjY+ffKVeB+bESy+U6A9K5AcAADAN3naGEJ/2gBT9QYv8IMTtDzq8/EKJ/qBEf1CiPwAAAAAAAKAS3u4DAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAUMLyQr0izOO1PgqEl/X6CBAuNvOjQJjbq48CYeuoPhKEoab8CBBGGvsjQFhozY/+YKA5PwLEcCfqI0GMdjY/CsRIHf3RIIbp7I8CMUR3f2SI667mR3+44nJ/BIh+1/OjP/SjPwgNyI/+0I38IER+UKI+KJEelMgPQuQHJfKDEv1Bif6gRH9Qor+yUgypv7/4e5tcihnRX1kpZkR/ZaUYEf1Bif4gRX9Qor+yUoyI/srKMSL6qyrLiOivrBzTor+yUoyK/spKMSj6gxL5QYn+IEV+UKI/KNEflOgPQh350R+GoT8okR+U6A9K9AehjvzoD2P0xEd+GIX+INSXH/3hqs7y6A9XXUqP/nDB9fboD93IDzpj6qM/9BhVH/mhA/VBiPogNCg/+kOPUfnRH84bVh8B4pyR6dEfzhjeHgHiBPqDkk1/BIgmRvlRII7ZxUeAOGSbHwFiH/1Bybo/CsQO+/woEBtc4iNArHPLjwCxwrE/CsQL+oOQa34EiCfO/REg7nnnR4C4458fAeJGkR8B4gP9QUiTHwHiG/qDkqo/AsRX9AchWX4EiDdpfxQIaX4EOD36g5A4P/qbmzo/ApyaOr6F/mambu9X6kOAjDq9b9SnABV1ee/UxwANdXcf1OcABXV1d9RHAQF1dPfUZwF/6ubuqc8C/tTN3ftlNerjgC91co+MdmhxWQyhLu6R0QYNLosx1MU9strd56/vfnf0k6GDKLQNZpv7/NX9bw9+OpznHNgRq709NGf0dOjgl1Yb86290V8gPlGdYb2tx3875NnQzbqm865u52hXT/92yCmil2VJfS5v52BbL/9yyDmij11HvS5vZ3VbO1sdcYzoYphRt4sbOr+vEQeJHkYFXea7qzFnidMs0hnDc1uDDhOnmaQzhOm2+AA4CLN8LrPc19OfunqI6GaeUTfDfd3/uWtPg4ucYupguLsRl8YQrkmdYrjDUZfGZe5ZNTPc4ahL4zL3rJrZbXLglXGNpKw2Hns3fw7sU0e2x2Xv1k+CXeLEdvns3fpZsEdb2D6XzZs/CfaoG9ulPhyYUye2S304MKdObJf6cGBOndgu9eHAmrqwferTgTV1YUfU5wNb6r4OqQ8IltR1NVAfEeyo22qhPiPYUbfVQn1GsKNuq4X6jGBH3VYT9SHBjDqtJupDghlZT6cfgIqMCjvO6fwjUJBNYA0x9TwG5ZgE1pJS14NQjUlhTSn1PQq1mBTWFFLnw1CKSWJtIXU+DKXYRNYSUufDUItNZS0d9T4OpRiF1hLRlceiCKvYGhO69GDkZ1lcS0KXHoz0TJNrSejao5GdaXMtAV19PHIzra6ln8sXQHKm4R3Wc/0KSM00vON4BlwCqZmmd9jOiGsgM9P2jtsZcQ0kZhvfYTpDLoK8bOs7LGfMVZCWbX5j+vM4B2ho+xtyESRmm9+Qjz7orzDT+ka897x/ESRnl95BOiOugfTs0ttvZ8Q1kJ9ZeQftDLgECrBLbyee61dAEVbhbddz+QIoxCa77XquXwGVWFS3U8/lC6AUi+j28rn6eFRi0txWPxcfjnqG97bZz8WHo6LBte0EdO3RqGlwbTsFXXowihqd23ZD3Q9EYQbBbXX0khTtTc8hv46vf/HbP7Q8+jsZoN/mIRevP7+9Q84lv1NfAO22dejJ83tcgtOuEYRPfi1Zkd+M4vSHGTn1R4BYRX9Qoj8o0R+U6A9K9Acl+oMS/UHIKz/6wxq3/ggQK+gPSvQHJb/+CBCvHPsjQLygPyh59keAeEZ/UKI/CLnmR4B4Qn9Qoj8o0R+U6A9CzvnRHx5490eAuOOeH/3hDv1ByD8/EsSNJj8CxDf0ByX6g5KqPwLEmzA/+sObsj8ChDI/AoQ0P/qbnTY/Apwd/UFInR8Bzk1dH/3NTV3fQoBTU8e30N/M1O39Sn0IkFGn9059DBBRh/dOfQzQUHd3oz4ISKizu1EfBATU0d1TnwXcqZN7pD4NeFMX90R9HHCmDu6Z+jzgSp3bC/WBwJU6t1fqE4EjdWwr1EcCP+rW1qjPBG7Uqa1Tnwq8qEtbpz4VeFGXtkV9LnChzmyb+mTgQV3ZDvXRwIE6sh3qo4EDdWQ71EcDB+rIdqiPBvbUje1SHw7MqRPbpz4dWBsYyqBLvVwXhY2sZMi1Vq+MmgY3MuRyWxdHOQaBjLjkzuVRiUkfAy66/wSowiSOAc0dPQVqMGpjRHZHz4H8zNIY0d3hkyA7uzDGpHf4NEjNsItB8R0/EdKyrWJQffRXlnEVg/Kjv6qsoxgVIAVWZB/EqPwIsCCPHEblR3/1+ORAgFjnFAP9YZVXDPSHFW4t0B9W+JVAgHjl2QH94ZlrBwSIJ74VECAeOTdAf7jn3wAB4pOgAPrDjWD+9IcbxfgJEO8006c//Eo3fPoD/UFKOXz6g3L2V/ojwBLEkyfAyannToBzk4+d/mamH/uF/ggwPf3Q6W9eEWZOf9OKMXMCnFWMidPfpIIMnP4mFWTe9DenKOOmvzlFGTf9TSnMuOlvRnHGTX8TCjTtK/0RYEqxRk1/kwk2afqbS7RJ099Uwg36Sn8EmE28OdPfTOKNmf4mEnHKBDiPiCOmv3lEHDH9TSPkhOlvElEHTH9ziDpf+ptC2PHS3xTCTpf+phB2uvQ3g7jDvdIfASYReLT0N4HAo6W/+kKPlv6qiz1X+isu+Fjpr7jgU6W/4oJPlf6Kiz5U+qst+kjpr7TwE6W/0sJPlP4qiz9Q+qss/kDpr64M06S/snIMk/6qyjFL+qsqxyjpr6ock6S/qnIMkv6KSjNH+ispzRzpr6Q8Y6S/ghKNkf7qSTVE+qsm1wzpr5pkI6S/WrKNkP5qyTZC+qsl3wTpr5KEE6S/QjIOkP7qyDhA+qsj5fzor4q046O/EvJOj/4KyDw9+ssv8/ToL73U0zvRX9QtTC778OgvtfSzo7/U0o+O/lLLPzn6yyz/4Ogvs/yDo7/ECsyN/hIrMDfyS6zA3Fr7C7+RCVUYG/1lVWNs5JdUkbHRX1JFpkZ/SRWZGv0lVWVo9JdSmaHRX0Z1ZkZ/+VSaWaW9zKLSzCrtZRaVZlZpL5NoHFmOmVXayxxa88sxs0p7mUJzfjlmVmgrUyiWH/3l0p5fkplV2kt9J/LLMrNKe6muYH5lPpE9g4L50V8eJ/JLNLJSm6msZn7cAHM4U1+qeVXbT1Fl++MFOINT+eWaV7X9lFQ3P/rLYO7+ku2ooML9cQOMr258b/SXAP1BqXJ+/AUwPvqDUuX8WnanXuHkKsf3Vd2d1VC7voP9qReHGV6hym6shtVhlBrSanQVNlZEjcoAZPf/UEsDBBQAAAAIAAAAIQAKD4lCnA8AAIBSCAAQABQAZ3JvdW5kX3RydXRoLm5weQEAEACAUggAAAAAAJwPAAAAAAAA7dy9jiVHHcbhnZSr2GxBcgCW5YALIAOREBAhg40IECAbkQBXwQ3jWe+u5+N0n/6oqreq+nki2zrdU/Wvn2tmx5L/95vf/fq3v3948683/3739Tff/enbd798++4/f/zFu8/evvvz37/957df/e0Pf//2628e//mvvvrrd998/8+/+8tX//jm+7//6eefvf3i8y8/e/vlFz//2Wdv//t2v5+8AQAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAICOPDykV8B1PTx6/xfpldzR+/o45OGZ9GpW9Lw2Dnt4GKPAflfGKYME2O3COGmIAHtdF+eNEGCny6KEAQLsdFmc9KG31wF2dtr6m9PLq+9jgF2dd7/XMucs5NfZefe2HkpZ7a+fA+9sOZSjP7IGCLCv1VDYIP11sxwK6znAvlZDcX3/BNjXaihuLb/8ife1GkpbrS9+5D2thQru5Zc9857WQhVd9+f+u4CeA+xoKdSkP6L67M9vX66i/wswug5qWw4wevJdLIL6VvpLnn58ATSyHmCqAPVdxr0AIxmo7zruB9i8BfldyZYAW+agvovZFGCzJuR3PRsLLNvFjbdFvtuTtzXAUnE8e1nkmqUv2ws8W0iLvhnOjgCPV1I5bUa2q8D9qdSrmjnsC3Cll70vkh/vnWjmWHPq45kj4ZxNT318UiAm+XGC+ohSH1nyI0p9ZKmPLPkRpT6i1EeW/IiSH1HyI0p/ZMmPKPkRJT+y5EeY/siSH2H6I0x+pOmPNPnRDf2RJD+i9EeS/IjSH1HyI0l/ROmPKPmRpD+inqX2+Bfyo6HXnemPrH7y62IRtNZNfzeuZ+bXTX41r+Ju9sgL/Vx/FX8Y7WePPNdRfi9+QVT6xSXfRyH9/OnjUa3fS3a1ST7p7ncvq78hP7rU7nbJB92dzN3+Diy3u03yQYcnsynAXWu++UB/G7+igfvbvO6Xnz11lVJUh4ewp78ta//0sTMNU0WXR7A3wPUNnO6Xero8ggP9LTclv451egQFAtwqvdVr6/QQmsUnw6xOp9+qv6cp0l63s9ffJXQ7e/1dQcej198FdDz65v11OYXJdTx5+c2v68nrb3pdT1520+v7CIQ3uc4PQn6T6/0k6gbY8cavYYhT0OC0hjmBk3FpsE8jTn9Pd1sfbbwFfjD++PeGpL+eXHL819txr/zrT5L+CPLjD0nyI0l/BPn2S5L+CPLbV5L0R5D/+kSS/gjyX99J0h9J8iPI9UeS/gh60B9B+iNJfyTpjyT9kaQ/gh70R9CDAMl5mZ/+aOhVfvqjIf0R9Do/AdLMrfz0Ryv6I+hmfvqjkdv9CZAmFvLTH03ojyT9EbSUnwBpQX8ELeenP+rTH0kr/QmQ6vRHkv5I0h9Ja/0JkNrkR5L+iNIfSfIjSn8k6Y8o/ZEkP6L0R5L+SNIfSfIjSX8k6Y8gP/6RJD+S9EeS/giSH0H+9EHQ7fz0Rxv6I0h+BC3kpz9aWMpPf7SgP5LkR5L+CPLtlyT5kaQ/gnz7JUl/JMmPINcfQYv56Y/65EfQcn76ozr5kaQ/kuRHkOuPJP2RJD+CXH8kyY8k/REkP5L0R5L+CJIfSfojSX4Euf5Ikh9J+iPIt1+S5EeS/giSH0n6I0l/BMmPJP2RJD+CXH8kyY8k/RHk2y9J8iNJfwTJjyT9kSQ/glx/JOmPJPkR5PojSX4k6Y8g+ZGkP5L0R5D8SNIfSfIjyPVHkvxI0h9Bvv2SJD+CXH8EyY8k/RG0kJ/+aEJ+BLn+SJIfQa4/guRHkv4Ikh9J8iPI9UeS/AiSH0G++5IkP4LkR5L+CJIfSfojSH4kyY8g1x9J8iNIfiTpjyD5kSQ/glx/JMmPIPmRpD+C5EeS/Ahy/ZEkP4LkR5L+CJIfSfIjyPVHkvwIkh9J+iNIfiTJjyDXH0nyI0h+JOmPIPmRpD8AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAgC0enkmvhst4uCm9KuZ3uzwB0sRqfgKkrjv5CZCq7vYnQOq5n58AqWZLfgKkkm35KZAqNucnQMrbkZ8CKW1ffgKkqL35yZCSjvYnQgo4lZ8COedsfgrkhAL56Y+jSuSnPw4qkp/+OKZMfvrjGP2RpD+SxEeS+EjSHknyI0l+JOmPJP2RpD+S9EeU/kjSH0n6I0l/JOmPKP2RpD+S9EeS/kjSH0n6I0l/ROmPJP2RJD+S9EeS/kjSH0nyI8j1R5L+CJIfQQfz0x9F6I+go/npj/MO16c/TjtRn/446VR98uOUc/XpjzPO1qc/jjtfn/w4qkB9+uOgIvXJj0PK1Cc/jihUn/w4QH3kqI+cUvXpj/3K1SdAdiuan/7YpWx9AmSX4vnpj83K1ydANquSn/7YpE59AmSTavkJkPsq5idA7qranwBZVzc/AbKqdn4CZE39/gTIogb5CZAlTfITIAsa9SdAbmmVnwC5oV1+AuSVlvkJkJf0R1Db/ATIM63zEyBPte9PgHwSyE+AfBTJT398kOlPgLwXyk+APIrlJ0DeRPsTIMn89Hd50fwEeHnh/gR4ben8BHhp6fge9Hdp6fgepWdATDq999JDICVd3gfpMRCSDu+D9BjISHf3SXoQRKSz+yQ9CBLS1T2RHgUB6eieSI+CgHR0T6VnQXPp5J5JD4Pm0sk9kx4GraWLeyE9DhpLB/dCehy0le7tlfRAaCqd2yvpgdBSurYb0iOhoXRsN6RHQjvp1m5KD4Vm0qndlB4KraRLW5AeC42kQ1uQHgttpDtblB4MTaQzW5QeDE2kM1uWngwtpCtblp4MLaQrW5EeDfWlG1uTng31pRtbk54N9aUbW5UeDtWlE1uVHg7VpRNbl54OtaULW5eeDpWlA7sjPR4qSwd2T3o+1JXu6570fKgr3dc96flQV7qvu9IDoqp0XnelB0RVua42f5CJ1UxsPavNH2RiNRO7U9X2TzKtqpGtV7Xjo8yqZmN3mtrzWSZVM7J7Te36MFOqGdm9onZ+nAlVjOxuUDs/zoQqRna3p90PMJ16kW2o6cAjzKVeZRtiOvIMU6kW2ZaUjj3FTGpVtqmkY08xkWqZbQnp6HNMo1ZmmzI6/iSzqNTZtopOPMocKnW2taEzzzKBaqltbejUwwyvXmxbCzr3NGOr2dvGgE4+zsiqFrctoJOPM7SqyW3q5+zzjKxqc5vyOf8GxlWzuW31nH8DA6sZ3aZ4CryCcdWsblM8BV7BwGpmt6WdEu9gXDW725JOkZcwrJrhbUmnzFsYVsXwNoRT6DUMq154W8Ip9BqGVS+8Dd2Ueg/Dqhbelm5KvYdhVQtvQzal3sPAqqV3v5tCr2Fk9dpb6qbQa5hC+egWuyn0GmZSsrfVcIq8hNmU7G0tnRLvYEIlg1tpp8ArmFDR4JbjOf8GplQ0uOV6Tr+AOZUtbimfs88zqbLBLfZz7mmmVTq55YLOPMukqmS3GNHxJ5lTtfKWKjr6HFOqm9/B/+1GywGQVDm/OyWp7+qy/d38+i22TR9q57e/vxa7phe99ddiz/Sjen/3ipLfldXPb09/TbZMR/L9vdn4MWbUoD9lsaRFfvpjif5I0h9JTfoTIAv0R5L+SGrTnwC5qVF++uMm/ZHUqj8Bcov+SGrWnwK5oWF/CuSlpvkJkBca9ydAntEfSa37EyBP6Y+k5v0JkB+1z09/fBLIT4B8oj+CIvkJkA/0R1AoPwHyKJaf/niT7E+AJPPTH9H+BHh50fz0d3XZ/AR4dfojKJ2fAK8tXd+DAK8s3d6j9AzISbf3XnoIpKTL+0F6CoSkw/soPQcy0t19lJ4DEensfpSeBAnp6p5Ij4L20s09lZ4FzaWTey49DVpLF/dceho0lg7upfQ8aCvd20vpedBUOrfX0hOhpXRtr6UnQkvp2m5Ij4R20q3dlB4KzaRTuyk9FJpJp3Zbeiq0ki7ttvRUaCQd2pL0XGgj3dmS9FxoI93ZovRgaCFd2Yr0aGggHdmK9GhoIB3ZmvRsqC6d2Kr0cKiuaC4lX/bDG5lc2VJKve35W5lW6UqKvO/2q5lQ8UAKNLf2euZSvo8S1a1+AeZRo44S1a1/BWZRJY4i3a1/CeZQJ44i4d35GsygUhplyrvzRRhetTLKpHfvqzC4emEUiu/u12FkFbMoVZ/+5lW3ilL9CXBWlZvQH6tqN6E/VtRPQoAsaxCE/ljUogf9saBNDgLktkYx6I+bWsUgQG5ol4L+eK1hCfrjpaYhCJAX2magP55pnYH+eKJ9BQLkR4EG9MdHkQQEyAeZAPTHe6kA9Mej2PkLkGB+R7+2/qaSP3z9XVgPZy/A6+ri6LtYBAGdnHwny6Cxbs69l3XQUjf56e+K+slPgBfUU35HVlNvLbTQVX4HllNzMVTXWX4CvJYD+dU+8M6WQ0Ud5ifA6ziSn/4opM/8dq+rwYqo4FB+TU67vxVRXrf56e8KDuXX6LA7XBJl9ZyfAKd3LD/9UUbf+e1bX7tVUcix/FqedJ+rooj+89PfxA7m1/agO10W542Qn/6mNUR+ApzVIPnpb1Kj9Ld9oe2XxmHD5Ke/KQ2T3/alRhbHIQPlp78JjdSfAOczUn76m85Q+elvOkPlt3m5wRWyx5H89Ecpg+W3dcHRJbLZcPnpbyrD5SfAqYyXn/4mMmB++pvIgPnpbx4D1qe/iYyY38ZVpxfJfWPmp79ZjJmfb8CTGDQ//U1i0Pz0N4lB8/MD4ByGzU9/Uxg2P/3NYNz89DeDcfPzB5AJjFuf/iYwcn76G9/I+elveEPnp7/hDZ2f/kY3dn76G90V+ksvkkWD56e/wemPoI359XuCY6/+8kbPT39D25hfxwc4+PKv7SL5dbz+a7tIf+lFctv4+elvZPojaIL89Dcw/RE0Q376G5f+CJoiP/2Namt+nR/e+Du4qDny09+gJslPf4PSH0Gz5Ke/Mc3S3xSbuJ5Z8tPfmGbJz7ffIW3Mb4CTm2MXFzNPfvobkf4Imig//Q1IfwRtzW+Ic5tlHxcyU376G87W/MY4tmk2chWb8xvj2GbZx1Vsz2+Mc5tlHxcxW376G8qO/AY5tmk2cgV78hvk2KbZyAXsym+QY5tmIxcwYX76G8eu/EY5tXl2Mrt9+Y1yavPsZHI78xvm1ObZydRmze/+xtIL5M3+/AY6tYm2Mi/9kTRvfgIcwN78hjqyqTYzp5nzu7+79Pq4dn+DbWc+c+fnAuzd3PW5ADs3e34uwL5Nn58LsGuz1/fGBdi9ufNzAY5g5kOaeGtTmfWM9DeQGU9IgGOZ7Xz0N56pTkeAQ5rmZJS3w/8BUEsBAhQDFAAAAAgAAAAhANrk42jqFAAAgFIIAA4AAAAAAAAAAAAAAIABAAAAAHByZWRpY3Rpb24ubnB5UEsBAhQDFAAAAAgAAAAhAAoPiUKcDwAAgFIIABAAAAAAAAAAAAAAAIABKhUAAGdyb3VuZF90cnV0aC5ucHlQSwUGAAAAAAIAAgB6AAAACCUAAAAA', 'reference/sample-15.npz': 'UEsDBBQAAAAIAAAAIQCS+hxReAgAAPB8BAAOABQAcHJlZGljdGlvbi5ucHkBABAA8HwEAAAAAAB4CAAAAAAAAO3dwW6c5RmG4XjLUXjnImVBUbtoD6C7IjZddIVCSdUFKihBbICj4IRLHAVISOMM/t7vfT8/172xPB7/o/+5/Hu8mfGPn/zj75/+8+rRt4++u/ni6fN/Pbv56/XN95//8ebx9c2/v3r2zbMn//3sq2dfPH1x+9+efPn86U+3P//Pk6+f/vT5Hz5+fP3xX/70+PrPH3304ePrH64v74NHWtjV/+m1O7zrjutr2yKju/ffYPzmQ6qy/aDvUfcoD71u37fUPcmDrxv4zbr3iKgb+fW618ioW/m1useIqRv657qHiKob+0XdG8TVDY68I+aZMY+MeGDM82KeGfLYkEfGPDHmqSHPDXlezANjnhjyxJjnxTwv5IExzwt5YMjzYp4X8ryY58U8rlXkzI8JeV7I82KeF/K4FpEzPyfkeTGPaxU583NinhfzuJaRMz8m5nkxz4t5XszzYh4Y87yYx7WMnPkxMY9rHTnzU2Ie10Jy5ke0Upz5Ea0lZ35Ai8mZz281OfPxLSdnPrz14siHV0DOfHYV5MxHV0LOfHA14sgHV0TOfHDIwyoCZz63OnLmU2MeVyE584lVgjMfWTE583lVkzMfF/K4ysmhT2sDOfNZIY9rBznzUW0hhz4q5nFtImc+KOZx7SJnPqdt5tDHxDyvfebQx8Q8L+Z57TOHPiXmed1TjvmB3Uuc+YndT/yyA2w/Ob21e1+nzI/r/mDQT2uBF/PDWqDF/LBWYDE/qyVSzE9qDRTzk1oExfygVkExP6dlUNCPiXlezONayAT9kBYaMT8k5nmtJGJ+RszzWirE/IiY57UWCPoJrdVhfkJrcZif0GIc5vNbbcN8fqtpmM+PeV7M81pOA31862GYT68ABvrwmOfFPK4Kl7vNoXdWwsJ8dCUqzEfXZA69swoT5rOrIHkPc+iNlYgwHx3zvGpEmI+uyRx6YyUezEdX48F8cl3m0PtinlcRB/PBMc+rzRx6W8zzqtJgPrcyDeZjq9NgPjXmefWZQ++q0Rx6U4UWzIfWaQ69J+Z5VVIwH1mtBfSJMc+rmIL5vKopmI+rnIL5uJjnxTyvegvo06q3YD4t13lezPPaQAF9WMwD2yDBfFrt5tD3146+8JH0njHPi3lezPMqp2A+rnoK6NNinhfzvDZIMB8W87x2SECfFfPA2s2hb495XjscoA+LeWD1CszHxTyvbnPoLdUKMB9ZLQDzkZXuz3xoheszz+sOc+gPMeZ5MQ8Mel7M87rDHPoDjHlgzPNinpdf7nkxz4t5YNDzussc+sOLeV7MA4OeF/O8mAfGPC8Xel7M82KeF/PAmOfFPC/meXlCz+tOc+gPLuZ5MQ8Mel53m0N/aDEPjHleLvTAmOfFPDDomTGPjXlgzPNiHhjzvJgHBj0v5oExz4t5Xszz8oSeF/O8mOfFPC/meb3VHPrDjnlezPNinhfzvJjnxTwv5oEhD4x5YMzzQh4Y88CQJ0ZckiRJkiRJkiRJkiRJkiRJkiRJkiRF9fKFlV5Vm5Q3TJjWK4Sr39x89eoSffTbl7///F2/fPnXH385jPfCGdgbv3gX9M7/6tx9urptKfg7fxC6z1Sv2kOuSSFPDHhiyANjnhfzvJjnhTwv5nkxD4x5XsjzYh5XETnzuVWRM59amTjzsTHPi3lcheTMh4Y8L+ZxFZJDHxrzuErJmY+MeV7M82IeVy0584kVm0MfGPO4qsmhz6veHPq0mMe1gZz5rHaQQx/VHnLmg9pEDn1M28SZD2mjOPMZbSVnPqG95MwHtJmceX+7yZm3t52ceXf7yaH31iHOvLMeceSNNZFD76uNnHlTfeLIm2okZ94U87g6yZm3RDyuTnLmLbWSM2+JeV7I42olZ94S87yYx9VLzryhZnLmDTGPq5uc+f66yZnvr5uc+f66yZlvr1uc+f66xZlvrxv8ivn2usGvmO+u2/tF3Ruk1e39ou4Nwurmvq17hLC6uW/rHiGsbu7bukcIq5v7tu4RsurWfln3Cll1a7+se4WsurVf1r1CVgu5mJ/ROvF7Hazp7DNbSc78iFaKMz+jheD3Otzu805uLfnvP97es86OeVyLxZkf0Hog5tNbz8N8eszzKtBhPrwCGubDq5BhPjvmeZWwMB9djQrzydWYMJ9cDQnzyRWJMB9cEQjzuVWBMJ9bmQfzsdVxQB8b87gKMZgPjXlehRaXmkPfU6UE85kxj6sWgvnAqiGgz6ucgfm4mOdVrsB8WhsQoM9qBwHzWTGPawvBhebQa9sjAH1QmwAuNYdeGPO8du0PfUzb1mc+pY3rMx8S87h2rs98RHvXhz4h5nkxz2u2OfSCdm8PvT/meTHPa/v00LtrWJ55d8zzYp5Xw/LQm2OeF/O8mAfWMDz05pjn1bE78+aY59WxO/TemAfGPDDmiW1fnfmANo9+mTn0kpjntXty5gNintfuyZn3t31y5u0xz4t5YNDzYp4X87z2T868u9Hm0EtqWJx5c8zz6hiceXPM8xptDr0k5nl17M28N+Z5Mc9rtjn0klrmZt5Zz9wu9M6mm1NfX9PWzPvq2voCc+hra9v6EnPoS2Oe1xnm0BfWuDTzpk4xh74u5oE1Ds28KeZ5Mc+rc2foTTEPrHFl5k0xD6xv40vMoS+tbWHmbZ1hDn1lbfsy76ttXeaN/WrZndsyb61nWOit9ezKPC/mgTHPi3lezANjnhnzzJhnxjww5oExz4t5XswDY54X87yY58U8L+aBMc+LeV7MA4MeGPO8mAfGPC/meTHPi3lezAODHhj0vJjnxTwv5oExz8uFnhfzvJgHxjwv5nkxz4t5Xv6Iy4t5XszzQp4X87yY58U8L+Z5Mc+LeWDME2MeGPO8kCdGXJIkSZIkSZIkSZIkSZIkSZIkSZIkSZKk+/Y/UEsDBBQAAAAIAAAAIQD9X6u7QQgAAPB8BAAQABQAZ3JvdW5kX3RydXRoLm5weQEAEADwfAQAAAAAAEEIAAAAAAAA7dyxbh7EFoXRpL1P4c5cKQUgKC4PQAeioaBCAYIorgAliAZ4Cl4YEKJAmDj/7zlzznivr4slTzJ7ZRKncH79+NOPPvns6ZMfn/x0+9WLV1++vP3g5vbnL965fXZz+/V3L394+fzbz797+dWLPz7+4fP/v3rx+8dfffP8+xe///itd5/dvPu/957dvP/22/99dvPLzeX954mkRT19w7p/nVrYm6K/ed030n2tN/dbY3rM80KeF/O8kAfGPC/keTHPC3lizANDnhjz7JhHhjwz5pkhz4x5YMgDYx4Y8sCYB4Y8L+aBIc+LeV7IA2OeF/O8kAfGPC7kaS0QZ35US8SZH9QiceTntIqc+TEtI2d+TMjjWkbO/JiYx7WOnPkpMc+LeV7I82KeF/O8mOfFPC/meTHPi3lgzPNinhfyvJjnxTwv5mmtEmd+TOvImR/SQnLmZ7SSnPkJLRVnfkKLyZnPbzU58/EhT2u5OPPpFZAzn10FOfPRlZAznxzytGrEmQ+uipz52JDHVUbOfGh14syHhjyuSnLmIyslhz6wYnHm8yonZz6tenLmw9pADn1WW8iZT2oPOfNJbTKHPijmeTHPa5c59Dkxz4t5Xg80hH5g12Jf/ulNF9Q/egj4ZZ/fcDnd2cPELzpg99X0Lz1QnPmBPVyM+Wk93Iv5aS3Qgn5YK6yYn9UKKuZntUSK+VGtgYJ+UmuYmJ/UIiXmB7UIiflBrTJifk7LiKCf0joh5qe0EIj5ITHPa6UP8zNiHtdSnjcwh97fWhzmJ8Q8r7U2zE9oMQ3z+a2W8dDnt9qF+fyWszAf33IV5uNjHleBCvPhFaB46MNjnleFCfPZMY+rxORec+id1ZAwnxzzvGpE7jWH3lkNCPPJMc+LeV5FINAHxzwv5nlVeUAfHPO8mOdVxcF8bl3m0PtinleZBvO5eeh5Mc+rTIP52Jjn1WYOvS3medVhMB8b87j63jn0rgoxmM+sEoP5zJjHVYvBfGDFGMwHxjyucg3m42IeV73G682h728DBvNZ7cBgPizmgW3AYD6sDRbMp+WdB+ahZ8Y8MOZ51VpAn1itBfOJNZpDb6qYgvnAmOdVbPFac+g9VVMwnxfzvFrNobdULsF8XMzz6jWH3lE9BPNpMc+r2Rx6R/UMzKfVbA69I+Z51Sswn1Y9goc+rnIB5vMq35/5yErX99DzYh4Y87w89LyYBwY9L+aBMc/LQ8+LeWDM8/LQ82IeGPO8PPS8mAfGPC/mefnDPTDmeTHPi3le/kLPi3lezPNintfrzaE/xpjnxTww6Hkxz4t5Xszzuscc+iOMeV7M87rPHPrji3lwzGODnhfzvJgHBj0v5nkxz4t5Xv80h/7oY54X87yY58U8L1/E5cU8L+aBMc/LQ8+LeV7M82KeF/O87jCH/shjHhjzvJjnxTww4nkhDwx5ZsgjQy5JkiRJkiRJkiRJkiRJkiRJkiRJkqRH013/zZVvnx/TfTw1dd86PN6J4Q6Md2DEE2MeGPLAmAeGPDHmeSHPi3lcyOOqImc+NuRxlZEznxryvJjHVUfOfGrM82KeF/O8mOfFPC/meTGPq5Cc+dCY58U8r0pz6DMrNYc+MuZx1ZJDH1g1OfRx1ZNDH9YOcuaj2kIOfVKbyKHPaRs58zExj2sfOfMhIY9rIznzGe0kZz4i5HFtJWc+oL3izAe0m5x5e9vJmXe3n5x5cw3k0HtrIWfeWI8488a6yJm3hTwu5HEhjwt5Wm3iyNsinhfzuLrImffFPK42cuZtMY+rj5x5V8zzajSH3lMnOfOemMfVSs68JeZ5MY+rl5x5R8zjaiZn3hDzvJjH1U3OfH/d5My31y3OfH/d4sy31w3+lPn2ltoxP6KF4szPaCH41ae1DhDYQvCrj2u7fGZrxZmf0FLwa8/ruHhwa8WvPHD7rbNbC37lgXuvHN9i8etO3HhfXW2+9sRtt9WTkn9PMx/ecvGrztxxU/1VhQ/z2VXoMJ9diQ3z0ZXIXGwOfWc1LMwnV4PCfHI1JMwnVyQCfXBFHswHxzyvKg7mc2OeV5XGpebQt1WHwXxqdRbMp1ZowXxmlRTMZzbIHPqeaiGYT6zWgfnAih2YD6yaAfq4yhWYj6segfmwNiAwH9YGg8vMoVe3hYD5qLYIXGYOvbY9AMwntWl/5nPatf9l5tAr27U+8zHtWx/6lJjntXF86DPauT3zGTHPa+v20Ce0d3rmExptDr2kzcszHxDzuLYvD7095nkxj6theOjdMc+LeWD7d2fe3nBz6BUNR1/7c+vPmOfFPLDdu0MfEPPA9u7OfER7V2c+ob2TX2QOvai9ezOf0ObBmU+IeV6bB4c+IOZ5bR6c+YB2D868P+aBbV6c+YB2L868v8Hm0Ivavjfz9pjnxTyv/Xszb495Xszz2r839O72r828O+Z5NazNvDnmeXXMzbw35oENNode1OSHTr2m2ebUK+rY+hJz6utrWZp5ay1TM2+tZ2ronfVMzbwz5oH1TM28s56tmXfWszTzzpqGZt5Y08zMG2tamXljXSND76trYeZ9tQ3MvK+ufZn31bUu877axmXeWc+2zJtrWJZ5YNDzYh4Y87yY58U8MOZ5Mc+LeWDM8/q7KPTAmOfFPC/mgTHPi3lezPNiHhj0vJjnxTww5nkxz4t5YMzzYh4Y9LyY58U8MOZ5MQ+MeV7M82IeGPS8mAfGPC/meTHPi3lcvobL6w5z6I885nkxz4t5XHeRM3/cMc+LeV7MA4OeF/O8mAfGPC/keTEPjHlezPNiHhjywJgnRlySJEmSJEmSJEmSJEmSJEmSJEmSJEmSpIf2G1BLAQIUAxQAAAAIAAAAIQCS+hxReAgAAPB8BAAOAAAAAAAAAAAAAACAAQAAAABwcmVkaWN0aW9uLm5weVBLAQIUAxQAAAAIAAAAIQD9X6u7QQgAAPB8BAAQAAAAAAAAAAAAAACAAbgIAABncm91bmRfdHJ1dGgubnB5UEsFBgAAAAACAAIAegAAADsRAAAAAA==', 'reference/sample-16.npz': 'UEsDBBQAAAAIAAAAIQC4ltsuDw0AAIBDCAAOABQAcHJlZGljdGlvbi5ucHkBABAAgEMIAAAAAAAPDQAAAAAAAO3csY4cSR3HcW/KUzgzSBfAgQh4ADIQCQEROrhDBIhDPkQCPAUvjM/gu7W3e7qquqp+PVOfT+bVdtV/ur7qHck7+59f//ZXv/nd06t/vPrnmy+/+uaPb9/84vWbf/3hJ28+e/3mT1+//fvbL/76+6/ffvnVt1//5Rd/+eard1//5s9f/O2rd//+4eefvf7Z5z/97PXPf/bjH332+t+v6/3gFQAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA8iKf30lOwoKePpcdhLU8vpUdiHRv5KZBJtuvTIDPcqk+AjHbQnwIZ6TA/ATJQQX8CZJCS+vTHIGX5CZAhSvMTICPoj6Ty/gRIdxX5CZDeqvLTH53V9SdAuqrMT3/0VJuf/uhJfwRV56c/OtIfSfojSX8k6Y8k/ZGkP5L0R5L+SNIfSfojSX8k6Y+k+v4ESD/yI0l9RDXGJ0S60B9JfviSpD+i2vLTIn20Pf30Rx9n+1MiZxSF96K/55dPHZcHU/bgkxlDFP7g1R8jFOT3oT8B0ltJfvpjiKL49McQpfXpj07Kk/ukv+87hEaN9X38HIQWp7rTH6ecfe7pj1bn2vuQnf5ocbq+D9npj1od4tMfzfRHUJ/89EeDTvHpjxb98vvQX/b1cFc65ic8qumPJP0R1DM/7/+o07W+p+8+/6ZASvTO73mHcER/JOmPJP0RNCw//XFsXH7649DA/PTHkZH56Y8jQ/sTILeNzW8jQFnyzOj+np79NuqtLFnThP62pF821xDKT3+8pz+CUvkJkGR9+kN/ZOmPoGR+AiTbnwQXl67vSYBLS8f3pL+lpeN70t/K0u29l74JhKTD+7/0bSAj3d330neChHR1z6RvBQHp6J5J3wrmSzf3kfTNYLp0cp9K3w+mSuf2UvqOMFO6tg3pW8I8E4uq/HYWMC65rZqaLuJxDWtut6TmC3k8w5q7UdGZa3kso5q7mdC5q3kco5o7DOj0Aty9YcmV1NNjDe7ZqOIK03n2HWeW4U6NrK6+m34rcR8m1FcRTc+1uAPj46tLpu9qXNz4+OqDEeAyrpifZ+A6xtfXksqYVbme4fWN6k+Aj2B0e62hjFqXSxmdXnMkQxfnEsaGd7qQ0esTNTi8DnnM2IOUod31aWPKJmQM7a5bGlM2YbrR4XUrY9I2TDU6vM5ZTN2M8Ya3178IAT6QO+zPLwY+kAn59c8hsCVDjI9vZAsKvG8z6psRggTv0aPUt/taJu1Nm0fqT4D352Y1+mOs29HcWX67804bgBqH1dxbf/sDzxuBYpP6u8SLmjsEJR6yPwHejaNs7jI/Ad6Lw2jutL+d0SNzsO+wljvuT4CXdxzLPfe3MX9uEDbM6k+AbCho5e77++RFZAfhe0WtdMrvAsd+mUF4r6iVXvld4tivM8ny3h3D3P6ucuyXGWRthan0y8+x84z+SCrsRH8MUViI/hiiMJCO/QmQ7+mPoNI+9McI+iOpsI+e+emPD0r70B8j6I8k/RFT3kfX/ATIq6oPZ3fOT4HUtNE/PwGubrcL/TFeVRf6o7N0fvpbWlUV+qOzqiaG9CfAlemPpKoi9EdfVUWMyU+BC6vKYVh/AlxTZQz6o6faGMb1J8AV6Y+kyhYG5qe/BV0oP/0tqLIE/dFVZQhD+xPgemo70B9dXak/AS7pOv0JcEUX6k+AK9IfSfoj6Tr56W9FF+pPgCvSH0n6I6uwAP0RND4//bFvQn8CZNeM/gTInin9CZAd+iNpTn8CZJv+SJrUnwDZpD+i9EfQpPz0xyb9kTSrPwGyRX8k6Y8k/ZE0rT8BsmFefwLkJf2RNLE/AfKC/giamZ/++JT+SNIfSfojSX8k6Y8k/ZGkP5Km9idAPtaYkf7o4kRE+uO0Uw0JkJNOFaQ/TjoZkAI55XQ9CqRdj3QaChQh73WKRn806RWNAGkR7E+BdAumqT8FLq5jLgKkVtdYGgNU4Lq6htLanwCXpT+C+mbS3J8Al/LdmXduRH8cGphIe38CXMTIQE7kp78VjO3jTH4KXMDgOE72J8BHNr6Ls/kJ8HFNyOJ8fvp7WBOq6NCfAB/TlCj0x5Y5TZSvpL+FnI2v48Ov6JuH3ARSTuc3pL8b3z3iJhBz0f4EuIhZ+dX2t39B7ztA0On8Bj3+9i/pfAOY5OQfBDrZX9tC8nsQPYNrCaJ1mcbtuJT+0dX20LpO635cSP/oantoXqZxPy6kf3PVObSv07YfFzIiu7oYTqzUuiVXMSK8yhbOLNW4JVcxpr2qGM6s1LglVzEovpoUzqwlv/uWr+9whOKLT90IEvL1Hc5QfnXzXSCkQ2n/O/Wn92sNmaH4+qbdSTpd3/gZilfoMgxTXT+/wv9A6TIKk6XbKxmh10ZcTza9kgl67sXFhNsrGKH3dlxJuL2CEUZsyVXE65PfynLZlY4wdnOSLnDw2faJyp/8UX76e2D5k5ffyvInL7+VxY9efSvLH77+VhY/e/mtLH/2+ltZ/Ojlt7L4watvZfFzl9/K4gcvv6XFT15+S4sfvfyWFj97+S0tfPiefovLnv6t/PS3Av2RdN389LcC/ZEUPPXb9elvCdfNT38riB26/HilP7JSp36Yn/6WcN3+BLgC/ZGUOnP98a3rPv70t4IL9yfAFVy4PwUu4NL9CfDh6Y8k/REVOe7S/gS4gCv3p8AFzD/oiv4U+Pimn3JVfwKks7r+FEhntQEqkJ6q+5MgPbUEqEB60R9R+iNLf1yCALkM/XEB8iNLf6SpDwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAoLOn9AAs6+kj6WlYytNL6ZFYx0Z+CmSW7fwUyAy79QmQ8W7lpz8Gu5mfABnsoD8BMtRRfwJkoMP8BMg4Bfnpj1FK8tMfoxT1J0AG0R9BZfnpjzH0R5L+SNIfSfojSX8k6Y+kwv4EyBD6I0l/BJXmpz9G0B9J+iNJfyTpjyT9kaQ/ogRIkv6Ikh9J+iNKf0TpjyT9kaQ/ovRHUtEDUIGMIkCSyvoTIGPoj6TC/gTIEKX9aZAhagJUIJ1V5adAOqvtT4L001CfAumisT0F0sOp/CTIKafr0x/tOuQnQFp1yU9/tOmTnwJp0i0/AVKvY34CpFbX/PRHnb75CZA6vfsTIBW65ydBKuiPpCH9CZBC+iNpTH8CpIz+SBrUnwApMqo/AVJiWH8CpMC4/gRIAf2RpD+CxuUnQAroj6SB/SmQQ0P7EyAH9EeS/kga258AOaA/svRH0sj+BMgh/ZGkP7JGFihBDumPKP2RNLA/AXJIf0QJkCgBkqQ/kkb2J0CODO1PgBzQH0lj+xMgt+mPJP2RpD+SBvenQG4a3p8CuUF/JOmPpAn9CZBd+iNJfyTpjyT9kTSjPwGyR38k6Y+kKf0JkB36I0l/JOmPJP2RpD+S9EeS/EjSH0nyI0l/JOmPJP2RpD+S9EeS/kh6kYr+mOhlKfpjno1O5Mc0G52Ij2n0R9JGK/pjmo1a5McsW7noj1k2c5Efk+iPJP0RtRWM+Jhluxn1McdONupjCv2RNKa/6EvijuylIz+m2KtHf8wwoL/sC+Ku6I+ovXz0xxQ7/eiPOfb6ER8z7DYkP2bQH0l7HemPGXZDkh8z7JakP2bYLUl+zKA/orZTkh+TnO4vOz5371x/B2vOexncqzP9Ha4472Vwr/RHkv6I+rSWc/XpjzqN/R2vJz9KNPV3vJz+KPW8Fv2RpD+CzsX3Sn+c0prfuy/XVAqbGvurfUrCJv2RpD+S9EdSW376ow/9kaQ/ktry0x99tMS3ednEmXkYLe1tXTdpXB5MY376owv9kdSpvznD8nD0R1Jjfp9eOGNUHlBzf1t/SgEqtfe38acUoNKJ/nzyl9PO9OcvD3FWe3zPrx45IY/sVH4f1hgyGSvo0J/HH8169DdmMlbQ4+cvtNIfSfojSX8k6Y8k/ZG02Z8CmWSnPwUyhf5I0h9J+iNptz8BMoP+SNIfSfojyRtAkvb7EyDj3ehPgAynP5LkR5L+SJIfSfojSX8k6Y8k/ZEkP5L0R5L+SNIfSfojSX8k6Y8k/ZGkP5L0R5L+SNIfSfojSX8k6Y8o/ZGkP5L0R5L+SNIfSfojSn8k6Y8kP4BJ0h9Je/0JkBl2+xMgM+iPJP2RpD+SvAEkSX8k6Y8o/ZGkP5L0R5L+SNIfSfojSX9E6Y8k+ZGkP5LkR5L8iJIfSfojSn4k6Y8o7QEAAAAAAAAAAAAAAADFfBKJAB9EJ0l/JH1ITn8keP6RpD+SXvYnQObRH0n6I0l/BMmPgI3s9Mc0+iNJfyTpj6R3lX3SnP6Y52Vw+mOej/p7X5z+mOfpxlvA9Gw8vk/f/2mPiZRHkvxI+q468ZHwXXLqAwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAO7LfwFQSwMEFAAAAAgAAAAhAITpFL8WDQAAgEMIABAAFABncm91bmRfdHJ1dGgubnB5AQAQAIBDCAAAAAAAFg0AAAAAAADt3c+uHEcZxuHjLVfhnUHyAkLEggtgB2LDghUKxIgFIpGD2ABXwQ2TEMXY50x3V09X1Vs19Tw7W5rur/z9VCP/ifKf3/zu17/9/aunfzz9882X77750/s3v3z95l9//Nmbt6/f/Pmr939//8Xf/vDV+y/ffffzv/rir9+8+/bnv/nLF1+/+/bHP/7s7evPP/v529e/+PynP3n7+t+vz/vREwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAMK9X30uPwZJefSI9DWt59UJ6Itbxsj4F0svt+gRIF5v5KZD29vJTII0d5CdAWjrMT4G0U5KfAGmlrD8B0kRhfgKkieL+BEh95fkJkOrO5Kc/ajvVnwCp61x+AqQu/ZGkP5LO9idAatIfSfojSX8k6Y8k/ZF0uj8BUpH4SJIfSeojSX4kyY8k+RElP6LkR5L6iJIfUfIjSn4kyY8o+ZGkPpLkR5L8SJIfSfIjxe89iNmKT3+0ttOe/mjgoDj90c6p+PRHVWfr0x/1nK9Pf1RyT3z6o44769Mf190dn/647Ep9+uOaa/Xpjyuu1qc/7ne9Pv1xtxr56Y/7VKlPf9ynUn764w616tMf99AfQfXy0x/n6Y+givnpj7Nq5qdBzqrenwYp1yQ/DVKoXX8S5FDT/DTIgfb9aZBNffKTIDf1y0+DvNS5Pw3ysUB+CuSDTH8C5H9C+QmQ78TyEyBP0f4ESDI//RHtT4Cry+YnwMWl8xPg2tL1vRLgyjoVJkBu6pOf/1sIN3XJ79Xxi9K/DmT0yk+A3NAvPwHyQs/8BMhzffvzm2A+0Tk/AfKJ3vkJkI/0z+/wlZFfByJ65PciqNMf4EF1iO9WTXd8hAfUIb7bLd33KR5M+/i2Qrr7gzyO5u3tZHThozyG1uXtN3Tt00yvcXhHAV1+AFNrml1RPTWewayadlfaTp2nMJ+23ZWHU+1BzKRtd+eqqfksptA4vLPJ1H0ao2td3vliaj+PkbUu765cGjySITVP785W2jyVwTRP70IozR7MKNq3dymTho9mAO3Tu9xI26cT1SG+64E0fwEhHeKrUkePd9Bdh/iqtdHnLXS0tckR89seV4CT2tzjmPntjKzAGc2X3+bQCpzP5gaH7k+BD2J7fWPndzhhm1dS187uhs/vaMZmb6WWnc3NkN/RnE1fzHU7a5ulPwXOa29n0+R3MGv7t3On3YVNlN/BuH0G4KxHym934m4jcML+rqbLb3forlNQZH9RU/a3PXbnMTh0sKc589ucPDAHex42v43hM5Ow4SiXmfO7OX5sFG44zGXy/m6cIDkLzxzWUqe/7NIHGoVP9MovvvWBRuGD41Iepb+PT5KehB+s1N9HZ0kPwvcKQqnW3xhLH2gUuuY3ytJHmmVxBZU8Xn+MoiSSiv0JkI8VJaI/Gumdn/74SFEgVfsTIB+U9aE/2ijLQ380kchPgPygrA390URhGvqjhdIyavcnQL6Tyk9/PAWvP/1x4j/Jqd+fACn99tUfLSTz09/yypNo0Z8AV1cehP6o7kQP+qO2Mzk06U+AS9MfQadi0B+VnWqhTX8CXNcI+elvXadC0B+VDdGfAJelP5L0R9QQ/QlwWfojSX8kncigXX4CXJb+SNIfUWP0J8BlFTegP5rQH1ED9CfApemPqOME2uanv9WF+xMgu/RHkv5Iat2fANmjP5L0R1Lz/gTItvb56Y9t+iNJfyR16E+AbNIfSfojqUd/AmSL/gjqkp/+2NCnPwFym/5I0h9JnfoTIDfpjyT9kdSrPwFyQ7f89McN+iOpX38C5CX9kaQ/kjr2J0Be0B9J94akPyq4N7/7Ppo9K+O5P797Pp07J2O6lN/pz4cOybCu9nfyCZEzMq7L+Z17SPfzMbYa+Z14TN/DMbw6+ZU/qd/JmEG9/ooe1e1cTKFifkUP63QsJlEzv5KndTkU06ia3/HjOpyImeiPpL756Y9PVc3P9cdJXfPTH8/UzE9/HHgegfzo52UINfPTH3tutdA1P/0t7HYO8qOHws6uhKM/Nlyrr1Z++lvTxfoKs5EfN1yNr7Saek/icVyurzga+fHc9frkx70q1Fcejfz4RI36Kuanv6VUqU9+3KVOfSeakR8fVKpPftyhVn1nmpEf30vU5/pb2cfLrVbfuWLkt67/r7defPKj0IcF16xPfpSpWt2dxchvXePnp78HNkB+rr91yY+kAfI7mKHFqRnEAPXJb13p9AqGqHtgRhIOr2SIWkdlQOHyCmaocEhGlQ2vZIhaL2FA6fKOh6j6JgYTLq9giAbvYxjZ8ApmaPZWBhAu73iIxq8mKxve8QxdBiBliMXLb1kjLN31t64Bli6/dY2wdPmta4Cty29hA2xdfusaYOuuv4UNsHX5rWuArctvYQOsXX4Ly69dfgvL792378rya5ffyuJrl9/K8nuX38rie3f9LS2+d/mtLL53+S0tvnj5rSy+ePktLb15375rSy9efmsb8/rrOwQ5Q15/nYcgaMTrT4DrGPD6E+BCRrz+BLiOIa8/Aa5jyOtPgcsYNT8BrmHIb18BLmPY60+ASxj3+lPgEsa9/gS4gtiiBchTsD8F8vRDBNm3C3Bh4R0LkCwBEqZAsgRImADJUiBhAiRLgITpjyz5ESY/svRHmPzIkh9h8iNLfgAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAANPXqpvRULOJ2fgKki638FEgHO/kpkNb28xMgbR31J0AaOsxPf7RznJ/+aKYgPwHSSlF++qONsvz0RxOF+emPJvRHUGl+AqSB8vz0R336I0l/JJ3oT4BUpz+S9EeU/kjSH0m+gEnSH0n6I0l/JOmPJP2RpD+S9EeS/kjSH1H6I0l/JOmPJP2RpD+i9EeS/kjSH1H6I8kFSFR5gAqkvhP9CZDqzvSnQKo7F6ACqetkfwKkrrMBKpCaTvcnQCo6358Cqeee/gRIHXfVp0CquLs+BXLZpfoEyCVX61Mg96tRnwC5T6X6FMg9KuYnQE6qWp8COad6fgqkWIv6BEihRvkJkCL6I6hZfgKkgP4IapifADnSND/9caBtfwJkV+P89Mcu/RHUOj8Bskd/BLXPT4Bs6pGf/tiiP4K65CdAtuiPKAGSpD+iBEiUAInSH1ECJEp/RAmQKAESpT+iBEiUAInSH1ECJEp/RAmQpOb9CZAd+iOpfX8CZFuH/gTIJv2R1KM/AbJFfyR16U+AbOjTnwC5TX8kdepPgNykP5L0R1Kv/gTILfojqVt/AuQG/ZHUrz8B8pL+SOrYnwB5QX8k9exPgDynP5K69idAntEfSfojSX9EyY8k/ZGkP5LkR5L+SNqMRX90oD+itlqRHz1sxqI/OtAfSb36S5+TMW3Xoj/a0x9R27Xoj/b69Bc+JMPSH0k7ueiP5rr0lz0iI9vJRX40176/5OkYXfP+kodjfDvFyI/m2vYXPBhTaNlf8FjMol2AwUMxjWb9Bc/EPHbKkR/ttekveSJmsh2P/GivRX/RAzGXzX7kRwebBcmPHir3Fz4Ns9mKSH50UbO/9FmYz1ZH8qOLav2lD8KUNlKSH33U6S99CqZVo7/0GZjXzZjkRy+X+0sfgKld7S89P5O7kZP+6OZaf+Hhmd/LnPRHRxf6yw7OY3iek/7oyvVHlOuPqLv6i07Mo9IfQfIjSX8kyY8k/ZEkP4LkR5L+SJIfSfojSH4k6Y8k+ZGkP4LkR5L+SNIfSfojSX4Euf5I0h9J8iNJfyTpjyT5kaQ/kvRHlPxI0h9J+iNJfkTpjyT9kSQ/kvRHkv6Ikh9J+iNJfyTJjyj9kaQ/kvRHkv5I0h9R+iNJfyTpjyT9kaQ/kjb7EyAd6I8k/ZGkP5L0R5L+SNIfSfojSX8k6Y8k/ZGkP5L0R9J2fwKkPf2RpD+S9EeS/kjSH0n6I2mnPwHSnP5I0h9JM/Y37GCcNmV/w07GWXP2N+xonLTX36hbHng0zpq0v1Fn46RZ+xt1OM6Zr7+xp+OcefsbdDxOmS6/p+EH5IyJ+xt1Qk6Yub9RR6TcbP3NMCPl5u5v0CEpNnl/g05Jscn7G3RMSs3e36BzUmj2/EadlDLz9zfoqBR5gP4GnZUiD9DfoMNSYqqVznVbU2CqjW71N+i4HJtpoZv5jTkuBWbap/4e0ET71N8Dmmif2/0NOjDHJlqn/ojSH1H6I8kFSJT+iNIfSfojSn5EyY8k/RElP6LkR5L8iNIfUfIDAAAAAJiaP9wlxd8ukOKvt4jxr5vI8Y+LCbqRnwLp5XZ+AqSLrfwESHvb9cmP5nby0x+tyY8g+ZGzV5/8aGw3P/3RlvwIkh85+/XJj6YO8tMfLcmPnKP65EdD8iPoMD/90Y78CFMfceojTX3EqY849QEAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAwJz+C1BLAQIUAxQAAAAIAAAAIQC4ltsuDw0AAIBDCAAOAAAAAAAAAAAAAACAAQAAAABwcmVkaWN0aW9uLm5weVBLAQIUAxQAAAAIAAAAIQCE6RS/Fg0AAIBDCAAQAAAAAAAAAAAAAACAAU8NAABncm91bmRfdHJ1dGgubnB5UEsFBgAAAAACAAIAegAAAKcaAAAAAA==', 'reference/sample-17.npz': 'UEsDBBQAAAAIAAAAIQBGgsOsEhsAAABjDgAOABQAcHJlZGljdGlvbi5ucHkBABAAAGMOAAAAAAASGwAAAAAAAO3dva5kWZ6X4UqXq2ivQWqDESMkuAA8EA4GFhqgEQZiUA/CAa6CG6a/qirznIg4+2Ot9UbseB5jpju78uRev/+rGqdH9f/+zb/71//233/76X/99L9/+59//w//6Q+//Ze/+e3/+Y9/89vf/ea3/+Xv//A///B3//0//P0f/vPv//Tr/+rv/ts//P6Pv/4P//Xv/sfv//jv//E/+91v/vZf/M3vfvPP//af/pPf/eb//ma/f/QTAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAGzx7Y76u7i8e+mJkNm+bk+EzLKjPgEy2r789MdIO+vTHyPtzk+BDHKkPQkyyIn8BMhZp/ITICed7E+BHHc6Pv1x3Ij8BMhR+iM0Jj/9ccig/ATIAcPqEyD7jcxPguw0Oj8BssP4/PTHZhPyUyBbzclPgGwyKz/9scW0/gTI1+blpz++pj9CE/PTH1+YWZ8C+crs/hTIA/Pz0x/36Y+S/ggtyE+A3KU/Qkvy0x936I/Qmvz0x02L8tMfN63qT4Dcoj9Ky/oTIJ+ty09/fLawPwHyycr+BMgHS/PTHx/oj5L+6KytT3/8SH+EVuenP763vD8B8h39UdIfofX56Y9f6Y+S/ijpj1CQnwD5RdKfBPkr/VHSHyX9Uar6EyB/oj9K+qOkP0r6o6Q/Sll/AuQn/dHSHyX9Uer6EyBpfwJEf6T0R0qAlPRHSX+U9EdJf5T0R0p/pPRHqMtPgOiPmP5o6Y+S/ijpj5L+KOmPlP4o6Y+S/ijpj5T+KOmPlP4o6Y+S/mjpj5L+SOmPkv4o6Y+S/kgJkJT+KOmPkv5I6Y+UAEnpj5L+KK3uT4B8b3l/AuR7+qO0vD8B8h39UdIfKf3R0h8l/VHSHyX9kdIfJf1R0h8l/VHSHyX9kdIfJf1R0h8l/VGSHyX5UVIfJf1R0h8l+VHSHyX9UZIfJf0RWpSf/rhJfpTkR0l+hJbkpz/u0B8h+RFak5/+uE1/lPRHZ1F98uMm8RFakp/+uEN+lPRHSX+EluSnP25bk5/+uE1/hBblpz9ukR+hVfnpj8+W1ac/btAfoYX56Y9P9EdIfpTkR0l/lPRHSX6E5EdJf4TW5ac/PtMfnYX16Y9P5EdoZX764yP9UZIfJfURkh+ddfXpj09W5qc/PtIfJfkRWpmf/vhIf5T0R0l/lORHSX2E1uWnPz7THyX9UdIfJf1R0h8l+VHSH6V1/QmQG/RHSX+U9EdJf5T0R0p/lPRHSX+U9EdKf7QESEqAlPRHSX+k9EdKf5T0R0l/lPRHS3+kBEhJf5T0R0l/lPRHSX+U9EdJf5T0xyhHLq0/xjh2af0xhv4o6Y+S/ig9bX8C5J4V+emPe/RHSX+U9EdJf3SW1Kc/bluUn/64SX+U9EdJf5T0R0l/hFblpz9u0R8l/VHSH6Fl+emPG/RHSX+U9EdJf4TkR0l/hNblpz8+WZif/vhEfZS0R0l/lORHaEl++uMO+VHSHyX9UdIfJf1RWtKfALnjCfo7+dt5ZfPb+zKgs7+fFzY9va/zOf0DeF3T2/syn9M/gBc2O72v8zn/E3hdk8vbUM+AH8HLmhvelnhG/Axe1dTutrQz4mfwumZ291U7I34G1zCrvTvtjPgZXMmE7j63M+JncEnjcnvQzoifwSWNy+1BOyN+Btc0rre77Yz4GVzXsOLutDPiZ3Bdw4q7E8+AH8GVDSvudjwDfgRXNqy4m/EM+BFc2qjgxquXYYW6svvqZVihruy+ehlWqCu7r16GFerK7quXYYW6svvqZVihruy+ehlWqCu7r16GFerK7quXYYW6svvqZVihruy+ehlWqCu7r16GBerIHqinYYE6sgfqaVigjuyBehoWqCN7oJ6GBerI7quXYYW6svvqZVihruy+ehlWqCu7r16G6erEHqm3Yb66sQfqaVigjuy+ehlWqCu7r16GFerK7quXYYW6svvqZVihruy+ehlWqCt7oJ6G+erGHqinYYE6sgfqaVigjuyeehfWqDu7rV6FVerSbqpHYZk6tZvqUVimTu2zehFWqmv7Xr0F69XNfaeegkAd3c/qHUjU2f2iHoJEnd3P6h1I1Nl9U94bUx0h7VESHyX5UVIfpXXFKY/P1vUHn+mPkPjoiI/Q9Pz0xwP6o6Q/SvojND8//XGf/ijpj5L+KMmPkvoIzc5PfzwgP0r6o6Q/UvqjpD9K+iPzxzz0R0Z/PKNb5fzya3eyEh8AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAK/n2x/V38B7+vaL+kt4M982qb+SS9oWnwQZb097ImSoQ/FJkEH0R+hEfgLktFP9SZATzranQE4Yk58AOWZUfwrkgHH56Y+dBsYnQPYanJ8E2WVCfwJkqxn5CZCt9EdnTn0CZItp9SmQDab2J0Aem5uf/nhMf3Qm1ydAHpmfnwC5a0V+AuQe/VFa058AuWlRfgLkhmX16Y/PFuYnQD5amp/++EB/lPRHaG1++uN7i+sTIN9bn5/++JX+KAX9KZCfJfkJkL+I8tMff6Y/SvojVOUnQH4q89MfZX4CRH+U0vz09/ba/hT45ur8BPje6vr099bq+L7p763V8X3T3zur2/uTegMydXp/Ua9ApS7vL+oVqNTl/UW9ApE6vL+qZyBSh/dX9QxE6vD+qp6BRt3dz+odaNTd/azegUbd3c/qHUjU2f2iHoJCXd2v6iUo1NX9ql6CQB3dd+opCNTRfaeegvXq5r5Xb8FydXI/qMdguTq5H9RjsFpd3I/qNVisDu6Deg7Wqnv7qN6DterePqjnYK26t4/qPVir7u2jeg/Wqnv7qN6DpercPqkHYaW6tg/qOViqzu2TehCWqnP7pB6EleraPqsXYaW6tk/qQVipru2TehBWqmv7qN6DpercPqkHYaW6ts/qRVioju2zehEWqmO7oZ6EderWbqgnYZ26tVvqTVimTu2mehSWqVO7pd6EZerUbqgnYaE6ts/qRVioju2GehIWqmO7oZ6EZerUbqk3YZk6tZvqUVilLu22ehVWqUu7rV6FRerQbqtXYZE6tDvqWViizuyOehbWqDu7o56FNerO7ql3YY26s3vqXVigjuyBehrmqxt7oJ6G+erGHqm3Yb66sUfqbZivbuyRehumqxN7pN6G2erCHqinYb66sUfqbZivbuyRehvmqxt7pN6G6erEHqm3Ybo6sYfqcZitLuyxeh0mqwP7Qj0Pk9WBfaGeh7nqvr5Qz8NkdWCP1eswWR3YY/U6zFYX9lA9DrPVhT1Wr8NcdV9fqfdhqjqvL9UDMVWd15fqgZiqzutr9UJMVMe1QT0RE9VxbVBPxDx1W5vUIzFLXdY29UpMUoe1Vb0Tc9RdbVXvxBx1V5vVQzFFndVm9VDMUFe1Qz0VE9RR7VBPxQR1VDvUUzFBHdUe9VYMVye1T70Wo9VF7VOvxWB1UDvVczFW3dNu9WCMVNe0X70YI9U1HVBPxjh1S0fUmzFO3dIR9WaMU7d0RL0Z49Qt/WLPp9SjMcq8nnbZ/THtaowyrag9jnxMORrDzEpqlyMfU47GMLOS2uXIx5SjMcqsovY58jHpbAwyKaidjnxMOhuDTApqpyMfk87GGJN62unQ15SzMcikoHY69C3tcIwwpab9Dn1LPB0DzIjpgEOfUm/HeRNaOuLQp9Tbcd6Elo448in1dAwwI6YDjnxKPR3nzWjpgEPfEk/HADNiOuDIt8TLMcKUmvY78i3xcowwpab9jnxKPR3nzWjpiAPfUk/HeVNaOmT/x9TbcdaUkFapx+OsuqBz6vU4qy7onHo9zqoLOqdej5PqgE6q5+OkOqCT6vk4p+7nrHo/zqn7Oavej1PqfE6rB+SMup7T6gE5pc7ntHpAzqjrOa0ekFPqfE6rB+SUOp/T6gE5o67nvHpBTqjjGaCekOPqdgaoJ+SEOp4B6gk5rm5nhHpDDqvTGaHekOPqdgaoJ+S4up0R6g05rm5nhHpDjqvbGaHekMPqdIaoR+Soupwx6hU5qi5njHpFDqrDGaSekYPqcAapZ+SgOpwx6hU5qA5nkHpGDqrDGaNekaPqcsaoV+Soupwh6hE5qi5njHpFjqrLGaNekaPqcsaoV+SgOpxB6hk5qA5nlHpHjqm7GadekgPqaMaq12SnOpjh6kHZpc5lgnpSdqhjmaHelO3qVuaoV2WrupRJ6lnZpu5knnpZtqgrmaielg3qSCaqp2WDOpKJ6mnZoI5kpnpbvlY3MlO9LV+rG5mp3pav1Y3MVG/L1+pGZqq35Wt1IzPV2/K1upGZ6m35Wt3ITPW2fK1uZKJ6WjaoI5monpYN6kjmqZdli7qSWepd2abuZJp6WDapM5mmHpYt6kqmqYdlkzqTaeph2aTOZJp6WDapM5mmHpZN6kymqYdlkzqTWepd2abuZJZ6V7apO5ml3pVt6k5mqXdlm7qTWepd2abuZJZ6V7apO5mknpWN6lAmqWdlozqUSepZ2abuZJZ6V7apO5mknpWN6lDmqFdlq7qUKepR2axOZYp6VDarU5miHpXN6lSmqEdlszqVKepR2a5uZYZ6U7arW5mgnpQd6lgmqCdlu7qVGepN2a5uZYJ6UnaoY5mgnpQd6lgmqCdlhzqWCepJ2aGOZbx6UXapcxmt3pN96l4Gq+dkpzqYweo52akOZrB6TnaqgxmsnpN96l4Gq+dkpzqYweo52akOZrB6TnaqgxmsnpOd6mAGq+dkr7qYseo12asuZqx6TfaqixmrXpO96mKGqsdktzqZoeox2a1OZqR6S/armxmnXpIj6mrGqZfkiLqaceolOaTOZpR6R06o49nr8wfXC3LS1gv+9a+718HHSv78l/782461du/T1PfOfm7jwd+bfq7uw28a1d9P3/81vJ0f/074w6/99W96d37fh05/+tDlh38HAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAABf17Rf1l/Buvn1UfxBv41N7EmSNe+lpkPm+rE+AzLMlPwEyx7b69McUW/PTIKPtak+DjHUoPwEyxsH89McQR/sTIOcdrk+CnHauPgFyyun8FMhxI/ITIEeN6U+AHDGoPgFyxLj89MduA/MTILvpj9DQ/PTHLmPrkyC7TMhPgGymP0JT8hMgG+mP0KT8FMgG8+oTIF+amp/++IL+KM3tT4A8pj9K+qOkP0KT89MfD+mPkv4o6Y/S7P4EyCP6o6Q/SvojND0//fGA/ijpj9D8/PTHffojtCA/CXKX/igt6k+A3KQ/SvqjpD9K+qOkP0Kr8tMft+iP0rL+BMgN+qOkP0r6o6Q/Suv6EyCf6Y+S/ijpj9LC/gTIJyv7EyAf6Y+S/ijpj5T+KOmPkv4o6Y+S/kjpj5b+KOmPkv4o6Y+U/ijpj5L8KC3qT4DcpD9K8iMlP0ryo6Q/SvojpT9S8iOlP0r6o6Q/SvIjpT5K8qOkP0oT+xMgX9IfpZn9CZAv6Y/SzP4EyFem9ncrQHHynbn9PVa/nSdQBqhABEhLgJT0R0l/lPRHKe1PgOiPkv4o6Y9S2t/HABX5fvIA//Q/b/8nHz+vnIlJ1nR214MPuPN58V6MNbGt825+Xb0YI61OaoR6M8apWzqono1R6pKOqndjjLqj4+rlGKGu6IR6OkaoKzqhno4R6orOqLfjvLSf8z+BFzcgo1I9HyfVAZ1Uz8dJdUAn1fNxVl3QSfV8nFQHdFa9H+fU/ZxV78c5dT+n1QNySp3PafWAnFLnc169IKc0zYz8WbyygSU06gE5pc7ntHpATqnzOa0ekFPqfE6rB+SUOp/z6gU5o67nvHpBzqjrOa9ekDPqes6rF+SMup7z6gU5o65ngHpCTqjjGaCekBPqeAaoJ+SEOp4B6gk5oY5ngHpCTqjjGaHekOPqdoaoR+SwOp0h6hE5rE5njHpFjqrLGaSekYPqcAapZ+SgOpxB6hk5qA5nlHpHDqrDGaSekaPqcgapZ+SoupxB6hk5qA5nkHpGDqrDGaXekWPqbkapd+SYuptR6h05qA5nkHpGDqrDGaSekYPqcAapZ+SgOpxR6h05pu5mlHpHjqm7GaYekkPqbMapl+SIupqB6ik5oI5mpHpL9qubGatek73qYgar52SnOpjR6j3Zp+5luHpQdqlzGa9elD3qWiaoJ2WHOpYZ6k3Zrm5linpUNqtTmaNelY3qUKaph2WLupKJ6mn5Wt3IXPW6fKUuZK56Xb5QBzJbvS+P1X3MVu/LQ3Ue89UL80hdxwr1xtxXt7FCvTH31W0sUY/MPXUZi9Qzc1vdxTL10NxQR7FSvTWf1U2sVG/NZ3UTS9Vj80mdxFr12nxQB7FavTc/qntYrd6bH9U9LFcPzg/qHJarB+d7dQ2FenN+VpdQqXfnz+oMMvXw/FmdQadenj+pK+jUy/MndQWhenreOr9vCuzVBbTq9akLiNXzv706gFg9/9urA6jV+7+5+vy9+gLvrb7+E6hP8Nbq4z+D+gZvrD79U6iP8Mbq0z+H+gpvqz78s6jv8K7quz+N+hBvqj7786gv8Zbqoz+T+hbvqL75M6lv8Ybqkz+X+hrvp774c6mv8X7qiz+Z+hxvpz74k6nP8Xbqgz+d+iDvpb72M6pv8kbqUz+l+ihvpD71U6qP8kbqUz+n+irvor7z06oP8ybqMz+x+jTvoL7xM6tv8w7qGz+z+jbvoL7xU6uP8wbqEz+1+jhvoD7xc6uvc3n1gZ9efaCLq8/79OoDXVx93udXX+jS6uO+gvpG11Vf9kXUZ7qs+rAvoz7URdVnfRn1oS6qPuvLqA91TfVVX0h9qkuqj/pK6ltdUX3TV1Lf6orqm76S+lZXVN/0ldS3uqT6qK+kvtUV1Td9LfW1rqe+6Iupz3U59UFfTH2uy6kP+mLqc11OfdBXU9/raup7vpr6XpdTH/TF1Oe6nPqgL6Y+1+XUB3019b0upz7oi6nPdTn1QV9PfbGLqc/5euqLXUt9zddTX+xa6mu+nvpi11Jf8wXVJ7uU+pivqL7ZldS3fEn10S6kPuVrqq92HfUlX1N9teuoL/mi6rNdRn3IV1Xf7TLqQ76o+myXUR/yRdVnu4z6kK+qvttV1Hd8XfXlrqG+4uuqL3cN9RVfWH26S6iP+MLq011CfcQXVp/uEuojvrD6dJdQH/GV1be7gvqGL60+3gXUJ3xp9fEuoD7ha6uv9/rqC766+n6vrr7fq6vv9+rq+726+n6vrr7fq6vv9+rq+726+n6vrr7fq6vv9/LqA764+nwvrz7gi6vP9/LqA764+nwvrz7gi6vP9/LqA764+nwvrz7gi6vP9/LqAw7TvCe93RUsvtc00XvK013C4ntNEz0ovNw1rD3XPNGDwstdw9pzTVO9KLvbVSy91jzVo5qjXci6U01VPao52oWsO9VU1auSm13JskvNVb0qudmVLLvUXNWzipNdyqpDTVY9qzjZpaw61GTVu4KLXcuiO81WPWz9wS5mzZmmq162/F4Xs+RIC1RvW32vq1lxoxWqxy0+1+UsONES1eMWn+tyFpxoiepxi891ObumnnbF884+buify2a7hp54x7POPW7wn8tm+4aeeMiTzj5v8J/LRvuG/uP/mnjLE86+bvQfzEb7d555zcPOPm/4H8w2T3rWvQ6+b+IfzCYHV5191r2yDx15i3d0dNTZd90p+85xl3hPR0edfdedqu8cd4g3dXTVyXfdK/rQYWd4W0dXnXrW/ZoPHXWEN3Z01plnPSD50FE3eGdHZ5141iOSDx10grd2dNaJZz2i+NBBF3hvh3edd9cjiu8cc4B3d3zOeZfdL/jM45PzvcNzzjrsEes/8/DejDLpskes/splG3PflMses/ozV03MAzMOe9Dq71y0MI9MuOtRqz90zcA8NP6shy3+0iXz8tjoo56w+lNXzMsXRh/1hNWfumJevjL6qset/tIV6/KV0Vc9bvGnLtiWL4296SlLP3f2rmw07KKnrfzcyaOy2aCDDrDwcydvynZjDjrCwu+dOynbDTnnGOu+eeKe7DOgm1EWffPEMdltTDpDrPnoeVNywKh4Bljy0fOW5Ihh9Zy34qun7cgxA/s5a/+n7/z6GftxzpyUDpn7+aOHY4h5Oe128Ps3PWHwaowyO6odZj5h5GQMNDuqHSa+YeBgjDS/qu0mPmLgYow0v6od5j1i5GSMM7+pPea9YuRmjDO/qT3mvWLkZgwzP6ld5j1j5GiMMr+oneY9Y+xwDDE7p3P+/H3jnjF7TPaaWs84o16xZFN2mBjNQKPesGZTtpvXzEij3rBmU7ab18xIw96wZlQ2m5bMUCPfsGZXtpnTy2hD37BmWDaZkssEI5+wbFy+NCGVOQY+Yd26fGFCKDMNesPikblpfB4LjHpFsDc/GtrFMqPe0GzOr8Y1sdKoNzSb86txTSw06gnN5HxnXBQr/fnDB7ygXJ5vr5rfX53//PoCb21EAi+uPsFbq4//BOoTvLX6+L36Am+rPvyzqO/wDuobP7P6NhdWn/Yl1Ee6sPq0L6E+0nXVl30N9ZUupz7oeDNfVV/rSmbdqDP9bd2xrmbSgVr33zfmxdWxrmfAMZ7Q/TeOeXJyqksacIwndP+RQ55cHOqazt/iKd1/5ZA3F5e6pPOneE6f3jn2qcWprmjkTZ7Ko8f6+9/TOH+JJ/Xguf7Rv8/j9CWe1dxnrznO9Y249HOa++4117m+EZd+TnMfvuQ41zfk0s9p6suXHOcNjDn1U5r68iXHeQNjTv2UZj5+/mHexLhzP52JL19ymrcw6thPaN7Ll1zmPQy79vOZ9/Ill3kPw679fOa9fMll3sOwaz+faU9fcZf3MfDiT2Xa06df5O0MOfUv//LsT7v/h3z80279q28//5dbNr/81gT3Zhk4Ot+58/8nduvXPh3j248/5Psivv8nT974L+R9/0O+/703Ovj1tzx6xHe/Zeu7b/653/3A7//DbT+T3e61duvX7vb37WF/N873w01/6OCnH2La9c9P/fGbtvuhtV9+7f7HM9L963+Z362ftv3/+t34pZ3VffjNO38LACP8f1BLAwQUAAAACAAAACEArE/tkAsaAAAAYw4AEAAUAGdyb3VuZF90cnV0aC5ucHkBABAAAGMOAAAAAAALGgAAAAAAAO3SzY5laXqW4copR1GzAqkHWLaQ4ACYgZh4wMhqcCMGCFvdiAlwFD5hXD9ZGRG5d8Ra6/u5I2Jf1yxT2lpfPO/9T//xb//Df/rPX3743z/8n5/+/k9/+a9//unf/fjT//0vf/XTH3786b/9w5//15//+D//7h/+/Pd/+vn///0f/8df/vTP//+X//7Hf/zTP//7X/71H378m3/7V3/48d/8zb/+V3/48f/9eN6/+AEAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA+KS+fPlSP4GH9eUX9St4SF++qZ/Co/nyQv0eHsnL+hTINrfiUyBb3I1PgCz3an0KZKU341MgyxyrT4EscDw+BTLZyfgEyEQX6lMgc1yMT4FMMdKfAhk0lp8AGTPanwIZMJ6fArluSn8K5JpJ+SmQS+b1J0BOm5ifAjltbn8K5JTZ+QmQM+b3J0AOW5Cf/jhqRX4C5Cj9EVqTnwA5Rn+EVuUnQI7QH6F1+QmQt+mP0Mr8BMhb9EdobX4C5HX6I7Q6PwHyGv0RWp+fALlPf5R29CdA7tEfoS35CZA79EdpU38C5Cb9UdrVnwC5RX+UtvUnQG7QH6V9/QmQ7+mP0Mb89Md3dvYnQF7SHyX9UdIfpa39CZAX9EdJf5T29idAntMfoc356Y9ndvcnQJ7SHyX9UdrenwB5Qn+U9Edpf38C5Bv9UdIfoSA/AfI7/VHSHyX9UUr6EyBf6Y+S/igl/QmQr/RHSX+U9EdKgJT0R0l/lPRHSn+Uiv4EyFf6I6U/SvqjpD9K+iMlQEr6o6Q/UvqjpD9KQX8C5HfyI6U/SvqjpD9S+qOkP0r6o6Q/UvKjpD9K+iOlP0r6o6Q/SvojJT9K+qOkP1L6o6Q/SvojpT9K8qOkP0r6I6U/SvqjpD9S+qMkP0r6o6Q/SvqjpD9K+qOkP0r6oyQ/SvqjpD9K+qOkP0r6I7QlP/1xh/wo6Y+S/Cjpj5L8KOmPkv4oyY+S/ijJj5L+KOmPkvwo6Y+S/Cjpj5L+KMmP0Nr89Mer5EdJf4QW56c/XiM/Qqvz0x+vkR+h1fnpj1csz09/3Cc/SvojJD9C6/PTH3dtyE9/3CU/Qhvy0x/3yI/Qjvz0xz36IyQ/Qlvy0x93yI/Qlvz0x2178tMft8mP0J789MdN8qOkP0LyI7QpP/1xk/wIbcpPf9ykP0LyI7QrP/1xi/wI7cpPf9wiP0K78tMfN8iP0rb+BMj39uWnP76nP0Ib8xMgL23NT3+8pD9KW/sTIC/oj5T+KG3tT4C8oD9K+iMlQEr6o6Q/Snv7EyAv6I+S/ijt7U+APKc/UvqjtLc/AfKc/ijpj5T+KOmPkv4o6Y+U/iht7U+AvKA/UvqjpD9K+qOkP1L6o6Q/SvqjtLM/AfKS/ijpj9DW/PTHC/qjpD9K+qOkP0p7+xMgz+mPkv4Ibc5PfzyjP0r6o6Q/SvqjpD9Ku/sTIE/pj9D2/PTHE/qjpD9K+qOkP0r6o6Q/SvIjtD0//fGE/ijJj5L+KMmPkv4oyY+S/gjJj5L+KMmPkv4oyY+S/ijJj5L+CO3MT3+8JD9K+qOkP0rvuD/tPoB329/Qj/ko3md/Qz/mI3mH/Q39mI/mnfU39GM+oHfU39CP+aDeS39DP+bjehf9Df2YDy3vb+jHfHgb8nsloaEf8xksr+9+QkM/5pNYHN+9hoZ+zCeyMLy7CQ39mM9lWXf3Ehr6MZ/OouxuJzT0Yz6lFdHdTmjox3xW85u7mdDQj/nEZid3I6GhH/PJTQ3uRkJDP+YBTAzuu4SGfsxjmNbbVPUqbFOndlu9CtvUqd1Uj8I+dWu31JuwT93aLfUmbFOndlM9CtvUqd1Uj8I2dWo31aOwS13abfUq7FKXdlu9CrvUpd1Wr8ImdWj31LuwR93ZPfUu7FF3dk+9C1vUmd1VD8MWdWZ31cOwRZ3ZXfUw7FBXdl+9DDvUld1XL8MOdWX31cuwQR3ZK+pp2KCO7BX1NGxQR/aKehrWqxt7RT0NG9SRvaKehg3qyF5RT8N6dWOvqbdhvbqx19TbsF7d2GvqbViuTuxV9TgsVyf2qnoclqsTe1U9DqvVhb2uXofV6sJeV6/DanVhv6hHoCIuSvqjJD9C6/PTH/fJj5L+CMmPkv4oyY+Q/Cjpj5L8CK3OT3+8Rn6U9EdIfpT0R0l+hBbnpz9eJT9K+iMkP0ovI9EfOx1tRH4sMNqI/hixqBD5AQAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAADvwZdv6qfweL4cVr+UT+h4fgJktjP1KZDJzuanQCa6kJ8AmeRSfQpkjsv5CZBxA/kpkEFj9SmQIeP5CZDLZuSnQK6ZVJ8AuWJefgrktKn5CZBTJtenQM5YkJ8COWpNfvrjiEX1CZAj1uUnQN60Mj/98Yal+QmQNyzuT4C8ZnV++uM1y/sTIPetz09/3LUhPwFy15b+BMhte/LTH7dt6k+A3LIrP/1xw7b8BMgNG/sTIC/tzE9/vLS1PwHy3N78BMgzu/PTH09t70+AfLM/P/3xTdCfAPmqyE9//CbJT3/8pulPgPwiyk9//EJ/hKr89McPYX764wf9kery0x/6IxXmpz/S/gT48NL89Pfw9EdJf4Ta/PT36PRHSX+E4vz09+D0R6jOT4CPra5Pf4+trk9/D62O74v+Hlod3xf9PbQ6vi/6e2R1ez+rN6BTt/ezegM6dXs/qzcgU6f3i3oEMnV6v6hHIFOn94t6BCp1eb+qV6BSl/eregUqdXm/qlcgUof3m3oGInV4v6lnIFKH95t6Bhp1d1/VO9Cou/uq3oFG3d1X9Q4k6ux+Vw9Bos7ud/UQJOrsflcPQaGu7pt6CQp1dd/USxCoo3uinoL96uaeqrdguzq5p+ot2K5O7ql6C7ark3uq3oLt6uSeqrdguzq5p+ot2K0u7pl6DHari3umHoPd6uKeqcdgt7q4Z+ox2K0u7pl6DHari3umHoPd6uKeqcdgt7q4Z+ox2K0u7pl6DHari3umHoPN6uCeq9dgszq45+o12KwO7rl6DTarg3uuXoPN6uCeq9dgszq45+o12KwO7rl6DTarg3uuXoPN6uCeq9dgszq4F+o52KrO7Tv1IOxU1/adehB2qmv7Xr0IG9Wx3VBPwj51azfUk7BP3dot9SZsU6d2Uz0Ku9Sl3Vavwi51aTfVo7BLXdpt9SrsUpd2W70Ku9Sl3VSPwi51abfVq7BLXdpt9SrsUpd2Uz0Ku9Sl3Vavwi51abfVq7BLXdpt9SpsUod2T70Le9Sd3VPvwh51Z3fVw7BFndl99TJsUEf2mnob1qsbe1U9DsvVib2qHofV6sLeUM/DYnVgb6n3Ya26rzfVA7FSXdcB9UQsVMd1RL0R69RtHVFvxDJ1WofUI7FMndYx9UqsUpd1TL0Si9RhHVTPxCJ1WAfVM7FIHdZR9U4sUWd1WD0US9RZHVYPxQp1VcfVS7FCXdVx9VIsUEd1Qj0VC9RRnVBPxXx1U6fUYzFbXdQ59VrMVhd1Tr0Wk9VBnVTPxVx1T2fVezFX3dNZ9V5MVed0Wj0YM9U1nVcvxkx1TefVizFRHdMV9WZMU6d0ST0a09QpXVKPxix1SdfUqzFLXdI19WpMUod0UT0bc9QdXVXvxhx1R1fVuzFFndFl9XDMUFd0Xb0cM9QVXVYPxwx1RdfVyzFBHdGAejomqCMaUE/HuLqhEfV2DKsTGlFvx7i6oRH1dgyrExpSj8ewOqEh9XiMqgsaU6/HoDqgMfV6jKoLGlOvx6A6oEH1fIyp+xlV78eYup9B9XyMqfsZVe/HmLqfQfV8jKn7GVXvx5i6n0H1fIyp+xlV78eYup9B9XwMqgMaVM/HmLqfQfV8DKoDGlTPx6A6oDH1egyqAxpUz8egOqAx9XoMqgMaU6/HqLqgMfV6jKoLGlKPx6i6oDH1eoyqCxpSj8ewOqEh9XiMqgsaUo/HsDqhIfV4jKoLGlKPx7A6oSH1eAyrExpUz8eYup8J6gkZUMczR70iV9XlzFMvyQV1NHPVa3JWXcx09aCcUucyX70oJ9SxrFBvynF1KyvUm3JYncoa9aocVZeyRr0qR9WlLFLPyjF1J8vUw3JIncky9bAcUmeyTr0sB9SRLFRPywF1JCvV2/KmOpGl6nF5U53IWvW6vKUuZK16Xd5SF7JWvS5vqQtZqh6XN9WJLFWPy5vqRFaqt+VtdSMr1dvytrqRlepteVvdyEL1tBxQR7JQPS0H1JEsVE/LAXUk69TLckRdyTr1shxRV7JOvSxH1JUsUw/LIXUmy9TDckidyTL1sBxRV7JMPSyH1JksUw/LIXUmy9TDckidySr1rhxTd7JKvSvH1J0sUs/KQXUoi9SzclAdyiL1rBxUh7JGvSpH1aWsUa/KUXUpa9SrclRdyhL1qBxWp7JEPSqH1aksUY/KYXUqK9Sbclzdygr1phxXt7JCvSnH1a0sUE/KCXUsC9STclzdygL1pJxQx7JAPSkn1LEsUE/KCXUs89WLckZdy3z1opxR1zJfvShn1LVMVw/KKXUu09WDckqdy3T1oJxS5zJbvSfn1L3MVu/JOXUvs9V7ck7dy2T1nJxUBzNZPScn1cFMVs/JSXUwc9VrclZdzFz1mpxVFzNVPSan1clMVY/JaXUyU9VjclZdzFT1mJxWJzNVPSan1cnMVG/JeXUzM9Vbcl7dzEz1lgAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAH86XL/ULeFhfflO/gwf05Zn6NTyWL9+rn8SjuBGfBtnjfnwSZLG34pMg6xyrT4IscCI+BTLX2fj0xzQX4hMgk1ysT3+MuxyfABk2VJ8AGTOan/4YMJyfALluQn7646oZ+QmQq+b0J0AumZSf/rhiVn4C5IJ5+emP0ybmJ0BOm9qfADlnbn7645TJ+QmQM6bnpz9OmN+fADluQX8C5KgV+emPo5b0J0AO0h+lNf0JkEMW5SdADtEfpWX9CZC3rctPf7xtYX8C5E0r+xMgb9EfoaX5CZA36I/S4v4EyGtW5ydAXqM/Suv7EyD36Y/Qhvz0x107+hMg9+iP0Jb89Mcd+qOkP0r6I7QnP/1xm/4o6Y+S/ijpj5L+KOmPkv4obepPgNykP0r6o6Q/SvqjpD9K+qOkP0r6o6Q/SvqjpD9K+iOlP0qb+hMgN+mPkv4o6Y+U/ijpj5L+KOmPlP4o6Y+S/ijpj5T+KOmPkv4o6Y+S/kjpj5L+KOmPkv5I7QhQgtyzpz8Fctuu/iTILRv7UyDf2dqfBHlpd4AS5Kn9/SmQb4r+JMhXUX8S5FddgArkhzRACdL2J0HqACX44Or8flZvQKiO71f1ClTq8n5XD0Gj7u6JegoKdXUv1HOwWR3cbfUqbFOn9qp6HNarGzugnoiV6roOq4diiTqr0+rBmKvu6bx6Maaqc7qgnoyJ6pguqUdjnrqla+rVmKZO6aJ6NmapS7qq3o056o4uq4djjrqj6+rlmKLOaEA9HRPUEQ2px2NYndCgej4G1QENqwdkSJ3PuHpBRtT1jKsXZERdz7h6QUbU9YyrF2REXc+4ekFG1PVMUE/IgDqeCeoJGVDHM0E9IQPqeCaoJ2RAHc8E9YQMqOOZoJ6QAXU8M9Qbcl3dzgz1hlxXtzNDvSHX1e3MUG/IdXU7M9Qbcl3dzhT1iFxWpzNFPSKX1elMUY/IZXU6U9QjclmdzhT1iFxWpzNHvSJX1eXMUa/IVXU5c9QrclVdzhz1ilxVlzNHvSJX1eXMUa/IVXU5c9QrclVdzhz1ilxVlzNHvSJX1eXMUa/IVXU5c9QrclVdzhz1ilxVlzNHvSJX1eVMUs/IRXU4k9QzclEdziT1jFxUhzNJPSMX1eFMUs/IRXU4k9QzclEdziT1jFxUhzNJPSMX1eHMUu/INXU309RDckmdzTz1klxQRzNRPSUX1NHMVG/JeXUzU9VjclqdzFz1mpxVFzNZPScn1cHMVu/JOXUv09WDckqdy3z1opxR17JAPSkn1LGsUG/KcXUrS9SjclRdyhr1qhxVl7JIPSsH1aGsUu/KMXUny9TDckidyTr1shxRV7JQPS0H1JGsVG/L2+pGlqrH5S11IYvV8/KGOpDV6n15VZ3HevXCvKauY716YV5Rx7FDvTH31W1sUY/MPXUZm9Qzc0cdxi71ztxUZ7FPvTS31FVsVE/N9+omtqrH5jt1EnvVa/NCHcRu9d48V/ewXT04T9U17FcvzhN1DIV6c76pW0jUo/NVXUKknp3f1CFU6t35RZ1Bp16eHx45PwG+B3UDqXp86gJi9fyPrr5/rj7Ag6vP36sv8NDq478D9QkeWX37d6E+wgOrT/8+1Fd4WPXh34v6Dg+qPvv7UV/iMdVXf0fqUzyi+ubvSn2MB1Sf/H2pr/Fw6oO/N/U9Hkx97venvshjqa/9/tQXeSj1sd+j+iYPpD71+1Rf5XHUl36n6rM8ivrO71Z9mMdQX/kdq0/zEOojv2f1bR5AfeL3rb7Op1cf+L2r7/PZ1fd99+oDfW71dT+A+kSfWn3cD6A+0adWH/cjqG/0qdXH/QjqG31m9W0/hPpIn1h92o+hvtInVp/2Y6iv9HnVl/0g6jN9WvVhP4r6Tp9VfdcPoz7UZ1Xf9cOoD/VJ1Wf9OOpLfU71VT+O+lKfU33VD6Q+1edUX/UDqU/1KdVH/UjqW31G9U0/lPpYn1B90g+lPtZnVN/0I6lv9RnVN/1Q6mN9QvVJP5T6WJ9QfdIPpT7WZ3Rt5u2nfx+WX+PxXBt5/+nfhdXHeEAXR95/+/dg8S0e0rWRg+O/B4tv8YiubRzc/j1YfItHdHHj4PjvwNpTPKSLGwfHfw/W3uIhXZu4OP47sPYUD+nixMX1e0sv8ZguTlxc/x1YeoqHdHHh4vjvwNJTPKaLCyfnz608xIO6uHBy/t7KSzymqwMn588tPMSDujpwcv7cwkM8qKsDJ+fvLbzEY7q8b3L+3LpDPKjL+ybnz607xIO6vG9y/t66Szym6/Mm588tO8SDuj5vcv7cskM8qOvzJufvLbvEYxpYNzl/btUhHtTAusn5c6sO8aAG1k3O31t1icc0Mm5y/tyiQzyokXGT8+cWHeJBjYybnD+36BAPamjc5P65NYd4UEPbJufPrTnEgxraNjl/bs0hHtTYtsn9c0sO8aDGpk3On1tyiAc1Nm1y/tySQzyowWmT+399a/dlZhlcNjn/72/tvswkg8sm5//21u7LzDG6bHL/398afpopRodNzv/treGnmWF02OT8T94afpoJRodNzv/0reW3GTa6a3L9Z28tv82o0V2T6z9/a/ltBo3uWhz/5VvTjzNkdNbi9t+9Nf04I0ZnDU5/463pxxkwOuv+y998a/t1Lhtddf/hb7+1/TpXjc66/e73ntp+nYtGZ9199vtPjT/PJaOz7r76K09tv84lg7tuvvnrL22/zhWDw+49+VsPjT/PeYO77r34mw+NP89pY8NuvfeRd8af56yhZXde++Az6+9zztCyO4999JX19zllZNmNpz7+yvr7nDIy7cZTn3hk/X3OGNh236HPvbH+PidcH3fbmU8/sf4+x10fd9eV76ufNvMOj+ryvJtu/Jr6cXMv8Zgu77vnxK+qXzf3Eo/p6r5bDvyG+n1zL/GYrg68475vuf6Hrf88x1yceP11Dxj5y5Z/nkMuTrz8uEcM/WnLP88RFzdefdtDxv601Z/niIsjr77tIYN/2uLPc8TFkVff9pDBP23x5znk2sqrb3vI4F+2+PMccm3j1bc9ZOgvW/55Drm28OrbHnP9L1v/dY65Nu/q2x5z8S/b8nWOuTbu6tse075x2gke2rVpV9/2mPSNk/Z/dNeWXXzbg9I3TlmfS7suPu1R6SNnjM8Pl0ZdfNqj0kcOL8/PLk26+LRHpY8c3J1fXRp08WmPSh85tDpfXZpz8WmPSh85sDnfXNpy8WmPCh95dW5euLTl0tMe1z3y2tZ879KUK097QvbIK5sxz8LTntK8cs/G3Lfqsmclz9yyMK9Zc9jzinfu2JfXrbjrFcFDN6zLW+af9aLdL92wLW+bfNXrNj91/bIcMfWoI/a+dfmuHDPxpmO2Pnb1qhw17aTD9r129aYcN+eiM2x77epJOWHKRafY9dzFg3LKhINOsum9a+fkpAnhzLLjwWvH5LQ56Uyx4cVLp+SCWfFMsPrJK2fkopkBDVr55oULMmJ6RQNWPXrhfAxaEtJFK169bjlmWFfTebOfvWw0plmc1Ckz371qL+ba0dVhsx6+aiym25XWITNevmgn1tja11tGH79iINYqOrtn5PXzl2GLLrfvXX3/7E3YqI7uiXoKCnV139RL0Ki7+109BI26u6/qHYjU4f2mnoFKXd6v6hXo1O39oh6BSB3eb+oZ2K9u7ql6C/aqe3up3oN96tZuqkdhh7qy++plWK0u7HX1OixUx3VAPRGL1GEdVM/EAnVUZ9RbMVEd0wX1ZExQRzSgno4BdTwT1BNyQR3NTPWWnFDHskA9KSfUsSxQT8oJdSwL1JNyXN3KEvWoHFanskQ9KofVqSxRj8phdSpL1KNyVF3KGvWqHFWXskQ9KofVqaxQb8pxdSsr1JtyXN3KAvWkHFe3skK9KcfVrSxQT8oJdSwL1JNyQh3LfPWinFDHskA9KSfUscxXL8oZdS3z1YtyhhAAAAAAAN70/wFQSwECFAMUAAAACAAAACEARoLDrBIbAAAAYw4ADgAAAAAAAAAAAAAAgAEAAAAAcHJlZGljdGlvbi5ucHlQSwECFAMUAAAACAAAACEArE/tkAsaAAAAYw4AEAAAAAAAAAAAAAAAgAFSGwAAZ3JvdW5kX3RydXRoLm5weVBLBQYAAAAAAgACAHoAAACfNQAAAAA=', 'reference/sample-18.npz': 'UEsDBBQAAAAIAAAAIQC9Ytd0BhgAAIBXCAAOABQAcHJlZGljdGlvbi5ucHkBABAAgFcIAAAAAAAGGAAAAAAAAO3cz45kV1ZG8copT1Ezg+QBtFog8QDMQEwYMEINbcQA0S0bMQGeghfGrkpnVmRE3Ljn7r3PuvvE+g3cabuccc53VpflP/L//d0//O3f/+PLp//69N/f/f6Hn/7lx+/++vN3//PPf/Hd95+/+9c//PifP/7uP/7pDz/+/odffvvf/O7ff/rh59/+07/97o8//Pzrf/qb7z//9jd/9f3nv/ztn//Z95//9/O4P/kkSZKkmV5e3r7a/nHvf8A3X25+tfVd9KRezoWeQ1PRud1AT6Jp6NRuokfRLHRpt9GraBa6tNvoVTQJHdod9CyahA7tDnoWzUF3dg+9i2agK7uPXkYT0JFtoKdRPbqxLfQ2KkcntokeR9XowrbR66gYHdgD9DyqRff1CL2PatF9PULvo1J0Xg/RA6kSXddj9EIqRMe1Az2R6tBt7UFvpDJ0WrvQI6kKXdY+9EoqQoe1Ez2TatBd7UXvpBJ0VrvRQ6kCXdV+9FIqQEc1gJ5K+eimRtBbKR2d1BB6LGWjixpDr6VkdFCD6LmUi+5pFL2XUtE5DaMHUyI6pgPoyZSIjukAejLloVs6gt5MaeiUDqFHUxI6pIPo2ZSD7ugoejeloDM6jB5OGeiKjqOXUwI6ogB6OoXRCYXQ4ymMTiiEHk9hdEIh9HiKoguKoddTEB1QED2fguiAguj5FEP3E0Xvpxi6nyh6P8XQ/UTR+ymEzieMHlAhdD5h9IAKofMJowdUCJ1PGD2gQuh8wugBFULnE0YPqBA6nzB6QIXQ+YTRAyqEzieMHlAhdD5h9IAKofMJowdUCJ1PGD2gQuh8wugBFULnE0YPqBA6nzB6QIXQ+YTRAyqG7ieK3k8xdD9R9H4KofMJowdUCJ1PGD2gQuh8wugBFULnE0YPqBA6nzB6QIXQ+YTRAyqEzieMHlAhdD5h9IAKofMJowdUBF1PHL2gIuh64ugFFUHXE0cvqAi6njh6QUXQ9cTRCyqCrieOXlARdD1x9IKKoOuJoxdUBF1PHL2gIuh64ugFFUHXE0cvqAi6njh6QUXQ9cTRCyqCrieOXlARdD1x9IKKoOuJoxdUBF1PHL2gIuh64ugFFUHXE0cvqAi6njh6QUXQ9cTRCyqCrieOXlARdD1x9IKKoOuJoxdUBF1PHL2gIuh64ugFFULnE0YPqBA6nzB6QIXQ+YTRAyqEzieMHlAhdD5h9IAKofMJowdUCJ1PGD2gQuh8wugBFUP3E0Xvpxi6nyh6P8XQ/UTR+ymG7ieK3k8xdD9R9H6KofuJovdTDN1PFL2fYuh+ouj9FEP3E0XvpyA6oCB6PgXRAQXR8ymIDiiInk9BdEBB9HwKogMKoudTEB1QED2fguiAguj5FEQHFETPpyA6oCB6PkXRBcXQ6ymKLiiGXk9RdEEx9HqKoguKoddTFF1QDL2eouiCYuj1FEUXFEOvpzA6oRB6PIXRCYXQ4ymMTiiEHk9hdEIh9HgKoxMKocdTGJ1QCD2e4uiGIujtFEc3FEFvpzi6oQh6O4XRCYXQ4ymMTiiEHk9hdEIh9HgKoxMKocdTHN1QBL2dMtAVHUcvpwx0RcfRyykDXdFx9HLKQFd0HL2cMtAVHUcvpxR0RofRwykH3dFR9G7KQXd0FL2bctAdHUXvphx0R0fRuykJHdJB9GxKQod0ED2bktAhHUTPpiR0SAfRsykLXdIx9GpKQ6d0CD2a0tApHUKPpjR0SofQoykNndIh9GhKQ6d0CD2a0tApHUKPpjR0SofQoykNndIh9GhKQ6d0CD2a8tAtHUFvpjx0S0fQmykRHdMB9GTKRNc0jl5MmeiaxtGLKRWd0zB6MOWiexpF76VcdE+j6L2UjA5qED2XktFBDaLnUjI6qEH0XEpHJzWEHksF6KgG0FOpAB3VAHoqFaCjGkBPpQp0VfvRS6kCXdV+9FIqQWe1Gz2UStBZ7UYPpRp0V3vRO6kIHdZO9EwqQ6e1Cz2SCtFx7UBPpEp0XY/RC6kSXddj9EIqRef1ED2QStF5PUQPpFJ0Xg/RA6kUnddD9ECqRff1CL2PatF9PULvo1p0X4/Q+6gW3dcj9D4qRgf2AD2PitGBPUDPo2J0YA/Q86gYHdgD9DwqRgf2AD2PitGBPUDPo2J0YA/Q86gaXdg2eh2VoxPbRI+jenRjW+htVI9ubAu9jerRjW2ht1E9urEt9DaqRze2hd5GE9CRbaCn0QR0ZBvoaTQBHdkGehpNQEe2gZ5GE9CRbaCn0QR0ZBvoaTQBHdkGehpNQEe2gZ5GE9CRbaCn0QR0ZBvoaTQBHdkGehpNQEe2gZ5G9ejGttDbqB7d2BZ6G9WjG9tCb6N6dGNb6G1Uj25sC72N6tGNbaG3UT26sS30NqpHN7aF3kb16Ma20NuoHt3YFnob1aMb20Jvo3p0Y1vobVSPbmwLvY3q0Y1tobdRPbqxLfQ2qkc3toXeRvXoxrbQ22gCOrIN9DSagI5sAz2NJqAj20BPownoyDbQ02gGurL76GU0B93ZPfQumoQO7Q56Fs1Cl3YbvYpmoUu7jV5F09Cp3USPopno2q7Ri2guureP6D00F93bR/Qemo5O7gI9hgh0de/oJcSgu/sVvYMgdHiv6BkEocN7Rc8gCl3eV/QKotDlfUWvIA7d3i/oDQSi43uxv6dGx/dif0+Nju/F/p4bXZ/9PTk6P/t7dvYnkv0JZX8i2Z9Q9ieS/QllfyLZn0j2JxCZn/3J/kSyP5HsTyT7EwjNz/6env2JZH8i2Z9I9ieS/YlkfyLZn0j2J5L9iWR/a3qd9/QL29+S2kzMtddgnLYuNj71zFR5r9soX6uZ5zfXZJiumu08Obg2u3TUcus5ofXapKeeW0/prNkmLTXdekZm3TbpqOvWEyprt0k/jceubqzhJM00H7u0sJ6T9NJ97NLEek7SSf+xKxNrOkkfC6xdF1jXRbpYZPGCtBqv0cfBxV+/PNEjpBV1YA0ddnDyE75CRk3j6Fs3d3DyU75CPKYD6Es3t9QzZF2m3cXbWuwlMq/T59Z9LfgUuVdqcumuVnyK3Ds1uXRPSz5F8qV6XLqn9Lc4xXNk3+rBnXXUos9Rca1TX7irVZ+j4l6nvnBTqz5Hxb1OfeGmVn2Oinud+sJNrfocFfc69YWbWvU5Ku516gv3tO5zlNzszBduad3nKLnZmS/c0sLvUXO18963pYXfo+Zq571vR0u/R9HlTnvfftZ+j6rbnfW+/az9HlW3O+t9+1n7Papud9b79rP4e1Rd76z3bWfx96i63lnv287i71F1vbPet53F36Pqeme9bzurv0fV/c56325Wf5Cy+53zuu0s/iaV1zvbXUFvG9waY3OhxR+l9Hrnuipne41vfuONlVZ/lNL7neuqlK01Hq60+qOU3u9cV50vYab8l9j6NEDpBc911amSpsp7gYcfRam94pluOlPOVmn7P/wkUPElT3TTiXK2Stv/4SeBii95optOlLNV2v4PP4lUfMsT3XSapLGS1t/1WaQnueY8OYMlDL/rc07iSa45Q+GUueihLr0d6O10S16zXtJs9eih7ng729rXLJM0Wz16qDveDrf2NcskzVaPHuqOJ7lmmez9ytBD3fEk1yyTvV8ZeqjbnuSaddIHrEIPdcOTXLNUxYY16KWuPMctq5WsWIEe6sJz3HKWsjUz0SN94zluOVfhpjnogd49xy1nq1w1BT3Qq6e4JKB21wT0QF89xSURxcuG0ft8ep5/zYxUv/FB9DD++8+TzNj5kCfYBLziecwae9j6i2A3PJNpa49afw/ohqczcfIRyy8BXPCs5g6/39K3n3O5JqA3eGDpm8+4XBvYK2xa+uITLtcI9w4blr51+eVaIV/ivpXvXHu3dtjHuGPl69bdrSn6QW5Y+aZFd+uLfpAbVr5nyd06ox/khpWvWXG35ugnubLyDdPvtgb6Wa6serfEey2Ffpcri14s61qrod/l2pr3yrnVeuh3ubbmtVJutSL6Ya6seCf7u4t+mFvWu5H9baDf5spat/ki6amWRL/NlaUu81XWW62IfpsrS13mq6y3WhL9OFfWucmvEl9rUfQLfWuBK3yQ+lRLol/oSvsLfKvixdZCv9CV7ue/UPJka6Gf6KPep/+g5smWQj/RldaH/6Dq0RZCP9ENrQ9/oe7ZFkI/0rW+J/+g8tnWQb/Sla7nvlL6bMugX+lK02Nfq323ZdDP9FHLQ99S/G4roZ/qQrPj3lX/bOug3+pCp7NumPJwi6Df6kKjo26Z83JroN/qQqOjbpnzcougH+tbbQ66bdLLrYF+rA/aHHTDvMdbAv1cl3qcctPMx1sA/VyXWhxy29TXWwD9Xhc6nPGBqY+3AvrBvtXgiI9MfbtV0I/25uTHu++XM3456sxnW0dk9kyb53v/8u2r99+/9dXFN0rK+e27hLfXpz1v8v6DLr56+/L6ZW691q3v93Lx/eD73rzbr1+9/cHzDvos7vTy6fVPKZuL3/g559ua3v/Yez8j3f+9dS7/r3B5EpF8BkmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJEmSJP3s5R76YFrd3fSMUOV21GeEqrE/PhtUsuH4jFBpDtdngwoKtmeDikjKzwR1QGJ9BqgxufHZoPYrac8GtU9pfgaobcX5GaA2lNdngrpvTn4GqFtm1WeAumFifiaoDybXZ4D61vz87E9vgPwsUF8w7VmgviDzM8Cnx+ZngM+Nru/FAJ8Z3d5X9AqC0OH9it5BCDq7N/QQItDVfYOeQvPRzV2gx9BsdHEf0XtoKjq3a/Qimomu7QZ6Es1Dt3YLvYmmoVO7jV5Fc9Cd3UcvoxnoyjbQ06ge3dgWehvVoxvbRI+janRh2+h1VIvu6yF6IFWi69qD3kh16LZ2oUdSFbqsfeiVVIQOayd6JtWgu9qNHkoF6KgG0FOpAB3VCHorpaOTGkOvpWx0UYPouZSMDmoUvZdS0TkNowdTIjqmI+jNlIdu6Qh6M+WhWzqC3kxp6JSOoVdTFrqko+jdlILO6DB6OKWgMzqMHk4Z6IoC6OmUgI4ogJ5OcXRDEfR2CqMTiqHXUxRdUAy9nqLogmLo9RREBxREz6cgOqAoej/F0P1E0fsphu4njB5QIXQ+YfSAiqDrSUBPqAA6ngT0hAqg40lAT6gAOp4E9IQKoONJQE+oADqeDPSGOoxOJwU9og6j00lBj6jD6HRS0CPqKLqcHPSKOoouJwe9oo6iy0lCz6hj6G6y0DvqGLqbLPSOOobuJgu9o46hu8lC76hj6G6y0DvqGLqbNPSQOoTOJg09pA6hs0lDD6lD6Gzy0EvqCLqaPPSSOoKuJhE9pcbRzWSit9Q4uplM9JYaRzeTid5SB9DRZKK31Di6mUz0lhpHN5OJ3lLj6GYy0VtqHN1MJnpLjaObSUWPqWF0MqnoMTWMTiYVPaaG0cnkotfUKLqYZPScGkQHk43eU4PoYNLRg2oInUsBelINoGOpQG+q/ehWStCjaj+6lRr0qtqLLqUIPat2okMpQs+qnehQqtC7aic6lCL0rNqJDqUIPat2okMpQs+qnehQitCzaic6lCr0rtqJDqUIPat2okMpQs+qnehQitCzaic6lCL0rNqJDqUIPat2okMpQs+qvehSatCraj+6lQr0ptqPbqUCvan2o1upQG+qAXQsBehJNYCOpQA9qQbQsRSgJ9UAOpYC9KQaQMdSgJ5UA+hYCtCTagRdSz56UY2ga8lHL6oRdC356EU1gq4lH72ohtC5pKMH1RA6l3T0oBpC55KN3lNj6F6y0XtqDN1LNnpPDaKDSUbPqVF0MbnoNTWKLiYXvaZG0cXkotfUKLqYXPSaGkUXk4teU8PoZFLRY2oYnUwqekwNo5NJRY+pYXQyqegxNY5uJhO9pcbRzSSip9QhdDZp6CF1DN1NFnpHHUN3k4XeUcfQ3SShZ9RRdDkp6BF1GJ1OCnpEHUank4IeUYfR6aSgR9RxdDsZ6A11HN1OAnpCRdD1xNELKoKuJ45eUBF0PXH0goqg6wmjB1QInU8YPaBC6Hyi6P0URAcURM+nIDqgGHo9RdEFhdDjKY5uKICeTgnoiA6jh1MKOqOj6N2UhA7pIHo2JaFDOoZeTVnokg6hR1MeuqUj6M2Uh27pCHoz5aFbOoLeTHnolg6gJ1MiOqYD6MmUia5pGD2YUtE5DaMHUyo6p2H0YMpF9zSK3ku56J5G0XspGR3UIHouJaODGkTPpWR0UIPouZSNLmoMvZay0UWNoddSNrqoMfRaykYXNYZeS9noosbQaykbXdQYei1lo4saRM+lbHRRY+i1lI0uagy9lrLRRY2h11I2uqhB9FzKRhc1iJ5L2eiixtBrKR2d1BB6LKWjkxpEz6VsdFHD6MGUis7pMHo45aA7OoweTinojI6jl1MGuqLj6OWUga4oiJ5PUXRBYfSAiqILCqLnUxAdUBS9n2LofqLo/RREBxRGD6gQOp8M9IYKoOPJQG+o4+h2Lhw9Dz2iDksuKOT4eegVdVhuQkdFzzV/N2WpKSqcTs530enVZBXvJu876cxKukqIJvN76bwqusooJve76awqysoo5uffn/ntdFY1cYV7efGnwOdQ0VZSKyXfVOdSUVdSKUXfVmdSUFdWJ1XfV2dS0FdmJnXfWaeQ0tOXX77s+IvWhPNtfkb6PKp3uLwb752d3/B5cz5CE9mfSIH+rp57Rn8GuJaE/l72fa8pB076EM0Sye/1vff1l3bg7c/J+hhNEcjv61+MfvnHtZ92/EOzaafO/CDViuR38eIvD//+y7RjZ36QSuXkt+vv/2UffOPTcj9KVdLS+/T1F/Py2z59/mcpW0p7L7v/+mP2FSo+T3nS2tv3XeffoeQTlSIlvluPfKL+DPC80vp7feTHf/4FrlHykUqQl9/rK788/MbEPUo+U2GZ+e3996+Qi5R8qILy0nt95bdfzE7h4cl0PinNXbzyrh+EXKXkUxURzu36kXf9IOYuJR+r46Kx3XrjXT8IukzJ5+qoYGp3Xvjl138INzmDfafTaaRFd/nCb+88O4N9x9NJpDX34YFff3nK/gzwNLKKu3rfL/8D/Pl3342KPluDcnK79bzvjzy5gd0nFC+ptxuv+/7EkxPYf0TRknK7fNiPjzu5gN3nFC4luDuPe/Kf/wyQl5Lanbc9+59/7Y+WEtqeV739d6JPcK2qI2iH1Or2Pei0t086rwpN6G3jI0uudOuD7O+c5tc3sT8DPD37sz8SkN8J+zNARDy+Qw839dkt8LyI+C4+N/Myjz7LAM+Gyu+c/RngbPZnfyQmvrcPzrrGrg8zwNMJ5dfnsZa81AqeI7+xa9KHfSKh/Bo91KLXas/+Wl+ruVh9nR5q1Xu1Fs2v0Tste7HOwv31ead1b9bXE+Vnfydkf0tcral4fZ1eaeGr9ZSSX6O3WvJSfaXm1+HBFrtOd/n1nfzBVrtPc+nxnf211rtRZ+n1nf7BFrtOb+nl9Xi0le7SWEV2LV7tZed/C/38N2mtqLsGr2Z/Z1AV3vmfzf5OoCq7Hg+3yj36Kmquy9utcIfOqnrr8nr9b9BbVW1dXq//DXqrqq3L6/W/QWtVsbV5vf43aK0qtj7v1/38rZWl1ucFe5++ubLQ+rxg79M3VxZanxfsffreyjpr9IStD99bWWXtnrHruXsr7qzXQ/Y8dWsTUuvzkj1P3dqE1Po8Zc9Td1bfWauXbHnozmaU1ugx+524uSmh9XnOdgdubkpmjZ6z3YF7m1JZp+dsd+DWpkTW6kF7nba5OYk9Qq9wqdFRu5vU10P0DlfaHLS3mY1toXe40uagrc1MbBM9xJU2B21sZmCP0Ftc6XLOvqb29RC9xg1NjtnV7MIeofe40uKQbc3O6yF6kCstDtnW7Lweoge51uKQPU2vawd6kysdztjS9LT2ooe5dPbztTU/rL3oZS6d/HhtAWHtRU9z4dynawvIajd6mwunPlxfRFd70dtcOPXhuiKqGkDP860zn60roqkx9ELvPv6H8enzLABqagQ90bsP/dHHWQEV1QB6oncvn+wvGRXVfvRCH533ZB1xXe1ED3TTeU/WzVAGZZFtf+z5nPhozdx+9QfP//57v/7gse9+I7Em2b07/wnbuKju01ZNOR+3+atfz1N7AkmSpOn+H1BLAwQUAAAACAAAACEAE+S/fHoXAACAVwgAEAAUAGdyb3VuZF90cnV0aC5ucHkBABAAgFcIAAAAAAB6FwAAAAAAAO3dsc6tx1WHcZ+WqzidQXIBUQQSF0AHoqGgQoYYUSAS2YgGuApumDh2bJ+z3/3umVlrzTMz+/l1iZJvv/NfT5EiSv7v7/7hb//+Hz988V9f/PeXv/nmu3/59su//vjl//zzX3z51ccv//W33/7nt1//xz/99tvffPP9P/83X//7d9/8/p//7t++/t03v//Hf/qrrz7++ld/9dXHv/z1n//ZVx//92O/P/lCkub58An6a/RmPiyFXkOT0cF9hp5Dc9G9fY7eQ1PRuT2gB9FMdG2P6EU0E13bI3oRTUTHdoGeRPPQrV2hN9E0dGqX6FE0C13aNXoVTUKH9gQ9i+agO3uG3kVT0Jk9RQ+jCejIbtDTqB7d2B16G5WjE7tFj6NidGAv0POoFt3XK/Q+KkXn9RI9kArRcTWgJ1Iduq0W9EYqQ6fVhB5JVeiy2tArqQbdVSt6J5Wgs2pGD6UKdFXt6KVUgI6qAz2V8tFN9aC3Ujo6qS70WMpGF9WHXkvJ6KA60XMpFZ1TN3owZaJr6kcvpkR0TAPoyZSHbmkEvZnS0CkNoUdTFrqkMfRqSkKHNIieTTnojkbRuykFndEwejhloCsaRy+nBHREAfR0iqMbiqC3UxidUAg9noLogILo+RRD9xNF76cQOp8wekBF0PXE0Qsqgq4njl5QEXQ9cfSCiqDriaMXVARdTxy9oCLoeuLoBRVB1xNHL6gIup44ekFF0PXE0Qsqgq4njl5QEXQ9cfSCiqDriaMXVARdTxy9oALoeBLQEyqAjicBPaEC6HgS0BMqgI4nAT2hAuh4EtATKoCOJwE9oQLoeBLQEyqAjicBPaEC6HgS0BMqgI4nAT2hAuh4EtATKoCOJwE9oQLoeBLQEyqAjicBPaEC6HgS0BMqgI4nAT2hAuh4EtATahzdTgZ6Q42j28lAb6hxdDsZ6A01jm4nA72hxtHtZKA31Di6nQz0hhpHt5OAnlABdDwJ6AkVQMeTgJ5QAXQ8CegJFUDHk4CeUAF0PAnoCRVAx5OAnlABdDwJ6AkVQMeTgJ5QAXQ8CegJFUDHk4CeUAF0PAnoCTWObicDvaHG0e1koDfUOLqdDPSGGke3k4HeUOPodjLQG2oc3U4GekONo9vJQG+oYXQ6KegRNYxOJwU9oobR6aSgR9QwOp0U9IgaRqeTgh5Rw+h0UtAjahidTgp6RI2iy8lBr6hRdDk56BU1ii4nB72iRtHl5KBX1Ci6nBz0ihpFl5ODXlGD6HCS0DNqEB1OEnpGDaLDSULPqEF0OEnoGTWIDicJPaMG0eEkoWfUIDqcJPSMGkN3k4XeUWPobrLQO2oM3U0WekeNobvJQu+oMXQ3WegdNYbuJgu9o8bQ3WShd9QgOpwk9IwaRZeTg15Ro+hyctArahidTgp6RI2j28lAb6hxdDsZ6A0VQMeTgJ5QEXQ9cfSCiqDriaMXVAidTxg9oGLofqLo/RRD9xNF76cgOqAgej5F0QXF0Ospii4ohl5PYXRCIfR4iqMbiqC3UxzdUAS9nRLQEQXQ0ykDXdE4ejlloCsaRy+nFHRGw+jhlILOaBg9nHLQHY2id1MSOqRB9GxKQoc0iJ5NWeiSxtCrKQ2d0hB6NKWhUxpCj6Y8dEsj6M2Uh25pBL2Z8tAtjaA3UyI6pgH0ZEpExzSAnkyJ6JgG0JMpE11TP3oxZaJr6kcvpkx0Tf3oxZSJrqkfvZhS0Tl1owdTKjqnbvRgSkXn1I0eTLnonnrReykX3VMvei/lonvqRe+lXHRPvei9lIwOqhM9l5LRQXWi51IyOqhO9FzKRhfVh15L2eii+tBrKRtdVB96LWWji+pDr6V0dFJd6LGUjk6qCz2W0tFJdaHHUj66qR70VspHN9WD3koF6Kg60FOpAl1VO3opVaCrakcvpRJ0Vs3ooVSD7qoVvZNq0F21ondSETqsRvRMKkKH1YieSVXostrQK6kMnVYTeiSVodNqQo+kOnRbLeiNVIiOqwE9kQrRcTWgJ1Iluq7X6IVUis7rJXoglaLzeokeSLXovl6h91Etuq9X6H1UjA7sBXoeVaMLu0evo2p0YffodVSOTuwWPY7q0Y3dobdRPbqxO/Q2qkc3dofeRvXoxu7Q26ge3dgdehvVoxu7Q2+jenRjd+htVI9u7A69jerRjd2ht1E9urE79DaqRzd2h95G9ejG7tDbaAI6shv0NJqAjuwGPY0moCO7QU+jCejIbtDTaAI6shv0NJqAjuwGPY0moCO7QU+jCejIbtDTaAI6shv0NJqAjuwGPY0moCO7QU+jCejIbtDTaAI6shv0NJqAjuwGPY0moCO7QU+jCejIbtDTaAa6sufoZTQDXdlz9DKaga7sOXoZzUBX9hy9jGagK3uOXkYz0JU9Ry+jGejKnqOX0RR0Zk/Rw2gKOrOn6GE0BZ3ZU/QwmoPu7Al6Fk1Ch/YEPYsmoUN7gp5Fs9ClXaNX0Sx0adfoVTQLXdo1ehXNQpd2jV5F09CpXaJH0TR0apfoUTQNndolehTNQ7d2hd5E89CtXaE30Tx0a1foTTQTXdsjehFNRgf3GXoOzUc390v0FkLQ2f2EHkIUurwf0CsIQof3I3oGEejqfkYvoeno5D5Bj6Hp6OQ+QY8hAB3dL9BTiEBX9zN6CSHo7H5CDyEIHd6P6BlEocv7Ab2COHR736M3EIiO74P9vTe6Pvt7c3R+9vfe6Pzs783Zn1D2J5T9CWV/QtmfSPYnlP0JZX8i2Z9AXH72J/sTCszP/kTmZ3+yP4HQ/Ozv3bH52d+bg/OzvzdnfwLR+dnfW6Prs7/3Rtdnf2+Ozs/+3hudn/21O3IvOL/j9qxy7F5kfCfuWeHkvbDyDt0zXWSuHfaFwhsb9M1E5tpm44m1PaIfv6zYXFvNXN3YHfrtS4rOtdnOZXE1oN++oOhc+w1dEVYj+unrCa615dLpWTWjX76c2FqbTp2aVBf65cuJrLXv1nlBdaIfvprIWjuPnZRTN/rdiwmstfnaCS2NoJ+9mOG19t87GNIg+tWLGVzriMEjGQ2jH72YoblOWXy0oQj6zYsZmCsy+WIn6M8njH7yYrrnCm2+3CFGXhNDv3gxnXOFRl/xGoMPGkc/eDGdcwVWX/Uco08aRD93MX1zHXmO/Ect/uCV9M115j0KXrX0e1fSNdepByl517rPXUrPXscepORh6z53JR17HXyRoqct+tqVdOx18kmq3rbkY1fSvtfZJyl73YqPXUj7XmffpO516711Jc17HX6Twuct99aVNO91+lEq37fYU1fSutfxV6l94EovXUrrXsdfpfaBK710Ka17HX+W4geu89C1tM51/F2qH7jKOxfTOtf5d6l+4SrvXEvrXOffpfqFq7xzLa1znX+X6heu8s61tM51/l2qX7jKO5fSPNcbHKb8iWs8cynNc73BYcqfuMYzl9I81xscpvyJazxzKc1zvcFhyp+4xjOX0jzXOxym/I1rPHMlzWudf5jyF67xzLU0r3X8ZeofuMIr19O41uGXqX/eCq/kXU3RNFftVT75qfnqH7fCK3GRudIP8fyndthlw1fCgnulHuH2lzYZZrNXsuJ75V3g1S9tM81OrySlDJazfssv7TXPJo/EZA2WMH3jL2230A6PpCQtlvBn2n5o043WfyRj6rRRbzES9UjE1GXD3mIm7JGAmbtmeIedqDcSZu6a4R2Gwt5ImLhrhneYinzjdLNGTULP9cU7/NfMpiqfMxW91s/e4Y1T1A1ZgB7rwTu8sVjBhHXosR68wROL5S9YiB7rE2/wxBlqZixCj/WT8184TdmUFeixfnT8Ayeq27ICvdb3Tn/fbJV7pqPHKp6LfhyjdNJkZ08Fvw5TPGumo3dCH4eqXjbPySOBb+OVr5vk4H2opy1iwsIZzh2HedlCZowcdu4eE1+2qsmLDzl2g+qHbQFbv92p7y991z7AC7Q59e2F79oLeoUGhz687FnbYe/w2pmvrnnVnuhb3DvzxRWv2hd9jVtHvjb9UZuj73HjyJcmP+oA9EWeO/GVmW86BH2Sp058Y+KbzkEf5ZkD35f1pMPQZ7l24OOSnnQc+i7XzntZxouORB/m2nHvij/oVPRlLh33qIxLHYo+zaXTnpRzqjPRt7nie94IfZwrZ73G/m7R17ly0FM+2N8L9HmuHPOQ7yXf6zj0fS6c8YofpR/sNPSBLpzwhj8quNhh6Atd2P4BP6u52VHoEz3a/PN/qehmJ6FPdGHrj/9E3dnOQd/owsaf/qnKux2DPtKFbT/8M7WHOwR9pAu7fvfnii93CPpKF/b86gflpzsDfaYLO37zowm3OwF9piv7ffGFOefbH32nC7t977VJ99sefacLe33tE9MOuDn6Tle2+tgnJp5wb/ShLm30qU/MPeLO6Etd2uZDn5l9xX3Rl3pil+98AjjkruhTPbPHVz7BnHJP9K2e2eEbn6FuuSP6Vjc2+MRr5D23Qx/rheU/8AJ90r3Q1zoPfdG90Nc6D33RzdDnOg990f3QFzsLfc1d0Xc7BX3HvdHX2x99wRPQN9wZfbtz0JfcE301SZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSZIkSdIxPnygv0Dv68Mz9IfpDTytzx5VrqM+G1Sy7vpsUGkG67NBJQjVZ4OKScjPBjUoqz4bVL/c+oxQPUrqs0E1KazPBvVCeX02qKcm1WeDujI1PwvUJ2bXZ3/62fz6DFB/hNRnf/oDqD4D1BdkfQYoNj/7e3dsfgb45uj87O+d0fF9j95AGDq9H9ArCEKH9yN6BjHo7n5CDyECXd3P6CUEoKP7JXoLzUYX9yl6DU1GB/c5eg9NRef2iF5EE9GxXaAn0Tx0a5foUTQJHdoT9Cyag+7sKXoYzUBXdoOeRvXoxu7Q26gcndg9eh0VowN7gZ5Hpei8XqMXUiE6rhb0RipDp9WEHklV6LIa0TOpBt1VK3onVaCr6kBPpXx0Uz3orZSOTqoPvZaS0UH1ovdSKjqnbvRgykTXNICeTHnolkbQmykP3dIQejSloVMaQ6+mLHRJY+jVlIUuaRA9m5LQIQ2iZ1MOuqNh9HBKQWc0jl5OGeiKxtHLKQNdUQA9nRLQEQXQ0ykBHVEEvZ3i6IYi6O0URzcUQo+nMDqhEHo8hdEJhdDjKYxOKIZeT1F0QTH0eoqiCwqi51MQHVAQPZ+C6ICC6PkURAcURe+nGLqfKHo/xdD9RNH7KYbuJ4weUCF0PmH0gIqg64mjF1QEXU8CekIF0PEkoCdUAB1PBnpDjaPbyUBvqHF0OxnoDTWObicFPaKG0emkoEfUMDqdFPSIGkank4NeUaPocnLQK2oUXU4OekWNostJQs+oQXQ4SegZNYgOJwu9o8bQ3WShd9QgOpwk9IwaRIeThJ5Rg+hwktAzahAdThJ6Rg2iw0lCz6hBdDhJ6Bk1iA4nCT2jBtHhJKFn1CA6nCT0jBpEh5OEnlGj6HJy0CtqFF1ODnpFjaLLyUGvqFF0OTnoFTWKLicHvaJG0eWkosdUNzqZdPSg6kLnko9eVD3oWgrQk6oDHUsBelL1oGspQE+qDnQsFehN1Y5upQQ9qprRqdSgV1UrupQi9KxqRIdShJ5VrehSStCjqhmdSgl6VDWjUylBj6p2dCsV6E3Vjm6lAr2p2tGtVKA3VTu6lQr0pupAx1KAnlQd6Fjy0YuqB11LPnpRdaFzSUcPqi50LunoQdWFziUdPai60LmkowdVH7qXbPSe6kP3koyeU53oYJLRc6oXXUwuek31oovJRa+pXnQxueg11Y1OJhU9prrRyaSix1Q3OplM9JbqRzeTid5SA+hoEtFTagAdTSJ6Sg2go0lET6kRdDV56CU1gq4mD72kRtDVpKGH1BA6mzT0kBpDd5OF3lFj6G6y0DtqDN1NFnpHDaLDSULPqEF0ODnoFTWKLicHvaJG0eXkoFfUMDqdFPSIGkank4IeUcPodDLQGyqAjicBPaEC6HgS0BMqgI4nAT2hAuh4EtATKoCOJwE9oQLoeOLoBRVB1xNHL6gIup44ekFF0PXE0Qsqgq4nit5PMXQ/QfR8CqIDiqHXUxRdUAg9nsLohALo6ZSAjmgcvZxS0BmNondTEjqkMfRqSkOnNICeTJnomrrRgykX3VMnei5lo4vqRM+lbHRRfei1lI5Oqgs9ltLRSXWhx1I6Oqku9FhKRyfVhR5L+eimetBbKR/dVA96K+Wjm+pCj6V0dFJd6LGUjk6qCz2W0tFJdaHHUjo6qS70WEpHJ9WFHkv56KZ60FspH91UD3or5aOb6kFvpXx0Uz3orVSAjqoHvZXy0U31oLdSPrqpHvRWKkBH1YGeSgXoqDrQU6kCXVU7eilVoKtqRy+lEnRWzeihVILOqhk9lGrQXbWid1INuqtm9FAqQWc1hl5NaeiURtG7KQfd0Th6OaWgMxpHL6cMdEUR9HZKQEcUQW+nOLqhGHo9hdEJxdDrKYouKIreT0F0QFH0foqh+4mjF1QInU8cvaAi6Hp+afR76A0VkB3RuPFv4tZTVEVJI0LfRQynHEU99Ql/2eTRlKYqqQ4ZnzZzMiWqiiqhnKy/o5XVVJVRTeKf0rJKssppJvWPaVElYaUEk/33tKKKspJqKfiTWk5FXDmp1PxVraWgrqxOyv6wFlIQWFIlhX9ay8gPLC2R4j+vJaQnlpZH/S+Il91YYhtTfkSw7MwSw3j+b879HYEmtDecxfN/rwEeY938fvi23o8eHkKIlfv7YuS/oTX8Y0Ks3N/QR+f/mspMqK8iiNm/pwIz2qvqYf4vKtWc9MpqAH5SeaZ0V9oC8qPKMaW72hSYX1WGGdVVlwD9rOImNDehA+yHFTOhuRkZcL+sgPrgJkVA/rZGlfc2rwH0xzWkPLeJCbC/rgHVsc0NAP8AdSqObfL58Q9Qn+LWZp+f/wL1qC1t/vFX+AY1Kw0Nuf0SH6FGpaEhp1/iI9SmtDPm9Gt8hVpUVvaLS0+9fPtnCZab2d2ZJx6+88vESS7t9sTz7j7wcSKk9NV+4Fl37/l6cYaje7he63VX688AQUntPflzdz9a+iz/j0L2kNnew9+8/5fkP+bJZ9jfutLj++nvvv7dpCfc/YQBrq0qv6Yfjv+Vlz9hgEvD8vvDT2f8mftfMMClDeZnf0oB5vf6PyFm/IIBrmwwv0P7M8DJ2Pxu/gck037AABdG52d/7w3vr9zBT9vf+fkZ4MJG89vqRkc/bmvj+e10o5PftrVIfhsd6eS3bc3+9n/cxmL5bXSjox+3rWh++9zo5LdtK57fPkc6+W27yuhvmysd/LRN5eS3y5mOfdi20vrb41CHPmtbifntcaojH7Wt3Py2ONaBT9pXen8bnOu4B+2rIL8NDnbYczZWkt/6JzvqMRuriW/9ox30lK3VpLfB2Y55yNaKwtvhcIc8Y29V3e1wuTNesbWy7La43Qlv2FpddHsc74AnbK0wui3Ot/8LtlYX3C7n2/4BWysMbpcD7v31eyvtbZcTbv3xe6vtbZcb7vztWyvObZsrbvvhmyvPbZM77vrdmyuPbZtL7vnVm6tPbZ9TbvnRm5uQ2j7H3O+LdzeltH3Oud0H725OadscdLPP3d6czDa66FYfu71ZmW10052+dXfTKrtBb/Bgo0/d3bzKbtAjfG6bD93exMhu0Cs82OZDNzczsjv0Dg92+c69TW3sFr3Eg00+c2tzE7tFT/Fgj6/c2+TGbtFbPNjiI7c2O7F79BoPdvjGnU0v7AV6j8+t/4Vbmx/YC/QgD9b/wo0Bgb1CT/Jg9e/bGBLYK/QoDxb/vI0xgb1Aj/Jg8c/bF9PXS/QsD5b+uH1Rfb1ED/Ng5W/bFpbXS/Qyjxb+tF2Bfb1Eb/Ng3S/bFpjXa/Q4D5b9sF2Rdb1Gr/No1e/aFJvXa/Q+jxb9rD3BdTWgF3qw5Edtio6rBb3RgxW/aVN0W5+h52i0zYeu7vxUJEmSJEmg/wdQSwECFAMUAAAACAAAACEAvWLXdAYYAACAVwgADgAAAAAAAAAAAAAAgAEAAAAAcHJlZGljdGlvbi5ucHlQSwECFAMUAAAACAAAACEAE+S/fHoXAACAVwgAEAAAAAAAAAAAAAAAgAFGGAAAZ3JvdW5kX3RydXRoLm5weVBLBQYAAAAAAgACAHoAAAACMAAAAAA=', 'reference/sample-19.npz': '...', 'reference/sample-2.npz': 'UEsDBBQAAAAIAAAAIQCscro4EAwAAIBgCQAOABQAcHJlZGljdGlvbi5ucHkBABAAgGAJAAAAAAAQDAAAAAAAAO3bwW5cSRkF4HjLU2QXkLIY0AghHoAdiA0LVmhgglggBmUQG+ApeGEmxI677dvuuvf+Vafa9/s2k+nY1dXnnOlM7OS/v/ndr3/7+7s3/3zzr3fffvj+Tx/f/fLtu3//8afv3r999+fvPv7j4zd/+8N3H7/98OnxX33z1+8//PD493/55u8ffvj3H//s/duvf/HV+7c///qrn7x/+5+36/3oDQAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAwLzudkrfn5u2d372xw6752d/bLR/exbIJlXTs0DONY2jeH4GyGcNMynfngEeXp9NrZPOgHINtaZX92hIIvSyut8Bk1ohERkl0tMpkQ6RbdK7qZLOkU3SsynzxgRvUXo2ldJZsl56M6XSYbJaejKl0mGyWnoypdJhslZ6MbXSabJWejG10mmyVnoxdT69lHSarJVeTZ03vgB4a9KTKeX97+akJ1MuHShrpNdSyy+/tya9mFo/vJ50oKySXkwt+7sx6cEUs7+b8bmo9GCKhTNlvfRkSqXDZLv0dgqkI2Sv9IL2SafHTukB7ZSOj53SA9opHR87pQe0Uzo+dkoPaKd0fOyT3s9e6fzYI72e/dIJskN6PAXSEbJTekA7peOjxosNX/iXNH/04HW5+/yn2Z+WvPCBj58QNDAZRmhu+eHRwYNruBk3rLnluyn+JNfQbOivuWX7o5M1JSdG13YzbtaKliOzu34tbtWaoodPrvlm3KjmlkdPbdHgcOhqQ8sjx9Z8KW7O0ledG6u+/5BBs/MNj1foauW1560Z26rfjHObpt3fwvtqh5dPWPH+6uZXcBmmdFJldeXmxzUnZVZ1/uXDqgdne6/Lk2GV7a/Pn8h6PLBLGIy2XO7W/Z2+6z07vmiA9veabJrAC6c9nrrx9PZr2OCNu5/I2TtKQ+8vHPh0f+UD7JoHY73Y52PhffZ35aft7/Vr7fOh+6sLeL6/i78SPz38xcmZ36v0cqFnE2p7C3r82eb9XZ3f2feid7xYpvNiowvvPtc3cDKrk3NOjnt6SsP8Wp+b2/P8/erJV04ef7R7A4vH2B7Pv/9x/ljVDvbtb+9rZFZnFT/84OlD9kcfZx0v1F42hOfHtK/P/l6nNQvYOYSFc+yP/ztpuOMQ1szN/o7rbsff/7h2sOlxTbf9bRpg1aviVvTbw5b9GeDR9NvDpv0Z4MH0W8O2/e34Lfeey5LR8d1o4/62PvPe65Iw7f7WPvn9pxjhTSmdwLqjWwa44gIbP42oufe3eIPlSxVenP42t1/+DI332PRJTKp3hVvWUq80Msp0rLDblrbokR27dayw15K26ZIeO3WssNOOtuqTH/v067DPinboFSHb9Wqxfj379U2SDbb1ePVvos9pVKi06lNl4WRKDQqVVp2qrFtMrUGp0qpPl2V7qTYsV9rUtfr4w5Iz+4hmzZun36yv63Wx6vXHnN3w4bH99zt7AoLOS6grdrHrzaed3K30/TQSOSfu+uyvWqerxWLn3lkJlbUuPFp1fJ1U6jw4LaG21ecPVp1fJ5M5j05aKG71+YObziq4z+XDSXusobrWZw/WPUGVXOzc+1JDfa2FJ/aSy53PTluob/T8JyrOLzU4a5570sLOXyYvPsmXJ5tK52y5bqmFjoWWTWe/Hmmy1lIRHTstW89+PdJkrSdN7PzftJZnm0W/TGn3pIk+//939myz6BgqzU6auNv/m9SWZ5tF31xpc9LE5x926XTC3/8OyJbrvlRx30qfUu2PZV+qGFBqyVNUGZAtLU6//XH/j/JSizZTaUS0bFPeaeVwigyMk7XuC/rS1d5KNy9k0yc2n86kPjdU1+nmhWz6xObTmVR1qVuP23qPlrOZVn2nlcupMS5NrvlcyN3ZV6JrKy2bTZkRudLmtJXt3wNpe4YV5227RuvpTOO0lT6N7t9LtVHZcl3/RmueodKobLlqTKHPP+SlU+56/42R6hDZauYea7Y2/t40mrzGiqVFLk6juVus2Vri5jSausWisQVuTrOJS6zb2/Cr02zeCksnN/ryNJq3wdrNjb49beZtsHZzo29Pm3kbrN3c4MvTaN4Ka0c3+PKsMGmH5cMbdG/W2vT91v636qD7pdlovipffO77G0x3aTZa2WPyS4B77s2s5qxx3XPf/6dx8oH3P7S8mzDN7DimC6uzPsbwpkeU6QEAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACj3D3z6aH0rXiFnk+tSfra3LiNuzNCKlTMzwB5pm0kNfOzPx4076VoegZ4DC9WXLqlzUZFQbmNRffZ0S7BEFktvZYu0qHSKD2UTtKxclV6Il2lw+Vl6X30ls6Xy9LbGCIdMhekhzFIOmYWpWcxTDpolqRXMU46aRakRzFQOmoWpEcxUDpqFqRHMVA6ahakRzFQOmoWpEcxUDpqFqRHMVA6ahakRzFQOmoWpEcxUDpqFqRHMVA6ahakRzFQOmoWpEcxTjpplqRXMUg6Zi5ID2OMdMpckl7GIOmYWZbexSjpnLkgPYxB0jFzQXoYg6Rj5oL0MAZJx8wF6WEMko6ZZeldjJLOmQvSwxgkHTMXpIcxRjplLkkvY4x0ylySXsYY6ZS5JL2MMdIpc0l6GWOkU+aS9DKGSIfMRelpDJEOmYvS0xgiHTIXpacxRDpkLkpPo790wrwkvY6e0tlyXXojvaRzpU16J72kc6VNeie9pHOlTXonnaRjpVF6KF2kQ6VZeir10omyRnot9dKJskJ6LB2kI6VVeildpEOlVXopxTaGUJsprdJzKbYjgfJouS65lR72JFCfLlfkhtLJngjq4+VluZ30sieD+nx5WW4nveyJoEPAvCg2k252vfSOSbNg9DgGWPfan2TQL2mWRBbS16rX/jSDbkGzKDORnva99I5R89TgZYyx65V3DpwzI2cxxt7X3D9zHo3ZxEh7X3P/zHk0ZhMj7X3N/TPn0ZhNjLbjNY+JnXuDBjHclpc6PHxe7f6er2ntxzPCiCWEbHihmQ6OrPMGkra9yFwVx9Sr/VuV7uOA0pVPJV3GMaVbn0e6iYNK1z6NdBFHle59GukiDitd/CTSNRxYuvoppEs4sHT1U0iXcGDp6qeQLuHA0tVPIV3CkaW7n0G6gyNLdz+DdAeHli5/AukKDi1d/gTSFRxauvwJpCs4tHT5E0hXcGjp8ieQruDQ0uVPIF3BoaXLn0C6gkNLlz+BdAWHli5/AukKDi1dfl66gWNLt5+XbuDY0u3HpQs4unT/aen8Dy5df1y6gINL1x+XLuDg0vWnpfM/vPQAwtLxH156AFnp9EkvICud/uH1bbbb6VXC6dO32W6nVwmnT+diex1fJZo99kdY3177nF4nmTyf9K21y+l1grlzr2+vPU6vk0udB3177XF6nVzqPOhba4fTC8VC51HnWuuPrxNKnDPDGm391MoLNT4lQdP1eFdxqcC92WTKGiv2Fbk4q01ZY8nCEhdnvRlbrFrZ+Juz2pQtlg1t+M1ZacoWy4Y2+uKsNWeLdVsbfXPWmbPEwrGNvjrrzFli5dwGX52VpmyxdHDjrs0WUxZZvLtBt2aDKYusXt6YW7PBnE1Wb2/MrdlkxiJt7Vjm7tamXr9Jp8dhGBsAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAMB+/wNQSwMEFAAAAAgAAAAhABRZ647XCgAAgGAJABAAFABncm91bmRfdHJ1dGgubnB5AQAQAIBgCQAAAAAA1woAAAAAAADt3L2OpMd5huGd1Eex2doAA0kQDMMH4MyCEwWKBEqi4cCwBFJQIvsofMIiRUqcn+6e6Z766q7euq5sN+n63ufGhvv/v/jlv//Hrx4+/OnDnz/97qtvfvv1p3/9+Ol/f/PTT198/PSfv//6j19/+T+//v3Xv/vqu7//ty//+5uvvv37b/7ryz989e2f//FnX3z8+b/85IuP//zzn/zTFx//7+P1/uEDAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAYz3UD2BjDw/6I/PwIEAyD/oj8/CgPzIPDwIk86A/Og/6I/PwVP0c9vKgPzrP89Mf87yoT4DMcyo//THJyfwEyBxn8tMfE5yrT39McD4/AXK4S/npj4NdzE9/HOpyfQLkUK/mpz+O83p++uMob6hPgBzlbfnpj0O8MT/9cYS35idAxnt7ffpjuGvyEyCDXZef/hjpyvr0x0hX5ydARtIfJf1Rur4/ATLODf0JkHH0R8k/gJT0R+mW/gTIKDf1J0BG0R8l/wBS0h+l2/oTIIMIkJL+KN3YnwAZQn+kBEjp1v4EyAj6o3RzfwJkAP2Ruj1AQfJ+4/qTJNc7tD9F8orJ/SmQJ6b3J0B+ND8//fF3QX7642+K/PTHD5L89Mf3mvwEyF9V+emPD2F++qPMb1p/za/yFk15R6dQ/S7XmtLZ7A7K3+YaRxdWRND+Olc4MK2qgPr3ebuDqgoDiH+eaxxS1HXiDxr781zlgJyu137QyF/nOuNbukX7PeN+nSuNDulW6ecM+nGuNjaid2i/Zsivc7WBAb1X+jHv/3FuMaye90u/ZcApud6odoYoP2XMObnSmHBGCT9k2EW5wohoBgq/Y+BRebP37zZW9xVj78rbjFhupPAjBl+Wtxgz3UDdJ4w/Lq8aNd4w3QcccV5eMW6+UbLnH3RhLhk54BjZ4w+7MWeNXXCI7N0Hn5oTjtjxvbIXzzk5jxy45s2yp069PN85ftQbVI8sBtjcrGmvUr0u3GFXkxd+I/8L4SaSmZdVr7GfevG11Gvsp158MfUc26kHX0w9x3bqwRdTz7GdevDF1HPspt57NfUeu6n3Xk29x27qvVdT77Gbeu/l1INspp57OfUgm6nnXk49yGbquZdTD7KZeu7l1INspp57OfUgu6n3Xk49yGbquZdTD7KZeu7l1INspp57OfUgu6n3Xk29x27qvVdT77Gbeu/V1Htspx58MfUc26kHX0w9x3bqwRdTz7GfevG11Gvsp158LfUa+6kXX0u9xn7qxddSr7GhevKl1GPsqN58JfUWW6pHX0g9xZ7q1RdST7GnevV11Etsqp59GfUQu6p3X0W9w7bq4RdRz7Cxevol1CNsrp4/Vw/A3hHWt+cHdQiR+uz8qG6hUN+cH9UtFOqb80gdQ6A+OY/UMQTqk/NIHUOgPjmP1DEE6pPzWF3DfPXFeayuYb764jxW1zBffXGeqHOYrj44T9Q5TFcfnCfqHOarL85jdQ3z1RfniTqH6eqD80Sdw3T1wXmizmG6+uA8VfcwW31vnqp7mK2+N0/VPcxW35tn6iAmq8/NM3UQk9Xn5pk6iMnqc/NMHcRk9bl5ri5irvraPFcXMVd9bZ6ri5irvjYv1ElMVR+bF+okpqqPzQt1ElPVx+aluomZ6lvzUt3ETPWtealuYqb61rxUNzFVfWxeqpuYqb41L9VNzFTfmpfqJmaqb80JdRQT1afmhDqKiepTc0IdxUT1qTmhjmKi+tScUlcxT31pTqmrmKe+NKfUVcxTX5qT6iymqQ/NSXUWs9R35rS6i1nqO3NGHcYk9Zk5ow5jjvrKnFOXMUd9Zc6py5ijvjJn1WnMUN+Y8+o2ZqhvzHl1GzPUN+aCOo7j1RfmkrqO49UX5pK6juPVF+aiOo+j1fflsjOTTe/kKPV9uewNi82PZqDgpFzhysGCgt5lzhG53TsWK4K60mFnY5BBgyV1vW7kpTjCAXs1qZ005Hs40rFzVeEd9T2MNnEv/fFCNJf++F6/lvx2ttZa+tvNwmvpbwN3Mpb+Pl/3tpX8Pjf3Npb+SOmPlP4oyY+U/kjpj5L+SKmPlPxIqY+S+oipj5z8yKkPAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAOg8fKt+A5+7hzeq38k9e2tlQuR2AyvT386mhHSj+jYcpA7rjeozMVpd1JXqczFMndJN6qPxfnVD71Hfjveo63m/+oLcpM5mmPqQXKkOZrT6nrxRHcpB6rPyqjqRQ9XH5RV1IAerz8tldR+Hqw/MRXUeh6sPzEV1HoerD8xFdR6Hqw/MRXUex6svzCV1HcerL8wFdRwT1CfmgjqOCeoTc0EdxwT1ibmgjmOG+sacV7cxQ31jzqvbmKG+MWfVaUxRH5mz6jTmqK/MOXUZc9RX5py6jDnqK3NOXcYk9Zk5re5ilvrOnFZ3MUt9Z06ru5imPjQn1VlMUx+ak+ospqkPzUl1FvPUl+aEOoqJ6lNzQh3FRPWpOaGOYqb61rxUNzFTfWteqpuYqj42L9RJTFUfm+fqIuaqr81zdRGT1efmmTqIyepz80wdxGT1uXmmDmK2+t48UecwXX1wHqtrmK++OI/VNQTqk/OjuoVCfXP+rk4hUR+dv6lLiNRn5wd1CJH67Hyv7qBS352/qjPo1Jfnw8756W8FdQSl+vZsnZ/+cnUBrfr626sDqNX33129f62+/+bq+Xv1Alurx19APcHW6vEXUE+wtXr8FdQbbK0efwH1BFurx19BvcHO6u1XUG+ws3r7FdQb7Kzefgn1CBurp19CPcLG6unXUK+wr3r5NdQrbKyefg31Cvuql19DvcK+6uXXUK+wr3r5RdQzbKsefhH1DNuqh19FvcOu6t1XUe+wq3r3ZdRD7KrefRX1Druqd19FvcOu6t2XUQ+xqXr2VdQ77KrefRX1Druqd19FvcOu6t1XUe+wq3r3VdQ7bKsefhH1DNuqh19EPcO26uEXUc+wrXr4RdQzbKsefhH1DNuqh19EPcO26uHXUK+wr3r5NdQrbKyefgn1CBurp19CPcLG6umXUI+wsXr6JdQjbKyefgn1CBurp19CPcLG6umXUI+wsYV2nvqUK97FkZZced6jrn0Zg6278ayXya+09MRLP44RVt939ffxPnew7vov5Gb3se09vJFb3M2y9/FKrnY3s97NQ7nO/Yx6xEv117ujSUc/VX9ruL9B7+y5vOIzmvPuHsx3Po/4LrrDJ2/l866P9amPlviIaY+W9ohJj1VoDwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAO7TXwBQSwECFAMUAAAACAAAACEArHK6OBAMAACAYAkADgAAAAAAAAAAAAAAgAEAAAAAcHJlZGljdGlvbi5ucHlQSwECFAMUAAAACAAAACEAFFnrjtcKAACAYAkAEAAAAAAAAAAAAAAAgAFQDAAAZ3JvdW5kX3RydXRoLm5weVBLBQYAAAAAAgACAHoAAABpFwAAAAA=', 'reference/sample-3.npz': 'UEsDBBQAAAAIAAAAIQD1GzTIJhAAAICuEAAOABQAcHJlZGljdGlvbi5ucHkBABAAgK4QAAAAAAAmEAAAAAAAAO3bv44s116GYU/KVezMIDkAZIHEBZBxREJAhAzHiABxkI1OAlwFN4y9/3jPnunuqq5aa71j9/NkXdGvPr1RSf1/v/uHv/v7f3z66o9f/ffXv//+x3/54eu/eff1//zzX3z9zbuv//UPP/zXD9/9xz/94Yfff//z87/97t9//P6n5z/+23f/+f1Pv//022/effuXf/3Nu7/69s//7Jt3//vufn/yFQAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAvypPH9V38HCeXqkv4mG8jk+DLHE1PQ0y13Z6GmSSO+KTIIPdXZ8CGeZQfQpkiMP1CZDzzuQnQM45V5/+OOVsfgLkhPP5KZDDhuQnQA4Y1J4COWJofgrkTqP7EyB3GJ6f/thtfH0CZLcp+SmQfWblJ0D20B+lef0JkC0T6xMgG+bWp0Bump+fALluRX8C5Jol/QmQy9bkJ0AuWVWfArlgZX4C5KW1/QmQL+mP0uL+JMhz6/MTIJ8U9emPT5r+BMh7UX4K5Gddfgok7k+Aj67NT3+PLu5PgA+u7k+Aj63OT4APrY7vZ/UGVOryPqpnIFKH90m9A4k6u8/qJSjU1T1TT0Ggju65eguWq5P7Ur0Gq9XFfaleg9Xq4r5Ur8FqdXEv1HOwVt3bK/UgLFXn9lq9CAvVsV1QT8JCdWwX1JOwTt3aJfUmrFO3dlE9CqvUpV1Wr8IqdWmX1auwSl3aZfUqrFKXdlm9CqvUpV1Wr8IidWjX1LuwRt3ZNfUurFF3dk29C2vUnV1T78IadWdX1cOwRJ3ZdfUyrFBXdl29DAvUkd1QT8MCdWS31NswX93YLfU2TFcndlM9DtPVid1Uj8N0dWI31eMwW13YbfU6zFYXdlu9DrPVhW2o52GyOrAN9TxMVgd2W70Ok9WBbajnYbI6sA31PExWB7ahnofJ6sA21PMwWR3YhnoeJqsD21DPw2R1YBvqeZisDmxDPQ+T1YFtqOdhsjqwDfU8TFYHdlu9DrPVhd1Uj8N0dWK31NuwQB3ZDfU0rFBXdl29DGvUnV1T78IidWiX1auwUl3ba/UirFTX9lq9CCvVtb1SD8JadW8v1XuwXt3cc/UWBOroflEPQaLO7pN6Bxp1d5/UO9Cou/uk3oFO3d6T/B5bXZ/+Hltdn/4eW12f/h5cnZ/+kB8l/ZFSHyn9EVMfNfVREh8p/VGSHyXxUdMeKfVRUh8l9VHSHqmfglEfpWHtyY8DhvUHB+iPlPxIyY+S/Iipj5T+SMmP0tPhBOvL+Y3QHyX1kdIfJfmRkh8l/dGSHyn9kZIfJfVRkh8l/VHSHyn9UZIfMf1R0h8l/VESHyntUZIfKflR0h+51/8Jri8CAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA2OHps/oUHsjTBfVNPIBL4WmQ+TbKUyAT7axPgsxwT376Y6i74hMgY92dn/4Y4/70BMgwR/OTIOedqU9/nHOuPgFyyun89MdB59vTIIeNy0+A3GlkfALkTqPz0x93GJ6fBNlrSnwCZJ9p+QmQbRPz0x+3zYxPgGyYnZ8EuWFFfgLksjX1aZCLVuanP15Ymp/+eGFtfwLks8XtSZDnmvz0x3tRfgLkPf2RyeLTH21+Gnx4dX1PAnxkdXs/qzegUpf3Qb0CkTq8T+odKNTVPVNPwXJ1ci/Uc7BW3dtL9R6sVff2Ur0HS9W5vVYvwjp1a1fUs7BG3dl19TIsUEd2Qz0NC9SR3VBPwwJ1ZLfU2zBbXdiGeh6mqvPaVi/ETHVd2+qFmKmua4d6Iuap29qj3ohp6rT2qVdilrqsfeqVmKQOa6d6Jiapw9qpnokp6qx2q4dihrqq/eqlmKGu6g71VExQR3WHeiomqKO6Qz0V49VN3aPeitHqou5WD8ZIdU0H1JMxUB3TAfVkDFTHdEA9GePULR1Rb8Y4dUtH1JsxTJ3SIfVoDFOndES9GePULR1Rb8Y4dUuH1KMxTJ3SIfVojFKXdEy9GqPUJR1Tr8YodUnH1KsxSl3SMfVqDFKHdFA9G4PUIR1V78YYdUdH1bsxRt3RUfVujFF3dEy9GsPUKd2tHoyh6pzuUo/FcHVS96i3Yry6qXvUWzFe3dRu9VBMUWe1Uz0Tk9Rh7VKPxDR1WnvUGzFP3dYO9URMVMe1rV6Iieq4NtTzMFkd2G31OsxWF3ZbvQ6z1YXdVq/DbHVhN9XjMF2d2HX1MixRZ3ZNvQtr1J1dU+/CGnVn19S7sEbd2RX1LKxRd3ZRPQrL1KldUm/COnVrl9SbsE7d2iX1JixUx3ZBPQkL1bG9Vi/CSnVtL9V7sFbd2wv1HCxWB/dCPQer1cV9oR6D9ermnqu3YL26uWfqKQjU0X1WL0Ghru6zegkSdXYf1TMQqcP7oF6BUB3fk/4eW12f/h5bXZ/+Hlydn/4oI6xfnbdBf6T0R0l/5NRHTn7k5EfqaX6D9SvyxumPN0F/vBHyoyY+Wvqjpj9y+uPN2f52WF/Ib57+iOmPlP7I6Y+c/Ejpj5b+SOmPlvgAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAIBfo6fX6pN4GBfqEyFLXG9Pgsy32Z8CmWRHe5pkkhP1iZFzxsWnQu42IT8BstuU/gTIPnPy0yDbJrYnQrasyE+BXLOoPwVy0bL+BMhr6/ITIC+sjE+CvKQ/Suv7EyC/CPLTH58U+QmQD5r6JMh7YX4CRH+k0v4E+Oja/PT36OL+JPjI6vTeq0cgU6f3Xj0CmTq9D+oViNThfVTPQKQO75N6BxJ1dr+ohyBRZ/dZvQSFurrP6iUo1NU9V2/BYnVwL9V7sFbd20v1HixV5/ZKPQhL1bm9Ug/CUnVur9WLsFAd2wX1JKxTt3ZRPQrL1KldVq/CKnVpl9WrsEbd2VX1MCxRZ3ZVPQwr1JVdVy/DCnVlN9TTsEAd2Q31NCxQR3ZDPQ3z1Y3dUm/DfHVjt9TbMF/d2C31NsxXN3ZLvQ3T1YndVq/DXHVfW+p9mKvua1M9EFPVeW2qB2Kmuq5t9ULMVNe1R70R89Rt7VKPxDR1WrvUIzFNndYu9UhMU6e1Sz0Ss9Rl7VOvxCx1WTvVMzFJHdZO9UxMUoe1Uz0Tk9Rh7VTPxCR1WPvUKzFLXdY+9UpMU6e1Sz0S09Rp7VKPxDR1WrvUIzFP3dYe9UbMVfe1pd6H2erCbqvXYb66sVvqbVigjuyGehoWqUO7op6FleraLqgnYak6t9fqRVisDu5L9RosVyf3hXoMEnV2v6iHoFO39yS/B1fnpz/0R0x/xPTHW6A/WgKkJj9S+iOnP3JPUzus345fA/mR0h81/ZHTH2+I+HgL9McbID/eBP3xZikPAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAeAuerqkP47fuanoCZLKN9mTIRPvrUyBD3ZeeBBnlWHkqZIDz8QmQ48b0J0CO0R+dQfVJkPuNjE+B3GV4fBJkv0n5CZA9puUnQLZNzE+C3DY5PgFyy4L8JMg1i/ITIJcsy0+AvLKwPv3x0tL8BMgLi/uTIM+tz09//CLIT4B8lNQnQD6o8pMgX7X5CfDRtfXp78HV+QnwsdX16e+h1fF9UK9ApS7vo3oGInV4H9Uz0Ki7+6TegUbd3TP1FCxXJ/eleg0Wq4N7qd6DtereXqr3YKk6t1fqQViqzu21ehEWqmO7rF6FVerSLqtXYZE6tCvqWVij7uyqehhWqCu7rl6GBerIbqvXYba6sA31PMxV97WpHoip6rw21QMxVZ3Xpnogpqrz2lYvxER1XDvUEzFRHdce9UbMU7e1Sz0S09Rp7VOvxCR1WHvVOzFFndVu9VBMUWe1Wz0UU9RZ7VcvxQR1VHeop2KCOqr71GsxWl3Ufeq1GK0u6j71WoxWF3Wfei1Gq4u6Uz0Xg9VB3amei7Hqnu5WD8ZQdU53qwdjqDqn+9WLMVJd0xH1ZoxTt3REvRnj1C0dUo/GMHVKh9SjMUyd0iH1aAxTp3RMvRqj1CUdVM/GIHVIR9W7MUbd0VH1boxRd3RUvRtj1B0dVg/HEHVGh9XDMUSd0WH1cAxRZ3RYPRxD1BkdVg/HEHVGx9XLMUJd0XH1coxQV3RcvRwj1BUdVy/HCHVFx9XLMUJd0XH1coxQV3RCPR0D1BGdUE/HAHVEZ9TbcV7d0Bn1dpxXN3RKPR6n1QmdU6/HaXVC59TrcVZd0Dn1epxVF3ROvR5n1QWdU6/HWXVBJ9XzcVId0Gn1gJxS53NaPSCn1PmcVg/IKXU+59ULckqdz2n1gJxS53NaPSCn1PmcVg/IKXU+p9UDckqdz3n1gpxS53NaPSCn1PmcVg/IKXU+p9UDckqdz2n1gJxT93NWvR8n1QGdVM/HWXVB59TrcVZd0Dn1epxVF3ROvR5n1QWdU6/HaXVCp9TjcV7d0Bn1dgxQR3RCPR0D1BGdUE/HCHVFx9XLMUSd0WH1cAxRZ3RUvRtj1B0dVe/GGHVHR9W7MUbd0VH1bgxSh3RQPRuj1CUdU6/GMHVKh9SjMUyd0iH1aIxTt3REvRkD1TEdUE/GQHVMB9STMVAd0/3qxRiqzule9V6MVfd0n3othquT2q0eiknqsPapV2KWuqw96o2Yp25rUz0Qc9V93VBPwwJ1ZM/VWxCoo/uonoFKXd4H9Qpk6vR+Vm9AqI5Pfg9OfpTUR0p+lORHSny01EdKe5TUR0l8pLRHTHmklEdJe5SUR0l6lFRHSnfE1EdJdrRUBwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAADAb8f/A1BLAwQUAAAACAAAACEAJwyAcW4TAACArhAAEAAUAGdyb3VuZF90cnV0aC5ucHkBABAAgK4QAAAAAABuEwAAAAAAAO3Uu45mZ1qGYXfKUTgzSA4AjUDiAMhAJAREaGAGESAGeRAJcBScMGO33e5NbdZa3+Ze5f+6sq6gen3Pe6v+72///m/+7h/effVfX/33N7/57e//+btv/urrb/7nn/7sm2+//uZffvfdf37363//x99995vffv/zv/71v/3+t3/4+e//9df/8ds//PuPf/Xt17/687/89uu/+NWf/sm3X//v1+f90VcAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAL9Q7/6g/gYe1rv36s/gMb37oP4SHs+7T9Rfw4N597n6g3gkX+SnQPZ5Kj8Jssdz9SmQDV7KT4Es9kp+CmSh1+tTIMscy0+BLHE4Pwky3an6FMhcp/NTIPNcyU+BzHGxPgUyw0B+CmTUWH4KZMhwfhLksin1KZBrpuWnQM6bmZ8COWdyfQrkjAX5CZCjluQnQA5a1J8AOWJVfgrkgIX5CZBXLe1PgLxsbX4K5EXL8xMgL9jQnwB5zo78BMgz9uSnQJ62rT8B8qV9+QmQL+zMT4B8bm9/CuQTu/MTIB/Zn58A+VnRnwL5UZOfAPlBlZ8A+V7XnwAp8xMgaX76e3htfwJ8cHF++ntsdX4CfGx1ffp7aHV836s3IFOn9169ApW6vPfqFYjU4f2k3oFEnd0H9RAk6ux+Vi9BoI7uY/UW7Fc397F6C/arm/tEPQa71cV9asnjJv9SZmp7+8LCF0341Uy3M64jFr9j5nSMW9rSFRu+ftGUXDA3nhm2fPXqWTloQjCT7frm9dvyqjmnnGvfJ2+ZmBdMPOY0W794z8w8Y/I159j6vdum5kvzzznD3s/dODefWXLQcZs/du/m/GzZScfs/tTds/PeypsO2f2p+6fnqzfU3/7/kQ3Wn/Wq/Z8ZneCRbbnrNcFXZmd4WLsue0XxjeUtHtHO256VfOHJpTad6Zdq721Pir5wYKHggm/bohNOEn3gnGHiy74N4zOvdPfvO6g+8n3Vl3kQ9Zlvqz7Mo6jvfFf1XR5Gfeh7qq/yQOpT31J9lEdS3/qO6ps8kvrWN1Sf5LHU176f+iIPpj737dQHeTT1ve+mvsfDqQ9+M/U5Hk998Xupr/GA6pPfSn2MR1Tf/E7qWzyk+ug3Up/iMdVXv4/6Eg+qPvtt1Id4VPXd76K+w6Oq734X9R0eVn34m6jP8Ljqy99DfYUHNvuCk37fXuH+D2/+7WYUsdf+1flgxdFmxrHBrql5yppzLchkmcUD87Jld1qYzFQrRuW4hRdaH88EE6fkinW32dbQiCkbct3Cy2zM6LrxZzJk4V22hnTVjIcyYOFRdrd0yaS3ctXCk+yv6YJpr+WahfdIgjpr5oO5YOE1oqTOmftkTlt4iq6qE6a/mlMWHiKs6oT57+aMhVdowzpqydM5bOEN6rSOWfR4Dlp4gDqtY9a9n1id1iH1SKxTt3VIPRLr1G0dUo/EOnVbh9QjsUyd1jH1SixTp3VMvRLL1GkdU6/EMnVax9QrsUyd1jH1Sm/K21ouKuqkeqW7e8PL7YtoRL3S/fxSplsYzUw3m2X35xz6qFt962HXHrXdzTa507fc6GMvmPC+Le62yK0+5iYfe8mcFy53uzlu9jl3+NhLJr1wuduNcbfv6b/2mnlvXOt+S9zug9r5Lpr5yKXut8P9vij82ssmP3OdG65wvy+qvva6+e9c5YYbrPukFXdZ+LXXLXjnKndcYNE3LTrLqq8dsealS9zx+Su+ad1R1nztkFVPXeGOj5/+TXM+a+fnjln42Olu+fK5HzXxw3Z97pilj53tlg+f+VFTP2zP545a/Ny57vnqaV81/cvWf+6w9e+d6Z6PnvRVC75s9eeO2/Hgie755ilfteTLFn/vBJuePMtNXzzhs1Z92srvnWDbkye56YMnXGLrLaZ87xQbHz3FXZ/71i4x/r1z7H31BHd97Vs7xOj3zrL73cNu+9a3doah750mePiouz71zZ1h5INnSR4+6K4vfXNXuP7B80RPH3Lbd765G1z84Im6tw+47zPf2gmufO9c5euvu+0j39wBzn/wXPHzr7rvG9/cAc5+8GT18y+68Qvf2vznepmtfv1l933gW1v/QjXz1I+/7sbve2PjX21nhvrtA+78vDe1/VBAo+rHD7jz497U9KMNjajfPuTOT3tDy09K6ZL67WNu/a43M/zcos6p3z7o1m96M7svKeuY+umj7v2gN7L7wr5eUz993L0f8yZm3xHac+q3j7v3O97E6Bt7u/sUF9z8EW9g7ya9++7xC3P7mfVHSn+U9Eep60+AfOUPIC39kdIfJf1R0h8pAZLSHyX9kdIfKf2R0h+lpD8B8hP9kdIfKf1R0h8p/ZESIC39kdMfNf1R0x81/VETIDX9UdMfNf3R0h8p/dESICn5kZIfLf2Rkh8p+dHSHyn5kZIfLf2Rkh8p/dGSHyn5kdIfLfmRkh8p/dGSHyn5kdIfLfmRUh8p8dESHy3xkRIfLfHREh8AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAKPefar+HB7Eu+fUH8YDeLY+BbLci/UpkKVerU+BrHIoPgWyxPH6FMhs5+pTIDOdr0+ATHIpPgEyxeX6FMiwofoEyJDR+gTIdRPqUyDXTIpPgFwwsT4BctLc+gTIGdPrUyCHLalPgByzKj8B8rp19QmQVy3NT4G8bHV+AuR56+sTIM/akp8Aedqm/BTIU/blJ0A+t7M+AfKZzfkJkI9tz0+AfBDUJ0B+0uQnQH5Q5SdAvirzEyBlfQJ8eHF+AnxsdX3fqzegUpf3o3oGGnV3H9RDUKir+0g9BfvVzX2iHoPN6uA+V+/BVnVuX6oXYaM6tqfUm7BLXdoz6lnYo+7sWfUw7FBX9oJ6GparE3tZvQ6r1YW9rF6HxerAXlPvw1p1X6+p92GpOq/X1QuxUB3XAfVELFTHdUS9EcvUaR1Sj8QydVrH1CuxSB3WYfVQLFFndVg9FCvUVZ1QT8V8dVNn1FsxX93UGfVWTFcndUo9FtPVSZ1Sj8VsdVHn1GsxW13UOfVaTFYHdVI9F3PVPZ1V78VcdU9n1XsxVZ3TafVgTFXndFo9GDPVNZ1WD8ZMdU3n1YsxU13TefViTFTHdEE9GRPVMV1QT8Y8dUtX1JsxT93SBfVkzFO3dEW9GfPULV1Rb8Y8dUsX1JMxUR3TBfVkTFTHdF69GDPVNZ1Uz8VkdVAn1XMxWR3UOfVazFYXdUo9FtPVSZ1Sj8V0dVJn1FsxX93UCfVULFBHdUI9FQvUUR1XL8UKdVWH1UOxRJ3VYfVQLFFndVS9E2vUXR1Uz8QidVgH1TOxSB3WMfVKrFKXdUg9EsvUaR1Sj8QydVpH1BuxTt3WEfVGrFO3dUA9EQvVcb2uXoiV6rpeVy/ESnVdr6oHYqk6r9fU+7BW3dcr6nlYrA7sZfU6LFcn9oJ6GnaoK3tWPQx71J09o56FXerSnlSPwj51a0+oJ2GnurYv1IOwVZ3b5+o92KwO7hP1GOxXN/eRegoKdXUf1EPQqLv7Sb0DkTq89+oVyNTpfa/egFAdn/weXJ2f/h6c/CjJj5j+iMmPmPyI6Y+Y/Ijpj5j8uAf9cUvy4170R01+1PRHTH7U9EdNfrT0R0p+pPRHSn609EdKfqTkR0p/pORHSX6k9EdKfpTkR0p/pOQHAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAABA49336o/gobx7Uv1V/JI93ZwKWehodCJkrqvliZAJJuSnQq6al58IOW92fxrkhBX5SZCDVuUnQY5Y2Z8EecXi/CTISzbkJ0Getak/AfKUXfkJkKfs60+AfGFjfgLkc1vzEyCf2dyfAPnY7vwEyEf25ydAflb0J0B+lOQnQN6L8hMgP8j6EyBlfgIkzU+AtP0J8MHF+QnwwdX5CfCh1fF9r96ATJ3ee/UKVOry3qtXIFKH95N6BxJ1dh/UQ5Cos/tZvQSBOrqP1FOwX93cJ+ox2K5O7hP1GOxWF/eZeg42q4P7TD0He9W9faEehJ3q2r5UL8JOdW1PqCdhn7q1p9SbsE/d2pPqUdilLu1p9SpsUof2nHoX9qg7e069C1vUmT2vXoYN6sheUE/DBnVkL6m3Ybk6sRfV47BcndjL6nVYrA7sFfU8rFX39ap6IJaq83pVPRAr1XUdUE/EQnVcB9QTsU7d1hH1RixTp3VIPRLL1GkdUo/EKnVZx9QrsUpd1kH1TKxRd3VUvRNL1FkdVg/FEnVWh9VDsUSd1WH1UKxRd3VUvRNr1F0dVe/EGnVXh9VDsUbd1VH1TqxSl3VMvRLL1GkdUo/EQnVcB9QTsVJd1+vqhVir7us19T6sVhf2inoeVqsLe1m9DuvVjb2k3oYd6sqeVy/DHnVnz6l3YZs6tSfVo7BVndsX6kHYr27uE/UYJOrsPqiHoFKX9169AqW6Pv09PP1R0x8x/VHTHzH9URMgMf0R0x81/RHTHzX9UdMfOf1REyA5/ZHTHzX9kdMfOf2R0x81/ZETGzm5AQAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAvFnv3qs/g0f07kn1V/EQno5Ph2zwanwaZJWj8WmQ6U7Gp0HmuRafBplgKD4FMmI8PgFy0Zz4FMgFE+MTIOdMjk+BHLciPgFyzKr6FMgBK/MTIK9Ym58AedHq/BTICzbkJ0CesyU/BfK0XfkJkCfsy0+BfGFrfgLkU5vzUyAf25+fAPmgyE+A/KjJT4H8IMtPgKT5KZA2PwE+uDq/dwp8ZHV7P6hHoFKX96N6Bhp1dz+rlyBQR/exegu2q5P7VL0Gm9XBfa7eg63q3L5UL8JOdW1fqhdhozq2p9SbsE/d2lPqTdimTu1p9SrsUpf2tHoVNqlDe069C3vUnT2rHoYd6sqeVy/DBnVkL6m3Yb26sZfU27BcndjL6nVYrS7sZfU6LFYH9pp6H9aq+3pVPRAr1XW9rl6Ileq6DqgnYp26rSPqjVinbuuQeiSWqdM6pB6JZeq0jqlXYpE6rKPqnVij7uqoeifWqLs6rB6KJeqsDquHYok6q+PqpVigjuqEeioWqKM6o96K+eqmTqnHYro6qVPqsZiuTuqcei0mq4M6qZ6Lyeqgzqr3Yq66p7PqvZir7um0ejCmqnM6r16MieqYLqgnY6I6pivqzZinbumKejPmqVu6pB6NaeqULqlHY5a6pIvq2ZikDumqejfmqDu6qt6NOeqOLquHY4o6o8vq4Ziizui6ejkmqCMaUE/HBHVEI+rtGFc3NKQej2F1QkPq8RhWJzSmXo9BdUCD6vkYVRc0qJ6PQXVAg+r5GFQHNKrej0F1QKPq/RhT9zOq3o8xdT/D6gEZU/czqt6PMXU/w+oBGVLnM6wekDF1P8PqARlS5zOuXpAhdT7D6gEZUuczrl6QEXU94+oFGVLnM65ekBF1PePqBRlR1zNBPSEj6nomqCdkQB3PBPWEDKjjmaHekAF1PBPUEzKgjmeGekOuq9uZod6QAXU8E9QTMqCOZ4J6QgbU8UxQT8iIup5x9YKMqOsZVy/IiLqeYfWAjKn7GVXvx5i6n1H1foyp+xlV78egOqBB9XwMqgMaU6/HqLqgMfV6DKsTGlKPx7A6oSH1eIyrGxpRb8e4uqER9XaMqxsaUE/HDHVF19XLMUNd0XX1csxQV3RdvRxT1BldVg/HFHVGV9W7MUfd0VX1bkxSh3RRPRuT1CFdVM/GJHVIF9WzMUtd0jX1akxTp3RFvRkT1TFdUE/GTHVN59WLMVWd0zn1WsxWF3VcvRRL1FkdUW/EQnVcr6jnYbW6sOfVy7BFndlT6k3YqI7tM/UcbFYH95F6Cgp1dT+oR6CjPVLaIyU9UtojpT1K0iOlPVLaIyU9UtojpT1S0qOkPVLaIyU9UtojpT1S0qOkPVLaIyU9UtojpT1S0iOlPUrqAwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAfsH+H1BLAQIUAxQAAAAIAAAAIQD1GzTIJhAAAICuEAAOAAAAAAAAAAAAAACAAQAAAABwcmVkaWN0aW9uLm5weVBLAQIUAxQAAAAIAAAAIQAnDIBxbhMAAICuEAAQAAAAAAAAAAAAAACAAWYQAABncm91bmRfdHJ1dGgubnB5UEsFBgAAAAACAAIAegAAABYkAAAAAA==', 'reference/sample-4.npz': 'UEsDBBQAAAAIAAAAIQA497YtZggAAIBXCAAOABQAcHJlZGljdGlvbi5ucHkBABAAgFcIAAAAAABmCAAAAAAAAO3YwYodRRTH4cnWp5jdKGShISj4AO4UNy5cSTQRF6IyETfqU/jCxoAQzJ3bdapO3X/G+b5Nc5uiuwp+HLj91xdfff7l14+ufrv6/eb5i5ff3d58en3zx7cf3Ty+vvn+59tfb5/99M3Pt89f/HP/s2c/vnzx6v7LH5798uLV7/efPL5++uSTx9cfP/3wg8fXf17XvXcFAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAMDV1aO3pbfEw3EiPwVyIafrkyAXcSY//bHbufwEyGbn89Mfe+mPnIP69MdGh/Xpj00G2hMgu4zmJ0E2qOSnP7qV+hMgzfRHkv5I0h9J+iNJfyTpjyj9EaU/wvRHkv5IqvUnR5oNB2ggssF0fwqkgwBJ0h9J+iNJfyTJjyT5kaQ/kuRHkvoI0h5B8iNJf+RM1Kc/ukzlpz96zOWnP3rojyT9kaQ/kvRHkv5I0h9J+iNJfyTpjyT9kaQ/kvRHkv5I0h9Jk/0JkBaz/QmQDvojSn8k6Y8k/ZEkP5L0R5T+SNIfUfojSX8k6Y8k+ZGkP5LUR5L8SFIfSfojSXwEyY8k+ZGkP5L0R5L+CCrkpz/a6Y8k/ZGkP5LkR5L+iNIfSfojSX9E6Y8o9RElP5L0R5L8SNIfSfojSX8k6Y8k/ZGkP5L0R9JAfwJkG/2RpD+S9EfQSH76Y5eh/gTIJvojSX8k6Y8k/ZGkP5L0R5T+SNIfSfIjyPgjSn8k6Y8g+ZGkP5L0R5L+SNIfSfIjSX8k6Y+ggfz0xzbyI8f0I0h+JMmPJP2RpD+CBvLTH7uM5Kc/dtEfQUP56Y895EfQWH76Y4vB/PTHFvojSX8EjeanP/oN16c/+hXy0x/t9EeS/kiSH0GV8SdAmpXy0x+9avnpj176I6iYn/7oVM1PgHTSH0n6I0l/JOmPoHp++qPPRH8CpI3+SNIfQTP5CZAu+iNnrj4B0uMoMgGy0fGM0x/7LPQnQJbpj6Dj/ATIPgP5nVsT3Tv330B+Q2tgwlBa5xYJkHmDYemPHUa7MgDZoFCV/mhXiMoApF0lKQHSrRLUuf4EyIRaTgKkV2N/AqSs2JL+aNXZnwApKockQPrMZKRAmkxFpD+azEUkQHrMJaQ/WkwmdL4/ATJoNqCDABXIkPl4BMiylXYEyKqVcvTHqqVy9MeatXIMQNYsdnMQoAI5a7kaAbJgPRr9MaulGQFS11eM/ijrDEaAFLX2oj9qmnsRIOM21HLwSAU+XEdpNLUiQE4p5zebyqbHcr9dKr+BF3Uei3viHepPgA/QxfozADnhcv0NvKzrUNwfF8xPgLztXepPgA/Q5fLTH6dcKj/9cdKF8vMfmLtcIj/9cSf9EaU/0tL56Y9dZaiPIVvSUB+jMv117Jz/hR1xyI5xWypRHXVqAQAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAADgTo/+vb4W3QoPzqNT0pviYTgZnwK5jLvzUyDbnc1PgOx0EJ8A2WkgPwGyzVB/CmQT/ZGkP5IG+xMgW+iPJP2RpD+S9EeS/kjSH0n6I0l/JOmPJP2RNNqfANlBfyTpj6Dh/PTHBvojabw/AdKv0J8Aaac/kir9CZB2pQAVSLNafwqkV7U/AdKp3J8AaVTvT4H0melPgrQRIEmT/SmQFtP9KZAuAuSdoEGSDEGi9EfSxABUIH0ESJYASTIBSdIfSTP9CZAuU/0pkCaT/QmQFvojabY/AdJBfyRN9ydBGqz0J0BWLfUnQBat9SdB1uiPpOX+BMiC9f40yLye/gTIHP2RpD+SmvoTIFO6+pMgMxr7EyBl+iOpsz8BUqY/kjr7kyBVzf0JkBL9kaQ/kvRHkv6IEiBJ+iNJfyR196dAigRIUnd/AqRCf0Tpj6Tu/gRIif5I6u5PgJQIkCj9ESVAovRHkv5I0h9RAiRJf2Tpjyj9kaQ/kvRHlP6IEiBhCiSprz8BMkGAJDX2J0DKOvtTIGUCJEqAROmPnNb69EdNc34KpKS/PwUyTn8k6Y+gHfkJkCF74tMfQ7blJ0CObcxPfxzSHzE749MfB+RHkPxI0h8xm+NTH+eoj5TN7amPc9RHkPxI0h9B8iNHegSJjyT1ESQ9knRHkPrIER9J8iNJfwTJjyTtkSQ/ksRHkv5Ikh9J8iNJfCTpjyT9kaQ/kvRHkv5IWupPgCzSH0lr/SmQNfojabU/AbJCfyTpj6Tl/gTIgvX+FMi8jv4EyKyW/gTIJP2RpD+S9EdST38CZI7+SGrqT4BMOQ5LgWwzVJUA2WSwKf2xhf5IGm1Kf+zQ2Z8AqWrtT4AUDRclQDZo7k+AlOiPJP2RNN6TAOlXyEl/tGvvT4AUVGrSH90qOY31J0DGlWoaDFCBjNIfSaWY9EezWkwCpJf+SKrFNNqfABmjP5KKLemPTtWWRvsTICOqLQ33J0AGlFMSII029idADtVD0h999EfSREgCpM/O/iTIoXpE+qNRuaJKfwLkgP6ImopIfzSZrkh/NJjPSH8sG8lPf2wylN/pjubDhdcG65vvT4Dcabg+/dGvkN/8J5iLn4p7opKf+Uc3/RFUym/6A8xlz8R9UatvuCP5MWRPfjBGfyTpj7Q32nojMvUBAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAADwH38DUEsDBBQAAAAIAAAAIQD7s26MrggAAIBXCAAQABQAZ3JvdW5kX3RydXRoLm5weQEAEACAVwgAAAAAAK4IAAAAAAAA7d2xbmNFFMfhTctTpAtIKWC1AokHoAPRUFChhQ2iQCzKIhrgKXhhQgesY8+ZOeM/WX9f5yvr3jPST8cuLPnPL776/Muvr579+uy3m1d3b767v/n0+ub3bz+6ub2++f71/S/3L3/65vX9q7u/r3/28sc3dw/X3/zw8ue7h9fvP7+9fvH8k9vrj198+MHt9R/Xde89AwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA4P/n6kF6Bi7V1b+kp+GyXB2WHouL8Eh9EuQMjtUnQPY6UZ8A2el0fgJkl5H6BMgeg/UJkA3G6xMg3Ur1CZBe1fwESKd6fwKkkQBJmuhPgPQRIEkz/QmQLlP9CZAuAiRprj8B0mQyQAXSYrY/AdJCgCRN9ydAOgiQpPn+BEgDAZK00J8AWaY/ouRHkv6Ikh9J8iNKfyTJjyj5kSQ/gqw/cibqkx9d5EfQTH76o8dUffqjx2R++qPDbH4CpMF8fvqjgf5I0h9J+iNpvj8Bsk5/JOmPKP2RpD+S9EfSfH8CZJ3+SNIfSfojSn8k6Y+k+f4EyDr9kaQ/kuRHlP5I0h9J+iNJfiTpjyT9kaQ/ouRHkv5I0h9J+iNJfiTpjyjhkaQ/kvRHkv5Ikh9J+iNJf0TpjyT9kaQ/kuRHkv5I0h9R+iNJfyTJjyT9kaQ/kvRHkv5Ikh9J+iNJfyTpjyT9kaQ/kuRHkv5I0h9J+iNoLD/9sYf8SNIfSfojSX8k6Y8k+ZGkP5L0R5L+SJIfSfojSX8kyY8k/ZGkP5L0R5L8CLL+SNIfSfIjSX8k6Y8k/REkP5L0R9Dp/PTHNgP56Y9dRvLTH7voj6Ch/PTHHmP56Y899EfQYH76Y4fR/PTHBvIjSX8EDeenP/qN56c/+umPoEJ++qNbJT/90ayUn/5opj+Cavnpj1bF/PRHK/0RVM1PfzQq56c/+tTz0x999EfQRH76o8tMfvqji/4ImspPfzSZ60+AtJjMT4C00B9B0/kJkAYL/QmQZSv9CZBFA43pj20GVpwFyC4D6+/4e9IH4EkbyE9/7DKSnwDZZSg/AbJJR38CZNJgfhYgW4zmJ0A2GM9PgPQr5OcrIO1KTemPXsWVJkBaFYMq5gpHlXMSII3qNemPPhMxCZAuMykd7U+AFEyFpD96TIYkQFpMZnS0PwEyaDoiAdJgPiEBsm4hIP2xaiUgC5BVS/kIkDWL8QiQJYvpHO9PgJywWo4AWbDejf6Y15CNAJnVEc3x/gTI41qaESBzmooRIFOaejnRnwA5rKsW/TGhLxYBUtPbyon+BMjgvytsunnrSXhCxrJbDkWAHFLMb74TAXLAufLTHwcU81upRIAccK78BMhh/5f+BHipzpKfAHncOfoTII87Q3++AnLM9josQI5K9yfAixfNT4B0/ex5+Mby4z9C+a0/g3dEfxvyo+Lc62/9Cbxjzrn+GsblndOWh/iYs3P9NYzHO699/XUMBcNkBwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAFyOqwfpGbhAV/+UHoYLcnVIeiguw8H4FMiZPN6fAtnuWH4KZLMT+SmQrU73p0C2GclPgGwylp8C2UN/BA3npz/6jecnQPrpj6BKfvqjm/6I0h9J+iNJfyTpjyT9EaU/kvRHkv5I0h9J+iNKfyTpj6TCB7ACaVfpT4B0K/WnQLoVA1Qgrar9KZBO9f4USJ+Z/gRIl6n+FEiTyf4USI/pABVIg/n+BEiDhQAVyLqVABXIOgWSpUCyBEiWAolaCVCBrFMgWQokS4AkLW1ABbJmLT8Bsma1PwGyYrk/AbJAfySt9ydA5jX0J0CmdfQnQGbpj6SW/gTInJ78BMicrv4EyAz9kdTWnwCZ0NefAKlr7E+AlOmPpM7+BEhVa38CpKi3PwFSoz+CmvMTIDUCJKm9PwFSoT+iBEiUAEnSH1ECJEqAJLX3J0Aq9EeUAElq70+AVAiQKP2R1N6fAKkQIFH6I6m9PwFSoT+iBEiS/ogSIFH6I0qAJOmPKP0RJUCS9EeUAInSH0m9/QmQIv0RJUCS9EeUAInSH0mt/QmQKv0RJUCS9EeU/ogSIEn6I6ezPgFS1Jyf/qjozk+AjOuvT38M25GfABm0Jz/9MWJTfQJkhPwIkh85++rTHyfJjyD5kbOzPv1xgvwIkh85e+uTH0dtzk9/HCM/cnbXJz+OkB9B2/PTH4/aX5/8eJT0CNIcObojSH0kqY8k9ZGkPpLUR5L6SFIfSeojSX0EWX8k6Y+kxf4EyBL9kbTanwBZoT+SlvsTIAvW+xMg8/RHUkN/AmSa/kjq6E+AzGrpT4BM0h9JPf0JkDn6I6mpPwEyRX8kdfUnQGa09SdAJgx1pT82GcxKgGyhP5JGsxIgO+iPpM7+BEjVcFQCZAP9kdTbnwCpGW5Kf2ww3pQA6ac/krr7EyAVhaIESDv9kdTfnwAZVwlKf3Tb0J8AGbajPwEyqpST/mi2pT8BMkh/JNVyEiC99EfSpv4EyJBiTAKklf5I2tafABlQTUl/dKq2NN6fADlNfyRt7E+AnFQuSYA00h9JW/sTICfUQ9Ifzbb1J0CG6I+cWkcCpFM5IwHSZqIi/dFkriIB0mG6Iv2xbGGLyY9Fpfreykh8rCjW93ZJ4mNeOb+3YxIf0/b2FzgQT0q9v+EAA6fhqeno79AdzzI8T197f1CgP5JkR5LoSFIdSaIDAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAIAn5y9QSwECFAMUAAAACAAAACEAOPe2LWYIAACAVwgADgAAAAAAAAAAAAAAgAEAAAAAcHJlZGljdGlvbi5ucHlQSwECFAMUAAAACAAAACEA+7NujK4IAACAVwgAEAAAAAAAAAAAAAAAgAGmCAAAZ3JvdW5kX3RydXRoLm5weVBLBQYAAAAAAgACAHoAAACWEQAAAAA=', 'reference/sample-5.npz': 'UEsDBBQAAAAIAAAAIQAimpjMYw4AAIBgCQAOABQAcHJlZGljdGlvbi5ucHkBABAAgGAJAAAAAABjDgAAAAAAAO3dv84cWUKH4XHKVUw2IE2woBVCXAAZiISACA3sIALErmYQCXAV3DDj+Wu7u6pPVdc57zfbzyM58Gerq86vXpU/y4H/7+/+4W///h/fffZfn/33F7/7+tt/+eaLv/78i//55z//4svPv/jX33/zn9989R//9Ptvfvf1+6//zVf//u3X333923/76g9ff/fzP/2LLz//7V/95svP//K3v/mzLz//38+P+5PPAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAB+Bd69V98Er+Tdjvre+GO3V5/+mGg/PQEyzUh6QmSG4+2JkMs8kZ8IedLT9emP8y7IT4CcdUl+AuSEi9qTISdcX58CGac/OnPq0x9DZuUnQB6bV58GeWB2fBpk26L69Mcdy+rTHzcW1idAPrY2PgnyoaI+CfKTrD8BUuYnQMr6FPjy6vwE+NLq+N7p76XV8b1Xb0Cnbu979Qhk6vR+UK9Ao+7uZ/UQBOroPlBPQaCO7gP1FATq6D5Ub8FydXIfq9dgtbq4G/UgLFXndqMehKXq3G7Ug7BUnduNehCWqnO7UQ/CUnVuN+pBWKrO7UY9CCvVtd2qF2GlurZb9SKsVNd2q16ElerabtWLsFJd2616EVaqa7tVL8JKdW236kVYqI7tjnoSFqpju6OehHXq1u6pN2GdurV76k1Ypk7trnoUlqlTu6sehWXq1O6qR2GZOrW76lFYpk7trnoUlqlTu6sehVXq0u6rV2GVurT76lVYpS7tvnoVVqlLu69ehVXq0u6rV2GVurT76lVYpS7tvnoVVqlLu69ehVXq0u6rV2GVurT76lVYpA5tQz0Li9ShbahnYZE6tA31LCxSh7ahnoVF6tA21LOwRt3ZlnoX1qg721LvwhJ1ZpvqYViizmxTPQwr1JVtq5dhhbqybfUyrFBXtq1ehgXqyHbU07BAHdmOehoWqCPbUU/DAnVkO+ppWKCObEc9DfPVje2pt2G+urE99TbMVze2p96G6erEdtXjMFkd2AP1PExWB/ZAPQ9z1X09Uu/DXHVfj9T7MFfd1yP1PsxV9/VIvQ9z1X09Uu/DXHVfj9T7MFfd1yP1PkxV5/VQPRAz1XU9Vi/EPHVbI+qNmKdua0S9EfPUbY2oN2Keuq0R9UZMU6c1pB6Jaeq0htQjMU2d1pB6JKap0xpSj8QsdVlj6pWYpS5rTL0Sc9Rdjap3Yo66q1H1TkxRZzWsHoop6qyG1UMxRZ3VsHoopqizGlYPxRR1VsPqoZiizmpYPRRz1F2NqndilrqsMfVKTFOnNaQeiWnqtIbUIzFP3daIeiOmqvN6qB6IyerAHqjnYba6sH31OsxWF7avXofZ6sL21eswXZ3Yrnoc5qsb21Nvw3x1Y3vqbZivbmxPvQ3z1Y3tqbdhvrqxPfU2LFBHtqOehgXqyHbU07BAHdmOehoWqCPbUU/DAnVkO+ppWKGubFu9DCvUlW2rl2GFurJt9TKsUFe2rV6GFerKttXLsEJd2bZ6GVaoK9tWL8MKdWXb6mVYoa5sW70MS9SZbaqHYYk6s031MKxRd7al3oU16s621LuwRt3ZlnoX1qg721Lvwhp1Z1vqXVij7mxLvQtr1J1tqXdhjbqzLfUurFF3tqXehUXq0DbUs7BIHdqGehYWqUPbUM/CInVoG+pZWKQObUM9C6vUpd1Xr8IydWp31aOwTJ3aXfUoLFOndlc9CkvVud2oB6FSl/eDegVKdX36e3F1fvp7cfojpj96+iOmP0r6I6U/Uvqjpj9S+qOkP1L6I6U/UvqjpD9S+qOkP1L6o6U/UvojpT9K+qOkP1L6o6Q/SvojpT9K+qOlP0r6o6Q/SvqjpD9Cs/PTHzum56c/duiP0Pz89Mc2/RFakJ/+2LIiP/2xRX+EluSnP+5bk5/+uGtRfvrjLv1R0h+hVfnpjzuW5ac/PrUuPv1xY2l++uMT+qOkP0r6I7Q2P/3xMf1R0h+hxfnpjw+srk9//GJ9ffrjR0V8+uMn+iMU5ac/vqc/SvojVOWnP97TH6EsP/1R5qc/wvr0h/4opfnp7+Xpj1Cbn/5eW1yf/l5cnZ/+Xludn/5eWl2f/l5bXZ/+Xlod3zv9vbK6vffqDejU7b1Xb0CmTu979QhU6vJ+UK9ApA7vR/UMROrwflTPQKPu7if1DjTq7n5S70Cgju4D9RSsVzf3oXoLlquT+0g9BsvVyX2kHoPl6uQ+Uo/BWnVvn6r3YKk6txv1IKxU13arXoSV6tpu1YuwTt3aPfUmLFOndlc9CqvUpd1Xr8IidWgb6llYpA5tQz0Li9ShbahnYY26sy31LqxQV7atXoYF6sh21NOwQB3ZjnoaFqgj21FPw2x1YfvqdZisDuyBeh7mqvt6pN6Hueq+Hqn3Ya66r0fqfZir7uuReh+mqvN6qB6Iqeq8HqoHYqo6r4fqgZiqzuuheiBmqut6rF6Imeq6HqsXYqI6rgH1RExUxzWgnoiJ6rgG1BMxT93WiHoj5qnbGlFvxDR1WkPqkZimTmtIPRLT1GkNqUdimjqtIfVIzFKXNaZeiTnqrkbVOzFH3dWoeiemqLMaVg/FFHVWw+qhmKLOalg9FDPUVY2rl2KGuqpx9VLMUFc1rl6KGeqqxtVLMUEd1QH1VFyvbuqIeiuuVzd1RL0V16ubOqLeisvVSR1Sj8XV6qKOqdfiYnVQB9VzcbE6qIPqubhYHdRB9VxcrA7qoHourlX3dFS9F5eqczqsHowr1TUdVy/GheqYTqgn40J1TCfUk3GhOqYT6sm4Tt3SGfVmXKdu6Yx6M65Tt3RGvRnXqVs6o96My9QpnVKPxmXqlE6pR+MydUqn1KNxmTqlU+rRuEgd0kn1bFyj7uisejcuUWd0Wj0cl6gzOq0ejkvUGZ1WD8cV6orOq5fjWXVBz6nX41l1Qc+p1+NJdUBPqufjSXVAT6rn4xl1Pc+rF+S4upkr1VtySJ3L5epBOaKu5Xr1ohxR13K9elEOqGOZoJ6UUXUpc9SrMqgOZZJ6VgbVoUxSz8qgOpRJ6lkZVIcyST0rg+pQJqlnZVAdyiT1rAyqQ5mknpUxdSez1Lsypu5klnpXxtSdzFLvypi6k1nqXRlTdzJLvStj6k5mqXdlTN3JLPWuDKpDmaSelUF1KJPUszKoDmWSelYG1aFMUs/KoDqUSepZGVWXMke9KqPqUuaoV2VYncoU9agMq1OZoh6VYXUqU9SjMq5uZYZ6Uw6oY5mgnpQD6lgmqCflgDqWCepJOaKu5Xr1ohxR13K9elEOqXO5XD0oh9S5XK4elEPqXC5XD8oxdS9Xq/fkqLqYa9VrctgnTy9L5xL1mBz2ydPL0rlEPSbHffz0unauUG/JCR89vLKe58VLctIvz64u6DntijyvLui0ejiuUXd0xnd3Xc/GReqWTqgn41p1T3fVowAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAALysd+9/vKvvgtf07hf1rfBi3m2o74uXsJWfAllgJz8FMtl+fQJkrof9CZB5HuenP6YZyE+BTDPWnwCZYjA/ATLDcH7643rj+QmQyx3JT39cTX+U9EfoUH7642L6o3SsPwFyqYP56Y8rHc1Pf1zocH764zrH8xMglzmTnwC5yrn+BMg19EfoZH764wpn89MfFzidn/64gP4o6Y+S/iid70+APE9/hJ7IT388TX+U9EdJf6T0R0l/lPRHSX+k9EdLf6T0R0l/lPRHSn+k5EdKf6T0R0l/lORHyOuPzvH65MdlTuQnQC5zqj8BcoVz8QmQK5yvT4E877n+BMhTnsxPgDzj6fz0x1nPxydATrsmPwFyzlX9CZDjLqtPgBx3ZX4C5KBr89Mfh1ycnwA5RH+ULu9PgIy7Pj/9MW5CfwJk1Iz8FMigSfnpjyH6ozSrPwEyYFp++mOA/ijN60+APKY/SvqjNLE/AfLQzP4UyANz89Mf+/RHSX+UJvcnQHbpj9Ls/gTIHv1Rmt6fANkxvz8Fsm1FfwJkk/4o6Y/Siv4EyBb9UVrSnwTZoD9Kq/oTIPfoj9Ky/gTIHev6kyC3lvYnQD6lP0pL+xMgn1jbnwL52Or+FMgH1uenP34R9CdAfqY/SkV/CuQn8/oSIAOm9bf/2emZeTsm5rf34eGJeUv0RynqT4B8b2p/u5/enZk3JOtPgHw2+7+fFiD79Edpbn/+DsK+2f3tXiM4L2/Liv4UyJY1/W1eZvVxeWPi/gT44vRHaU1+AuS+Vf3tXGrhaXlr3kB/Anxh6/LzAuTWwvwEyKeW5rd9wQUn5Q26Or+BkATIz95MfwJ8SW+nPwG+oqA/AfKzor/Ny849Km9Q1J9vAvneG+tPgC+m6k+AfLb0H39HL37/VyecnV7a38Grz9qAzPX5Hatk6ofz5umPUt3f0VuYsQGVCfWdSER/L2pKfucS0d8LekP9jd7LtQOQ+rX1d+3pqf3K+rv28NTm5Hc2E9G9mrfVHy9mUn76Y4j+KOmPkv4o6Y/QrPz0xwj9UdIfJf1R0h8l/VHSHyX9kdIfKf1R0h8l/ZHSHyX9UdIfJflR0h8l/VFSHyXtURIfJflRkh8l+VHSHyX5kRIgJf0Ruio//XGG/Cjpj5L6KMmPkvooqY+S/iiNFaY/5hh8u8mPKYbq2/592X3zx2H4vaY+JjjyVlMfVzv2WpMf1/KnKiXtUZIeKflR0h4p+QEAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAADwiv4fUEsDBBQAAAAIAAAAIQCBUClyKw8AAIBgCQAQABQAZ3JvdW5kX3RydXRoLm5weQEAEACAYAkAAAAAACsPAAAAAAAA7di9jmRZWobRLperKK9AamNAI4S4ADwQDgYWaphGGIgZdSMc4Cq4Yaa6+icrM07EPhF77yercy2vUpkZ33nPozTq//7uH/727//x3Vf/9dV/f/jdt9//y3cf/vr9h//55z//8PX7D//6++/+87tv/uOffv/d7779+PW/+ebfv//2j1///t+++cO3f/z3n/7F1+9/+1e/+fr9X/72N3/29fv/fX/en3wFAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAB8Qd59VB/Bm/Puc/U5vB3vLqvP4m04yE+CPGS0niv9KZB7jRZ0NT8Jcpfhhm7mp0DOOpHRSH8S5IQzEQ3mp0AGnUpoPD8FMuJcQaf6kyA3nMznbH4K5Iqz8dyRnwQ5cLqcO/MTIBecLufu/ATIS2fDeSA/AfLCyW4eyk+APHcunAfzUyDPnOrm8fwEyFNnuplRnwB5alJTAuQuRX8C5Cf6o5T0J0B+pD9CTX4C5BP9Uar6EyAf6Y9S1p8A+Up/tLr+BEjanwDRH6myPwGiP0ppfwJ869r89PfGxfnp722r89Pfm1bXp783rY7vnf7esrq9j+oNyNTp/aAegUpd3if1CkTq8H5Uz0Cj7u4n9Q4k6ux+Vg9Boa7uF/USBOronqinYL+6uafqLdiuTu4z9RjsVhf3uXoNNquDe6aeg73q3p6r92Cvurfn6j3Yq+7tuXoPtqpze6EehK3q3F6oB2GrOrcX6kHYqs7thXoQtqpze6EehJ3q2l6qF2GnuraX6kXYqa7tpXoRdqpru6CehH3q1i6pN2GfurVL6k3Yp27tknoT9qlbu6TehH3q1i6pN2GbOrWL6lHYpk7tonoUtqlTu6gehW3q1C6qR2GXurTL6lXYpS7tsnoVdqlLu6xehV3q0i6rV2GXurTL6lXYpA7tQD0Lm9ShHahnYZM6tAP1LGxSh3agnoVN6tAO1LOwSR3agXoWNqlDO1DPwiZ1aAfqWdikDu1APQub1KEdqGdhkzq0A/UsbFKHdqCehU3q0A7Us7BJHdqBehY2qUM7UM/CJnVoB+pZ2KPu7Ei9C3vUnR2pd2GLOrND9TDsUFd2rF6GHerKjtXLsEEd2RX1NGxQR3ZFPQ3r1Y1dU2/DenVj19TbsFyd2FX1OCxXJ3ZVPQ6r1YVdV6/DanVh19XrsFgd2A31PCxWB3ZDPQ9r1X3dUu/DWnVft9T7sFbd1y31PqxV93VLvQ9L1XndVA/EUnVeN9UDsVSd1031QKxU13VbvRAr1XXdVi/ESnVdt9ULsVAd14B6Ihaq4xpQT8RCdVwD6olYp25rRL0Ry9RpDalHYpk6rSH1SKxSlzWmXolF6rAG1TOxRt3VqHon1qi7GlXvxBJ1VsPqoViizmpYPRRL1FkNq4dihbqqcfVSrFBXNa5eihXqqsbVS7FCXdW4eikWqKM6oZ6KBeqoTqinYoW6qnH1UixRZzWsHoo16q5G1TuxRt3VqHonFqnDGlTPxDJ1WkPqkViojmtAPRFL1XndVA/EYnVgN9TzsFpd2HX1OixWB3ZDPQ+L1YHdUM/DWnVft9T7sFbd1y31PixV53VTPRBr1X3dUu/DYnVgN9TzsFpd2HX1OqxXN3ZNvQ0b1JFdUU/DDnVlx+pl2KGu7Fi9DDvUlR2rl2GHurJj9TLsUFd2rF6GHerKjtXLsENd2bF6GXaoKztWL8MGdWRX1NOwQ13ZsXoZdqgrO1Yvww51ZcfqZdiizuxQPQxb1JkdqodhizqzQ/UwbFFndqgehi3qzA7Vw7BFndmhehj2qDs7Uu/CHnVnR+pd2KPu7Ei9C3vUnR2pd2GPurMj9S5sUod2oJ6FTerQDtSzsEkd2oF6FnapS7usXoVd6tIuq1dhmzq1i+pR2KZO7aJ6FPapW7uk3oR96tYuqTdhn7q1S+pN2KrO7YV6EGr6o6U/YvqjpT9i+iMmP1L6I6U/SvojpT9S+qOkP1L6I6U/UvqjpD9S+iOlP0r6o6Q/SvqjpD9K+qOkP0r6o6Q/SvqjpD9K+qOkP0r6o6Q/SvqjpD9K+qOkP0r6o6Q/SvojtCk//XGR/gjtyk9/XKI/SvojtC0//XGB/gjty09/vKQ/SvojtDE//fGC/ijpj9DO/PTHc/qjpD9CW/PTH8/oj9De/PTH5/RHSX+ENuenPz6jP0r6I7Q7P/3x1Pb+BMgT+iO0Pz/98Qv9UdIfoSA//fEz/REq8tMfP0ry0x+fNPnpjx9E+emPj6r89MdXYX76o8xPf5T56Y8yP/29eWl++nvr2vz098bF+envbavz09/bVuenvzetrk9/b1nd3kf1BmTq9H5Qj0ClLu+TegUadXc/qXcgUWf3s3oIAnV0T9RTsF/d3FP1FuxWF/e5eg02q4N7pp6Dverenqv3YKe6tpfqRdioju2CehK2qVO7qB6FXerSLqtXYY+6syP1LmxRZ3aoHoYN6siuqKdhvbqxa+ptWK5O7Kp6HBarA7uhnoe16r5uqfdhqTqvm+qBWKmu67Z6Idap2xpRb8QydVpD6pFYpS5rTL0Si9RhDapnYo26q1H1TqxRdzWq3ok16q5G1TuxRt3VqHonlqizGlYPxRJ1VsPqoViizmpYPRRL1FkNq4dihbqqcfVSrFBXNa5eihXqqsbVS7FCXdW4eikWqKM6oZ6KBeqoTqinYoE6qhPqqVigjuqEeirmq5s6o96K+eqmzqi3Yr66qTPqrZivbuqMeiumq5M6pR6L6eqkTqnHYro6qVPqsZitLuqcei1mq4s6p16L2eqizqnXYra6qHPqtZisDuqkei4mq4M6qZ6LyeqgTqrnYq66p7PqvZir7umsei/mqns6q96Lueqezqr3Yqo6p9PqwZiqzum0ejCmqnM6rR6MqeqcTqsHY6o6p9PqwZiprum8ejFmqms6r16MmeqazqsXY6a6pvPqxZiprum8ejFmqms6r16MmeqazqsXY6a6pvPqxZiprum8ejFmqms6r16MieqY7lBPxkR1THeoJ2OiOqY71JMxUR3THerJmKiO6Q71ZExUx3SHejImqmO6Qz0ZE9Ux3aGejInqmO5QT8aD6oAeVM/Hvepy5qhX5Ly6mZnqLTmhjmWBelLG1a2sUG/KsDqVJepRGVWXska9KoPqUNaoV2VQHcoi9ayMqTtZpd6VEXUl69TLMqCOZKF6WgbUkSxUT8uIupJ16mUZUVeyTD0sY+pOVql3ZUzdySr1rgyqQ1mknpVBdSiL1LMyqi5ljXpVRtWlLFGPyri6lRXqTTmhjmWBelJOqGNZoJ6UM+pa5qsX5Yy6lvnqRTmlzuVeh4fXg3LKzmYmOry83pOT9nYzyfHl8ZqctbmcOY4vj9fktM3pzHDl8nRK7rA7nsddOz0ckvts7+dR107vZuRO+wN6zLXbqw15QBHRA67dHi3II5KK7nb1+GQ/HnT0CjdWNWz4eL4cR29wb1pDrh6vvy/T0Rvc3NaAM9fzxTh6g9v7uuHc9XwpDt/g/sSuOnk9X4qjV7g/sWtuXr96JhY5fMdBZcdunr92JNY5fMdJaJfdPn/hQKx19JKj1i4YOH/ZOqx3+RV2vX1u6Po1w7DJhVeYNvfEXcfzxXml/wM9drr+fmXq7H4yeu3aNdit7u5Hw+eu3ILt6u5+VM9AQ0aEFAQAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAK/Wu3f1Bbwl716qT+JtuJCeANnlMD8BssGV/gTIatfyEyCrXe9PgCx1Iz8BstTN/gTIQrf7EyDLDOQnQJYZ6k+ArDGWn/5YY7A/AbLCaH76YwX9ERrOT38soD9K4/0JkOlO5Kc/ptMfoTP56Y/ZTvUnQOY6l5/+mOpkfvpjprP56Y+JTucnQOa5Iz/9MY3+CN2Tn/6Y5a7+BMgc9+WnP+bQH6E789MfM9ybn/6Y4e7+BMjj7s9PfzxOf4QeyE9/POyR/gTIw/RHSn+U9EdKfqT0R0l/pPRHSn6U9EdKf6T0R0l/pORHSn+U/AEkpT9Cp//+CZB5zucnQKa5Jz8BMsl9+QmQGe6tT39McH9+AuRhj+QnQB70WH4C5BGP1idAHjAhP/1xrxn5CZA7zclPgNxjVn364w7z8hMgp83MT4CcM7c+AXLK9Pz0xwnz+xMg4/RHSX+UFvQnQIbpj5L+KOmP0or+BMgo/VHSH6Ul/QmQQfqjpD9Ka/oTIGMW9SdAhqzqT4AMWJafABmwsD8BctPK/gTILUv7EyA3rO1PgFy3uD8BctXq/gTINcv7EyBXrO9PgBzb0J8AObSjPwFyZEt/AuTAnv4EyGWb+hMgF+3qT4Bcsq0/AXLBvv4EyEtLOhMgg1bkJ0BGrelPgIxZk58AGbOqPwEyYlV+AmTEsvwEyICF/QmQWybn927ktzdPymu0tj8Bct3s/gTIGav7EyDXTO9PgJywvj8Bcmx+f4MBBs/K67OjPwFyZEt/AuTAgv4EyLBN/QmQi3b1d/GDNj8rr8+2/gTIBfv6EyAvrehPgIza2Z8AeW5rfwLkmb39Xfq4fc/K67M3PwHyud39CZCntvcnQJ7Y358A+UXQ34XP3PGkvEJFfgLkJ01/AuSTqD8B8tGK/AZLEiBlfy8+/fq3zXhaXpkl+Z1J5cbP3flb+ULk/X1+w/XLpj45r8Ca/E6H8vlPzvu9vHKvpL/xS6YvQGhRfvdVIsA351X1N3jPzOen9dr6G7po3uPTWpXfY4kI8K14nf3dvmvKw1Nblt+EQFT36/ea++PXT3+U9EdJf5T0R0l/lPRHSX+U9EdJf5T0R0l/lPRHSX+U9EdJf5T0R0l/lPRHSX+U9EdJf5T0R0l9lNRHSXyUxEdJfZTER0l8lNRHSX6kBEhpSn8C5E5z+lMg95nVnwC5x7T+BMgd5EdJfZTkR0l+lNRHaTQw+bHE+F83+THfYHwH35uczK/IeH1Pvnv7lfxanYgPplMfJfFRUh8l8VFSHyXxURIfJfUBAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAGz3/1BLAQIUAxQAAAAIAAAAIQAimpjMYw4AAIBgCQAOAAAAAAAAAAAAAACAAQAAAABwcmVkaWN0aW9uLm5weVBLAQIUAxQAAAAIAAAAIQCBUClyKw8AAIBgCQAQAAAAAAAAAAAAAACAAaMOAABncm91bmRfdHJ1dGgubnB5UEsFBgAAAAACAAIAegAAABAeAAAAAA==', 'reference/sample-6.npz': 'UEsDBBQAAAAIAAAAIQByhmfEPQkAAIBXCAAOABQAcHJlZGljdGlvbi5ucHkBABAAgFcIAAAAAAA9CQAAAAAAAO3dzW4cWRmA4XjLVXhnkLKA0QgkLoAdiA0LVigwQSwQgxLEBrgKbpgZMcqME3d1/ZxTrzt+np2tqu6vP706dlu2/N/f/O7Xv/393at/vvrXw1dv3//p3cMv7x/+/cefPby+f/jz1+/+8e7N3/7w9buv3n77+V+9+ev7t998/v1f3vz97Tcf//iL1/dffvGL1/c///KnP3l9/5/77X70CgAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAODZuvtePQovzt3H6oF4QT6pT3+c54n8BMhp9Efpyf4EyDmezk+AnONSfwLkBBfz0x8n0B8l/VHSHyX9UdIfJf1R0h8p/VHSHyX9UdIfJf1R0h8l/VGSHyX9UdIfnYvf/QmQ+Rby0x/T6Y+S/ggt5SdAZlvuT4DMdaU/BTLV1f4EyET6I3Q9PwEyzZr8BMgk6/JTIHPoj9Dq/ATIeBvyEyDD6Y/QpvwEyGD6o6Q/Shv7kyBDbe9PgIyzoz8BMsqe/PTHKPqjtKs/CTKKACnt7U+AjKA/SvqjtLs/ATLA/v4UyGFH8tMfR+mPkv4o6Y/Qofz0x0H6I3QsP/1xjP4IHcxPfxwiP0ryo6Q/SvojJD9K8qOkP0r6o6Q/SvqjJD9K+qMkP0ryoyQ/SuojJT5K+qOkP1LfFSU/evLjGVMfAMDn5Lu3wdffhNSD8lla/yMYBTKe/ijpj9KG/gTIBPojJUBK+iO0Pj/9Md6G/gTIaFvy0x+jbepPgIy1LT/9MZb+KOmP0Mb89MdQ+qOkP0r6o6Q/SvIjJD9K+qOkP0r6IyQ/SvojtDE//TGU/ghtzU9/DCQ/QvKjpD9S8qMkP1LyI6U/UvKjpD9K+qMkP0ryo6Q/SvqjJD9K+qOkP0r6o6Q/SvqjJD9K+qOkP0r6I6U/SvKjpD9K+iO0Ij/9MY38KKmPkv4o6Y+S/Ag5/ijpj9DV/PTHRPqjpD9K+iN0PT/9MY/+KOmPkv4o6Y/Qivz0xzT6o6Q/Qmvy0x+TrMpPf8yxLj/9McXK/ATIFPKjpD9K+qOkP0r6o6Q/SvqjpD9K+qOkP0r6o6Q/SvojtDY//TGD/ijpj5L+KOmPkv4o6Y+S/ijpj5L+KOmPkv4o6Y+S/ijpj9Lq/gTIBPqjpD9K+qOkP0r6o7S+PwUy3pb+BMho+qO0qT8FMtTG+jTIQLvqkyCj6I+SA5CW/kjpj5T+KOmPkv5IXW5MfMy3MTL1MdT/Y3PCUfvwVbceBAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAICX5e6DehJemruP1PPwonycnwQ5zVPxCZCTXMpPgJzgcn4SZLrl/ATITNfqEyAz6Y/QivwkyCwr8xMgU+iPkv4Irc5PgIy3IT/9MdqW/ATIaNv6UyBj6Y+S/ijpj9LW/gTISPqjpD9K+qOkP0r6o6Q/SvqjpD9S+qOkP1L6o6Q/SvqjpD9S+qOkP0r6oyQ/SvojpT9K+qOkP0r6I6U/SvqjpT9K8iOlP0r6o6Q+SvqjpD9K+iO0Kj/9McW6+gTIFKvzEyAT6I/Qhvz0x2hb8pMgQ22NT4SMtLe/gQXK+eXan9+oZpynL9mR/gY0M6Fnbseh+gYkMydqbsPh+g4HM6NpbsSg/A4kM7xnbsXA+HY3M7hmbsfg/HY1MzZmbsjw/LZnM+pxuDlT6ttQz+EH4KZN7e9aR3vv47NxQnxPl7TvLj4zk7M7qN4Os9WFLau3w3R1Yovq5TBdndiiejlMVye2qF4Os9WFLaqXw3R1Yovq5TBbXdgF9Vo4R93Z0+qtcJa6tKfVW+EsdWlPqpfCaerUPlVvhBPVsX2kXgcnq4N7rN4GZ6uLe6zeBmeri3ukXgZnq4t7pF4GJ6uDk9zLJj5C8qOkP0ryIyQ/SrPDkh9LJue3/ATZq+a5mN7f0nM0L5nnY05+T4YlOj4yqT6JsYr8CE3KT3+soj9K+iMkP0r6o6Q/QrPy0x/XTatPf1w3MT/9cY38CM3MT39cMTU//bFobn36Y9Hs/PTHRdPj0x+XnZGf/rjglPz0x5POqU9/POWs+vTHp7bUIz/G2paP/hhLf4Q21nOwPwHyQ1vTOZqf/vje9nL0xzj6I7S9nMP56Y8PdkSjPwbZlYv+GGJnKvpjCP0R2luK/hihyk+AfGtnI/pjhEdB3D36vYLV9wmQvS73t/4+/bGX/ijtTUJ/jKA/UvqjpD9K+qO0swj9McTOHPTHGPqjtK8G/TGI/rg58qMkP1L6o6Q/SvojpT9S+iMlP0r6o6Q/SvojpT9K+qOkP0r6o6Q/SvqjpD9K+qOkP0r6o6Q/SvqjpD9K+qOkP0r6o6Q/SvqjpD9K+qOkP0r6o6Q/SvqjpD9K+qOkP0r6o6Q/SvqjpD9K+qOkP0r6o6Q/SvqjpD9K+qOkP0r6o6Q/SvqjpD9K+qOkP0pH+hMgR+mPkv4o6Y+S/igd6k+AHHSsPwFyjP4o6Y+S/ijpj5L+KOmPkv4o6Y/Swf4EyCH6o3Q1L/0x0fW89Mc81+vSH/Mc7K8enxu34nTTH9Poj9LB/gTIEcv56Y+59EdJf5T0R0l/lPRHSX+U9EdJf5T0R+lwfwLkAP1R0h8l/VHSHyX9UdIfJf1R0h8l/VHSH6Xj/QmQ/fRHSX+U9EdJf5RWtaU/JtEfJf1R0h+lAf0JkN1G9CdA9tIfJf0RWleW/phjXVj6Y44h/aWvgBu27mBz/DHHmP4EyD76I7QyLPkxw9pzTX/MsLYr+TGD/iit7Up/zDDm+NMf++iP0Nqu5McM+qOkP0pr+/PnR8yw+lyTHxPoj5L+KOmP0ur+Fq/MxufGrc5v6cpqeG6e/ijpj9L6/hYujWbn5i3nt7K/aHZu3JX41vYXTc+tG9NfNDw3b1t/jj/G0h+h6/npj3m29beqUVhNf5T0R0l/lDbEt3R1Mzy3b1NP+mO0LTXJj0l+8A9+Fy7SHyH9kZIfz4f0AAAAAAAAAAAAAAAAAJ6V/wFQSwMEFAAAAAgAAAAhAE1K0AOJCQAAgFcIABAAFABncm91bmRfdHJ1dGgubnB5AQAQAIBXCAAAAAAAiQkAAAAAAADt3b2OXEkZgGFPylU4M0gOYLUCiQsgA5EQECHDGhEgFtmIBLgKbpjdldj1z0x3nVNV/Xo8z5N165yjrz+9qlZH/d/f/O7Xv/393bN/PvvXi69ev/3Tmxe/fP7i33/82YuXz1/8+es3/3jz6m9/+PrNV6+/ff9Xr/769vU377/9y6u/v/7m9Y+/ePn8yy9+8fL5z7/86U9ePv/P8+N+9AwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAVrqrB+AJu/tBPQpPzt2d/qjcfaCehyflw/z0x+18VJ8AuaH7+hMgN3JvfvrjNu7PT4Dchv4IPZSfALmBh/PTH/td6E+A7HYpP/2x28X+BMhm+qN0uT8BstWV/PTHVtf6EyA7Xe1PgGykP0LX8xMg++iP0kh/AmSXof4EyB5j+emPPQb7EyBbjPYnQDYYzk9/bDDenwBZ70B/AmQ5/RE6kp8AWe1YfwJkqYP5CZCl9EfocH4CZJ0T+QmQVU7lJ0DWOJmfAFnhdH76Y975/ATItJn8BMikufwEyJTZ/PTHhOn8BMh5C/ITIKct6U+AnKQ/UgKkpD9Ka/oTIOfoj5L+KC3qT4Ccoj9Kq/oTIGfoj9Ky/gTICfqjpD9K6/oTIMfpj5L+KOmPkvwo6Y+S/ijpj5L8KOmPkv4o6Y+S/Cjpj5T+KMmPkv4o6Y+S/ijJj5L+KOmPkv4oyY+S/ijpj5L8KOmPkv4oyY+S/ijpj5L+KMmPkv4o6Y+S/Ejpj5j+aOmPmPyI6Y+Y/mjpj5j8SOmPkv4o6Y+S/ggN56c/NtAfJf1R0h+h8fz0x3r6o6Q/SvIjJD9K+qMkP0ryo6Q/QvKjJD9K8qOkP0K+finpj5L+KMmPkOOP0GB++mOH0fz0xwbD+emPDfRHaDw//bGc/AgdyE9/rHYkP/2xmv4o6Y/Qofz0x2L6o6Q/SvKjpD9K+iN0KD/9sZj+KOmPkv4oyY+S/ggdyk9/LKY/SvqjpD9K+qOkP0ryo6Q/SvqjpD9K+qOkP0r6oyQ/SvqjpD9K+qOkP0r6o6Q/SvojJD9K+qOkP0r6I3QkPwGymv4o6Y/Qsfz0x1r6o6Q/SvojdDA//bHU0f4EyEr6I3Q4P/2xkP4o6Y/Q8fz0xzr6o6Q/Sif6EyCrnMlPf6yiP0r6o6Q/SvqjdKo/AbKI/iid60+ArKE/Sif7EyBL6I/S2f4EyAr6o3S6PwGygP4one9PgMzTH6WJ/gTItJn+BMgs/VGa6k+ATNIfKQFS0h+luf4EyCT9UXIAUtIfpcn+BMgc/VFyAFLSHyXlkdIdJdlRkh0p2ZHSHSmHHwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACRu4/UE/FkfByfArmRB+ITIPtdiE+BbHetPwWy0fX8BMg+I/0pkF3G+hMgewz2J0B2GM1PgexwoD8BstyR/gTIaof6EyBrHctPf6ylP0oH+xMgS+mPkv4IHc1Pf6ykP0r6o6Q/UgKkpD9K+qMkP0r6I6U/SvqjJD9K+iOlP0r6o6Q/SvKjpD9S+qOkP0ryo6Q/SvojJT9K+qMkP1L6oyQ/UvqjJD9K+iMlP0r6IyU/Ss4/Sle/gBXIRtf7EyC7DNQnQDYZq0+A7DBcnwZZ7mB9EmShM/WtTvD9p6n76Thd38IE332U0/UpmatvTSrfP2d92HzKFsS3oJZNVfOJW1ffVDAbiuaTtzi+080sz5lHYEt9Z6pZ9yQejW31HQ5Hfk/S3gCH41nwCB6l3QEO9DN5O4/a/gAvNDRzL5+H7fU9UNShi/ls7Y1tVr0ddqsLu6heDrvVhV1Wb4fd6sIuqpfDdnViF9SrYb+6sYfVm+EW6soeUK+F26g7u1+9FW6lLu0+9U64nbq1j9QL4Zbq2j5U74Pbqnt7X70NbqwO7n31Nri1urh31bvg5urk3lGvgturm/tBvQkCdXT/V++BRt3dt+odkKnTE9+TJj5C4qO0vyvt8aA9+V1qS3x8b1N+8mKI/ijpj9Cu/PTHCP1R0h8l/VHSHyH5UdIfoW356Y/r9uWnP67amJ/+uGJnffLjiq356Y/L9uanPy7anJ/+uGR3fvrjAvnR2V6f/njYDfLTHw+5RX764343qU9/3OtG9cmPexzoR38sdigf/bHUwXr0x0KH49Efy5xIR38scqoc/bHEyXAm+xMg3zmbjf5Y4HQ0+mPe+WZm+xMgZ373nrpVf9xjJhf9MWsmlun+BPjkzaSiP2bNhDLfnwCfuplK9Me0iUj0x7SJRPTHtIlA9Me0iT70x7zzdeiPeeej0B/zzjehPxY4nYT+WOB0EfJjgdNF6I8VTvYwn5/++M6pHPTHMidi0B8l/VHSHyX9UZIfJf1R0h8l/VHSHyX9kdIfJf1R0h8p/VHSHyX9kdIfJf2R0h8l/VHSHyn9UdIfJf1R0h8l/VHSHyX9UdIfJf1Rkh8l/VHSH6GZ/PTHLP1R0h8l/VHSHyX9UdIfJf1Rkh8l5x8l/VHSHyX9UdIfJf1R0h8l/VGa6k+ATNIfJf1R0h8l/VG6nJf+2OtyXfpjr8tx6Y+9rsQlQLbSHyX9UbrWlv7YSX+U9EdJf5SutqU/NtIfJf1Rut6W/thHf5T0R2mgLf2xjf4o6Y+S/iiNtKU/dtEfpcn+BMgU/ZHSHyX9UdIfJf1R0h8l/VHSHyX9UdIfJf1R0h+lgbb0xy4jbemPXfRHSX+UZvsTIDP0R0l/lPRHSX+UpvsTIBP0R2ioLP2xif4oDaWlPzbRH6UF/QmQ0/RHSX+U9EdJf5RW9CdAztIfJf0RGgtLf+yhP0r6ozRW1rWrBMg5+qOkP0r6o6Q/QoNd6Y8t9EdJf5QW9Zd+Bh6t0a70xw76o6Q/SvqjNJqV/thgOCv9scGq/sKPwCOmP0rDWemPDcazkh/r6Y/Smv6y8XncxvMb+Y9WOORKfvpjp2v5DfZXjc/jdjW/u6HLo+l55K7nN3YAVvPzqI3kpz920R8l/ZE6GqD+WEp/lNb0l43PY6c/UocC1B+LLTj/stn5DAzH5/hjh7H2LlwZzMznYzgq+bHDgaT0R05+9PRHT38AAAAAAAAAAAAAAAAAn4z/AVBLAQIUAxQAAAAIAAAAIQByhmfEPQkAAIBXCAAOAAAAAAAAAAAAAACAAQAAAABwcmVkaWN0aW9uLm5weVBLAQIUAxQAAAAIAAAAIQBNStADiQkAAIBXCAAQAAAAAAAAAAAAAACAAX0JAABncm91bmRfdHJ1dGgubnB5UEsFBgAAAAACAAIAegAAAEgTAAAAAA==', 'reference/sample-7.npz': 'UEsDBBQAAAAIAAAAIQA22izaBg0AAIC2DQAOABQAcHJlZGljdGlvbi5ucHkBABAAgLYNAAAAAAAGDQAAAAAAAO3YvY4cWRmAYU/KVUxmkByAWK0QF0AGIiEgQoY1IkAsshEJcBXcMAu7ixbcP1WnzqnXM/M8WVe3ur8jvfqqu//5i1/9/Je/fnj111d/e/3Fuw+/e//6p4+v//7bH71+8/j691++/8v7t3/6zZfvv3j37+s/e/vHD+++uv7hD2///O6rx9//8ZvHzz7/yZvHzz/74Q/ePP7jcb/vvQIAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAePIevlUPwovz8JF6Il6Oj+sTIOe52J8COceV/ATIGa7mJ0BWuxGfAFnuTn8KZKW7+emPhe73J0DW0R+hDfnpj2X0R0l/lPRHSoCU9EdJf5T0R0l/lPRHSn+U9EdJf5T0R0l/pPRHSoCk9Edpw1dACbKOAEltCVCDrLKxPwmyxPb+FMh8e/pTILPt60+CzLW7PwEykf4o6Y/Q/vwEyDz6ozTSnwCZZag/ATKJ/iiN9SdAphjMT39MMFqf/phhvD8BctiB/PTHYfojdCQ//XHUof4EyDHH8tMfhxzMT38ccrQ/AXLA4fz0x7jj+emPcRP6EyDD9EdJf4Rm5Kc/Rk3pT4AM0h8l/VHSHyX9UdIfJf1R0h8l/RGak5/+GKM/SvqjpD9K+qOkP0ryo6Q/SvqjJD9CU/LTH4PkR0l/lPRHSX+U9EdJf5T0R0p/lPRHSX+kBEhpRn8CZJT+KOmPkv5ICZCS/ijpj5L+KE3pT4AM0h8l/VHSHyX9UdIfpTn9CZAx+qOkP0qT+hMgQ/RHaVZ/AmTAtPz0x4B5/QmQ/fRHSX+U9EdJf5T0R0l/lPRHSn+k9EdLf6T+E4/+KOmPkv4oyY+S/EjJj5L+KMmPkv4o6Y+S/ijpj5T8SOmPkv4o6Y+U/ijpj5L+SOmPkv4o6Y+S/Cjpj5L+KOmPkv4o6Y+S/Cjpj5L+KOmPkvwo6Y+S/giN5Kc/ZtEfIflR0h+hofz0xyT6IzSWn/6YQ3+EBvPTH1Poj5L+KOmPkv4o6Y+S/ijpj5D8KOmPkv4oyY+S/ijJj5D8KOmPkvwo6Y+U/ijJj5T+CNl/lPRHSX+U9EfpamP64wTXG9Mfy91oTH8spz9KNxLTH+vpj5L+KOmP0PXEruanP6YZ6U+AzKI/SvqjpD9K+qN0NbEb+QmQaW52pj8W0x+lof4EyCRj/QmQOfRHabA/ATKF/iiN9idAZtAfoeH89McE4/0JkOPkR0l+lLRHSXuU1EdJfJQ2p6c+5rPvCMmPkvgoyY+S/khJj5zoAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAIg9fK0egxfo4X/U0/CyPPy/eiBejo/i0yDnuZaf/jjD1f4EyHrX89Mfy93IT3+sdis/AbLa7f4UyFr6o3SvPwGykv4o6Y+S/gjdzU9/LKQ/UgKkpD9S+iNk/1HSHyn9kdIfJf1R0h8l/ZHSH6V7C1CBrHS3PwGy0P3+BMg6G/oTIMts6U+ArLKpPwmyyNb+BMgK+iO0OT/9Md32+gTIdLvy0yCT7e9PgEwzkJ/+mGakPwkyyWB+AmSC4foUyHGH8lMgxxzNT38ccDg/ATJsQn0CZNSc/BTIkGn5CZAB+iM0MT/9sdfM/ATIPnPr0x/76I+S/gjNzk9/7DG9PwGyw/z+BMh2C/oTIFutyE+AbKU/Qmvy0x+bLMpPgGyiP0r6o6Q/Ssv6EyD3rctPgdynP0pL+xMgd+iPkv4ore1PgNymP0r6o6Q/QovzEyA3Le9PgNywvj8Bct0J/QmQq/RH6Iz89Mc1+iN0Sn764wr9UdIfoXPy0x+X6Y+S/ijpj9BJ+emPi/RHSX+U9EdJf5T0R0l/lPRHSX+U9EdJf5T0R0l+lPRHSX+U9EdJf5TkR0l/lORHSX+U9EfpQir64zT6o6Q/SvqjpD9KF0qRH6fRH6FLqeiPs+iPkv4ondFffUY+WRdb0R8nOaO/+ox8uvRH6YT+6iPyKbsUi/g4y9L+4rPx6VvZX3w0noBLxciPs6zrrz0XT8Si/tpD8WSs6a89E0+H/ihd6EZ/nOZSNfqjpD9K+qOkP0r6o6Q/SvqjdLw/ATJOf5T0R2lCfwJknAApzehPgIzSHyX9UdIfJf1R0h8l/VHSHyn9UZrRnwAZNaU/ATJIf5Tm9CdAxuiPkv4o6Y+U/ijpj9Kc/gTIGP1R0h8l/VHSHyX9UZIfJflR0h8l/VGSHyX9UZIfKf1R0h8l/VHSHyX9UdIfJf1R0h8l/VHSHyX9UdIfJf1R0h8l/VHSHyX9UdIfIflR0h8l/VHSH6Hj+emPcfIjJD9K+qMkP0LWHyX9UdIfJf1R0h8l/VHSHyX9UdIfJf0ROp6f/hinP0r6o6Q/SvKjpD9K+qOkP0r6o6Q/SvojpT9K+qOkP0r6o6Q/SvqjpD9K+qOkP0ryo6Q/SvqjJD9K+iOlP0ryoyQ/SvqjpD9K+qN0pzD9sZT+KOmPkv4o3env5vP17Dx99xac/ljqXmD6Y6F7+09/rHQvvxsviCfnebjTl/5Y63Zc+mOxm23pj8X0R0l/lAb7EyBT6I+S/ijpj5L+KN1q60Z++mOKwf7KkXlGbsRl/bGc/ijdikt+LDfSXzkvz8zO+286K8/QlcSUxzmu3n+/82w2HM/enUWnPpa6dKf9TonyY63LvzT0x6tvv4Wt/5irAervRUv789vjxcv7O+fzeeG0R0p+fBIevvkHpp4DAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACZ6+Fo9Bi/Tw399dL0ZiJfk4aZ6Op652/kJkKXu5adAFtqQnwBZZVN+AmSRjf0JkCX0R2lrfwpkhe39CZAlBEhJf5T0R0l/lPRHSX+U9EdJf6QESEl+lPRHSn+E7D9K+qOkP0r6o6Q/SvqjpD9K+iOlP0r6o+QGTGlrfxJkhR39CZDp9vSnQCbbl58CmUt/lHb3J0Cm2V+f/phkJD4FMslwfgpkAv1ROtKfADnoUH7645hj+QmQQ47mpz+OONyfADngeH8CZNyE/gTIqBn5CZBRc/pTIGP0R2lWfwJkxLT+BMh+8/LTH/vpj9DE/PTHbjP7EyA7Tc1Pf+wzNz/9scvk/PTHLvqjNLs/AbKH/ihN70+A7DC/PwWynf4o6Y+S/iit6E+AbKU/SvqjpD9CS/LTHxut6U+AbKM/Qovy0x+b6I/QqvwEyAbr8hMgd63MT4Dcoz9K+qOkP0Jr89Mfty3uT4DcpD9K+qOkP0r6o6Q/SvqjpD9Cq/MTILfoj9D6/PTHdfqjpD9KJ/SnQK46pT8Bco3+KOmPkv4InZKfALlCf5T0R0l/lPRHSX+U9EdJf5T0R0p/lPRHSX+U9EfonPz0x2X6I6U/SvojpT9K+qOkP1L6IyVAUvojpT9K+iMlP0raIyU/SvqjpD9K+qO0JDv5sdWK/r554/BUPBXL+nulP+5b1x9sID9a+qOkP0r6o6Q/UvqjpD9K+qOkP0r6I6U/SvojpT9K+qOkP0LT8tMfA+b1J0D20x8l/VHSHyX5UZIfJf1R0h8l/VGSHyH1UdIfKf1R0h8l+VGSHyXxUdIfJf1R0h8l/RGSHyX9UdIfJflRkh8l4VHSHyX9ERrNT3/MID9K+qOkP0r6o6Q/QqP56Y8Z5EdJf5T0R0l/lPRHSX6U9EdJfnQGt5/+mEF+lPRHSX+U9EdoND/9MYP+KOmPkvwo6Y+S/kjpj5L+CA3mpz+m0B8l/VGSHyX9UdIfJf1Rkh8h9VGSHyX9UdIfJf1R0h8l+VHSHyX9ERrIT39MIz9C1h8l+VHSHyX9EZIfJf1R0h8l/VHSHyH5UdIfJf1R0h8l/VGSHyHrj5L+KOmPkvwIqY+S/ijpj5L+KOmPkvwo6Y+U/ijpj9DO/PTHVPqjpD9K+iOlP0r6o6Q/SvojtDM//TGV/ijpj9De/PTHTPqjpD9K+qOkP0r6oyQ/SvqjpD9Ce/PTHzPpj5L8KOmPlP4oyY+S+khJj5L8SKmPkPgoSY/Q5e7kxznsPVLyI2T9UdIfJfkRsv4o6Y+S/ghdyE9/nONSfPrjLPIjZP1R0h+lK/0JkHPoj5L+iOmPlP5I6Y+S/Cipj5b2aKkPAAAAAAAAAOCF+BdQSwMEFAAAAAgAAAAhAGMZZTwODgAAgLYNABAAFABncm91bmRfdHJ1dGgubnB5AQAQAIC2DQAAAAAADg4AAAAAAADt282KJFkZgOHOrVfRu1aYheIg4gW4U9y4cCWj0+JCHJkRN+pVeMOOKNp2VcbvOfF2ZT3PqioyEr4DL18kReU/fv7Ln/3iV7c3f3nz13dfvv/mt1+/+8nbd3/7zQ/effb23e+++vrPX3/xx19/9fWX7/91/adf/OGb999e/+b3X/zp/be/f/eHn739/Ec//uztjz7//vc+e/v3t/t95w0AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAADyw27fqGXitbv9Wj8FrdPufehRem9v/q8fhNbk9VY/Ea/FMfALkGnfi0x+XuN+fAJluIT8BMt1ifwJksuX+BMhUK/kJkKlW+xMgE633J0Cm2ZCfAJlGf5Q29SdA5tiWnwCZY2t/AmQG+ZHSHyX9kZIfJf2R0h8l+ZHSHyX9kZIfJf2R0h8lC5DUWoASZKb1/jTIPNv6kyCTbA5Qg0ywpz8JMtzOACXIULv7kyAj6Y+Q/UfoSH4CZJBj+emPMfRH6GB++mOEo/npjwEO5ydABtAfoRP56Y/T9EdJf4TO5CdAztIfJf0ROpef/jhHf5RO9idAzjibn/44Q3+U9EfodH7644Tz/QmQ4/RHaEB++uMw/VHSH6ER+QmQo/RHSX+ExuSnP44Z1J8AOUR/hEblpz+O0B8l/REalp/+OEB/lORHSX+U9EdJfpT0R0l+lPRHSn+U5EdJf5T0R0l+lPRHSX6U9EdpQH8C5DD9URrRnwA5TH+URvQnQI7SHyX9URrSnwA5SH+UxvQnQI7RH6VB/QmQQ/RHSoCU9EdJf5RG9SdAjtAfpWH9CZAD9EdpXH8CZD/9UdIfJf1RGtifANlPf5T0R0l/lAb2J0B20x8l/VHSH6WR/QmQveRHSX+09EdPfqT0R0l/lORHSn6U5EdJfpTkR0l+lORHSX6U5EdKf5TkR0l/lPRHSX+U1kLTHzPpj5L+SOmPkv4oLeenP+Y60l89M49Df5T0R0l/lJbz0x9z6Y+S/ijpj9CR/PTHKPqjpD9KR/qrZ+ZhLOWnP2Y7kp/+GEV/lPRHaCk//TGb/igdyk9/DKI/SvojtJSfj3/Mpj9KK/npj6kOrT/9McZSfvpjNv1RWslPf0ylP0or/S2lCacd66+emkehP0rL+emPufRHSX+UlvtbShPO0x+lxfw8fplMf5T0R2mxv6U0YQD9UVrKz+OX2fRHSX+E7ubnu29cYDE//TGZ/igd6C+emEeiP0qL+T3/cjovj0V/lJbz0x9z6Y+S/igt56c/5tIfpeX89Mdc+qPk+UtJf5Se78/fX7jGgf4EyDB3+lsOUIEMcqw/ATLGvf6W/gFGgIxytD8BMsLdvm7bXocz7ve19A0k/THG8f4EyHkLfVmATLfa3/17yrF5EEv9+T8YZtvY3zN3RgPzULb39/Ht18/K49nZ3wdvuXZOHtOB/v7ztutm5HEt9qcx5tMfLf0R0x+xj5ITIBeTHDEB0hIgAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAABA6Xa71SPwWt0+UM/CK3N7Tj0Ur8Oz8emPa9zLT4BcQX+E7ucnQObTH6Gl/ATIbPqjtNyfAJlqJT/9MdVafwJkJv0RWs1Pf0ykP1L6o6Q/Svqj5AMgKf1R0h8l/ZHSHyX9UZIfJf2R0h8l/VHSHyn5UbL/KC0/gCXIXGv9KZCJ1vNTIPNs6k+CzLE1PwUywY78BMhwu/pTIGPtzE+ADLW7PwUyzoH8FMgox/ITIEMczU+BnHeiPgVy0sn6BMgJ5+vTH8eN6E+AHDQkPwFykP4IDcpPgByiP0LD8hMgB+iP0MD8BMhu+iM0ND8BspP+CA3OT4Dsoj9Cw/MTIDvoj9CE/ATIZlP6EyDbzMlPf2wzqT8BssWs/PTHFtP6EyDr5uWnP9ZN7E+ArJmZn/5YM7U/AbJCf5Tm9idAlumP0uT+BMgi/VGa3Z8AWaI/QtPzEyAL9Efpgv4EyF36o6Q/SvqjJD9K+qMkP0rz+xMg9+mPkv4o6Y+S/ijpj9IF/QmQe67IT3/coz9K+qOkP0pPUpEfF9IfpQv6q4/IJ2x+f/UJ+ZQ9iWV4fwLkPv1R+riV8fkJkPv0R+njUvTHlfRH6YL+6iPyCdMfoSel6I8L6Y+S/ijpj9KTUPTHhS7orz4inzD9UdIfpSeh6I/rPA1Ff1xnfn/xAfmkTe8vPh+ftGdSkR+Xea6V08nJj22e3VUD+0sPxyfvufzO9/fmvz/Ago+7ee7q8f7Kk/ESPJvfgP7eyI91T7J57vLR/rpj8UI8n9+I/mDVvWr0xxXuRaM/LnC3Gv1xgbvZ6I8rzOpPgGxyJxv9cRH9kXquG/1xnQkBpufhpXlSzdn+BMgZ+qOkP0r6o6Q/Sqf7EyAn6I+S/ijpj9L5/gTIcfqjpD9K+iOlP0rn+xMgx+mPkv4o6Y/SWlv6Y6YB/QmQw/RHaa0s/TGT/ijpj9JqWQJkIv1R0h8l/VHSH6Uh/QmQg/RHaTUr/TGR/kjpj5L+KOmP0oj+6jPwcumPkv4o6Y+S/ijpj9KA/uoj8ILpj5L+KOmPkv4o6Y/USlfyYyr9UdIfJf1R0h+ls/3F4/PC6Y/QWln6Y6bVsvTHRPqjpD9KZ/trp+el0x8l/VHSHyX9EVpPS3/Mc7a/dnpeOv1R0h8l/VHSHyX9UTrZXzs8L57+CG2IS39Msx7X4h364xT9UdIfpZP9xdPz0umPkv4o6Y+S/ggtxqU/JtMfpZP9xdPz0umP0np/64HCUfqjpD9K+qN0rr96el46/VFazU9/TKQ/SvqjpD9Kp/qLZ+flW+3P+mMi/VHSHyX9UdIfJf1ROtNfPTsvn/4o6Y+S/iit9rdwRzw6D0B/lE70F0/OI1jv7+4t7eA8hPX89Mc8G/q7d085Ng9Cf5T0R2lDfvdu6obmYWzpz/5jFv1ROp6f/jhPf5Q2pKU/ptnQ1vqGhMP0R0p/lPRHSX+EVtPSHxPpj5L+CK2npT/mWS9Lf8xzvL9oYB7KhrT0xyxb0pIfs2xr64NX5MdAW3fbhxf1xyBHP9rJjxGO9gcj6I+S/gipj5L6KImPkvYoSQ8AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAOC1utUD8KrdBEjnpj9Ct2X1eDy2lfwEyFSr/QmQedbzEyDTbMlPgMyyrT8BMsXG/ATIHJsDVCAT6I+U/ijpj5T+KOmPlP4oWYCk9EdJf5T0R0l/lPRHaWN/AmQK/VHSH6Wt/SmQGfRHSX+UtvcnQMbTHyX9UdIfJf1R2tGfABlOf5T0R0l/lPb0J0AG25Wf/hhrX34CZKi9+QmQgfbnJ0BGOVKfAhnkaH4CZIDj+SmQ007lJ0DOOZmfAjlFf4TO56c/jtMfoQH5CZDD9EdoSH764yD9ERqTnwA5Rn+U9EdoVH4C5Aj9ERqXn/7YT3+EBuYnQHbTH6Gh+QmQnfRHaXB/AmQX/VEa3Z8A2WN4fwJkB/1RGt+fANlOf5Qm9CdANpvRnwDZSn+UpvQnQDbSH6U5/QmQTSblJ0A20R+laf0JkHXz8hMg6/RHaGZ+AmSN/ijN7U+ALJqcnwBZpD9K+iM0PT/9sUB/hObnpz/u0x8l/RG6ID/9cdcV/QmQOy7JT3/coT9K+iOlP0r6o6Q/SvqjpD9Kl/QnQO7QHyX9kdIfJf1R0h8l/VHSHyX9UdIfJf2R0h8l/VHSHyX9kdIfJf1R0h8l/ZHSHyX9UdIfJf2R0h8l/ZHSHyn9kdIfKf2R0h8p/VGSHyn5UZIfKfVRkh8p9VGSHyX1kZIfJfWRkh8l/ZGSHyX5UZIfKf1Rkh8p+VHSHyX9UdIfJflR0h8l/VGSHyHrj5L+KMmPkv4o6Y+S/gjJj5L+KOmPkvwo6Y+S/gjJj5L+KOmPkvwo6Y+S/gidyE9/nKY/SvqjpD9K+qOkP0Jn8tMfZ+mPkv4o6Y+S/gidyk9/nKQ/SvqjpD9K+iMkP0r6o6Q/SvojdC4//XGO/ijpj5L+CJ3MT3+cIj9K8iNk/VGSHyX5EbL+KMmPkPwo6Y+S/AjJj5L+CMmPkvwo6Y+S/Cjpj5L8KOmPkvwo6Y+S/Cjpj5L+KMmPkv4oyY+S/ijJj5L+KMmPkv4o6Y+S/Cjpj5L8KOmPkvwo6Y+S/ijpj5L+KOmP0N789MdI8qOkP0r6o6Q/Qnvz0x8j6Y+S/ijpj5L8CFl/lPRHSX+U9EdJf4R256c/BtIfJflR0h8l/VHSH6Hd+emPgfRHSX+U9EdJf4TkR0l/lPRHSX+U9EdJf5T0R0l/lPRHSX+UdmYnQIbau+30x0h7H7b6Y6S9/b3RH+Pszk9/DLS/vzf6Y5j9+VmAjHOgPwuQYQ7kpz+GOdIfDCI/SvqjJD9K+qOkP0ryI2T9UZIfJf1Rkh8l/VGSHyHrj5L8KOmPkvwo6Y+S/kjJj5L+KMmPmP6IyY+Y/Gipj5b6AAAAAAAAAACAmf4JUEsBAhQDFAAAAAgAAAAhADbaLNoGDQAAgLYNAA4AAAAAAAAAAAAAAIABAAAAAHByZWRpY3Rpb24ubnB5UEsBAhQDFAAAAAgAAAAhAGMZZTwODgAAgLYNABAAAAAAAAAAAAAAAIABRg0AAGdyb3VuZF90cnV0aC5ucHlQSwUGAAAAAAIAAgB6AAAAlhsAAAAA', 'reference/sample-8.npz': 'UEsDBBQAAAAIAAAAIQAya/0nvxcAAICAEQAOABQAcHJlZGljdGlvbi5ucHkBABAAgIARAAAAAAC/FwAAAAAAAO3dzW5kSV7GYeeWq+hdg9QLkFoIcQHsQGxYsEIDM4gFAtSD2ABXwQ0z1VVl58f5iIgTEe/J9PMsSl12+kTEP37jKruqe/7vb/7ur//27y9v//X23z/+9ne//6dffvzLH378n3/8sx9/+uHHf/73X/7zl9/82z/8+y+//d2Xt//Vb/7197/7w9t//y+/+Y/f/eHnf/zzTz/8/PNf/PTDn//8p3/y0w//+0O9P3oDAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACA13L5Jr0PPqXLRYDEXC63ASqRFl+aWQnnsmTpne8/m7VpntpiWBXWH5E+Ged3tL6dMmHNyPT0x4rh2emP7+ZU9vVr3vvlsicnZkJ0a43p77OaGN1mY9r7TDLVXVnc0/QxMFE6uVvpadBbuqg66Wlx3OVs0dVsJz08mgxrp4eaDaYHSa2R5fRRs83wMCk0NJjOKnacHSpbBlcyTvn+o/Plw9fL+LiXUWlMcXWmktcRNSGIyUrPlZ07X4yvYb7ik2VH/+kNDyGk9HBLA5l9B5/N6Ls/g8avfVffcT5PsMdRl/sECs++PbDIpRU6+fbO9ydic7V9/tt+77mcd3ufPL0vfh1D4euuJ7f1zohvJ0lvo8aI+3w2hXPYGN23qz+F5f7Os78vRt3kcyoYyPYQJ99euet/Yz65jysD7/FZFUxle5KTr7Dc9R/tRDfyYeA9Pqv9sdwN8H6S829xw9V+Lvp7BvtjuZ/f7SQT17juakeX6xRD27k37Baf2PZQFgb4McjADT614XeZ1P14eyOcfHn9LR9i4NE6X9AMRRsfcsKdIQ66oom25zd2xadRsvMxJ9we4ojrmW1zfiMXfB5FOx9zwu0hjrie2TbnN3C9J1K09yEn3BnhgOs5hZEH7Hg7s+zv/21Mfw8Lv73W1x4bRp2w3+XMs3+E2z9nGrLutzetber1XB+y12k7Xs5E+4cY39/Hmxb39HqGnLLj5Uz0eIq7oww64ePkPv65982kLf+vfMQ6T2jnMLWHK33l40M/1ul/M1FLJ/Id6O82TlJ/qkNz+P6Rd9uov4Lqjxhmb8YD171e/8xKz1P7sE47Kd7i/bpVH9TV++p1M+68g+JfibIqDlT3tE5buXnnzn5v/oS67GAjXG83spf60ef0PdGRIWwv/f6PxbtceUnZgQ/4WLxyyD3Xfx5dD3VkCjsLLzxxe49Nl9fBx+KJvdTPPa3nsY6MYX3dzcetr3jwJttcrX2/lY+DDF/+mfQ8Xusc1r5ofX/ozpqHD9bL1dqb5x20u8qxn0LP87XO4Uh/u5uZ6+NE07fXMqQT6HjCxkm8bfbX+p2EY5fZ7Nvi0zfYMqJT6HjMhlEMm+ntY+oOeUTrJqeueyYdD1o9ioEzvXtK7SmbrW1g+dP4gHWfTr+zFk9iwkzvnlJ/yEbbZ3x/T90gatZ9Og1nXfnGXOkkZsz07iH1h2y0c8jv72gf/t66T+fYcRv6mzLT+4c0nbJB6SnvX3d0g0dmFXXwuI/9dfvWQ49jHT1nvcIFe2/q0LDmOzqBjwfdPnRvGNNGuvCExqO22f0d6P3bDl7KoWHNcnfqm590OO/Oy6aOdOkJrWc95mPx7b++8yn666bw8UdWP3zUkeevPkrBa76/tOf3JM6k/lR15915WY8lqg47eASdXX/51vDRJ9d3VvV/5F+/gSeZRHetWxw1rx46z2fYs9dXOe88zmLsxA4YfMiej99a57xDqTrXwEef1NhD9nz61jonHkrNuQY++qQGH7Lr47cWOvFUKo5V/MLJX64NU3uMykP2ffz6OqeeSlffvw1T/XFn1H04qwuM//bOicfS0ff9vUR/Y6az8Pzeqz3bXLpp39/gkTXpPZ27U77dftf0Of44vd8mR7jaY+VuB46s2bABvT/99i9dDVnm9CPp5naTlfsdNrF2o+a0d/jez2s++s1bjm1sxjDf/zrR15/cHmZ7sIcH1lPT0S8HvoZY38HOnlbe0/P0bfP4/sEfT2l/SMOy1Td7aGK9bW1wZ/ud5rXwjtWPK3newcPXH+f9n47+T7N1A5WnOzKx3jY3uLP9fvO6f8fqxxU+r/n0Td/JuNnI2+z+6j8DHplYb5v7295+72lt72rnHZ1O3/adtJt9zO+vtsAjE+tte39r26+fSeXR157WeZqPD1t5/s4h7/dRMp+Oak64/fLZdrbXdxbt+9ra0fF1Vh/98c7qgzZMrlntPg5NrJ+Cs/ScxJG9be2o25plyx543DGNh19+0iksn7L589+4rW3uqOeyq+u2LVc1vpLxtpx+8MQOWPrtTnt/nbe29ujB01x5csNqvb8MWTx801dzVQMZZ+WMj6/Z+IBRB9p6/rSlF97Y8JRelh/aspm6eYzScJS1uQze3MY7Ry7++KbSD5731Uf1oYYNrVLTUY5NoHV7G+8cuPjum7Y+eJZL0X+98n5nFaMYZfEsLR+lv8WPnqnyYE/c37yvpTbXGL36wpPbLnqS2n2Vv36Y1lO0HvzA/rbeO27xhbfVPmKC5b3Wni1h6SyVHzlle4Et9HjymNxar+yMzn6Y7V2N3HKXJ+tvx+kP87YdwsAtd3lyh7peur+zflf8xs7f7Bi1ap8n62/bSx2mp77DOJrhywb4Qkfpatg0JHjj6Q8wyMChtDf2sgGmN3E+Y6fSlt+r9pfewglNuNSG/mb+5SOSJl1qZYDy+yxm3urx/iZtlFkmX2vx58CO9Yn2xKZ/XtEfV6b3d/+Xzb6uvhZch/z0d2IHr/bgqg+FDemP8wpd7mJrj+8Y/i9fkZW63qslH1e/2ZH2Xlm4v+ufL7xi6ZUtix35YEbK9tf7pWtPOPLBDJXsb95is1aiWq6/iYtNW4pqif7mLqe/M9MfQZFffyf3N20lqr3+b/84s0R++uObSH764xv9kRTJT398E+tv7oKcVKa/uV76cE9OfyTpj6TP0B8nlvjDj4mLcXaBP/udtxinN/kTYOAXe72f2uy/DKU/rumPz0N/BCW++tAfX2W+9vUFN1/pj6RQf4E1OSPfeybInzST5K86kKQ/kvRHkvwISuUneb7QH0n6I0l/JPnygyT5AcDn4Rd9kvRHkv5I0h9J+iNJfwAAAAAAjOD7zwAAAAAAAADAEf6bW6S8Tf6/dodr+iNJf6S8XUnvhc/l7U56P3wm9/Xpj5n0R5L+yHmsT3+MtlSdABnkI6ud8AT4ydyHMXCRL08vzO/sAX7Z39v9m/Y3/nb14q2Hf3/OwwMXxnM3sobZLbx+/Rm/nnzh3Vdve7t+7dKNf3xEt6tvKGzP3oo3P7x9ncv3Q1/u/2lpxgs/3L7j7f7J90fcPvbGiBdeXDKIwmdtPONmJDVb+/jhk7ma/f0Pb2/Lc7l72/Vnjbfrmu4esPrDSRzey+lO9BTWP48feWjfx/HaeucHAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAADAs7v8Kr0LPqXLg/SO+Ewe+1Mg8yz1J0JmWe1Pg8yxmaAMGWy3PwkyUkmA9RWqljLF/RUWuhisz6Ss6NjfYo/3i+ROyhkN7O+9tsUqh7Qo8Gcztr/FIH/98THITgfp9DgmmV3fQ399mpHfk5rZ33UkS2/rcIwOE2GmueU9VrL+npZjdBgIs03tbquRA/mo74mNrw7WjcvuQH7q/TTG9HcdYm1HSy3zqgb3d51RWUsbD9h6fc+ZMNGU+C4d+lv8Rf3mnTyh4dlV/k6w4rn+dPkl9I/uLourOjb/DPjbJ8nyZ98vxDPqmdtWLVffqFtI5vo1VY99fzzPqUdcjZqre3+AL0CeX4eQDivYxMJOP7afnSCHdK+p1O13mtdfdrPT0JAYZ0Jqy2HdFLb+Ml7blNiWwroubONl75tMDolhxpe2Z3MLd3sMD4vuRtd10N0W09Oiv1hb9dKjYoR0VRXSo2KAdFQ10rOiv3RTTdJDo590S+0O/dsjnEU6o0PSw6OHdEXt0pOjh3RF7dKTo4d0Re3Sk6OHdEXt0pOji3RGzdKDo4t0Rs3Sg6OLdEbN0oOji3RGzdKDo490R63Sc6ObdEpN0kOjm3RKTdJDo5t0Sk3SQ6ObdEpN0kOjn3RLLdIzo590Sw3SI6OndE310hOjp3RN9dITo6t0TtXSA6OzdFB10tOiu3RSVdLDort0UlXSw6K7dFJV0sOiu3RSddLTort0UlXSw2KAdFQV0qNihHRVFdKjYoB0VBXSo2KAdFQV0qNihHRV5dKTYoh0VuXSk2KMdFfF0oNijHRXpdJzYox0V4XSY2KQdFiF0mNilHRZZdJTYpR0WWXSU2KUdFll0lNilHRZZdJTYph0WmXSU2KcdFtF0kNinHRbRdJDYph0WkXSQ2KYdFpF0kNinHRbJdIzYpx0W0XSQ2KcdFsl0jNinHRbJdIzYpx0WyXSM2KodF670gNirHRfe9LzYax0X3vS82GwdGA70uNhsHRgO9LjYbjiBibktro2L6zi/qeFt7I+r6jm9mfWp7/Pofb2BUhndXevPzqrvnf90VP9jeuPMP2RpD+S9EeU/kjSH1H6I0l/JOmPpMH9CZBNo/sTIB8eaxjenwDZoD+S9EeS/kjSH0Hj89MfG/RHkv5I0h9J+iNJfyTpjyT9kSU/otRHkv5I0h9R8iNJf8TJjyjxEaY/ovRHmvw4EflxDuIjSX8k6Y8k+ZGjPpLER5T+iNIfSeojSnokiY8k/RElP9LUR5b6CJMfSfojSn4AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAADAk7hcLukt8CldPqS3wmdzeZDeEZ/IY34CZJ6l/gTIHIv1SZDB1rsTIcOt1yZABtv+bKc/htr51VZ/jFT6mz390d16fPpjvPL8BEh/FZ//9Edva9lt9CdAuin5Zff25dO2xjPbrqkyO/iivJHtrORHi079rf2mD7aUR7Idlv5oURjJZluL7QmQAmWNrBe2bfTueXKFheiPoNb89EcP+iOkOT0Jcszh9ARIq07xSZAmXfsTIJX69idAqnTOT4PUGJGfCCk1rj8Bsk9/BA3MT4BsGxqfCLl2KfyLfDpkgNsE5ud3vbwkP433Sz5BfwsiM2Gajzu+ve5odNdCc2GOy83nvxP2d5HgS3u/3durDid3KzIYJru56nBxKz42F5oRwzxc8alFR0UXt5d4e7PJtooE5sVI79ea7apUelz09v1S02WVCQ+L7p7lt35fRUdFf5en+a3fV9lp0dtFfyQ9z9cev0qPi/7STdVJT4sOwn/p75DY0OhGf5xGOqha6XnRWTqoWul50Ve6pxbpmdFPuqVG6bHRSTqkZunB0UU6o3bpydFDuqJD0sPjsHRCx6Snx1Hpgo5JT4+j0gUdk54eR6ULOig9Pg5KB9RJeow0ur3DWD+HpedIm7sbzAV0VHiOtLm7wFw/R4XnSJv7C8wFdFB2jDR6uL9gQockh0irhcuLVtQuNUGOWLq8aEbNQgPkkKW7y3bU6v6/rsRT+HZ3j297RpEJcsTSvYUraheZIIcsXFo6o2bzp8cI6Y5a+G/0vo50S83Sg6OLdEbN0oOjh3RFx6Snx0HpgNq8Pf5ZNs8nF9BBt5tPj5FStzeWKKeP290nJ0qFmyubX00/H4d5k9/zyEbT0bfTLP6NCk4p2ktvj6cKTpYCwVgGWDhWcLbsy7UywuPZMlOlVLqYvh6PFxgpNX79S5uXAyGeqeH0MDmi/bKHxNQgNjq6aL7pcUlViQyNftovemRWxeYPjK4OXPTIrkpNnxedrVxoySVP7GzNnBkx1sJ9Fl3x/N7ujB8NMzxeZ+kNh8Ir3B3P4fEyiy84F1/R9ngGC5dZfMHB/PT3IhbusuJ+9cchh+9SfxygP5I63KX+aNblLvVHo053qT+adLxL+VGr+2XqjwpjLlN/lBl5mfpjx+jL1B/rplym/lgx7zb1x6OZt6k/7k29Tf1xZ+p16o87cy9Uf9zSH0mT++ua4PDtMtz0C9Uf12ZeaMf29PciJl5oQUT6+3RO1V9NgIN3yxzn6m/1dfJ7Uefrr6TAwVtlmomXWteR/D6FiVdaX5H8Xt6p+/v4sKEbI2fm5xSfxbg3Mwb9cW9qDPrjzuQY9Me16THojyu5/qatyJnpj6jU7/9mrcjZzYtBfSTJjyD5kaQ/kvRHkK8+SNIfSfojSX8kyY8k/ZEkP5L0R5L8SNIfQX75JUl/JOmPJP0RpT4AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAocLmS3gufzeVOej98Kvf5He9Pw5R76O9wPfp7OT26WGhsob3b17Ss6VfxF3P4E9NOZrvm7paTab/Ro+G1rNvaLWd1+BOR/mjXcqEd29PfZ9bUQef6Ftdd3U5rt5zPUgk719u9vaWFdrqU4IuozGL/IxqtLVKw/oixMEdtFyUf0mRtkYLlx0yGGZq76G11nYIdjxsPQ9W20SO0vTXuVyrY8+gxMcbAnOqtbu0PP9nZ89Sh0c2ksspsbW1nz1OHRjczsiq2ubPtPU8eG52Mj6rC5sa29zx7bnQxPKk6e7vaeHdshBwwKqRR1vecniT15rbTw9au09Ok1sxyOtn9xZmnMTGbKdLzpEo6l/7SE6VGupYB0iOlXLqVEdIzpVy6lRHSM6VcupUR0jOlTLqTcdKTpUS6koHSo2VfupGR0rNlX7qRkdKzZV+6kZHSs2VXOpGh0sNlVzqRodLDZU+6kLHS02VPupCx0tNlRzqQwdLjZUc6kMHS42Vbuo/B0uNlRzqQsdLTZU+6kLHS02VPupCx0tNlVzqRkdKzZV+6kYHSo6VAOpKB0qOlRLqScdKTpUg6k2HSg6VMupNB0mOlVLqUIdJDpUo6l87S46RSOpjO0uOkUjqYztLjpFa6mK7Sw6RNupte0nOkUTqcTtJjpFW6nD7SU6RZOp0u0kOkXbqdDtIj5Ih0PQelx8dR6YIOSQ+Pw9IJHZIeHoelEzokPTyOShd0THp6HJUu6JD08DgqXdAR6dlxXLqhA9Kjo4N0RAekR8dx6YYOSI+O49INNUsPjh7SFbVKz40+0h21Ss+NTtIhNUqPjV7SJTVJD41+0i1VSw+MrtI5VUsPjK7SOdVKz4u+0j3VSs+LztJBVUkPi+7SSRVLD4ox0l0VSQ+JYdJpFUkPiYHSce1LT4ix0n3tSI+HCdKRrUiPhZnStT1ID4S50r3dSk+D6dLJXUvPgoR0dd+kx0CM8EhSHkn6I0p+ZEmPMOkRpj6i5EeU/IhTHmkFPYmNccrq0h6DlH1ykx+jFP3iKj5GKf3FVXwAAAAAAAAAAK/l/wFQSwMEFAAAAAgAAAAhACZvxbSKDgAAgIARABAAFABncm91bmRfdHJ1dGgubnB5AQAQAICAEQAAAAAAig4AAAAAAADt3LFuZVcZhmG75SqmG5CmAGmEEBdAB6KhoEKBDKJABE0QDXAV3DAJJGTi8Tln773W2u/Yfp7Op/E63/92lvzvX/3ml7/+7f3d3+/+8frzd1/+4f3rn796/c/f/+T1m1ev//jF+7+9/+wvv/vi/efvvv78F5/9+ct3X33+5Z8+++u7r37+4ds3r96+/dmbVz99++MfvXn1r1f7/eAOAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAIBl7j9Qv4WX4v6S+mE8YxerkyCrbItOgsx0pDoJMma4Ogmy19zoJMgWK6uTII87rToF8q0kuu+rJ+B0dXIP1Xtwgjqy6+p1WKPuaod6KqapUzqq3o0BdTxz1CuyT93LAvWk3FQnslg9L4+pqzhVPTbfqEPo1Mu/XPXlPxn1IV6O+tKfrPowz1d92SejPtQzUp/yqarv9oTVp3su6js+IfWpnqn6rJ+0+jgvQ33lT0p9jJepvnqqHp+v1RWcq16bR9RRrFWvywZ1JHPVa3JAHc2gej6G1QntVe/FfHVTt9T7sF7d2AP1HAQkR0101PRHTX/U9EdNf9T0R01/1PRHTX/U9EdNf7T0R0p/tPRHSn+k9EdLf6T0R0p/tPRHSn+k9EdLf6T0R0p/tPRHSn+k9EdLf6T0R0p/tPRHSn+k9EdLf6T0R0p/pPRHS3+k9EdKf7T0R0p/pPRHS3+k9EdKf7T0R0p/pPRHS3+k9EdKf7T0R0p/pPRHS3+k9EdKf7T0R0p/pPRHSn+U9EdJf5T0R0l/lPRHSn+U9EdJf5T0R0l/lPRHSX+U9EdJf5T0R0l/lPwBmJL+KOmPkv4o6Y+S/ijpj5L+KOmPkv4o6Y+S/ijpj5L+KOmPkv4o6Y+S/ijpj5L+KOmPkv4o6Y+S/ijpj5L+KOmPkv4o6Y+S/ijpj5L+KOmPkv4o6Y+S/ijpj5L+KOmPkv4o6Y+S/ijpj5L+KOmPlP4o6Y+S/ijpj5L+KOmPkv4o6Y+S/ijpj5L+KOmPkv4o6Y+S/ijpj5L+KOmPkv4o6Y+S/ijpj5L+KOmPkv4o6Y+S/ijpj86d/z9E5u5Of0TuvlG/g5fo7k5/RO4+VD+GF+buTn9U7qarvxFPxvz4nrP6Ws9NfU8CdXTfqncAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAgI/c/0/9DF6e+wfq9/BSPCxPgpzkYnoaZKnb5WmQJfakJ0GmOhKfBplhoD0NMmRCexLkkHntaZCd5senQbZZ1p4GuWF5exLkgrPaEyAfOTU+BT4ly691fnsKfEqWnqyKb9HXYYF1ZyuiW/NNWGjR8c5t7XET1mG1JQc8q7AbJs7EKvPPuD6sjVbMxWyTj7m0qF1WjsY0U4+6pqRjTlmPYfNOOz2hIaeOyHGTLjwxnRmaLdlvyqGnNDNTuyk71KmsUG/KdnUrK9SbskMdywL1pOxR1zJdPSi71LlMVw/KPnUvs9V7sk/dy2T1nOxVFzNXvSa71cnMVG/JfnUzM9VbckAdzUT1lBxQRzNPvSSH1NlMUw/JIXU209RDckzdzSz1jhxUhzNJPSMH1eFMUs/IUXU5c9QrclRdzhz1ihxWpzNFPSLH1e3MUG/IcXU7M9QbMqCOZ4J6QgbU8UxQT8iIup5x9YIMqfMZVg/IkDqfYfWAjKn7GVXvx5i6n1H1fgyqAxpUz8eouqAx9XqMqgsaU6/HsDqhIfV4DKsTGlKPx7i6oRH1dkxQRzSgno4J6ogG1NMxQ13RcfVyzFBXdFy9HFPUGR1WD8cUdUaH1cMxR93RUfVuTFKHdFA9G5PUIR1Uz8YsdUnH1KsxS13SMfVqTFOndEg9GvPULR1Rb8Y8dUtH1JsxUR3TAfVkTFTHdEA9GTPVNe1XL8ZUdU671YMxVZ3TfvViTFXntFs9GFPVOe1WD8ZcdU971XsxWR3UTvVcTFYHtVM9F7PVRe1Tr8V0dVK71GMxX93UHvVWLFBHtUM9FSvUVW1XL8USdVab1UOxRt3VVvVOLFKHtVE9E6vUZW1Tr8QydVqb1COxTt3WFvVGrFO3tUW9EQvVcW1QT8RKdV231QuxVJ3XTfVArFX3dUu9D4vVgd1Qz8NqdWHX1euwXJ3YVfU4rFc3dk29DSeoI7uinoYz1JVdVi/DKerMLqqH4Rx1Z5fUu3CSOrQL6lk4y5UATi3u41/Pi3Dl+Od399ETePau3V1/LHft5vpjuav31h8t/ZHSHy39kdIfLf2R0h8t/ZHSHy39kdIfLf1R0h8p/ZHSHyn9kdIfJf2R0h8p/ZHSHyX9kdIfKfWRkh8p+dGSHy350VIfLfnRkh8t+dGSHy350VIfLfnRkh8t+dGSHy310ZIfLfnRkh8t+dGSHy310ZIfLfnRkh8t+dGSHy31AQAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAABwrvvvq5/DS3F/Q/0+nqlb4SmRJXaHp0RmGA9PihyxojwlctMZ4cmQj5wfngS5a8P7Tr0Cp6uTe6Ceg7PUpV1UD8Mp6syuqKfhBHVk19XrsFgd2G31QqxU17VFvRHr1G1tVM/EInVYm9VDsUSd1R71VsxXN7VPvRaz1UXtVg/GVHVOR9SbMU/d0jH1asxSl3RYPRxT1BkNqKdjgjqiMfV6jKoLGlYPyJA6nwnqCRlS5zNBPSFj6n7G1QsyqA5oXL0gg+qAhtUDMqouaFS9H8PqhEbV+zGsTmhQPR/j6obG1OsxQR3RkHo8ZqgrGlKPxwx1RSPq7ZiizmhAPR1z1B0dVy/HLHVJh9XDMUtd0lH1bkxTp3RQPRsT1TEdU6/GRHVMh9SjMVNd0xH1ZsxV97RbPRiz1UXtU6/FAnVUe9RbsUbd1Wb1UKxSl7VRPRPr1G1tUo/EQnVcG9QTsVbd1031QKxWF3ZDPQ/r1Y1dVY/DCerIrqm34RR1ZhfVw3CWurQL6lk4T93ao+pROFEd22PqTThVndvH6kU4WR3cA/UcnK9u7nvqMSjU1X2gnoJEnd136iWI1OF9q96BTJ3ef9UjUKrruxfgC1fnp78XT3/EBEhMf7T0R0x/tPRHTH/EBEhLf8T0R0t/xPRHS4DE9EdMf7T0R0x/tPRHTIDE9EdLf8T0R0t/xPRHTH601EdMfrTUR0x+tORHTH3E5EdLfcTkR0t+xNRHTH605EdMfbTkR0x9AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAPGn3k9TfgydmVnhSZJd14enxxdpy59PDu+yMSTjNjQOf39ct0U6sUMd0RL0Zs9QlHVTPxhx1R4fVwzFBHdGIejtG1QWNqddjTN3PsHpARtT1TFBPyGF1OnPUK3JM3c009ZAcUEczU70le9XFzFWvyU51MLPVe7JLnct89aLsUdeyQD0p29WtLFGPylZ1KWvUq7JVXcoi9axsU3eyTD0sm9SZrFMvywZ1JCvV23JTnchS9bjcVCeyVD0ut9SFrFWvyy11IWvV63JDHchi9bxcV/exWD0vN9SBLFbPy3V1H6vV+3Jd3cdq9b5cVeexXD0w19R1LFcPzFV1HsvVA3NNXcd69cJcUcdxgnpirqjjOEE9MZfVbZygnpgr6jhOUE/MZXUbZ6g35qI6jVPUI3NRncYp6pG5oo5jvXphrqrzWK4emBvqQBar5+WmOpGl6nHZoI5koXpaNqkzWaXela3qUtaoV2WzOpUl6lHZoY5lgXpSdqlzma4elJ3qYOaq12S/upmZ6i05oq5mnnpJDqmzmaYekoPqcCapZ+SwOp0Z6g0ZUdczrB6QIXU+o+r9GFP3M6iejzF1P4Pq+RhUBzSkHo9hdUIj6u0YVzc0oJ6OcXVDA+rpmKCO6Lh6OWaoKzqsHo4p6owOqmdjkjqkY+rVmKUu6ZB6NGapSzqkHo1p6pSOqDdjnrql/erFmKmuabd6MKaqc9qr3ou56p52qudirrqnneq5mKwOapd6LKark9qj3ooV6qo2q4dikTqsbeqVWKdua4N6Ipaq87ql3ofV6sKuqbfhDHVlF9XDcJI6tMfVq3CeurVH1JNwqjq3h+o9OFtd3IfqLSjU1f1fPQSROjzpvXS6o6U7WsKjpTtawqOkO1K6oyQ8AAAAAAAAAAAAgPP9B1BLAQIUAxQAAAAIAAAAIQAya/0nvxcAAICAEQAOAAAAAAAAAAAAAACAAQAAAABwcmVkaWN0aW9uLm5weVBLAQIUAxQAAAAIAAAAIQAmb8W0ig4AAICAEQAQAAAAAAAAAAAAAACAAf8XAABncm91bmRfdHJ1dGgubnB5UEsFBgAAAAACAAIAegAAAMsmAAAAAA==', 'reference/sample-9.npz': 'UEsDBBQAAAAIAAAAIQBZis2figoAAIBXCAAOABQAcHJlZGljdGlvbi5ucHkBABAAgFcIAAAAAACKCgAAAAAAAO3dQW8kRxnH4e0rn8K3BWkPEEUg8QG4gbhw4IQWsogDItEu4gJ8Cr4w8UbejO3xeKr7rfpXTz/PKXF2u6rf+nlmLCsz//vdH377+z8ub/715t9vv/nw6S8f3/767u1//vyLt+/u3v7124///Pj+H3/69uM3H+6//pv3f//04fuvf/rb++8+fP/vP/3q3d3XX/3q3d0vv/75z97d/feu3U/eAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAADArixLegcc17Loj5xFgATpj6RFgATpjyz9EaU/ogRIkFeAJOmPJD8Ck6Q/kvRH0iJAchb9kaQ/kvRHkv5I0h9J+iNJfyTpjyT9kaQ/kvRHkl/AkaQ/ovRHkMc/khYBErQIkKBFgAQtAiRIfyQtAiRoeSq9IQ7lWX8nCUqS3s71dyFJqHRlfgqki+v7UyD1WvqTIOUaA1QgpVr7UyCV2vsTIGVW5CdA6uiPJA+AJOmPoFX5KZAia/sTIBVW96dACmzoT4BstqU/BbKV/gjalp/+2EiAJOmPpI393ViBt3Y/89vc300d2G3dzR5s72+nR7bTbd+agv52eZJ73fetGdnfBMHer73vb5wbU9HfC+e4nC4wx/vMrP3OoZeS/p4d5cnXzvUXudOTzeQ3wg/K8nvN6ZPvNP3F9sGDrs19WSJ6iycedjXNhg6vX3ZzmG0/PHYb+b24j+m+H3jsZvPT3y7sOL8rtjJ6S7Taa37X7kV/c9thfVNthm32199Um2GjHfX36Pd56c1QYy/9PSwQ3wil5u5vgi3Q1Yz9ae1A5unvtWvr7wZV5rctiMHLMYXS/jYUMXY1ZhHvb+hizKauvfYkti2jv1tQGN/S0N+QRZhfVXlNaQxZhD2o6e7KNLZf//U12JXtzT0OpNtSowbCUDXZXeqk+nrckpLmzhRTeCluWE0nz6qpuxI3rSiUTtLTobd0YS/zZnxHkK7srPRQGCad2hnpkTBQOrZn0gNhqHRuj6SHwXDp5L5ID4KIdHa6OzbhkaQ/kvRHUq4/ARL/+SN9+6QJkKRwfwI8uHR/iwQPLR3fvfQMyOlZlQB5Vbf6GhfJ3D1hQ/K7apXI7RM2pr/rlkncP1kj4rt+wUE3zTwG5/fqmm9O/8cjSd68QH7XvRx8/E/ecO1GJfprW/T049S5MeX5dXwTti9/sds0GK46v6vjqL4euxTr78rFe903c4j299rqne6ZeeiPpJn763TLzGTe/jrdMFOJ9vfS6n1ulRkl8zu3eJebZF7JAPVH8lMw9UfyU4C1x71Afw9/Wn3cG9zflz+tP+7F++t4b8wv3F/HO2MXAgH+uG63u2I3igK8siXR8dTIAOXHMwP7g+f0R1JJfwJkLf0Rpz/S9EeW/gjTH3n6I05/ROmPJP2RpD+S9EdUa4BCpFTz45/+KOT5lyT5ESU+suRHlv5Ikh9J8iNLfqQpjyT5kaQ/sj5npz4AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAaLQsS3oLHNeyCJCcRYDkLF+kd8IBLYsAiVn0R86iP4IWARKkP4KWp9Ib4lCe9adABtIfSfojSX9ECZAk/ZGkP5L0R5L+SDrTnwAZRn8k6Y8k/RF1LkAFMowAidIfQWcf/wTIIPojSX9E6Y8k/ZGkP5K8ACRJf0TpjyT9keQJmCT9kXS+PwEyxgv9CZBKLzalP/p7uSv90d2FsvRHdxfqeqm/GQKcYhNsdymvKfubZR+UuFTYdP3NsxOK7Ka/ibZCnQuJvZTf8EN/cSP6278LhzvFoc+0Fzqo7K8wkM17YR/K+qusZOte2I2a/opKKdkLe1LRX1Eu7VsR4P5tPfOaXNp38coF2Y+1Z14SzdqLnLkU+7Ty0LeVUyM6N4qsO/f6mtqlJ0eFdEXrpSdHhXRFq6UHR410R2ul50aVdEnrpKdGlXRJ66SnRpl0Squkh0aZdEprpGdGoXRMK6RHRqV0Tc3SA6NUOqdW6XlRLB1Um/S0KJdOqkl6WNRLN9UiPSs6SEfVID0qOkhH1SA9KnpIV3W99KToIp1Vi/Ss6CFdVZP0sKiXbqpVel7USve0QnpkFErHtE56apRJp7RWem5USZe0QXp0VEhXtF56clRIV7ReenJUSFe0XnpyVEhXtF56cpRIZ7RaenCUSGe0Wnpw1Eh3tFZ6btRId7RWem5USZe0Tnpq1Em3tEZ6ZtRJt7RGemYUSse0QnpkFErHtEJ6ZFRK19QuPTFqpXtqlZ4XxdJBNUqPi2LpoBqlx0WxdFCN0uOiWDqoRulxUSwdVKP0uCiWDqpRelwUSwfVKD0uiqWDapQeF8XSQTVKj4ta6Z5apedFqXROzdIDo1Q6p2bpgVEqnVOz9MAolc6pWXpglErn1Cw9MGqle2qVnhe10j21Ss+LWumeWqXnRbF0UI3S46JcOqkm6WFRL91Ui/SsqJduqkV6VtRLN9UiPSs6SEfVID0qekhXdb30pOghXdX10pOih3RV10tPii7SWV0tPSi6SGd1tfSg6CVd1nXSU6KbdFpXSQ+JftJtXSM9IzpKx3WF9IjoKp3Xq9IDoq90X69Jz4fe0oVdlp4O/aUbuyQ9GwZIR3ZBejSMki7tvPRUGC6d3CPpYbDVukNMd/egfByMtfIQ0909KJ8HY609xXB3D8rnwUjrzzDc3YPieTDUphNMp/dZ4TAYbesJhtu7VzYLxtt8guH4Nu6esM0nGI5v4+4J236Umeg2bpo5FB1lpLutmyas9Cx/+MsXLqc/Hutylpcuoz9OjD9L/fGjwFnqjwfBs9Qfyf5KAxy1Z2rpjyT9kRQ9S/0dXfgw9Xdozw5xvwEO2zGF4v2VBThuxxSKn6b+Du3JEQYOU39H9vQIx5+l/g7r5AXfaX/Dd6G/49IfSfojKn6C+ju68AHqjyT9EaU/ovRHlP5I0h9J+juSCQ9Lf8cx42Hp7zCmPCz9Hcach6W/g5j2sPR3CPMelv4OYdqz0h9R+iNJf0TpjyT9EaY/ovRHmv6I0h9R+iNMfyTpjyj9kaQ/ovRHlP7I0h9R+iNJfyTpjyT9EaU/kvRHkv4I0x9J+iNKf6TpjyT9EaU/kvRH0vLk44r1R4T+iNIfUfojS39k6Y80/RGlP6L0R5j+iNIfAAAAAAAAAAAAAAAAAAAAsIE3LiDIO2eQpD+S9EeU/siSHwCM5vUfOd46nAAfXUTKSx8dKENGuNCfAOnuXH+LX4Qwxtn8ls//Ib01DuDF/mAA+ZGkP5L0R5L+SNIfKed/9tUfY+iPJP2RJD+y9EeU/EjSH1H6I0h2hPxYnfwY7vRRT34M96i/8F44oJOnXf0xnqddAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA+D9QSwMEFAAAAAgAAAAhAC3M2cvLCgAAgFcIABAAFABncm91bmRfdHJ1dGgubnB5AQAQAIBXCAAAAAAAywoAAAAAAADt3b3OHVcVx2FPy1W4M0guIIpA4gLoQDQUVCgQIwpEkINogKvghklI4jj2+Zi9Z+39n4/n6WK958yatX56rTTJf3/zu1//9vfLi3+++Nerz998+ae3r3758tW///izV69fvvrzF2//8fazv/3hi7efv/n6z3/12V+/fPPVn3/5l8/+/uarf/7xJ69ffvrJL16//PmnP/3J65f/ednuRy8AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAKDEkh6AS9MfSfojSX8kLQIkSH8k6Y8k/ZG0CJAg/ZGkP5L0R9IiQIL0R5L+SNIfSYsACdIfSfojSX8kLQIkSH8k6Y8k/ZG0CJAg/ZGkP5L0R9IiQIL0R5L+SNIfSYsACdIfSfojSX8kLQIkaFkUSM7ygfQ8XMuH/UmQmW70J0Gmud2fBJnjbn8SZIJH/UmQ0Z70J0GGet6fBBlnVX8SZJC1/UmQERr6kyDl2vrTILXa+1MgdXr6EyBVuvpTIEU6+xMgJXr7EyAVuvsTIAX6+zt7gVd4x7wt/Z31OBd4xd3Y1N/prnOBV9yZbf2d5zoXeMVd2tjfGc5z+hfcs839Hfk+5367Q9je3yFPdNoXO5qC/o51pzO+04FV9HeQa53rbU7iEv2d501O5+z9neIlTqymvz3e7ujzX8MZ+zvy7FdT1N9OjnjQsS/sLP0dcGRenKK/g43L+6r6i1z0MINyz0H7O8SQPFfW36zb7n0+mhyovz3PRqey/EbeeK9zsVlRfMMOvcORKFST3ohj72saxigpr/jkuxmE4QqyKzz7LoZgooKLl9w+/XwySu6+6f7JZ5NWdP2+CEKPZT+qEmgsYfoD2aeyENb2MPNZ7F5hDs+imPQYjqQ0intljH8CB1Wcxod1jP12Dq88kGX8N3MeIyqp/07pndWAVqqlV8RA6bieSK+HwdKBPZBeDROkI7sjvRYmSYd2Q3olTJSO7YfS22C2dHHfS2+ChHR130hvgZR0edq7Nu2RpD2SxEdSqD8F8n+x/hTIi/C/f6Rfnrhofwq8vHB/Ary4dH8KvLZ0fV9L74CcdHvfSG+BlMKCBEizqv6WrV+XXQMhZf0tm78ztwRi6vor+A8fhHZATmF/H+cjQJ6o7O9mPgrkgdL+7tWjQO6Y0l/jk2a9O3nz+mt42owXZxfm9rf6kYPfmr0I9LfusQPfmf0I9ff82WNel51J9vfg+fUvyi59d+9Uf+8PIb/reXfwZH/vDaK+a3l38XR/PxiGq3h38T30x+W8q0Z+BHxfjfyYT38kFf7/ipKvwUF9X438SJIfUfIjSX8kyY8k+ZEkP6LUR5L2yBIfYdojTHyEiY8w7REmPsK0R5j4AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAhYlvQEXNey6I+YRX/ELN9Kz8ElLfojZnlPehauR38k6Y+kRYAE6Y8oAZKkP5L0R9IiQIL0R5L+SFoESJL+SPILkCT9kaQ/khYBEqQ/ogRIkv5I0h9JH/YnQGbSH0n6I+mj/gTITPojyS9AkvRHkv5I+rg/ATKP/ki60Z8AmUd/JOmPJH8Bk6Q/ZrnV1j7728UQlLlf163+orffxxRUepTXjvrbyRhUexTYPn4B3pxCgWfx4LJtdx9RxZ32FHgeDy7bcvXyMh6mp8DzuH/ZhqOXxrEmPQGext3Trr55YSDr25PfSdy97sqjV2XSlJ78zmPT2UtqaZxgxTdyIBvOvjmZxi94+F0cU/fhN4bT9/GPpuDo+i6/KZ+tkuuiWM/px2TVNgMn0Xz8MVW1TMCZNF5/UFbrB+BcwkG1SS+LeummGqRXxQjpqlZLL4oh0lmtld4Tg6TDWie9JUZJl7VKekmMk25rhfSKGCld11PpBTFUOq9n0vthsHRgj6W3w3DpxB5J74bx0o09kF4NM6Qruyu9GKZIZ3ZPei9Mkg7ttvRWmCad2grpFTFQOq5m6YVRK93TNuntsVk6oU3Sy2OzdEKbpJfHdumGtkjvju3SDW2R3h0F0hFtkF4dBdIRbZBeHQXSEW2QXh0F0hFtkF4dFdIV9UtvjhLpjLqlF0eNdEe90nujSDqkTum1USQdUqf02qiSLqlPemuUSafUJb006qRb6pHeGYXSMXVIr4xK6ZrapTdGqXROzdILo1a6p1bpfVEr3VOr9L6ole6pVXpf1Er31Cq9L2qle2qV3he10j21Su+LUumcmqUXRql0Ts3SC6NSuqZ26Y1RKV1Tu/TGKJSOqUN6ZRRKx9QhvTIKpWPqkF4ZddIt9UjvjDrplnqkd0aZdEpd0kujTDqlLumlUSVdUp/01qiSLqlPemsUSYfUKb02iqRD6pReGzXSHfVK740a6Y56pfdGjXRHvdJ7o0Q6o27pxVEinVG39OKocOusgZg6pDdHhdtnDeTULLw4Kty7aqCnVtG9UeP+VQNFtQlujSKPrjq/qDa5rVHl4VEDTbVI7Ywyz446P6oGmZVR6OlNA1mtllgYlVbcdH5WqwUWRqk1Jw2EtdL0dVFs3UkDaa0yeVlUW33RQFwrzFwVAzRcNJDXU/MWxQhtBw0E9sSkNTFI40EDhT02Z0sM0n7PQGOPTNgR4/TcM1DZfcM3xECd5wx0ds/Y/TBW9zkDpd02cjsMtuGagdRuGrcchtt0wW256I+tF9wSif7YfsD+PKr6E+BxFZyvuwv9XV7N9TqT0N/lVV2vqwb9XV3h8TpC0N/VZW+nv4sLn05/F6c/gtKX09+1pQ+nv0vL301/V5a/m/6ubAdn09917eFq+iNKf1e1j6Ppjyj9EaU/kvRHlP5I0h9R+iNJf0TpjyT9EaU/ovRHkv6I0h9J+iNKfyTpjyj9kaQ/ovRHlP5I0h9Z+iNMf2Tpjyj9EaU/ovRHlP6I0h9R+iNKfyTpjyj9EaU/ovRHkv7IWf/Xr/4o15Cf/qjWkp/+qNVUn/6o1Zif/ijUWp/+KNSen/6o0lGf/qjSlZ/+qNGXn/6o0Fmf/qjQnZ/+2Ky/Pv2x2Zb89Mc2awrTH6Os+w2nP4ZYl9/9n4sOz+Gt7e/eTwZH5/jW53fnh1ODcwpN/d368czYnENjfjc+ERia02jv78PPTB+Z8+jJ74OPzR2YU+ns7/0PzhyXc+nO773PThuW09nS33efnjQq57Mtv2+/YMagnNLm/r7+ivFjck4F+UE3/REkP5L0R5D8SNIfQfIjSX8EyY8k8REkPpLER5D4SNIeQeojSH4EyY8g+ZGkP4LkR5D8CJIfQfIjSH4EfZyf/phGfiTd6E+AzHIrPwEyye38BMgU9/ITIBPcz0+BDPcwPwEy2JP+BMhIz/JTICOt6E+ADLOmPwEyiPxIkh9Bfv2RJD+S5EeQX38kyY8k+REkP5L0R5L8CJIfSfojSXoEKY8k5ZGkPIKUBwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAADX9j9QSwECFAMUAAAACAAAACEAWYrNn4oKAACAVwgADgAAAAAAAAAAAAAAgAEAAAAAcHJlZGljdGlvbi5ucHlQSwECFAMUAAAACAAAACEALczZy8sKAACAVwgAEAAAAAAAAAAAAAAAgAHKCgAAZ3JvdW5kX3RydXRoLm5weVBLBQYAAAAAAgACAHoAAADXFQAAAAA='}
for name,encoded in payload.items():
    path=phase/name; path.parent.mkdir(parents=True,exist_ok=True); path.write_bytes(base64.b64decode(encoded))
sys.path.insert(0,str(phase))
configuration=json.loads((phase/'config.json').read_text())
configuration['STOP_AFTER_IMAGES']=None
configuration['RESUME_INPUT']='flmm-phase1-gates'
(phase/'config.json').write_text(json.dumps(configuration,indent=2))
from bootstrap import initialize,stage
gate0_only=configuration.get('STOP_AFTER_GATE')==0


## 2. Environment and checkpoint audit
Restore the verified CPU-prepared environment once; no reinstall or network access during inference. G0 requires tensor provenance and exact 20-image evaluator equivalence.

In [ ]:
cache=initialize()
print('Using prior G0; setup validates resume fingerprint before reusing outputs')


## 3. Dataset validation and caching
Image/ref/sentence/mask mapping comes from official annotations and is checked against installed loader. 50 probe and 100 stability images are nested in development; holdout is disjoint and untouched.

In [ ]:
manifest=json.loads((Path('/kaggle/working/results/dataset_manifest.json')).read_text())
print(manifest['counts']); print(manifest['distribution'])


## 4. Resource preflight
Real GPU/RAM/disk, pinned cache hashes, observed quota snapshot and persistence limitations.

In [ ]:
print(Path('/kaggle/working/results/resource_preflight.json').read_text())


## 5. 50-image feasibility probe
Synchronized post-warm-up latency, peak allocated/reserved VRAM, RAM, output bytes and data preparation costs measured separately.

In [ ]:
if gate0_only:
    print('UNVERIFIED: user requested stop after Gate 0; probe not run')
else:
    print('Reusing completed probe from prior committed output')


## 6. Go/No-Go decision
No expansion while G0 is unverified. Stale/unknown quota limits scope; disk, timing and cached-data gates apply.

In [ ]:
if not gate0_only:
    stage('gate')
else:
    print('Not run: stop after Gate 0')


## 7. 100-image stability test
The first job stops at 75 validated stability images and commits output; the resume job skips them and completes the remaining 25. Already completed 50 probe images are reused. This deliberately tests actual cross-session persistence. Currently disabled by user request: stop after G0.

In [ ]:
if not gate0_only:
    stage('stability')
else:
    print('UNVERIFIED: stability and resume not run')


## 8. Resumable baseline evaluation
Development/holdout/full validation are disabled by default. Enabling them also requires G2, a fresh GO decision, sufficient prepared cache and the configured image cap.

In [ ]:
if not gate0_only and Path('/kaggle/working/results/G2.json').exists():
    stage('baseline')
else:
    print('Not run: expansion disabled until user audit and later gates')


## 9. Metrics and qualitative inspection
Global sums of intersections/unions; one first sentence per selected annotation. Per-expression CSV and detailed masks remain auditable. No claim of all-expression coverage.

In [ ]:
stage('export')
print(Path('/kaggle/working/results/baseline_metrics.json').read_text())


## 10. Export artifacts and final audit
Atomic per-image records and periodic archives survive normal stage errors. Completed Kaggle Output is required for cross-session durability; scratch is not persistent.

In [ ]:
print(Path('/kaggle/working/results/run_summary.json').read_text())
